In [2]:
import pandas as pd
import re
import os
import ezdxf

## Loading data to change

In [354]:
df = pd.read_csv('labeled_regions_df.csv')
df.head()

C:\Users\artem\AppData\Local\Temp\ipykernel_49416\1329734249.py:1: DtypeWarning: Columns (2,5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('labeled_regions_df.csv')


,Source_File,Handle,Layer,Type,BlockName,Text,l1,l2
0,(3) старое Симферопольское шоссе.dxf,A7,013,INSERT,BL_333,NaN,Неизвестно,Неизвестно
1,(3) старое Симферопольское шоссе.dxf,A8,013,INSERT,BL_333,NaN,Неизвестно,Неизвестно
2,(3) старое Симферопольское шоссе.dxf,A9,013,INSERT,BL_333,NaN,Неизвестно,Неизвестно
3,(3) старое Симферопольское шоссе.dxf,AA,013,INSERT,BL_333,NaN,Неизвестно,Неизвестно
4,(3) старое Симферопольское шоссе.dxf,AB,013,INSERT,BL_333,NaN,Неизвестно,Неизвестно


## Taxonomy

In [ ]:
unified_taxonomy_ru = {
    "Водные объекты": ["Неизвестно"],

    "Водопроницаемые покрытия": [
        "Галька",
        "Газон спортивный",
        "Гравий",
        "Гранитный отсев",
        "Деревянные настилы",
        "ПГС",
        "ОПГС",
        "Песок",
        "Щебень",
        "Щепа",
        "Мульча",
        "Грунт",
        "Дренажная призма",
        "Ультимакс",
        "Геомат",
    ],

    "Здания и сооружения": [
        "Неизвестно",
        "Подземные",
        "Мосты",
        "Подпорные стенки",
        "Лестницы и пандусы",
        "Фонтаны",
        "Пирс"
    ],

    "Инженерные сети": [
        "Водопровод",
        "Газоснабжение",
        "Дренаж",
        "Инженерная инфраструктура",
        "Канализация бытовая",
        "Канализация напорная",
        "Канализация самотечная",
        "Ливневая канализация",
        "Наружное освещение",
        "Прочие кабели",
        "Прочие трубопроводы",
        "Сети связи",
        "Теплосеть",
        "Электроснабжение",
        "Столбы и опоры",
        "ЛЭП"
    ],

    "МАФ": [
        "Велоинфраструктура",
        "Дорожные знаки",
        "Игровое оборудование",
        "Информационные конструкции",
        "Калитка",
        "Ворота",
        "Ограждения",
        "Памятники",
        "Прочее",
        "Спортивное оборудование",
        "Уличная мебель",
        "Шлагбаумы",
        "Павильоны",
        "Флагштоки",
        "Вазоны",
        "ИДН",
        "Проверить",
        "Фото"
    ],

    "Озеленение": [
        "Газон",
        "Деревья",
        "Кустарники",
        "Цветники",
        "Газонная решетка",
        "Выноски"
    ],

    "Рельеф": [
        "Горизонтали",
        "Отметки высот",
        "Откосы",
    ],

    "Служебные элементы": [
        "Зоны СЗЗ",
        "Зоны ПК",
        "Зоны ООПТ",
        "Зоны ОКН",
        "Зоны Береговая полоса",
        "Зоны Водоохранная зона",
        "Зоны Полоса отвода",
        "Зоны Прочее",
        "Границы",
        "Кадастр",
        "Контуры",
        "Координатные кресты",
        "Координаты",
        "Красные линии",
        "Нумерация",
        "Номера деревьев",
        "Прочее",
        "Размеры",
        "Размеры деревьев",
        "Условные обозначения",
        "Штамп",
        "Динамические блоки",
        "Выноски",
        "Автоименованные блоки"
    ],

    "Твердые покрытия": [
        "Асфальт",
        "Бетон",
        "Брусчатка",
        "Гранит",
        "Камень",
        "Плитка",
        "Кирпич",
        "Резиновая крошка",
        "Архитектурный бетон",
        "Дорожка из плит",
        "Дорожка из бруса",
        "Каменный ковер"
    ],

    "Сохраняемые покрытия" : [
        "Неизвестно"
    ]

    "Элементы благоустройства": [
        "Бортовые камни",
        "Дорожная разметка",
        "ЖД пути",
        "Парковка",
        "Трамвайные пути",
        "Металлические настилы",
        "УЗП",
        "КДО",
        "Тактильная плитка",
        "Колодцы"
    ],

    "Неизвестно": ["Неизвестно"],
}

## Labeling function

In [2]:
def apply_labels(
    df,
    l1_val: str,
    l2_val: str,
    section: str,
    strict: list = [],
    non_strict: list = [],
    regex: str = None,
    exclude: list = [], 
    exclude_strict: list = [],
    total_changed: int = 0,
    total_nan: int = 0,
) -> tuple[pd.DataFrame, int, int]:
    """
    Присваивает (l1_val, l2_val) строкам, у которых:
      - l1 is None  → строка ещё не размечена
      - Text попадает под strict (точное совпадение) ИЛИ
        non_strict (вхождение подстроки) ИЛИ regex
      Пустой список → проверка пропускается.

    Сохраняет CSV как NEW_REGIONS_in_progress_{section}.csv
    и выводит статистику.
    """

    def _match(row):
        # Защита: пропускаем уже размеченные строки
        if pd.notna(row['l1']):
            return row['l1'], row['l2']
        if not isinstance(row['Text'], str):
            return row['l1'], row['l2']

        text = row['Text'].lower()

        if exclude and any(ex in text for ex in exclude):
            return row['l1'], row['l2']
        if exclude_strict and any(text == ex.lower() for ex in exclude_strict):
            return row['l1'], row['l2']

        if strict and any(text == s.lower() for s in strict):
            return l1_val, l2_val
        if non_strict and any(ns in text for ns in non_strict):
            return l1_val, l2_val
        if regex and re.search(regex, text):
            return l1_val, l2_val

        return row['l1'], row['l2']

    _before = df['l1'].notna().sum()
    df[['l1', 'l2']] = df.apply(_match, axis=1, result_type='expand')
    _delta = int(df['l1'].notna().sum() - _before)

    total_changed += _delta
    total_nan     -= _delta
    assert len(df) == total_changed + total_nan, "Contradicton found"

    fname = f'NEW_REGIONS_in_progress_{section}.csv'
    df.to_csv(fname, index=False)

    print(
        f"[{section}]  +{_delta:>5}  |  \n"
        f"размечено: {total_changed}  |  \n"
        f"осталось: {total_nan}  →  {fname}"
    )

    return df, total_changed, total_nan


def show_unique_texts(
    contains=None,
    regex=None,
    only_letters=False,
    strict=False,
    export=None,
    full=False,
    layers=None,
):
    s = df['Text'].dropna().astype(str)

    if layers:
        mask = df.loc[s.index, 'Layer'].isin(layers if isinstance(layers, list) else [layers])
        s = s[mask]
    
    if only_letters:
        s = s[s.str.contains(r'[а-яА-ЯёЁa-zA-Z]', regex=True)]
    if contains:
        items = [contains] if isinstance(contains, str) else contains
        if strict:
            s = s[s.str.lower().isin([x.lower() for x in items])]
        else:
            pattern = '|'.join(items)
            s = s[s.str.contains(pattern, case=False)]
    if regex:
        s = s[s.str.contains(regex, regex=True)]
    
    result = df.loc[s.index].drop_duplicates() if full else sorted(s.unique())
    if export:
        (result if full else pd.DataFrame(s.unique(), columns=['Text'])).to_excel(export, index=False)
    return result

### 1. Reseting l1 & l2

In [357]:
df['l1'] = None
df['l2'] = None

Tracking the progress

In [358]:
total_changed = 0
total_nan = len(df)
print(f'total_changed: {total_changed}, total_nan: {total_nan}')

total_changed: 0, total_nan: 3069680


### 2. Make "POINTS" -> elevation points

In [359]:
mask = df['Type'] == 'POINT'
df.loc[mask, ['l1', 'l2']] = ['Рельеф', 'Отметки высот']
_delta = int(mask.sum())
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_2.csv', index=False)
print(f"[2]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[2]  +436410  | размечено: 436410 | осталось: 2633270


### 3. Pattern for elevation points

216613

In [360]:
df, total_changed, total_nan = apply_labels(
    df, 'Рельеф', 'Отметки высот',
    section='3',
    regex=r'^\d{3,4}\.\d{1,2}$',
    total_changed=total_changed, total_nan=total_nan,
)


[3]  +216613  |  
размечено: 653023  |  
осталось: 2416657  →  NEW_REGIONS_in_progress_3.csv


## 4. TEXT
### 4.1. Pattern TEXT = 'A'  
Just changed: 7692  
Rows changed: 660715  
Rows remained: 2408965  

Checking the layers & visual exploring the files with uncleared Layers

In [361]:
df, total_changed, total_nan = apply_labels(
    df, 'Твердые покрытия', 'Асфальт',
    section='4.1',
    strict=['А'],
    total_changed=total_changed, total_nan=total_nan,
)

[4.1]  + 7695  |  
размечено: 660718  |  
осталось: 2408962  →  NEW_REGIONS_in_progress_4.1.csv


### 4.2. Pattern TEXT "Ц" / "ПЛ" -> Плитка

Just changed: 4483  
Rows changed: 665198  
Rows remained: 2404482  

In [362]:
df, total_changed, total_nan = apply_labels(
    df, 'Твердые покрытия', 'Плитка',
    section='4.2',
    strict=['ц', 'ц.', 'пл', 'пл.', 'бр'],
    total_changed=total_changed, total_nan=total_nan,
)


[4.2]  + 4483  |  
размечено: 665201  |  
осталось: 2404479  →  NEW_REGIONS_in_progress_4.2.csv


# TEXT Analysis function

Улицы (шоссе, проспекты, проезды) - границы - кабели - карусель - котельная - рельеф - "строительная площадка" - флигель - Спортивно-развлекательный центр - КАЧЕЛЬ

In [364]:
# show_unique_texts(contains='паравоз', strict=False, full=False)
# export='TEXT_ANALYSIS.xlsx'

### 4.3. TEXT -> (МАФ, Игровое оборудование)

Just changed: 339  
Rows changed: 665537  
Rows remained: 2404143  

In [365]:
df, total_changed, total_nan = apply_labels(
    df, 'МАФ', 'Игровое оборудование',
    section='4.3',
    strict=['тир'],
    non_strict=[
        'карусел', 'качел', 'горк', 'песочниц', 'кораблик',
        'гамак', 'змейк', 'батут', 'колокольчик', 'аттракцион',
        'бабочк', 'космос', 'ралли', 'паравоз', 'кенгуру',
        'лодочк', 'скороход', 'игровой', 'маф', 'малые',
    ],
    total_changed=total_changed, total_nan=total_nan,
)

[4.3]  +  339  |  
размечено: 665540  |  
осталось: 2404140  →  NEW_REGIONS_in_progress_4.3.csv


In [366]:
# _before = df["l1"].notna().sum()

# df[['l1', 'l2']] = df.apply(assign_labels, axis=1, result_type='expand')

# _delta = df["l1"].notna().sum() - _before
# total_nan -= _delta
# total_changed += _delta
# assert len(df) == total_changed + total_nan
# print(f'Just changed: {_delta}')
# print(f'Rows changed: {total_changed}')
# print(f'Rows remained: {total_nan}')

# df.to_csv('NEW_REGIONS_in_progress.csv')

### 4.3.1. "Атракцион" -> (МАФ, Игровое оборудование)

In [634]:
df, total_changed, total_nan = apply_labels(
    df, 'МАФ', 'Игровое оборудование',
    section='4.3.1.',
    non_strict=[
        'атракцион',
    ],
    total_changed=total_changed, total_nan=total_nan,
)

[4.3.1.]  +   17  |  
размечено: 674259  |  
осталось: 2395421  →  NEW_REGIONS_in_progress_4.3.1..csv


### 4.3.2. "колесо" -> (МАФ, Игровое оборудование)

In [667]:
# show_unique_texts(contains='детское кафе', strict=True, full=True)
# export='TEXT_ANALYSIS.xlsx'

In [668]:
df, total_changed, total_nan = apply_labels(
    df, 'МАФ', 'Игровое оборудование',
    section='4.3.1.',
    non_strict=[
        'колесо',
    ],
    strict=[
        'Детская беседка ',
        'детское кафе'
    ],
    total_changed=total_changed, total_nan=total_nan,
)

[4.3.1.]  +    5  |  
размечено: 674264  |  
осталось: 2395416  →  NEW_REGIONS_in_progress_4.3.1..csv


### 4.4. TEXT -> (МАФ, Спортивное оборудование)

Just changed: 175  
Rows changed: 665712  
Rows remained: 2403968  

In [367]:
# show_unique_texts(contains='гамак', strict=False, full=False)
# export='TEXT_ANALYSIS.xlsx'

In [368]:
df, total_changed, total_nan = apply_labels(
    df, 'МАФ', 'Спортивное оборудование',
    section='4.4',
    non_strict=[
        'снар', 'турник', 'брусья', 'трибун', 'баскетбольно',
        'волейбольно', 'теннисные', 'пинг', 'шахмат', 'настольн',
        'шведск', 'тенстол', 'сетк', 'футбольные',
    ],
    total_changed=total_changed, total_nan=total_nan,
)

[4.4]  +  175  |  
размечено: 665715  |  
осталось: 2403965  →  NEW_REGIONS_in_progress_4.4.csv


### 4.5. TEXT -> (МАФ, Уличная мебель)

In [369]:
# show_unique_texts(contains='урн', strict=False, full=False)
# export='TEXT_ANALYSIS.xlsx'

In [370]:
df, total_changed, total_nan = apply_labels(
    df, 'МАФ', 'Уличная мебель',
    section='4.5',
    strict=['(урн)'],
    non_strict=['вело', 'лавоч', 'скамей', 'урна', 'урны', 'контейн', 'urna', 'вазон', 'клумб'],
    total_changed=total_changed, total_nan=total_nan,
)

[4.5]  +  398  |  
размечено: 666113  |  
осталось: 2403567  →  NEW_REGIONS_in_progress_4.5.csv


### 4.6. TEXT -> (Здания и сооружения, Неизвестно)

Just changed: 1650  
Rows changed: 667714  
Rows remained: 2401966  

In [371]:
# show_unique_texts(contains='котель', strict=False, full=False).
# export='TEXT_ANALYSIS.xlsx'

In [372]:
df, total_changed, total_nan = apply_labels(
    df, 'Здания и сооружения', 'Неизвестно',
    section='4.6',
    strict=[
        'ДК Пушкино', 'Кинотеатр "Амигос"', 'железнодорожная платформа Пушкино',
        'кинологический центр', 'кинотеатр', 'здание театра',
        'Воскресенский собор', 'Преображенский собор',
        'Собор Распятия Христа', 'Успенский собор',
        ' гараж кирп', ' гараж кирп ', ' гараж мет', ' гараж мет ',
        'Гараж', 'Гаражи', 'Гаражный комплекс', 'КН     Гаражи',
        'ТП 98-гаражи', 'гараж', 'гаражи',
        'Котельная 15', 'Котельная 3', 'Котельная ОАО Металлист',
        'котельная', 'отель', 'отель Марк',
    ],
    non_strict=[
        'школа', 'быт', 'сарай', 'храм', 'монастырь', 'магазин',
        'баня', 'пятерочка', 'гимназия', 'фонд', 'сооруж', 'туалет', 'дк ',
        'дом культуры',
        'дом быта',
        'манеж'
        ],
    regex=r'^(?:\d(?:кн|км|кж|кд|ж|м|тп|дн|смж|мн)|\d{1,2}[а-яё]|\d+/\d+)$',
    total_changed=total_changed, total_nan=total_nan,
)

[4.6]  + 1647  |  
размечено: 667760  |  
осталось: 2401920  →  NEW_REGIONS_in_progress_4.6.csv


### 4.6.1. Насосная станция -> (Здания и сооружения, Неизвестно)

In [490]:
show_unique_texts(contains='насосная станция', strict=False, full=False)


['Насосная станция']

In [491]:
df, total_changed, total_nan = apply_labels(
    df, 'Здания и сооружения', 'Неизвестно',
    section='4.6.1',
    non_strict=[
        'насосная станция'
        ],
    total_changed=total_changed, total_nan=total_nan,
)

[4.6.1]  +    2  |  
размечено: 669764  |  
осталось: 2399916  →  NEW_REGIONS_in_progress_4.6.1.csv


### 4.6.2. Детский сад -> (Здания и сооружения, Неизвестно)

In [672]:
show_unique_texts(contains='детский сад', strict=False, full=False)


['Детский сад {\\fArial|b0|i0|c204|p34;№} 45 "Колосок"']

In [673]:
df, total_changed, total_nan = apply_labels(
    df, 'Здания и сооружения', 'Неизвестно',
    section='4.6.2',
    non_strict=[
        'детский сад'
        ],
    total_changed=total_changed, total_nan=total_nan,
)

[4.6.2]  +    2  |  
размечено: 674266  |  
осталось: 2395414  →  NEW_REGIONS_in_progress_4.6.2.csv


### 4.6.3. Полиция / спортивный зал -> (Здания и сооружения, Неизвестно)

In [692]:
# show_unique_texts(contains='дворц', strict=False, full=True)


In [693]:
df, total_changed, total_nan = apply_labels(
    df, 'Здания и сооружения', 'Неизвестно',
    section='4.6.3',
    non_strict=[
        'полиц',
        'спортивный',
        'центр',
        'комплекс',
        'дворец',
        'дворц'
        ],
    total_changed=total_changed, total_nan=total_nan,
)

[4.6.3]  +   51  |  
размечено: 674317  |  
осталось: 2395363  →  NEW_REGIONS_in_progress_4.6.3.csv


### 4.7. TEXT -> (МАФ, Шлагбаум)

Just changed: 44  
Rows changed: 667758  
Rows remained: 2401922  

In [373]:
# show_unique_texts(contains='шлагб', strict=False, full=False)
# export='TEXT_ANALYSIS.xlsx'

In [374]:
df, total_changed, total_nan = apply_labels(
    df, 'МАФ', 'Шлагбаум',
    section='4.7', non_strict=['шлагб'],
    total_changed=total_changed, total_nan=total_nan,
)

[4.7]  +   44  |  
размечено: 667804  |  
осталось: 2401876  →  NEW_REGIONS_in_progress_4.7.csv


### 4.8. TEXT -> (МАФ, Калитка)

Just changed: 66  
Rows changed: 667824  
Rows remained: 2401856  

In [375]:
df, total_changed, total_nan = apply_labels(
    df, 'МАФ', 'Калитка',
    section='4.8', non_strict=['калитк'],
    total_changed=total_changed, total_nan=total_nan,
)

[4.8]  +   66  |  
размечено: 667870  |  
осталось: 2401810  →  NEW_REGIONS_in_progress_4.8.csv


### 4.9. TEXT -> (Здания и сооружения, Мост)

In [376]:
df, total_changed, total_nan = apply_labels(
    df, 'Здания и сооружения', 'Мост',
    section='4.9',
    strict=[
        ' мост', ' мост верх', ' мост низ', '(дермост)', '(мостик)',
        'Мост', 'мост', 'мост дер на мет', 'мост мет.',
        'на мосту в лотках', 'пешеходный мост',
    ],
    total_changed=total_changed, total_nan=total_nan,
)

[4.9]  +   34  |  
размечено: 667904  |  
осталось: 2401776  →  NEW_REGIONS_in_progress_4.9.csv


### 4.10. TEXT -> (МАФ, Памятники)

In [377]:
# show_unique_texts(contains='мемориал', strict=False, full=False)
# export='TEXT_ANALYSIS.xlsx'

In [378]:
df, total_changed, total_nan = apply_labels(
    df, 'МАФ', 'Памятники',
    section='4.10',
    non_strict = [
        'мемориал',
        'скульптур',
        'пушка',
        'танк',
        'бтр',
        'оруди',
        'памятн',
        'погибш',
        'солдат',
        'пам'
    ],
    total_changed=total_changed, total_nan=total_nan,
)

[4.10]  +   63  |  
размечено: 667967  |  
осталось: 2401713  →  NEW_REGIONS_in_progress_4.10.csv


### 4.11. TEXT -> (Твердные покрытия, Асфальт)

In [457]:
# show_unique_texts(contains='ш.', strict=False, full=False)[:30]
# show_unique_texts(contains='сквер', full=False)

# export='ул._TEXT_ANALYSIS.xlsx'

['Сквер им. В.Н. Леонова, Пл. Урицкого, ',
 'Славы, сквера с прудом, территория вокруг',
 'и Сквер у ДК"Победа" (Центральный парк) в',
 'сквер']

In [458]:
df, total_changed, total_nan = apply_labels(
    df, 'Твердые покрытия', 'Асфальт',
    section='4.11',
    non_strict = [
        'а/б',
        'крошк',
        'аразр',
        'улица',
        'просп.',
        'проезд',
        'шоссе',
        'бульвар',
        'ул.'
    ],
    strict = [
    ' асф',
    ' асф/борд',
    ' асф/смешанное покрытие',
    ' асф/смешанное покрытие/щеб',
    ' асф/ц пл',
    ' асф/щеб',
    ' борд/асф',
    ' дом асф',
    ' дор асф',
    ' край асф',
    ' троп асф',
    ' троп асф/заб мет',
    ' ц пл/ асф',
    ' ц пл/асф',
    '(асф а/ц плит)',
    '(асф)',
    '(асф15)',
    '(асф158)',
    '(асф8)',
    '(асфбет)',
    '(асфгравий)',
    '(асфтр)',
    'Асф. кр.',
    'Асф. крошка',
    'Асф.кр',
    'асф',
    'асф отмостка',
    'асф троп',
    'асф троп/грунт троп',
    'асф троп/щеб троп',
    'асф.',
    'асф. пл.',
    'асф.лоток',
    'асф/бордюр(асф)',
    'бордюр (асф)',
    'заасф.',
    'заасфальт.',
    'под асф.',
    'ц пл/асф', 
    'ц пл/асф троп',
    'А разв.',
    'Аразв.',
    'разв',
    'разв.',
    'развал',
    'развал.Н',
    ],
    total_changed=total_changed, total_nan=total_nan,
)
    

[4.11]  + 1795  |  
размечено: 669762  |  
осталось: 2399918  →  NEW_REGIONS_in_progress_4.11.csv


### 4.12. TEXT -> (Озеленение, Дерево)

In [562]:
# show_unique_texts(contains='iva', strict=False, full=False)

# analysis[analysis['l1'].isna()]['Text'].tolist()
# analysis = show_unique_texts(contains='ива', full=True, strict=False, export='ива._TEXT_ANALYSIS.xlsx')
# export='ул._TEXT_ANALYSIS.xlsx'

In [563]:
df, total_changed, total_nan = apply_labels(
    df, 'Озеленение', 'Дерево',
    section='4.12',
non_strict = [
    'лиственн', 
    'листв',
    'липа', 
    'лип',
    'берез',
    'рябин',
    'каштан',
    'ясен',
    'ольх',
    'клен', 
    'туя',
    'сосна',
    'ива', 
    'осин',
    'тополь',
    'акаци',
    'слива' 
    'lipa',
    'el ',
    'klen',
    'olha',
    'osina',
    'yasen',
    'tuya',
    'dub',
    'iva',
],
strict = [
    ' дер сухое',
    'дер листв',
    'дер листв х2',
    'дер листв х3',
    'дер листв х4',
    'дер листв х6',
    'кусты и дер контур',
    'ряд дер туя',
    'Деревья',
    'ель',
    'дуб',
    'дуб2',
    'дуб4',
    'дуб6',
    'дуб7',
    'дуб8',
    'дуб9',
    'дуб10',
    'дуб 40',
    'вяз',
],
regex=r'(?i)(?<![а-яё])ива(?:\d+(?:[хx]\d+)?(?:\s*сух)?(?:зав|мнгст\d+|двуст)?)?(?![а-яё])'
      r'|(?<![a-z])el(?=\s|$)',
total_changed=total_changed, total_nan=total_nan,
)

[4.12]  + 2281  |  
размечено: 672045  |  
осталось: 2397635  →  NEW_REGIONS_in_progress_4.12.csv


### 4.13. TEXT -> (Озеленение, Кустарник)

In [572]:
# show_unique_texts(contains='kust', strict=False, full=True)

# analysis[analysis['l1'].isna()]['Text'].tolist()
# analysis = show_unique_texts(contains='ива', full=True, strict=False, export='ива._TEXT_ANALYSIS.xlsx')
# export='ул._TEXT_ANALYSIS.xlsx'

In [573]:
df, total_changed, total_nan = apply_labels(
    df, 'Озеленение', 'Кустарник',
    section='4.13',
non_strict = [
    'куст',
    'kust'
],
total_changed=total_changed, total_nan=total_nan,
)

[4.13]  +  198  |  
размечено: 672243  |  
осталось: 2397437  →  NEW_REGIONS_in_progress_4.13.csv


### 4.14. TEXT -> (Водопроницаемые покрытия, Щебень)

In [577]:
show_unique_texts(contains='щеб', strict=False, full=True, export='щеб_TEXT_ANALYSIS.xlsx')

# analysis[analysis['l1'].isna()]['Text'].tolist()
# analysis = show_unique_texts(contains='ива', full=True, strict=False, export='ива._TEXT_ANALYSIS.xlsx')
# export='ул._TEXT_ANALYSIS.xlsx'

,Source_File,Handle,Layer,Type,BlockName,Text,l1,l2
208669,077-22-ИГДИ-tp.dxf,93D3,Топонимика,TEXT,NaN,ЩЕБ.,None,None
249997,077-22-ИГДИ-tp.dxf,1C83B,Code,MTEXT,NaN,усил откос щеб,None,None
249998,077-22-ИГДИ-tp.dxf,1C83D,Code,MTEXT,NaN,усил откос щеб,None,None
249999,077-22-ИГДИ-tp.dxf,1C83F,Code,MTEXT,NaN,усил откос щеб,None,None
250000,077-22-ИГДИ-tp.dxf,1C841,Code,MTEXT,NaN,усил откос щеб,None,None
...,...,...,...,...,...,...,...,...
3043186,Топосъемка.dxf,18EA7,Отметка,TEXT,NaN,щеб.,None,None
3043219,Топосъемка.dxf,18EE4,Отметка,TEXT,NaN,щеб.,None,None
3043247,Топосъемка.dxf,18F13,Отметка,TEXT,NaN,щеб.,None,None
3043276,Топосъемка.dxf,18F46,Отметка,TEXT,NaN,щеб.,None,None


In [578]:
df, total_changed, total_nan = apply_labels(
    df, 'Водопроницаемые покрытия', 'Щебень',
    section='4.14',
    non_strict = [
        'щеб'
    ],
    total_changed=total_changed, total_nan=total_nan,
)

[4.14]  +  150  |  
размечено: 672393  |  
осталось: 2397287  →  NEW_REGIONS_in_progress_4.14.csv


### 4.15. TEXT -> (Водопроницаемые покрытия, Песок)

In [583]:
# show_unique_texts(contains='пес', strict=False, full=True, export='пес_TEXT_ANALYSIS.xlsx')

# analysis[analysis['l1'].isna()]['Text'].tolist()
# analysis = show_unique_texts(contains='ива', full=True, strict=False, export='ива._TEXT_ANALYSIS.xlsx')
# export='ул._TEXT_ANALYSIS.xlsx'

In [584]:
df, total_changed, total_nan = apply_labels(
    df, 'Водопроницаемые покрытия', 'Песок',
    section='4.15',
    non_strict = [
        'пес'
    ],
    total_changed=total_changed, total_nan=total_nan,
)

[4.15]  +  101  |  
размечено: 672494  |  
осталось: 2397186  →  NEW_REGIONS_in_progress_4.15.csv


### 4.16. TEXT -> (Водопроницаемые покрытия, Грунт)

In [592]:
# show_unique_texts(contains='грунт', strict=False, full=True)

# analysis[analysis['l1'].isna()]['Text'].tolist()
# analysis = show_unique_texts(contains='ива', full=True, strict=False, export='ива._TEXT_ANALYSIS.xlsx')
# export='ул._TEXT_ANALYSIS.xlsx'

In [593]:
df, total_changed, total_nan = apply_labels(
    df, 'Водопроницаемые покрытия', 'Грунт',
    section='4.16',
    non_strict = [
        'грунт'
    ],
    strict = [
        'гр.'
    ],
    exclude_strict = [
        'каб. в грунте ПАО "Ростелеком"'
    ],
    total_changed=total_changed, total_nan=total_nan,
)

[4.16]  + 1493  |  
размечено: 673987  |  
осталось: 2395693  →  NEW_REGIONS_in_progress_4.16.csv


### 4.17. TEXT -> (Здания и сооружения, Лестницы и пандусы)

In [599]:
# show_unique_texts(contains='лестниц', strict=False, full=True)

# analysis[analysis['l1'].isna()]['Text'].tolist()
# analysis = show_unique_texts(contains='ива', full=True, strict=False, export='ива._TEXT_ANALYSIS.xlsx')
# export='ул._TEXT_ANALYSIS.xlsx'

In [600]:
df, total_changed, total_nan = apply_labels(
    df, 'Здания и сооружения', 'Лестницы и пандусы',
    section='4.17',
    non_strict = [
        'лестниц',
        'пандус'
    ],
    total_changed=total_changed, total_nan=total_nan,
)

[4.17]  +   93  |  
размечено: 674080  |  
осталось: 2395600  →  NEW_REGIONS_in_progress_4.17.csv


### 4.18. TEXT -> (Здания и сооружения, Подпорная стенка)

In [630]:
# show_unique_texts(contains='Атракцион', strict=False, full=True)

# analysis[analysis['l1'].isna()]['Text'].tolist()
# analysis = show_unique_texts(contains='ива', full=True, strict=False, export='ива._TEXT_ANALYSIS.xlsx')
# export='ул._TEXT_ANALYSIS.xlsx'

In [614]:
df, total_changed, total_nan = apply_labels(
    df, 'Здания и сооружения', 'Подпорная стенка',
    section='4.17',
    non_strict = [
        'подпорн',
        'стена',
        'блок'
    ],
    total_changed=total_changed, total_nan=total_nan,
)

[4.17]  +   38  |  
размечено: 674118  |  
осталось: 2395562  →  NEW_REGIONS_in_progress_4.17.csv


### 4.19. TEXT -> (Здания и сооружения, Неизвестно)

In [624]:
# show_unique_texts(contains='бытовк', strict=False, full=True)

# analysis[analysis['l1'].isna()]['Text'].tolist()
# analysis = show_unique_texts(contains='ива', full=True, strict=False, export='ива._TEXT_ANALYSIS.xlsx')
# export='ул._TEXT_ANALYSIS.xlsx'

In [625]:
df, total_changed, total_nan = apply_labels(
    df, 'Здания и сооружения', 'Неизвестно',
    section='4.19',
    non_strict = [
        'навес',
        'павильон',
        'здани',
    ],
    total_changed=total_changed, total_nan=total_nan,
)

[4.19]  +  124  |  
размечено: 674242  |  
осталось: 2395438  →  NEW_REGIONS_in_progress_4.19.csv


### 4.20. TEXT -> (Здания и сооружения, Неизвестно)

In [654]:

set(show_unique_texts(regex=r'(?i)(?:рез(?:ин(?:овое)?)?|иск|спец)[.\s]+покр(?:ытие)?\.?|покр\.', full=False))


{'Иск.покр.',
 'Спец. покрытие',
 'Спец.покр.',
 'Спец.покрытие',
 'покр.',
 'рез.покр.',
 'резин.покр.',
 'резиновое покрытие',
 'спец покр',
 'спец. покрытие'}

In [694]:
# show_unique_texts(regex=r'(?i)(?:рез(?:ин(?:овое)?)?|иск|спец)[.\s]покр(?:ытие)?\.?|покр\.'
# , strict=False, full=True)

# show_unique_texts(contains='спорт', strict=False, full=True, export='спорт_TEXT_ANALYSIS.xlsx')

# analysis[analysis['l1'].isna()]['Text'].tolist()
# analysis = show_unique_texts(contains='ива', full=True, strict=False, export='ива._TEXT_ANALYSIS.xlsx')
# export='ул._TEXT_ANALYSIS.xlsx'

In [695]:
df, total_changed, total_nan = apply_labels(
    df, 'Твердые покрытия', 'Резиновая крошка',
    section='4.19',
    non_strict=[
        'дет',
        'спорт'
    ],
    exclude=[
        'Игровое поле'
    ],
    regex=r'(?i)(?:рез(?:ин(?:овое)?)?|иск|спец)[.\s]+покр(?:ытие)?\.?|покр\.',
    total_changed=total_changed, total_nan=total_nan,
)

[4.19]  +  219  |  
размечено: 674536  |  
осталось: 2395144  →  NEW_REGIONS_in_progress_4.19.csv


### 4.21. TEXT -> (Инженерные сети, -)

In [705]:
# show_unique_texts(regex=r'(?i)(?:рез(?:ин(?:овое)?)?|иск|спец)[.\s]покр(?:ытие)?\.?|покр\.'
# , strict=False, full=True)

# analysis[analysis['l1'].isna()]['Text'].tolist()
# analysis = show_unique_texts(contains='ива', full=True, strict=False, export='ива._TEXT_ANALYSIS.xlsx')
# export='ул._TEXT_ANALYSIS.xlsx'

In [706]:
layer_map = {
    ('канализация', 'Канализация',  
     'канализация напорная', 'ИИ_ТРУБ_Канализация_025'): ('Инженерные сети', 'Канализация бытовая'),
    ('канализация ливневая', 'Ливневая канализация', 'ВОДООТВЕДЕ-КОЛОДЦЫ_И_'): ('Инженерные сети', 'Ливневая канализация'),
    ('Водопровод', 'водопровод', 'ИИ_ТРУБ_Вода_025'): ('Инженерные сети', 'Водопровод'),
    ('газопровод', 'ГАЗОПРОВОД-КОЛОДЦЫ_И_'): ('Инженерные сети', 'Газопровод'),
    ('теплотрасса', 'теплотрасса надземная'): ('Инженерные сети', 'Теплосеть'),
    ('Дренаж', 'дренаж'): ('Инженерные сети', 'Дренаж'),
    ('SVAZ', 'SVYAZ'): ('Инженерные сети', 'Сети связи'),
    ('CABEL',): ('Инженерные сети', 'Электроснабжение'), 
    (
        'неизвестно',
        'Неизвестно',
       '0', 'Высоты',
        'текст', 'Отметки', 'отметки', 'Рельеф',
        'выс2',
       'ИЛОПРОВОД', 'Высотные отметки точек', 'TEXT',
        'ИИ_ОТМЕТКА_025',
    ): ('Инженерные сети', 'Прочие кабели')
}

_layer_to_labels = {layer: labels for layers, labels in layer_map.items() for layer in layers}

_kl_regex = re.compile(r'к\.л\.\d+(?:\.\d+)?', re.IGNORECASE)

def assign_labels(row):
    if row['l1'] is not None:
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    if _kl_regex.search(row['Text']):
        labels = _layer_to_labels.get(row['Layer'])
        if labels:
            return labels
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_4.21.csv', index=False)
print(f"[4.21]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[4.21]  +3517  | размечено: 678053 | осталось: 2391627


### 4.21.1. TEXT -> (Инженерные сети, -)

In [ ]:
_pipe_re = re.compile(
    r'(?i)'
    r'чуг|'
    r'ст[\.\s]|'
    r'лот\.|'
    r'в\.тр|'
    r'в\.г\.|'
    r'н\.д\.|'
    r'дно\s|'
    r'd-\d+|'
    r'\d+\.\d+[а-яёa-z]|'
    r'[а-яё]+\.\d+'
)

_pipe_exclude_re = re.compile(
    r'(?i)'
    r'канава|'              # канава-0.6м
    r'цоколь|'             # 112.45цоколь
    r'авт\.ост|'           # авт.ост.
    r'^наст\.$|'           # наст. — только целая строка
    r'крест на камне|'     # (крест на камне)
    r'горизонтал|'          # Сплошные горизонтали...
    r'^уч\.\d+[а-яё]?$'
)

_pipe_layer_map = {
    # Канализация бытовая
    (
        'канализация', 'Канализация', 'КАНАЛИЗАЦИЯ',
        'канализация напорная', 'Канализации',
        'ВОДООТВЕДЕ-КАНАЛИЗАЦИ', 'КАНАЛИЗАЦИ-КАНАЛИЗАЦИ',
        'КАНАЛИЗАЦИ-РЕШЕТКИ_И_', '30_Канализация',
        'ИИ_ТРУБ_Канализация_025', 'Бытовая канализация',
        'KANAL_33', 'SEWAGE', 'TG_KANAL',
        'канализация2', 'Напорный коллектор',
        'Водопровод_и_канализация'
    ): ('Инженерные сети', 'Канализация бытовая'),

    # Ливневая канализация
    (
        'канализация ливневая', 'Ливневая канализация',
        'Ливневые канализации', 'ВОДООТВЕДЕ-КОЛОДЦЫ_И_',
         'Водостоки', '+Канава', 'TG_LKANAL'
    ): ('Инженерные сети', 'Ливневая канализация'),

    # Водопровод
    (
        'Водопровод', 'водопровод', 'ВОДОПРОВОД',
        'Водопроводы', 'ВОДОПРОВОД-ВОДОПРОВОД',
        'ИИ_ТРУБ_Вода_025', 'WATER_PIPE',
        '31_Водопровод', 'TG_VODA', 'CK_Водопровод',
        'Водопровод подземный', 'Технический водопровод',
        'WODOPROV_32', 'Водопровод2',
    ): ('Инженерные сети', 'Водопровод'),

    # Газоснабжение
    (
        'газопровод', 'ГАЗОПРОВОД', 'Газопровод',
        'Газопроводы', 'ГАЗОПРОВОД-ГАЗОПРОВОД',
        'ГАЗОПРОВОД-КОЛОДЦЫ_И_', '33_Газопровод',
        'TG_GAS', 'GAZ_PIPE', 'GAZ_35',
    ): ('Инженерные сети', 'Газоснабжение'),

    # Теплосеть
    (
        'теплотрасса', 'теплотрасса надземная',
        'Теплосеть', 'Теплосети', 'ТЕПЛОСЕТИ-ТЕПЛОСЕТИ',
        'ТЕПЛОСЕТИ-КОЛОДЦЫ_И_', 'ТЕПЛОФИКАЦИЯ',
        '32_Теплосеть', 'TG_TEPLO', 'HEAT_PIPE',
        'GEO_TEPLO_PIPE', 'Теплосеть подземная',
        'Теплосети_Опоры Назем',
    ): ('Инженерные сети', 'Теплосеть'),

    # Дренаж
    (
        'Дренаж', 'дренаж',
    ): ('Инженерные сети', 'Дренаж'),

    # Сети связи
    (
        'SVAZ', 'SVYAZ', 'TG_SVAZ',
        'Кабели связи', '35_Кабели связи',
        '35_Телефон', 'СВЯЗЬ-КОЛОДЦЫ_И_',
        'катодная защита',
    ): ('Инженерные сети', 'Сети связи'),

    # Электроснабжение
    (
        'CABEL', 'TG_CABEL', 'ELEKT_KAB_37',
        'Электрокабели', 'ЭЛЕКТ_КАБЕЛИ',
        'Кабели низкого напряжения', 'Кабели высокого напряжения',
        '37_Кабели низкого напряжения', '38_Кабели высокого напряжения',
        '37_Кабель низкого напряжения',
        'Объекты электропередачи', '07_Объекты электропередачи',
        'Кабельные столбики',
    ): ('Инженерные сети', 'Электроснабжение'),

    # Прочие трубопроводы
    (
        'трубы под дорогой', 'ИЛОПРОВОД',
        'Трубопроводы спецназначения', 'PROCHIE',
        'ПРОЧИЕ_СЕТ-СЕТИ', 'ПРОЧИЕ_СЕТ-КОЛОДЦЫ__К',
        'Инженерные сооружения', '06_Инженерно-технические сооружения',
        'Коммуникации', 
    ): ('Инженерные сети', 'Прочие трубопроводы'),

    # Прочие кабели — неясные/общие слои
    (
        'неизвестно', 'Неизвестно', '0',
        'текст', 'TEXT', 'Отметки', 'отметки',
        'Рельеф', 'Высоты', 'Высотные отметки точек',
        'ИИ_ОТМЕТКА_025', 'топоОтметка', 'Топонимика',
        'NAD_MDEFAULT', 'NAD_M34', 'NAD_M37', 'NAD_M32',
        'PI_OTDEFAULT', 'PI_OTD-Комков2', 'PI_OT0',
        'LAND', 'GOR_B0', 'RELIEF_1', 'MOST_26',
        'CONSTRACTION', 'Слой1', 'Code', 'ramka', '2_Ramka',
        '03b-5', '12492', '12493', '12494', '12117',
        'GEO_BUILDINGS_TEXT', 'GEO_TEXT', 'GEO_HVC',
        'Survey.td2_point_desc_Mark', 'Survey.td2_Mark',
        'TG_KOLOD', '44_Крышки колодцев', 'Крышки колодцев',  '11b', 'кабели', 'ИИ_КАБЕЛЬ_025',
    ): ('Инженерные сети', 'Прочие кабели'),
}

_pipe_layer_to_labels = {
    layer: labels
    for layers, labels in _pipe_layer_map.items()
    for layer in layers
}

def assign_pipe_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    if _pipe_exclude_re.search(row['Text']):
        return row['l1'], row['l2']
    if _pipe_re.search(row['Text']):
        labels = _pipe_layer_to_labels.get(row['Layer'])
        if labels:
            return labels
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_pipe_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_4.21.1.csv', index=False)
print(f"[4.21.1]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[4.21.1]  +10586  | размечено: 887077 | осталось: 2182603


: 

### 4.22. TEXT -> (Служебные элементы, Номера деревьев)

In [18]:
# show_unique_texts(regex=r'(?i)ном[.\s]*дер|номер[а]?\s*дерев|деревья\s*и?\s*номер'
# , strict=False, full=True)

# analysis[analysis['l1'].isna()]['Text'].tolist()
# analysis = show_unique_texts(contains='ива', full=True, strict=False, export='ива._TEXT_ANALYSIS.xlsx')
# export='ул._TEXT_ANALYSIS.xlsx'

In [26]:
_tree_num_layer_re = re.compile(
    r'(?i)ном[.\s]*дер|номер[а]?\s*дерев|деревья\s*и?\s*номер',
)

_tree_num_regex = re.compile(r'^\d{1,6}$')
total_changed = int(df['l1'].notna().sum())
total_nan     = int(df['l1'].isna().sum())

def assign_tree_numbers(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    if (isinstance(row['Layer'], str)
            and _tree_num_layer_re.search(row['Layer'])
            and _tree_num_regex.match(row['Text'].strip())):
        return 'Служебные элементы', 'Номер дерева'
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_tree_numbers, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_4.22.csv', index=False)
print(f"[4.22]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[4.22]  +161498  | размечено: 839551 | осталось: 2230129


### 4.23. TEXT -> (Служебные элементы, Кадастр)

In [ ]:
# total_changed = int(df['l1'].notna().sum())
# total_nan     = int(df['l1'].isna().sum())

In [7]:
df, total_changed, total_nan = apply_labels(
    df, 'Служебные элементы', 'Кадастр',
    section='4.23',
    regex=r'^\d{2}:\d{2}:\d{7}:\d+(?:\(\d+\))?$',
    total_changed=total_changed, total_nan=total_nan,
)

[4.23]  +  975  |  
размечено: 869582  |  
осталось: 2200098  →  NEW_REGIONS_in_progress_4.23.csv


### 4.24. TEXT -> 
    'Heights':   ('Рельеф', 'Отметки высот'),
    'Positions': ('Служебные элементы', 'Координаты'),

In [16]:
_coord_layer_map = {
    'Heights':   ('Рельеф', 'Отметки высот'),
    'Positions': ('Служебные элементы', 'Координаты'),
}

def assign_coord_layers(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    labels = _coord_layer_map.get(row['Layer'])
    if labels:
        return labels
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_coord_layers, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)

total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_4.24.csv', index=False)
print(f"[4.24]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[4.24]  +18470  | размечено: 876491 | осталось: 2193189


### 4.24.1. TEXT -> ('Рельеф', 'Отметки высот')

In [8]:
df, total_changed, total_nan = apply_labels(
    df, 'Рельеф', 'Отметки высот',
    section='4.24.1.',
    regex=r'^\d{3}\.\d{1,5}$',
    total_changed=total_changed, total_nan=total_nan,
)

[4.24.1.]  +12666  |  
размечено: 882248  |  
осталось: 2187432  →  NEW_REGIONS_in_progress_4.24.1..csv


### 4.24.2. TEXT -> ('Рельеф', 'Отметки высот') - пробелы, запятая, / Р, две точки

In [17]:
df, total_changed, total_nan = apply_labels(
    df, 'Рельеф', 'Отметки высот',
    section='4.24.2',
    regex=r'^\s*(?:\\P\s+)?\d{2,3}[.,]\d{1,5}\.?\s*$',
    total_changed=total_changed, total_nan=total_nan,
)

[4.24.2]  + 8359  |  
размечено: 890607  |  
осталось: 2179073  →  NEW_REGIONS_in_progress_4.24.2.csv


In [3]:
# df = pd.read_csv('NEW_REGIONS_in_progress_4.21.1.csv')

C:\Users\artem\AppData\Local\Temp\ipykernel_22864\2040617742.py:1: DtypeWarning: Columns (2,5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('NEW_REGIONS_in_progress_4.21.1.csv')


In [ ]:
df[(df['Layer'] == '') & (df['l1'].isna())]['Text'].drop_duplicates().to_excel('PI_NUDEFAULT.xlsx', index=False)

### 4.25. TEXT от 2 до 5 чисел -> ('?', '?') - в зависимости от слоя

In [94]:
_digits_layer_map = {
    # Служебные элементы / Прочее
    (
        '00p', 'PI_NUD-Бонд', 'PI_NUDEFAULT', 'GOR_BDEFAULT',
        '12493', '12494', 'PI_NUD-Рома', 'границы зданий',
        'Топонимика', '03_Здания и строения', 'растительность',
        'PI_NUD-Комков2', 'строения', 'КОД', 'R_contorlines',
        'текст', 'GEO_BUILDINGS_TEXT', 'дороги', 'Коды', 'text',
        '0_Point_Height 2', 'ЗД_ТОРГОВЫЕ', 'Слой1', 'NAD_MDEFAULT',
        'ИМЕНА', 'ИМЕНА_14', 'Растительность', 'ИМЕНА_H',
        'z_Условные знаки_2', 'z_Условные знаки_ПАША', '0',
        'C-TOPO-MAJR-N', 'PI_OTDEFAULT', 'name',
        'Здания и строения', 'здания и сооружения', 'GILYO_3',
        'KANAL_33', 'KAN_LIVN_34', 'WODOPROV_32', 'GAZ_35',
        'Podpis', 'MOST_26', 'TEXT', 'ЗДАНИЯ_И_С-ЗДАНИЯ',
        'ДОРОГИ-ТРУБЫ_ПОД_', 'РАСТИТЕЛЬН-ЛЕСА_И_ПОЛ',
        'РАСТИТЕЛЬН-ХАРАКТЕРИС', '13_Растительность',
        'ИМЕНА Nicon', 'TG_DOM', 'TG_RAST', 'Коды пикетов',
        'Растительность общая', 'ЗДАНИЯ_ЖИЛЫЕ', 'ЗДАН_ОБЩЕСТ',
        'ТРОТУАРЫ', 'Примечания к деревьям',
    ): ('Служебные элементы', 'Прочее'),

    # Служебные элементы / Номер дерева
    (
        'Подеревка Номер', 'Номер_дер', 'номер подеревки',
        'Подерёвка', '00p подеревка', 'дерНомер',
        '1_дерево', 'Деревья', 'Подеревка',
    ): ('Служебные элементы', 'Номер дерева'),

    # Служебные элементы / Размеры дерева
    (
        'Диаметры деревьев', 'Высоты деревьев', 'TREE DIAMETR',
    ): ('Служебные элементы', 'Размеры дерева'),

    # Служебные элементы / Нумерация
    (
        'Номер', 'Номер_2', 'Номер_к',
        'POINTS', 'ИМЯ ТОЧКИ', 'НОМЕРА ТОЧЕК',
        'номер', 'топоНомер', 'архНомер', '1Номер', '2Номер',
        'Без_Номер', 'номер пикета 1', 'НОМЕР', 'Points',
        'Подписи точек', 'Имена точек', '12492', '12493-1',
        'ТОЧКИ', 'Комков Номер', 'Козин Номер', 'К3_Номер',
        'номер 3', 'Номер3', 'Текст_строения', 'Номер2', 'T-Rastit',
    ): ('Служебные элементы', 'Нумерация'),

    # Инженерные сети / Прочие кабели
    (
        'Номер_каб', '45_Номера колодцев',
    ): ('Инженерные сети', 'Прочие кабели'),

    # Инженерные сети / Водопровод
    (
        'Водопровод',
    ): ('Инженерные сети', 'Водопровод'),

    # Инженерные сети / Сети связи
    (
        'Кабели связи',
    ): ('Инженерные сети', 'Сети связи'),

    # Рельеф / Отметки высот
    (
        'Высоты', 'Гидрография', 'Горизонтали--LBL', 'высоты',
        'Горизонтали', 'Отметки', 'отметки', 'высоты текст',
        'Рельеф', 'Горизонтали-О-LBL', 'Горизонтали целые',
        'подпись горизонталей', 'Основные горизонтали', 'GOR_B0',
        'C_Горизонтали', '11329-1(gor)', '20_Горизонтали утолщенные',
        'RELIEF_1', 'ДОРОГИ-ВЫСОТНЫЕ_О', 'РЕЛЬЕФ-ВЫСОТНЫЕ_О',
        'РЕЛЬЕФ-ГОРИЗОНТАЛ', '19_Горизонтали', 'Б горизонталь',
        'ИИ_РЕЛЬЕФ_050', 'точки высота', 'Отметка', 'Горизонтали_00',
        'Goriz_2',
    ): ('Рельеф', 'Отметки высот'),

    # Служебные элементы / Координаты
    (
        'Сетка координат',
    ): ('Служебные элементы', 'Координаты'),

    # Служебные элементы / Размеры
    (
        'Диаметр',
    ): ('Служебные элементы', 'Размеры'),
}

_digits_layer_to_labels = {
    layer: labels
    for layers, labels in _digits_layer_map.items()
    for layer in layers
}

_digits_re = re.compile(r'^\d{2,5}$')

def assign_digits_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    if _digits_re.match(row['Text'].strip()):
        labels = _digits_layer_to_labels.get(row['Layer'])
        if labels:
            return labels
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_digits_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_4.25.csv', index=False)
print(f"[4.25]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[4.25]  +128019  | размечено: 1018626 | осталось: 2051054


### 4.26. TEXT от 2 до 5 чисел + буквы (латиница) -> ('?', '?') - в зависимости от слоя

In [103]:
_weird_layer_map = {
    # Служебные элементы / Номер дерева
    (
        'Номера деревьев',
    ): ('Служебные элементы', 'Номер дерева'),

    # Озеленение / Дерево
    (
        'Code деревья',
    ): ('Озеленение', 'Дерево'),

    # Здания и сооружения / Неизвестно
    (
        'Строения',
    ): ('Здания и сооружения', 'Неизвестно'),

    # Служебные элементы / Размеры дерева
    (
        'Диаметры деревьев',
    ): ('Служебные элементы', 'Размеры дерева'),

    # Рельеф / Отметки высот
    (
        'Survey.td2_Mark',
    ): ('Рельеф', 'Отметки высот'),

    # Инженерные сети
    (
        'газопровод', 'ГАЗОПРОВОД', 'GAZ_PIPE',
    ): ('Инженерные сети', 'Газоснабжение'),

    (
        'канализация', 'SEWAGE',
    ): ('Инженерные сети', 'Канализация бытовая'),

    (
        'теплотрасса', 'Теплосети', 'Теплосеть', 'ТЕПЛОФИКАЦИЯ',
    ): ('Инженерные сети', 'Теплосеть'),

    (
        'ЛЭП',
    ): ('Инженерные сети', 'Электроснабжение'),
}

_weird_layer_to_labels = {
    layer: labels
    for layers, labels in _weird_layer_map.items()
    for layer in layers
}

_weird_re = re.compile(r'^\s*[a-zA-Z]{1,4}\d+\s*$|^\s*\d+[a-zA-Z]{1,4}\s*$')

def assign_weird_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    if _weird_re.match(row['Text']):
        labels = _weird_layer_to_labels.get(row['Layer'])
        if labels:
            return labels
        return 'Служебные элементы', 'Прочее'
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_weird_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_4.26.csv', index=False)
print(f"[4.26]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[4.26]  +30899  | размечено: 1049525 | осталось: 2020155


### 4.27. TEXT к. л. 123.45 -> ('?', '?') - в зависимости от слоя

In [109]:
_misc_re = re.compile(
    r'(?i)'
    r'к\.л[\.\s]|'          # к.л. 179.17 / к.л156.57
    r'каб[\.\s]|'            # каб. 139
    r'^втр\s|'               # втр 159,36
    r'\\fD\d+\||'            # {\fD431|c1|...}
    r'd=\d+|'                # d=2000 ж.б.
    r'н\.кол\.|'             # н.кол.148.48
    r'н\.к\.\d'              # н.к.153.67
)


_misc_layer_map = {
    # Рельеф / Отметки высот
    (
        'Рельеф', 'рельеф'
    ): ('Рельеф', 'Отметки высот'),

    # Канализация бытовая
    (
        'канализация', 'Канализации', 'канализация напорная',
        'канализация2', '30_Канализация', 'TG_KANAL',
    ): ('Инженерные сети', 'Канализация бытовая'),

    # Ливневая канализация
    (
        'канализация ливневая', 'Ливневые канализации',
        'Водостоки', 'ВОДООТВЕДЕ-КОЛОДЦЫ_И_',
    ): ('Инженерные сети', 'Ливневая канализация'),

    # Водопровод
    (
        'Водопровод', 'Водопроводы', 'CK_Водопровод',
        '31_Водопровод', 'ВОДОПРОВОД-КОЛОДЦЫ__К',
    ): ('Инженерные сети', 'Водопровод'),

    # Газоснабжение
    (
        'Газопроводы', '33_Газопровод',
    ): ('Инженерные сети', 'Газоснабжение'),

    # Теплосеть
    (
        'теплотрасса', 'теплотрасса надземная',
        'Теплопроводы', '32_Теплосеть',
        'ТЕПЛОСЕТИ-КОЛОДЦЫ_И_',
    ): ('Инженерные сети', 'Теплосеть'),

    # Дренаж
    (
        'Дренажи',
    ): ('Инженерные сети', 'Дренаж'),

    # Сети связи
    (
        'SVAZ', 'TG_SVAZ', 'Кабели связи',
        'TEL_CABLE', 'СВЯЗЬ', 'СВЯЗЬ-КОЛОДЦЫ_И_',
    ): ('Инженерные сети', 'Сети связи'),

    # Электроснабжение
    (
        'CABEL', 'TG_CABEL', 'ЭЛЕКТ_КАБЕЛИ',
        'Кабели электрические', 'кабели',
        'Кабели высокого напряжения', 'Кабели низкого напряжения',
        '38_Кабели высокого напряжения',
        'Объекты электропередачи', '07_Объекты электропередачи',
        'ЛЭП',
    ): ('Инженерные сети', 'Электроснабжение'),

    # Прочие кабели — неясные/общие
    (
        '12492', 'люки', 'Code', 'Code_3',
        'Ответы на вопросы', 'GEO_HVC', 'текст', 'TEXT',
        'GEO_TEXT', 'Топонимика', 'LAND', 'NAD_MDEFAULT',
        'Слой1', '0', 'Коммуникации',
        'ОГРАЖДЕНИЯ-ОГРАЖДЕНИЯ', 'Элементы благоустройства',
        'Автодорожное хозяйство', 'МАФ',
        'ИИ_ТЕКСТ_025', 'Текст_обозначение', 'Отметки', 'отметки',
        'Высотные отметки', 'Высотные отметки точек',
        'Отметки высот', 'Отметка_2',
        '61_Отметки высоты поверхности',
        'высоты текст',
    ): ('Инженерные сети', 'Прочие кабели'),
}

_misc_layer_to_labels = {
    layer: labels
    for layers, labels in _misc_layer_map.items()
    for layer in layers
}

def assign_misc_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    if _misc_re.search(row['Text']):
        labels = _misc_layer_to_labels.get(row['Layer'])
        if labels:
            return labels
        return 'Инженерные сети', 'Прочие кабели'  # fallback
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_misc_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_4.27.csv', index=False)
print(f"[4.27]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[4.27]  +3498  | размечено: 1053023 | осталось: 2016657


### 4.28. TEXT к. л. 123.45 -> ('?', '?') - в зависимости от слоя

In [114]:
_next_re = re.compile(
    r'(?i)'
    r'^\s*\\P\s+\d+[.,]\d+\s*$|'           # \P 187.47
    r'^\s*\d{7}\.\d+;\d{7}\.\d+\s*$|'      # 2186351.177;496642.094
    r'^\s*\d{1,7}\s*$|'                     # 30 / 222 / 2224050
    r'^\s*\(\d+\)\s*$|'                     # (0) / (600)
    r'^\s*с\d+\s*$'                         # с693
)



_next_layer_map = {
    # Рельеф / Отметки высот
    (
        'Height', 'Height  SOUHT',
        'Height_1',
    ): ('Рельеф', 'Отметки высот'),

    # Служебные элементы / Координаты
    (
        'Сетка', 'сетка', 'Сетка координат', 'Координатная',
        'SETK_NAD', 'SETKR', '02_Сетка',
        'координат_сетки_крестов', 'Координатные_кресты',
        'Координаты', 'ИИ_ГЕОСЕТКА_025',
    ): ('Служебные элементы', 'Координаты'),

    # Служебные элементы / Номер дерева
    (
        'Деревья', 'Подерёвка', 'Подеревка', '00p подеревка',
        '1_дерево', 'Новые_номера_деревьев',
    ): ('Служебные элементы', 'Номер дерева'),

    # Служебные элементы / Размеры дерева
    (
        'Диаметр', 'Диаметры деревьев', 'Высоты деревьев',
        'TREE DIAMETR', 'Количество стволов', 'кол-во стволов',
        'Height деревья', 'Примечания к  деревьям',
    ): ('Служебные элементы', 'Размеры дерева'),

    # Служебные элементы / Нумерация
    (
        'Номер', 'НОМЕР', 'номер', 'номер пикета 1', 'номер 3',
        'Номер3', 'топоНомер', 'Без_Номер', 'POINTS', 'Points',
        'ИМЯ ТОЧКИ', 'ТОЧКИ', 'Подписи точек', 'Имена точек',
        'ИМЕНА', 'ИМЕНА_14', 'ИМЕНА_H', 'ИМЕНА Nicon',
        'PT_ID', 'CODE', 'Коды', 'GEO_HVC',
        'Комков Номер', 'Козин Номер', 
    ): ('Служебные элементы', 'Нумерация'),


    # Служебные элементы / Прочее — всё остальное включая здания/границы
    (
        '0', '013', '00r', '00p', '08b', '594',
        'PI_NUD-Бонд', 'PI_NUDEFAULT', 'PI_NUD-Комков2',
        'DEFAULT', 'NAD_MDEFAULT', 'NAD_M0',
        '12492', '12493', '12494',
        'границы зданий', 'Здания и строения', '03_Здания и строения',
        'ЗДАНИЯ_И_С-ЗДАНИЯ', 'ЗДАНИЯ_ЖИЛЫЕ', 'ЗДАН_ОБЩЕСТ',
        'Строения', 'Текст_строения', 'GILYO_3',
        'Топонимика', 'Ответы на вопросы', 'Территория Лиры',
        'TEXT', 'STEXT3', 'name', 'ramka', 'TG_RAMKA', 'TG_DOM',
        '10_РАМКА', 'z_Условные знаки_2', 'z_Условные знаки_ПАША',
        'SETKR', 'SETK_NAD', 'Сетка',
        '45_Номера колодцев',
        'ИИ_ГЕОСЕТКА_025',
        'Height  SOUHT', 'Высоты', 'высоты текст', 'Рельеф','отметки', 
        'RELIEF_1', 'Доп.отметки', 'Survey.td2_Mark',
        'Survey.td2_point_desc_Mark', 'PROCHIE', 'MOST_26','Кабели связи', 'СВЯЗЬ-КАБЕЛИ_СВЯ',
        'CABEL', 'Кабели низкого напряжения',
        'Кабели высокого напряжения', 'ЭЛЕКТРОСЕТ-КЛ_РАСПРЕД',
        'CK_ЛЭП_высоковольтная', 'ЛЭП и освещение','канализация', 'канализация напорная',
        'Растительность', 'растительность', 'Растительность общая',
        'Растительные_объекты', 'РАСТИТЕЛЬН-ЛЕСА_И_ПОЛ',
        'РАСТИТЕЛЬН-ХАРАКТЕРИС', '13_Растительность',
        'РАСТИТЕЛЬНЫЕ_ОБЪ','Характеристики покрытий и дорог', 'SIT_LДОРОГИ', 'Тропы',
        'Откосы',
    ): ('Служебные элементы', 'Прочее'),
}

_next_layer_to_labels = {
    layer: labels
    for layers, labels in _next_layer_map.items()
    for layer in layers
}

def assign_next_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    if _next_re.search(row['Text']):
        labels = _next_layer_to_labels.get(row['Layer'])
        if labels:
            return labels
        return 'Служебные элементы', 'Прочее'  # fallback
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_next_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_4.28.csv', index=False)
print(f"[4.28]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[4.28]  +35810  | размечено: 1088833 | осталось: 1980847


### 4.28.1. TEXT к. л. 123.45 -> ('?', '?') - в зависимости от слоя + ДОБАВЛЕНЫ СКОБКИ

In [117]:
df, total_changed, total_nan = apply_labels(
    df, 'Служебные элементы', 'Прочее',
    section='4.28.1.',
    regex=r'^\s*\d+\s*\(\d+(?:\s+\w+)?\)\s*$',
    total_changed=total_changed, total_nan=total_nan,
)

[4.28.1.]  + 1671  |  
размечено: 1090504  |  
осталось: 1979176  →  NEW_REGIONS_in_progress_4.28.1..csv


### 4.29. Слой "Code деревья" -> (Озеленение, Дерево)

In [127]:
def assign_code_trees(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if row['Layer'] == 'Code деревья':
        return 'Озеленение', 'Дерево'
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_code_trees, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_4.29.csv', index=False)
print(f"[4.29]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[4.29]  +1047  | размечено: 1095077 | осталось: 1974603


### 4.30. TEXT люки / кабели / пнд / пэ итд.

In [118]:
_eng_re = re.compile(
    r'(?i)'
    # Материалы труб с размерами: ПЭ, ПВХ, ЖБ, ПНД, кер, бет
    r'п[эь]|пвх|жб|ж[./]б|пнд|кер[\.\s]|бет[\.\s]|'
    # Кабели и напряжение
    r'кабел|кв\b|\bкл\b|сип|пр\.\s*\d|н[\\]н|'
    # Статус люка/колодца
    r'недейств|засып|заварен|зацемент|заклин|завал|затоплен|залит|засыпан|замусорен|замурован|зал\.|нед\.|'
    # Отметки с префиксом
    r'н\.тр\.|в\.тр\.|люк|дно\b|реш\.|лот\.|'
    # Габариты: 2х75, 2х150ст, ГНБ, обойма
    r'\dx\d|\dх\d|гнб|обойма|'
    # Газ
    r'газ\s+в\s+зем'
)

In [120]:
mask = df['Text'].dropna().str.contains(_eng_re, regex=True)

pd.DataFrame(df[mask & df['l1'].isna()]['Layer'].unique()).to_excel('люки_слои.xlsx', index=False)

print(f"Неразмеченных: {(mask & df['l1'].isna()).sum()}")

Неразмеченных: 4071


In [121]:
_eng_exclude_re = re.compile(
    r'(?i)'
    r'ограды?\s+кам|'
    r'ограждения\s+из\s+гладк'
)

_eng_exclude_layers = {'Code', 'Code деревья', 'дороги', 'Ограждения'}

_eng_layer_map = {
    # Канализация бытовая
    (
        'канализация', 'Канализация', 'КАНАЛИЗАЦИЯ',
        'канализация напорная', 'канализация2', 'Канализации',
        '30_Канализация', 'KANAL_33', 'SEWAGE', 'TG_KANAL',
        'ВОДООТВЕДЕ-КАНАЛИЗАЦИ', 'КАНАЛИЗАЦИ-КАНАЛИЗАЦИ',
        'КАНАЛИЗАЦИ-РЕШЕТКИ_И_', 'ИИ_ТРУБ_Канализация_025',
        'Бытовая канализация', 'Напорный коллектор',
        'CK_Канализация', 'КАНАЛИЗАЦИЯ',
    ): ('Инженерные сети', 'Канализация бытовая'),

    # Ливневая канализация
    (
        'канализация ливневая', 'Ливневая канализация',
        'ВОДООТВЕДЕ-КОЛОДЦЫ_И_', 'KAN_LIVN_34', 'TG_LKANAL',
        'КАНАЛ_ЛИВНЕВ', 'WATER_DRAIN',
    ): ('Инженерные сети', 'Ливневая канализация'),

    # Водопровод
    (
        'Водопровод', 'водопровод', 'ВОДОПРОВОД',
        'Водопроводы', 'Водопровод2', 'WATER_PIPE',
        'ВОДОПРОВОД-ВОДОПРОВОД', 'ВОДОПРОВОД-КОЛОДЦЫ__К',
        '31_Водопровод', 'WODOPROV_32', 'TG_VODA',
        'CK_Водопровод', 'Водопровод_и_канализация',
        'Водопровод подземный',
    ): ('Инженерные сети', 'Водопровод'),

    # Газоснабжение
    (
        'газопровод', 'ГАЗОПРОВОД', 'Газопроводы',
        'ГАЗОПРОВОД-ГАЗОПРОВОД', 'GAZ_35',
    ): ('Инженерные сети', 'Газоснабжение'),

    # Теплосеть
    (
        'теплотрасса', 'теплотрасса надземная',
        'Теплопроводы', 'Теплосети', 'ТЕПЛОСЕТИ-ТЕПЛОСЕТИ',
        '32_Теплосеть', 'TG_TEPLO', 'GEO_TEPLO_PIPE',
        'СПЕЦЭКСПЛУАТАЦИЯ',
    ): ('Инженерные сети', 'Теплосеть'),

    # Дренаж
    (
        'Дренаж', 'TG_DRENAZ',
    ): ('Инженерные сети', 'Дренаж'),

    # Сети связи
    (
        'SVAZ', 'SVYAZ_38', 'TG_SVAZ', 'Кабели связи',
        '35_Телефон', 'СВЯЗЬ-КОЛОДЦЫ_И_',
    ): ('Инженерные сети', 'Сети связи'),

    # Электроснабжение
    (
        'CABEL', 'Электрокабели', 'ЭЛЕКТ_КАБЕЛИ',
        'электрика', 'Кабели электрические',
        'Кабели высокого напряжения', 'Кабели низкого напряжения',
        'CK_Кабели высокого напряжения', 'CK_ЛЭП_высоковольтная',
        '37_Кабели низкого напряжения', '38_Кабели высокого напряжения',
        'Объекты электропередачи', '07_Объекты электропередачи',
        'Кабельные столбики', 'ELEKT_KAB_37',
        'ЭЛЕКТРОСЕТ-КЛ_ПИТАЮЩА', 'ЭЛЕКТРОСЕТ-КЛ_РАСПРЕД',
        'ЛЭП', 'LEP', 'LEP_17', 'ЛЭП__ЛЭС_и_воздушные_каб_линии',
        'ИИ_ЛЭП_025', 'ИИ_КАБЕЛЬ_025',
        'STREET_LAMP', '51324200_Электрокабели_НН_подземные',
        
    ): ('Инженерные сети', 'Электроснабжение'),

    # Прочие трубопроводы
    (
        'ИЛОПРОВОД', 'Инженерные сооружения',
        '06_Инженерно-технические сооружения',
        'Инженерно-технические_объекты', 'PROCHIE',
        'MOST_26', '11b', 'NAD_M33',
    ): ('Инженерные сети', 'Прочие трубопроводы'),

    # Прочие кабели — fallback для неясных слоёв
    (
        '00r', '12117', '12492', '12493', '0',
        'PI_OTDEFAULT', 'PI_OT0', 'NAD_MDEFAULT',
        'границы зданий', 'Топонимика', 'Ответы на вопросы',
        '44_Крышки колодцев', 'Крышки колодцев',
        'GEO_HVC', 'GEO_BUILDINGS_TEXT', 'GEO_KLS_LINE',
        'текст', 'TEXT', 'ТЕКСТ', 'ИИ_ТЕКСТ_025',
        'Слой1', 'LAND', 'CONSTRACTION', 'MOST_26',
        'неизвестно', 'Здания и строения', 'C_Здания и строения',
        'ЗДАНИЯ_И_С-ДОПОЛНЕНИЯ', 'ЗДАНИЯ_И_С-СООРУЖЕНИЯ',
        'Автодорожное хозяйство', 'ДОРОГИ-ДОРОГИ',
        'Дороги', 'ROADS', 'TG_DOROGA', 'TG_USL_ZN',
        'Коды пикетов', 'Характеристики покрытий и дорог',
        'Растительность', 'Гидрография', 
        'Podpis', 'GEO_BUILDINGS_TEXT', 'времен_подпись',
        'Текст_обозначение', 'АС_Размеры осевые',
        'Код', 'Code_1', 'Code_2', 'Code_3', 'Code_8', 'code',
        '03122(3n)', '20_Социальные_объекты',
        '11b', 'Коды пикетов',
        'Отметка', 'Отметка_2', 'Отметки', 'отметки',
        'Отметки высот', 'Высотные отметки', 'Высоты',
        'высоты текст', 'топоОтметка', 'RELIEF_1',
        '61_Отметки высоты поверхности',
        '64_Отметка высоты крышки колодца',
        'Отметки высот_пруда 2022', 'Рельеф_имена точек',
        'Survey.td2_point_desc_Mark', 'GEO_LVC',
    ): ('Инженерные сети', 'Прочие кабели'),
}

_eng_layer_to_labels = {
    layer: labels
    for layers, labels in _eng_layer_map.items()
    for layer in layers
}

def assign_eng_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    if row['Layer'] in _eng_exclude_layers:
        return row['l1'], row['l2']
    if _eng_exclude_re.search(row['Text']):
        return row['l1'], row['l2']
    if _eng_re.search(row['Text']):
        labels = _eng_layer_to_labels.get(row['Layer'])
        if labels:
            return labels
        return 'Инженерные сети', 'Прочие кабели'  # fallback
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_eng_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_4.30.csv', index=False)
print(f"[4.30]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[4.30]  +3526  | размечено: 1094030 | осталось: 1975650


### 4.30.1. TEXT люки / кабели / пнд / пэ итд.

In [130]:
_misc2_re = re.compile(
    r'(?i)'
    r'^\s*\d+[.,]\d+\s*[лд]\.?\s*$|'       # 172,56л. / 170,59д / 150.44 л.
    r'^\s*\d+[.,]\d+-[лд]\.?\s*$|'          # 197.10-л. / 193.70-д.
    r'^\s*дно\s*\d+[.,]\d+\s*$|'            # дно132.73 / дно135,01
    r'^\s*ц\s+\d+[.,]\d+\s*$|'              # ц 133,7
    r'^\s*к\.?\s*л\.?\s*\d+[.,]\d+\s*$|'   # к. л. 141,45
    r'^\s*з\.\s*\d+\s*$|'                   # з. 139
    r'^\s*тр\.\d+[.,]\d+\s*$|'             # тр.121.39
    r'^\s*в\.кр\.\d+[.,]\d+\s*$|'          # в.кр.117.34
    r'^\s*\d{4,7}[.,]\d+\s*$'              # 1422,27
)

mask = df['Text'].dropna().str.contains(_misc2_re, regex=True)

pd.DataFrame(df[mask & df['l1'].isna()]['Layer'].unique()).to_excel('MISC2_layers.xlsx', index=False)


print(f"Неразмеченных: {(mask & df['l1'].isna()).sum()}")

Неразмеченных: 575


In [131]:
_misc2_layer_map = {
    # Рельеф / Отметки высот
    (
        'Слой1'
    ): ('Рельеф', 'Отметки высот'),

    # Канализация бытовая
    (
        'канализация', '30_Канализация', 'КАНАЛИЗАЦИ-РЕШЕТКИ_И_',
    ): ('Инженерные сети', 'Канализация бытовая'),

    # Ливневая канализация
    (
        'канализация ливневая',
    ): ('Инженерные сети', 'Ливневая канализация'),

    # Водопровод
    (
        'Водопровод', 'Водопроводы', '31_Водопровод',
    ): ('Инженерные сети', 'Водопровод'),

    # Теплосеть
    (
        'теплотрасса', 'Теплосети',
    ): ('Инженерные сети', 'Теплосеть'),

    # Дренаж
    (
        'Дренаж',
    ): ('Инженерные сети', 'Дренаж'),

    # Сети связи
    (
        'SVAZ', 'Кабели связи', 'СВЯЗЬ-КОЛОДЦЫ_И_',
    ): ('Инженерные сети', 'Сети связи'),

    # Электроснабжение
    (
        'Кабели низкого напряжения',
    ): ('Инженерные сети', 'Электроснабжение'),

    # Прочие кабели
    (
        '44_Крышки колодцев', 'NAD_MDEFAULT', 'TEXT',
        'ЗДАНИЯ_И_С-ЗДАНИЯ', 'ГАЗОПРОВОД-КОЛОДЦЫ_И_',
        'Отметка', 'Отметка_2', 'Отметка_к', '1Отметка',
        'отметки', 'ВЫСОТЫ', 'Отметки высот',
        '60_Отметки высот на зданиях и сооружениях',
    ): ('Инженерные сети', 'Прочие кабели'),
}

_misc2_layer_to_labels = {
    layer: labels
    for layers, labels in _misc2_layer_map.items()
    for layer in layers
}

def assign_misc2_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    if _misc2_re.search(row['Text']):
        labels = _misc2_layer_to_labels.get(row['Layer'])
        if labels:
            return labels
        return 'Инженерные сети', 'Прочие кабели'  # fallback
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_misc2_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_4.30.1.csv', index=False)
print(f"[4.30.1]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[4.30.1]  +575  | размечено: 1095652 | осталось: 1974028


In [124]:
print(df['l1'].value_counts())
print()
print(df['l2'].value_counts())

Рельеф                      710755
Служебные элементы          341142
Инженерные сети              20536
Твердые покрытия             14192
Озеленение                    2560
Здания и сооружения           1994
Водопроницаемые покрытия      1744
МАФ                           1107
Name: l1, dtype: int64

Отметки высот              710755
Номер дерева               176545
Прочее                     114358
Нумерация                   31331
Асфальт                      9490
Размеры дерева               7533
Прочие кабели                7234
Координаты                   6200
Плитка                       4483
Размеры                      4200
Канализация бытовая          3511
Водопровод                   2785
Дерево                       2362
Электроснабжение             2067
Ливневая канализация         1930
Неизвестно                   1829
Грунт                        1493
Кадастр                       975
Сети связи                    967
Теплосеть                     893
Газоснабжение   

In [ ]:
_eng_exclude_layers = {'Code', 'Code деревья', 'дороги', 'Ограждения'}

In [125]:
mask = df['Layer'].isin(_eng_exclude_layers)

df[mask & df['l1'].isna()][['Layer', 'Text']].drop_duplicates().to_excel('EXCLUDE_LAYERS_text.xlsx', index=False)

### 4.31. ТЕКСТ паттерн 1111-ААА-111

In [134]:
df, total_changed, total_nan = apply_labels(
    df, 'Служебные элементы', 'Координаты',
    section='4.31',
    regex=r'^\s*\d{3,4}-[A-Za-z]{2,4}-\d{2,4}\s*$',
    total_changed=total_changed, total_nan=total_nan,
)

[4.31]  + 1729  |  
размечено: 1097381  |  
осталось: 1972299  →  NEW_REGIONS_in_progress_4.31.csv


### 4.31.1. ТЕКСТ паттерн 2186377.187;496579.211

In [141]:
df, total_changed, total_nan = apply_labels(
    df, 'Служебные элементы', 'Координаты',
    section='4.31.1',
    regex=r'^\s*\d{6,7}\.\d+;\d{6,7}\.\d+\s*$',
    total_changed=total_changed, total_nan=total_nan,
)

[4.31.1]  +  900  |  
размечено: 1098281  |  
осталось: 1971399  →  NEW_REGIONS_in_progress_4.31.1.csv


### 4.32. ТП / КТП / ЦРП - здания и сооружения

In [145]:
df, total_changed, total_nan = apply_labels(
    df, 'Инженерные сети', 'Электроснабжение',
    section='4.32',
    regex=r'(?i)(?:к\s+)?(?:ктп|ртп|црп|тп)[\s\-]?\d+|тп\d+-тп\d+|ао\s+.{0,30}тп-\d+',
    total_changed=total_changed, total_nan=total_nan,
)

[4.32]  +  426  |  
размечено: 1098707  |  
осталось: 1970973  →  NEW_REGIONS_in_progress_4.32.csv


### 4.33. доп. паттерны -> кадастр

In [149]:
df, total_changed, total_nan = apply_labels(
    df, 'Служебные элементы', 'Кадастр',
    section='4.33',
    regex=r'(?i)^\s*кн\s+\d{2}:\d{2}:\d{7}:\d+\s*$|^\s*\d{2}:\d{2}-\d+\.\d+\s*$',
    total_changed=total_changed, total_nan=total_nan,
)

[4.33]  +  160  |  
размечено: 1098867  |  
осталось: 1970813  →  NEW_REGIONS_in_progress_4.33.csv


### 4.34. h=0.6 -> инж. сети

In [150]:
_h_re = re.compile(r'(?i)^\s*h=\d+[.,]\d+\s*$')

mask = df['Text'].dropna().str.contains(_h_re, regex=True)

df[mask & df['l1'].isna()][['Layer', 'Text']].drop_duplicates().to_excel('H_layer_text.xlsx', index=False)

print(f"Неразмеченных: {(mask & df['l1'].isna()).sum()}")

Неразмеченных: 58


In [151]:
_h_layer_map = {
    'канализация': ('Инженерные сети', 'Канализация бытовая'),
    'Водослив':    ('Инженерные сети', 'Ливневая канализация'),
}

def assign_h_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    if _h_re.search(row['Text']):
        labels = _h_layer_map.get(row['Layer'])
        if labels:
            return labels
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_h_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_4.35.csv', index=False)
print(f"[4.35]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[4.35]  +58  | размечено: 1098925 | осталось: 1970755


### 4.35. имя деревьев - (Озеленение, Дерево)

In [153]:
_tree_re = re.compile(
    r'(?i)'
    r'сухосто|поросль|вырублен|вырубленн|молод\.|повален|накл|облом|'  # статус кириллица
    r'povaleno|naklon|oblom|zaval|odinsuh|smor|leza|povrezd|povr|poval|plohoe|naklom|stelyash|'  # статус латиница
    r'sux|cyx|suh|zav|'  # суффиксы состояния
    r'берёз|бер\d|бер\s|ber\d|ber\s|'  # берёза
    r'ель\s|ел\s|el\d|el\s|'            # ель
    r'сосн|sos[n]?\d|sos[n]?\s|'        # сосна
    r'дуб\d|дуб\s|dub\d|'              # дуб
    r'ол\d+|ol\d|ol\s|'                # ольха
    r'ос\d+|os\d|osin|'                # осина
    r'кл\d+|кл\s|kl\d|klen|'          # клён
    r'лп\s|lip\d|lip\s|lipa|'          # липа
    r'ив\d|iv\d|ива\d|'                # ива
    r'яс\d+|yas\d|'                    # ясень
    r'лист\d|list\d|'                  # лиственница
    r'черем|черём|'                    # черёмуха
    r'яблон|вишн|облепих|хвойн|фруктов|'  # прочие породы
    r'kyst|rel\b|bub\d|саженц'         # кустарник, саженцы
)

_tree_exclude_strict = {
    'СНтТ Машиностроитель 1',
    'СНТ Машиностроитель 1',
    'Узел Связи от насосной станции',
    'Водозаборный узел N3',
    'shmel113@mail.ru',
    'SOL1.1',
    'SOL2.1',
}

def assign_tree_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    if row['Text'].strip() in _tree_exclude_strict:
        return row['l1'], row['l2']
    if _tree_re.search(row['Text']):
        return 'Озеленение', 'Дерево'
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_tree_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_4.35.csv', index=False)
print(f"[4.35]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[4.35]  +882  | размечено: 1099807 | осталось: 1969873


### 4.36. vt 1 - (Служебные элементы, Прочее)

In [157]:
df, total_changed, total_nan = apply_labels(
    df, 'Служебные элементы', 'Прочее',
    section='4.36',
    regex=r'^\s*vt\s*\d+\s*$',
    total_changed=total_changed, total_nan=total_nan,
)

[4.36]  +  290  |  
размечено: 1100097  |  
осталось: 1969583  →  NEW_REGIONS_in_progress_4.36.csv


### 4.37. 1+2+3+4+5 - (Служебные элементы, Прочее)

In [159]:
_plus_re = re.compile(r'^\s*\d+(?:\+\d+)+\s*$')

mask = df['Text'].str.contains(_plus_re, regex=True, na=False)

df[mask][['Layer', 'Text', 'l1', 'l2']].drop_duplicates().to_excel('PLUS_layer_text.xlsx', index=False)

print(f"Всего: {mask.sum()}, неразмеченных: {(mask & df['l1'].isna()).sum()}")

Всего: 642, неразмеченных: 642


In [160]:
df, total_changed, total_nan = apply_labels(
    df, 'Служебные элементы', 'Размеры дерева',
    section='4.38',
    regex=r'^\s*\d+(?:\+\d+)+\s*$',
    total_changed=total_changed, total_nan=total_nan,
)

[4.38]  +  642  |  
размечено: 1100739  |  
осталось: 1968941  →  NEW_REGIONS_in_progress_4.38.csv


### 4.38. ФИО / Ген.директор и т.д. - (Служебные элементы, Штамп / Условные обозначения)

In [162]:
_usl_re = re.compile(
    r'(?i)'
    r'^\s*[A-Za-z]\d+\.\d+\s*$|'                        # A2.1 A1.3
    r'^\s*уч\.\d+[а-яё]?\s*$|'                          # уч.11 уч.12а
    r'сплошные горизонтали|'                             # Сплошные горизонтали...
    r'система\s+(высот|координат)|'                      # Система высот / координат
    r'мск-\d+'                                           # МСК-50
)

_stamp_re = re.compile(
    r'(?i)'
    r'стадия\s+лист|'
    r'м\s*1[:]\s*\d+|'                                   # М 1:500
    r'ооо\s*[«"]|'                                       # ООО «...»
    r'ип\s+[А-ЯЁ][а-яё]+|'                              # ИП Киркин
    r'ген\.?\s*директор|директор|нач\.?\s*отдела|'
    r'должность|исполнитель|заказчик|инженер|инж\.|'
    r'топографическая\s+съемка|'
    r'\+7\(\d{3}\)\d{3}-\d{2}-\d{2}|'                  # телефон
    r'@|'                                                # email
    r'[А-ЯЁ]\.[А-ЯЁ]\.[А-ЯЁ][а-яё]+'                  # инициалы + фамилия
)

_exclude_Lenin = re.compile(
    r'(?i)ленин'
)

for name, pattern in [('USL', _usl_re), ('STAMP', _stamp_re)]:
    mask = df['Text'].str.contains(pattern, regex=True, na=False)
    mask_excl = df['Text'].str.contains(_exclude_Lenin, regex=True, na=False)
    df[mask & ~mask_excl & df['l1'].isna()][['Layer', 'Text']].drop_duplicates().to_excel(f'{name}_layer_text.xlsx', index=False)
    print(f"{name}: неразмеченных {(mask & ~mask_excl & df['l1'].isna()).sum()}")

C:\Users\artem\AppData\Local\Temp\ipykernel_22864\2788778390.py:29: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = df['Text'].str.contains(pattern, regex=True, na=False)


USL: неразмеченных 145
STAMP: неразмеченных 305


In [164]:
_stamp_exclude_strict = {
    'кип каб', ' кип каб', ' кип охр каб', 'кип охр каб',
    '(Хип хоп)',
    'ООО "РСК"', 'ООО "Водоканал"', 'ООО "ТЭК"', 'ООО "ЧАЙКА"',
    'ООО"Бэлс-Энергосервис"', 'ООО"ГрадИнвест"', 'ООО "РЭСС"',
    'ООО "НВП"ДИАМЕТ"', 'ООО"НТК"', 'ООО"БКС"', 'ООО"Тепловодосервис"',
    ' кип газ', 'кип газ',
}

_stamp_exclude_re = re.compile(
    r'(?i)'
    r'ленин|'
    r'\\f@Arial Unicode|'        # AutoCAD-форматирование
    r'ооо\s*["\'](?:рск|водоканал|тэк|чайка|рэсс|нвп|нтк|бкс)'
)

def assign_usl_stamp_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']

    text_stripped = row['Text'].strip()

    if text_stripped in _stamp_exclude_strict:
        return row['l1'], row['l2']
    if _stamp_exclude_re.search(row['Text']):
        return row['l1'], row['l2']

    if _usl_re.search(row['Text']):
        return 'Служебные элементы', 'Условные обозначения'
    if _stamp_re.search(row['Text']):
        return 'Служебные элементы', 'Штамп'

    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_usl_stamp_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_4.38.csv', index=False)
print(f"[4.38]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[4.38]  +399  | размечено: 1101138 | осталось: 1968542


### 4.39. 650а / 120б - (Служебные элементы, Номер дерева)

In [166]:
_num_letter_re = re.compile(r'^\s*\d+[а-яё]\s*$', re.IGNORECASE)

mask = df['Text'].str.contains(_num_letter_re, regex=True, na=False)

df[mask & df['l1'].isna()][['Layer', 'Text']].drop_duplicates().to_excel('NUM_LETTER_layer_text.xlsx', index=False)

print(f"Всего: {mask.sum()}, неразмеченных: {(mask & df['l1'].isna()).sum()}")

Всего: 392, неразмеченных: 66


In [167]:
def assign_num_letter_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    if _num_letter_re.match(row['Text'].strip()) and row['Layer'] == 'Номер дерева':
        return 'Служебные элементы', 'Номер дерева'
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_num_letter_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_4.40.csv', index=False)
print(f"[4.40]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[4.40]  +62  | размечено: 1101200 | осталось: 1968480


### 4.41. (5 84) - (Служебные элементы, Прочее)

In [168]:
_paren_re = re.compile(r'^\s*\(\d+\s+\d+\)\s*$|^\s*\(кус\)\s*$')

mask = df['Text'].str.contains(_paren_re, regex=True, na=False)

df[mask & df['l1'].isna()][['Layer', 'Text']].drop_duplicates().to_excel('PAREN_layer_text.xlsx', index=False)

print(f"Всего: {mask.sum()}, неразмеченных: {(mask & df['l1'].isna()).sum()}")

Всего: 78, неразмеченных: 78


In [170]:
df, total_changed, total_nan = apply_labels(
    df, 'Служебные элементы', 'Прочее',
    section='4.41',
    regex=r'^\s*\(\d+\s+\d+\)\s*$|^\s*\(кус\)\s*$',
    total_changed=total_changed, total_nan=total_nan,
)

[4.41]  +   78  |  
размечено: 1101278  |  
осталось: 1968402  →  NEW_REGIONS_in_progress_4.41.csv


### 4.40. (5 84) - (Служебные элементы, Прочее)

In [173]:
df, total_changed, total_nan = apply_labels(
    df, 'Служебные элементы', 'Прочее',
    section='4.40',
    regex=r'^\s*\d+\(\d+-\d+(?:\s+end)?\)\s*$',
    total_changed=total_changed, total_nan=total_nan,
)

[4.40]  +  139  |  
размечено: 1101417  |  
осталось: 1968263  →  NEW_REGIONS_in_progress_4.40.csv


### 4.42.  - (Служебные элементы, Прочее)

топографические полевые коды (field codes) — сокращения которые геодезисты вводят при съёмке. Расшифровка очевидна:
CNT = контур, STR = строение, OGRS/OGRD = ограждение, BRL/BRR = бровка, RST = рельеф, TROPA = тропа, GAZON = газон, ASPH = асфальт, PLITKA = плитка, DER = дерево, KAB = кабель, LEP = ЛЭП, KOLODEC = колодец, TURNIK = турник, KASHELI = качели, LAVKA = лавка, YAMA = яма, BESEDKA = беседка и т.д.

In [177]:
_field_code_re = re.compile(
    r'(?i)^\s*(?:'
    r'cnt|str|ogrs|ogrd|ogrm|ogrb|ogr|rst|brl|brr|brd|brd12|'
    r'tropa|tropa1|gazon|asph|plitka|plti|cement|bruwatka|'
    r'der|kab|lep|luk|nvs|nvs12|stb|strl|'
    r'kolodec|yama|besedka|teplica|sarai|fund|'
    r'turnik|kasheli|lavka|brus|pirs|blok|'
    r'boloto|rodnik|holm|pesok|lug|'
    r'pk|pd|pd12|vdm|vdm1|rdm|rd4|'
    r'fnrb|fnrm|rstd|rstm|razdel|rzdel|razel|'
    r'kr|kr1|kr\s|ob\s|e\s|s\s|c\s|cr\s|dn\s|mn\s|'
    r'porosl|plod|mol|nakl|verh|sredn|ugol|'
    r'grp|lst|lst1|nws|rstd|stl'
    r')[\s\w.+=/\-]*$'
)

mask = df['Text'].str.contains(_field_code_re, regex=True, na=False)

df[mask & df['l1'].isna()][['Source_File', 'Layer', 'Text']].drop_duplicates().to_excel('FIELD_CODE_layers.xlsx', index=False)

print(f"Всего: {mask.sum()}, неразмеченных: {(mask & df['l1'].isna()).sum()}")

Всего: 1590, неразмеченных: 1458


In [179]:
df, total_changed, total_nan = apply_labels(
    df, 'Служебные элементы', 'Прочее',
    section='4.42',
    regex=(
        r'(?i)^\s*(?:'
        r'cnt|str|ogrs|ogrd|ogrm|ogrb|ogr|rst|brl|brr|brd|brd12|'
        r'tropa|tropa1|gazon|asph|plitka|plti|cement|bruwatka|'
        r'der|kab|lep|luk|nvs|nvs12|stb|strl|'
        r'kolodec|yama|besedka|teplica|sarai|fund|'
        r'turnik|kasheli|lavka|brus|pirs|blok|'
        r'boloto|rodnik|holm|pesok|lug|'
        r'pk|pd|pd12|vdm|vdm1|rdm|rd4|'
        r'fnrb|fnrm|rstd|rstm|razdel|rzdel|razel|'
        r'kr|kr1|kr\s|ob\s|e\s|s\s|c\s|cr\s|dn\s|mn\s|'
        r'porosl|plod|mol|nakl|verh|sredn|ugol|'
        r'grp|lst|lst1|nws|rstd|stl'
        r')[\s\w.+=/\-]*$'
    ),
    total_changed=total_changed, total_nan=total_nan,
)

[4.42]  + 1458  |  
размечено: 1102875  |  
осталось: 1966805  →  NEW_REGIONS_in_progress_4.42.csv


### 4.43.  - (Служебные элементы, Прочее)

In [183]:
_obj_re = re.compile(
    r'(?i)^\s*(?:'
    r'livnevk|livne|trotu|naves|veldor|svetofor|'
    r'ostanovka|reklama|opora|shlgbaum|palatka|'
    r'dom|stolb'
    r')\d*\s*$'
)

mask = df['Text'].str.contains(_obj_re, regex=True, na=False)

pd.DataFrame(df[mask & df['l1'].isna()]['Layer'].unique(), columns=['Layer']).to_excel('OBJ_layers.xlsx', index=False)
print(f"Всего: {mask.sum()}, неразмеченных: {(mask & df['l1'].isna()).sum()}")

Всего: 138, неразмеченных: 102


In [184]:
_obj_prefix_map = {
    'livnevk': ('Инженерные сети', 'Ливневая канализация'),
    'livne':   ('Инженерные сети', 'Ливневая канализация'),
    'trotu':   ('Твердые покрытия', 'Асфальт'),
    'naves':   ('Здания и сооружения', 'Неизвестно'),
    'veldor':  ('Твердые покрытия', 'Резиновая крошка'),
    'svetofor':('Инженерные сети', 'Инженерная инфраструктура'),
    'ostanovka':('Здания и сооружения', 'Неизвестно'),
    'reklama': ('МАФ', 'Информационная конструкция'),
    'opora':   ('Инженерные сети', 'Столбы и опоры'),
    'shlgbaum':('МАФ', 'Шлагбаум'),
    'palatka': ('МАФ', 'Павильон'),
    'dom':     ('Здания и сооружения', 'Неизвестно'),
    'stolb':   ('Инженерные сети', 'Столбы и опоры'),
}

_obj_prefix_order = sorted(_obj_prefix_map, key=len, reverse=True)

def assign_obj_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    text = row['Text'].strip().lower()
    for prefix in _obj_prefix_order:
        if re.match(rf'^{prefix}\d*$', text):
            return _obj_prefix_map[prefix]
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_obj_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_4.43.csv', index=False)
print(f"[4.43]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[4.43]  +102  | размечено: 1102977 | осталось: 1966703


In [185]:
pd.DataFrame(df[df['l1'].isna()]['Text'].unique()).to_excel('TEXT_left.xlsx')
pd.DataFrame(df[df['l1'].isna()]['Layer'].unique()).to_excel('LAYERS_left.xlsx')

## 5.0 Left labels 

In [187]:
new_mapping = {
    'Стадия': ('Служебные элементы', 'Штамп'),
    'Лист': ('Служебные элементы', 'Штамп'),
    'Листов': ('Служебные элементы', 'Штамп'),
    'ИЗ': ('Служебные элементы', 'Штамп'),
    'г.Красногорск': ('Служебные элементы', 'Штамп'),
    '087-19-ИЗ': ('Служебные элементы', 'Штамп'),
    'Владиславлев П.Н.': ('Служебные элементы', 'Штамп'),
    'Ухина М.В.': ('Служебные элементы', 'Штамп'),
    'Шамарина А.А.': ('Служебные элементы', 'Штамп'),
    '06-19г.': ('Служебные элементы', 'Штамп'),
    'Иошкин В.А.': ('Служебные элементы', 'Штамп'),
    'низк.,выс. напряжения': ('Инженерные сети', 'Электроснабжение'),
    'Условные обозначения': ('Служебные элементы', 'Условные обозначения'),
    'Теплотрасса': ('Инженерные сети', 'Теплосеть'),
    'Канализация': ('Инженерные сети', 'Канализация бытовая'),
    'К': ('Инженерные сети', 'Канализация бытовая'),
    'В': ('Инженерные сети', 'Водопровод'),
    'Г': ('Инженерные сети', 'Газоснабжение'),
    'Водопровод': ('Инженерные сети', 'Водопровод'),
    'Газопровод': ('Инженерные сети', 'Газоснабжение'),
    'Т': ('Инженерные сети', 'Теплосеть'),
    'Сведения, полученные из КПТ': ('Служебные элементы', 'Кадастр'),
    'Границы земельных участков': ('Служебные элементы', 'Границы'),
    'Подземные коммуникации': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Зона с особыми условиями использования территорий': ('Служебные элементы', 'Границы'),
    'въезд в г.о. Серпухов со стороны г. Москва': ('Служебные элементы', 'Прочее'),
    '(на границе с г.о. Чехов).': ('Служебные элементы', 'Прочее'),
    'дор. указ.': ('МАФ', 'Дорожные знаки'),
    'г. Чехов': ('Служебные элементы', 'Прочее'),
    'г. Серпухов': ('Служебные элементы', 'Прочее'),
    'ОАО"Ростелеком"': ('Инженерные сети', 'Сети связи'),
    'АО "СМУ-5"': ('Служебные элементы', 'Штамп'),
    'АО "Риал Ком"': ('Инженерные сети', 'Сети связи'),
    'Западный ТЦТЭТ': ('Инженерные сети', 'Сети связи'),
    'б/пр.': ('Инженерные сети', 'Прочие кабели'),
    'инф.': ('МАФ', 'Информационная конструкция'),
    'Д': ('Инженерные сети', 'Дренаж'),
    'Н': ('Инженерные сети', 'Наружное освещение'),
    '"Полет"': ('МАФ', 'Игровое оборудование'),
    '"Мини Джет"': ('МАФ', 'Игровое оборудование'),
    'Тренажеры': ('МАФ', 'Спортивное оборудование'),
    '"Каное Ривер"': ('МАФ', 'Игровое оборудование'),
    '"Автодром"': ('МАФ', 'Игровое оборудование'),
    '"Бамперные': ('МАФ', 'Игровое оборудование'),
    '"Ракушки"': ('МАФ', 'Игровое оборудование'),
    'мусор': ('МАФ', 'Прочее'),
    'МН': ('Здания и сооружения', 'Неизвестно'),
    'Сцена': ('Здания и сооружения', 'Неизвестно'),
    'флагштоки': ('МАФ', 'Флагштоки'),
    'КН': ('Здания и сооружения', 'Неизвестно'),
    'площ.': ('Служебные элементы', 'Прочее'),
    'стб.': ('Инженерные сети', 'Столбы и опоры'),
    'Теннисный корт': ('Твердые покрытия', 'Резиновая крошка'),
    'ворота': ('МАФ', 'Ворота'),
    'для уборочной техники': ('МАФ', 'Прочее'),
    'Тартан': ('Твердые покрытия', 'Резиновая крошка'),
    '"Железная дорога"': ('МАФ', 'Игровое оборудование'),
    'тартан': ('Твердые покрытия', 'Резиновая крошка'),
    'Московская область, г.о. Серпухов, между': ('Служебные элементы', 'Штамп'),
    '026-19-ИЗ': ('Служебные элементы', 'Штамп'),
    'Ревина М.В.': ('Служебные элементы', 'Штамп'),
    'Бондаренко И.В.': ('Служебные элементы', 'Штамп'),
    '03-19г.': ('Служебные элементы', 'Штамп'),
    'Резиновая фигура': ('МАФ', 'Игровое оборудование'),
    '"Формула 1"': ('МАФ', 'Игровое оборудование'),
    'МУП"Водоканал-Сервис"': ('Инженерные сети', 'Водопровод'),
    'ориент.': ('Служебные элементы', 'Прочее'),
    'н/н': ('Инженерные сети', 'Инженерная инфраструктура'),
    '1отв.': ('Инженерные сети', 'Прочие кабели'),
    'Коломенский филиал АО"Мособлэнерго"': ('Инженерные сети', 'Электроснабжение'),
    'Серпуховское ПО': ('Инженерные сети', 'Электроснабжение'),
    'МБУ"Комбинат благоустройства"': ('Служебные элементы', 'Прочее'),
    'Серпуховская РЭС': ('Инженерные сети', 'Электроснабжение'),
    'АО"Мособлгаз"': ('Инженерные сети', 'Газоснабжение'),
    'машинки"': ('МАФ', 'Игровое оборудование'),
    'цепочка"': ('МАФ', 'Игровое оборудование'),
    'MK54.1': ('Инженерные сети', 'Прочие кабели'),
    'в зем.': ('Инженерные сети', 'Инженерная инфраструктура'),
    '6пр.': ('Инженерные сети', 'Прочие кабели'),
    'разр.': ('Твердые покрытия', 'Асфальт'),
    'Т1': ('Инженерные сети', 'Теплосеть'),
    'газ': ('Инженерные сети', 'Газоснабжение'),
    'дер.': ('Водопроницаемые покрытия', 'Деревянный настил'),
    'ковер': ('Инженерные сети', 'Газоснабжение'),
    'иск.': ('Неизвестно', 'Неизвестно'),
    'яма': ('Рельеф', 'Откос'),
    'КУ': ('Инженерные сети', 'Сети связи'),
    'ГРП': ('Инженерные сети', 'Газоснабжение'),
    'л': ('Неизвестно', 'Неизвестно'),
    'чаща': ('Озеленение', 'Кустарник'),
    'сцена': ('Здания и сооружения', 'Неизвестно'),
    'КУАЗ': ('Инженерные сети', 'Электроснабжение'),
    'терраса': ('Здания и сооружения', 'Неизвестно'),
    'б.': ('Здания и сооружения', 'Неизвестно'),
    'Полка с книгами': ('МАФ', 'Уличная мебель'),
    'летняя кухня': ('Здания и сооружения', 'Неизвестно'),
    'стр.': ('Здания и сооружения', 'Неизвестно'),
    'газ из земли': ('Инженерные сети', 'Газоснабжение'),
}

def assign_mapping_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    labels = new_mapping.get(row['Text'].strip())
    if labels:
        return labels
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_mapping_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_4.45.csv', index=False)
print(f"[4.45]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[4.45]  +7837  | размечено: 1110814 | осталось: 1958866


In [190]:
new_mapping = {
    'Канатная дорога': ('МАФ', 'Игровое оборудование'),
    'флагшток': ('МАФ', 'Флагшток'),
    'Дерево для замков': ('Служебные элементы', 'Условные обозначения'),
    'Московская область, г.о. Серпухов, между ': ('Служебные элементы', 'Штамп'),
    '027-19-ИЗ': ('Служебные элементы', 'Штамп'),
    'Ханова Е.И.': ('Служебные элементы', 'Штамп'),
    'Комков А.М.': ('Служебные элементы', 'Штамп'),
    '04-19г.': ('Служебные элементы', 'Штамп'),
    'Линия сводки с листом 1': ('Служебные элементы', 'Условные обозначения'),
    'Линия сводки с листом 2': ('Служебные элементы', 'Условные обозначения'),
    'Схема расположения листов:': ('Служебные элементы', 'Условные обозначения'),
    'пруд': ('Водные объекты', 'Неизвестно'),
    'закрыт': ('Инженерные сети', 'Инженерная инфраструктура'),
    'вент.': ('Инженерные сети', 'Инженерная инфраструктура'),
    '148.92гидр.': ('Инженерные сети', 'Водопровод'),
    '151.04гидр.': ('Инженерные сети', 'Водопровод'),
    'н/д': ('Инженерные сети', 'Инженерная инфраструктура'),
    'МУП "Водоканал-Сервис"': ('Инженерные сети', 'Водопровод'),
    '"Подольскмежрайгаз"': ('Инженерные сети', 'Газоснабжение'),
    'ПАО"МОЭСК" Южные эл.сети': ('Инженерные сети', 'Электроснабжение'),
    'Коломенский ф-л АО"Мособлэнерго"': ('Инженерные сети', 'Электроснабжение'),
    'АСБ200': ('Инженерные сети', 'Прочие трубопроводы'),
    'ПАО"Ростелеком"': ('Инженерные сети', 'Сети связи'),
    'Серпуховской МЦТЭТ': ('Инженерные сети', 'Сети связи'),
    'Примечание:': ('Служебные элементы', 'Прочее'),
    'Промежуточный материал. В плане возможны изменения.': ('Служебные элементы', 'Прочее'),
    'Территория автостоянки': ('Твердые покрытия', 'Асфальт'),
    'КТ': ('Инженерные сети', 'Сети связи'),
    'Московская область, г.Ступино': ('Служебные элементы', 'Штамп'),
    '034-18-ИЗ': ('Служебные элементы', 'Штамп'),
    'ООО НПП «Пассат»': ('Служебные элементы', 'Штамп'),
    '02-18г.': ('Служебные элементы', 'Штамп'),
    'выполнялась при высоте снежного покрова +1м.': ('Служебные элементы', 'Прочее'),
    'инвентаризации:': ('Служебные элементы', 'Прочее'),
    'Вазы для цветов': ('МАФ', 'Вазоны'),
    'Тарасенков Ю.В.': ('Служебные элементы', 'Штамп'),
    'МУП "ПТО ЖКХ" "Тепловые сети"': ('Инженерные сети', 'Теплосеть'),
    'МУП "ПТО ЖКХ" ': ('Инженерные сети', 'Инженерная инфраструктура'),
    '08-17г.': ('Служебные элементы', 'Штамп'),
    'Стелла': ('МАФ', 'Памятники'),
    'арка': ('МАФ', 'Прочее'),
    'рекл.': ('МАФ', 'Информационная конструкция'),
    'Московская область, г.Ступино,': ('Служебные элементы', 'Штамп'),
    '035-18-ИЗ': ('Служебные элементы', 'Штамп'),
    'ООО НПП "Пассат"': ('Служебные элементы', 'Штамп'),
    'Амерханов Р.Р.': ('Служебные элементы', 'Штамп'),
    'проспект Победы': ('Служебные элементы', 'Прочее'),
    'В.И.Ленин': ('МАФ', 'Памятники'),
    'свеча': ('Неизвестно', 'Неизвестно'),
    'в зд.': ('Инженерные сети', 'Инженерная инфраструктура'),
    'газ по стене': ('Инженерные сети', 'Газоснабжение'),
    'кафе': ('Здания и сооружения', 'Неизвестно'),
    'пож.лестн.': ('Здания и сооружения', 'Лестницы и пандусы'),
    'ков.': ('МАФ', 'Ограждение'),
    'фунд.': ('Здания и сооружения', 'Неизвестно'),
    'Щ': ('Водопроницаемые покрытия', 'Щебень'),
    'Московская область, Рузский район, рабочий': ('Служебные элементы', 'Штамп'),
    '039-19-ИЗ': ('Служебные элементы', 'Штамп'),
    'Безрук П.Г.': ('Служебные элементы', 'Штамп'),
    'Славы, сквера с прудом, территория вокруг': ('Служебные элементы', 'Штамп'),
    '22/с1': ('Здания и сооружения', 'Неизвестно'),
    'г. Руза': ('Служебные элементы', 'Штамп'),
    'Суворов К.М.': ('Служебные элементы', 'Штамп'),
    '2отв.': ('Инженерные сети', 'Прочие кабели'),
    '9отв.': ('Инженерные сети', 'Прочие кабели'),
    'ОАО "Ростелеком"': ('Инженерные сети', 'Сети связи'),
    '8отв.': ('Инженерные сети', 'Прочие кабели'),
    'АО"Жилсервис"': ('Служебные элементы', 'Прочее'),
    'Зап.эл.сети Филиал': ('Инженерные сети', 'Электроснабжение'),
    'ПАО"МОЭСК" Рузская РЭС': ('Инженерные сети', 'Электроснабжение'),
    'Одинцовский ф-л': ('Инженерные сети', 'Электроснабжение'),
    'АО"Мособлэнерго"': ('Инженерные сети', 'Электроснабжение'),
    'Филиал АО"Мособлгаз"': ('Инженерные сети', 'Газоснабжение'),
    '"Одинцовомежрайгаз"': ('Инженерные сети', 'Газоснабжение'),
    'Преображенская церковь': ('Здания и сооружения', 'Неизвестно'),
    'Заболоченная местность': ('Водные объекты', 'Неизвестно'),
    'Конюшня': ('Здания и сооружения', 'Неизвестно'),
    'Гр': ('Водопроницаемые покрытия', 'Грунт'),
    'Погреб': ('Здания и сооружения', 'Неизвестно'),
    'Вентиляционные трубы': ('Инженерные сети', 'Инженерная инфраструктура'),
    'ТБО': ('МАФ', 'Прочее'),
    'Остановка': ('МАФ', 'Павильоны'),
    'камера': ('Инженерные сети', 'Инженерная инфраструктура'),
    'дор.': ('МАФ', 'Дорожные знаки'),
    'указ.': ('МАФ', 'Дорожные знаки'),
    '4пр.': ('Инженерные сети', 'Прочие кабели'),
    '1пр.': ('Инженерные сети', 'Прочие кабели'),
    'реконструкция': ('Служебные элементы', 'Прочее'),
    '2пр.': ('Инженерные сети', 'Прочие кабели'),
    '7пр.': ('Инженерные сети', 'Прочие кабели'),
    '5пр.': ('Инженерные сети', 'Прочие кабели'),
    'почта': ('Здания и сооружения', 'Неизвестно'),
    'часовня': ('Здания и сооружения', 'Неизвестно'),
}


def assign_mapping_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    labels = new_mapping.get(row['Text'].strip())
    if labels:
        return labels
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_mapping_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_5.02.csv', index=False)
print(f"[5.01]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[5.01]  +3083  | размечено: 1113897 | осталось: 1955783


In [191]:
new_mapping = {
    'БТИ': ('Служебные элементы', 'Прочее'),
    'изрыто': ('Водопроницаемые покрытия', 'Грунт'),
    'Базар': ('Здания и сооружения', 'Неизвестно'),
    '3пр.': ('Инженерные сети', 'Прочие кабели'),
    '10пр.': ('Инженерные сети', 'Прочие кабели'),
    '9пр.': ('Инженерные сети', 'Прочие кабели'),
    'доска почета': ('МАФ', 'Информационная конструкция'),
    'Одинцовомежрайгаз': ('Инженерные сети', 'Газоснабжение'),
    'АО "Мособлгаз"': ('Инженерные сети', 'Газоснабжение'),
    'Рузская РЭС': ('Инженерные сети', 'Электроснабжение'),
    'АО "Жилсервис"': ('Служебные элементы', 'Прочее'),
    '6 отв.': ('Инженерные сети', 'Прочие кабели'),
    'ПАО "Ростелеком"': ('Инженерные сети', 'Сети связи'),
    '2ВОЛС АО "Фортекс"': ('Инженерные сети', 'Сети связи'),
    'Одинцовский межрайгаз': ('Инженерные сети', 'Газоснабжение'),
    '3отв.': ('Инженерные сети', 'Прочие кабели'),
    'от РП-57': ('Инженерные сети', 'Электроснабжение'),
    'Рузская яшма': ('Служебные элементы', 'Прочее'),
    'ПЛ25': ('Инженерные сети', 'Прочие трубопроводы'),
    'АСБ 200': ('Инженерные сети', 'Электроснабжение'),
    'летний': ('Здания и сооружения', 'Неизвестно'),
    'АСБ150': ('Инженерные сети', 'Электроснабжение'),
    'нет данных': ('Служебные элементы', 'Прочее'),
    'пл. Партизан': ('Твердые покрытия', 'Асфальт'),
    'ДЖ': ('Здания и сооружения', 'Неизвестно'),
    'дер. настил': ('Водопроницаемые покрытия', 'Деревянный настил'),
    'КЖ': ('Здания и сооружения', 'Неизвестно'),
    'канава': ('Водные объекты', 'Неизвестно'),
    'ТП': ('Инженерные сети', 'Электроснабжение'),
    'плодовое': ('Озеленение', 'Дерево'),
    '{\\C0;ЛЭП}': ('Инженерные сети', 'Электроснабжение'),
    'ПАО "Россети Московский регион"': ('Инженерные сети', 'Электроснабжение'),
    'ориентировочно': ('Служебные элементы', 'Прочее'),
    '+7 495 748 13 49': ('Служебные элементы', 'Штамп'),
    'охранная зона': ('Служебные элементы', 'Кадастр'),
    'Грибок': ('МАФ', 'Игровое оборудование'),
    'кор500': ('Инженерные сети', 'Прочие трубопроводы'),
    'корс630': ('Инженерные сети', 'Прочие трубопроводы'),
    'корс1200': ('Инженерные сети', 'Прочие трубопроводы'),
    'корсис1200': ('Инженерные сети', 'Прочие трубопроводы'),
    'корсис 1200': ('Инженерные сети', 'Прочие трубопроводы'),
    '\\A1;корс 315': ('Инженерные сети', 'Прочие трубопроводы'),
    'кор315': ('Инженерные сети', 'Прочие трубопроводы'),
    'корсис 100': ('Инженерные сети', 'Прочие трубопроводы'),
    'корс 200': ('Инженерные сети', 'Прочие трубопроводы'),
    'МУП "Подольская электросеть"': ('Инженерные сети', 'Электроснабжение'),
    'РП-55': ('Инженерные сети', 'Электроснабжение'),
    'корс500': ('Инженерные сети', 'Прочие трубопроводы'),
    'Бр.': ('Твердые покрытия', 'Брусчатка'),
    'м': ('Служебные элементы', 'Размеры'),
    'МУП"Водоканал"': ('Инженерные сети', 'Водопровод'),
    '2тр.': ('Инженерные сети', 'Прочие трубопроводы'),
    'корсис 400': ('Инженерные сети', 'Прочие трубопроводы'),
    'кор. 1000': ('Инженерные сети', 'Прочие трубопроводы'),
    'пешеходный переход': ('Элементы благоустройства', 'Дорожная разметка'),
    'Б.плиты': ('Твердые покрытия', 'Бетон'),
    'кор400': ('Инженерные сети', 'Прочие трубопроводы'),
    'кор200': ('Инженерные сети', 'Прочие трубопроводы'),
    'кор160': ('Инженерные сети', 'Прочие трубопроводы'),
    'к РП 24ф18': ('Инженерные сети', 'Электроснабжение'),
    'к РП 21ф10': ('Инженерные сети', 'Электроснабжение'),
    'АО "Мособлэнерго"Подольское ПО': ('Инженерные сети', 'Электроснабжение'),
    'к РП 38': ('Инженерные сети', 'Электроснабжение'),
    'к РП 21 ф7,10': ('Инженерные сети', 'Электроснабжение'),
    'к РП 24 ф 18': ('Инженерные сети', 'Электроснабжение'),
    'ПАО "Ростелеком" ': ('Инженерные сети', 'Сети связи'),
    'Р.': ('Водные объекты', 'Неизвестно'),
    '———': ('Служебные элементы', 'Условные обозначения'),
    'РУЧЕЙ': ('Водные объекты', 'Неизвестно'),
    'Р. СХОДНЯ': ('Водные объекты', 'Неизвестно'),
    '0.1': ('Служебные элементы', 'Размеры'),
    'Р.СХОДНЯ': ('Водные объекты', 'Неизвестно'),
    'X-2019Г.': ('Служебные элементы', 'Штамп'),
    'ЗАБОЛОЧЕНО': ('Водные объекты', 'Неизвестно'),
    'Ц РАЗР.': ('Твердые покрытия', 'Плитка'),
    'Н/Н': ('Инженерные сети', 'Инженерная инфраструктура'),
    'ГАЗОН': ('Озеленение', 'Газон'),
    '——2': ('Служебные элементы', 'Условные обозначения'),
    '0.10': ('Служебные элементы', 'Размеры'),
    '0.10        ': ('Служебные элементы', 'Размеры'),
    '9+2ПР': ('Инженерные сети', 'Прочие кабели'),
    '6+1ПР': ('Инженерные сети', 'Прочие кабели'),
    '9+2ПР.': ('Инженерные сети', 'Прочие кабели'),
    '6+1ПР.': ('Инженерные сети', 'Прочие кабели'),
    '6+1 ПР.': ('Инженерные сети', 'Прочие кабели'),
    'АГИТ.': ('МАФ', 'Информационная конструкция'),
    'ЯМА': ('Рельеф', 'Откос'),
    '-2.0': ('Рельеф', 'Отметки высот'),
    '-1.5': ('Рельеф', 'Отметки высот'),
    '0.5': ('Служебные элементы', 'Размеры'),
    'РЕКОНСТРУКЦИЯ БОРТОВОГО КАМНЯ': ('Элементы благоустройства', 'Бортовой камень'),
    'ВИДЕОКАМЕРА': ('Инженерные сети', 'Инженерная инфраструктура'),
    '0.4': ('Инженерные сети', 'Электроснабжение')
}


def assign_mapping_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    labels = new_mapping.get(row['Text'].strip())
    if labels:
        return labels
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_mapping_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_5.03.csv', index=False)
print(f"[5.03]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[5.03]  +2417  | размечено: 1116314 | осталось: 1953366


In [192]:
new_mapping = {
    'РЕКОНСТРУКЦИЯ': ('Служебные элементы', 'Прочее'),
    '-0.3': ('Служебные элементы', 'Прочее'),
    '-1.0': ('Служебные элементы', 'Прочее'),
    '0.3': ('Служебные элементы', 'Размеры'),
    'ИЗРЫТО': ('Водопроницаемые покрытия', 'Грунт'),
    'КИП': ('Инженерные сети', 'Инженерная инфраструктура'),
    'дор.знак': ('МАФ', 'Дорожные знаки'),
    'канава-1,30': ('Водные объекты', 'Неизвестно'),
    'верх тр.207.82': ('Инженерные сети', 'Прочие трубопроводы'),
    'Воздухопровод': ('Инженерные сети', 'Прочие трубопроводы'),
    '6пр.+1пр.': ('Инженерные сети', 'Прочие кабели'),
    '9пр.+2пр.': ('Инженерные сети', 'Прочие кабели'),
    'Ручей': ('Водные объекты', 'Неизвестно'),
    'в.тр204.32': ('Инженерные сети', 'Прочие трубопроводы'),
    '0,1': ('Служебные элементы', 'Размеры'),
    '0.25': ('Служебные элементы', 'Размеры'),
    '0.20': ('Служебные элементы', 'Размеры'),
    '0.30': ('Служебные элементы', 'Размеры'),
    '\u00a0люк': ('Инженерные сети', 'Инженерная инфраструктура'),
    '\u00a0знак дор большой': ('МАФ', 'Дорожные знаки'),
    '\u00a0заб мет': ('МАФ', 'Ограждение'),
    '\u00a0воздух': ('Инженерные сети', 'Прочие трубопроводы'),
    '\u00a0заб мет/заб раб': ('МАФ', 'Ограждение'),
    '\u00a0заб раб': ('МАФ', 'Ограждение'),
    '\u00a0столб мет камера': ('Инженерные сети', 'Столбы и опоры'),
    '\u00a0кип каб': ('Инженерные сети', 'Прочие кабели'),
    '\u00a0столб мет кам': ('Инженерные сети', 'Столбы и опоры'),
    '\u00a0выход труб 1000 ст': ('Инженерные сети', 'Прочие трубопроводы'),
    '\u00a0люк квадр': ('Инженерные сети', 'Инженерная инфраструктура'),
    '\u00a0ящ мет': ('Инженерные сети', 'Инженерная инфраструктура'),
    '\u00a0опора труб': ('Инженерные сети', 'Столбы и опоры'),
    '\u00a0поворот труб': ('Инженерные сети', 'Прочие трубопроводы'),
    '\u00a0труб верх': ('Инженерные сети', 'Прочие трубопроводы'),
    '\u00a0труб поворот': ('Инженерные сети', 'Прочие трубопроводы'),
    '\u00a0 ет': ('Неизвестно', 'Неизвестно'),
    '\u00a0бет': ('Твердые покрытия', 'Бетон'),
    '\u00a0выход труб': ('Инженерные сети', 'Прочие трубопроводы'),
    '\u00a0люк квадрат': ('Инженерные сети', 'Инженерная инфраструктура'),
    '\u00a0труб бет 800 верх': ('Инженерные сети', 'Прочие трубопроводы'),
    '\u00a0труб бет 800 низ': ('Инженерные сети', 'Прочие трубопроводы'),
    '\u00a0r': ('Неизвестно', 'Неизвестно'),
    '\u00a0верх': ('Рельеф', 'Откос'),
    '\u00a0урез': ('Неизвестно', 'Неизвестно'),
    '\u00a0низ': ('Рельеф', 'Откос'),
    '\u00a0болото': ('Водные объекты', 'Неизвестно'),
    '\u00a0рел': ('Рельеф', 'Отметки высот'),
    '\u00a0верх откоса': ('Рельеф', 'Откос'),
    '\u00a0ель35': ('Озеленение', 'Дерево'),
    '\u00a0ель15': ('Озеленение', 'Дерево'),
    '\u00a0ель30': ('Озеленение', 'Дерево'),
    '\u00a0ель50': ('Озеленение', 'Дерево'),
    '\u00a0насыпь': ('Рельеф', 'Откос'),
    '\u00a0насыпь верх': ('Рельеф', 'Откос'),
    '\u00a0канава верх': ('Рельеф', 'Откос'),
    '\u00a0канава низ': ('Рельеф', 'Откос'),
    '\u00a0верх отк': ('Рельеф', 'Откос'),
    '\u00a0заболоченность': ('Водные объекты', 'Неизвестно'),
    '\u00a0низ отк': ('Рельеф', 'Откос'),
    '\u00a0овраг верх': ('Рельеф', 'Откос'),
    '\u00a0овраг низ': ('Рельеф', 'Откос'),
    '\u00a0верх канавы': ('Рельеф', 'Откос'),
    '\u00a0труб мет 130': ('Инженерные сети', 'Прочие трубопроводы'),
    '\u00a0труб мет 130 низ': ('Инженерные сети', 'Прочие трубопроводы'),
    '\u00a0лэп': ('Инженерные сети', 'Электроснабжение'),
    '\u00a0бет разр': ('Твердые покрытия', 'Бетон'),
    '\u00a0труб мет': ('Инженерные сети', 'Прочие трубопроводы'),
    '2к.': ('Инженерные сети', 'Прочие кабели'),
    '4к.': ('Инженерные сети', 'Прочие кабели'),
    '4к.бд.': ('Инженерные сети', 'Прочие кабели'),
    'напорная': ('Инженерные сети', 'Канализация бытовая'),
    '3тр.': ('Инженерные сети', 'Прочие трубопроводы'),
    'бр.к.': ('Инженерные сети', 'Прочие кабели'),
    'бр.к.Б.Д.': ('Инженерные сети', 'Прочие кабели'),
    '2бр.к.': ('Инженерные сети', 'Прочие кабели'),
    '3бр.к.': ('Инженерные сети', 'Прочие кабели'),
    'd=ВЧШГ': ('Инженерные сети', 'Прочие трубопроводы'),
    '1к.': ('Инженерные сети', 'Прочие кабели'),
    'обр.': ('Служебные элементы', 'Прочие кабели'),
    '6отв.': ('Инженерные сети', 'Прочие кабели'),
    '2обр..': ('Инженерные сети', 'Прочие кабели'),
    '6к.': ('Инженерные сети', 'Прочие кабели'),
    '4тр.': ('Инженерные сети', 'Прочие трубопроводы'),
    '6к.обр.': ('Инженерные сети', 'Прочие кабели'),
    '8тр.стр.': ('Инженерные сети', 'Прочие трубопроводы'),
    '4к.тр.': ('Инженерные сети', 'Прочие кабели'),
    '8к.аб-та.': ('Инженерные сети', 'Прочие кабели'),
    '10к.аб-та.': ('Инженерные сети', 'Прочие кабели'),
    '6к.аб-та.': ('Инженерные сети', 'Прочие кабели'),
    '4к.аб-та.': ('Инженерные сети', 'Прочие кабели'),
    '-21450': ('Служебные элементы', 'Координаты'),
    '-21650': ('Служебные элементы', 'Координаты'),
    '-21350': ('Служебные элементы', 'Координаты'),
    '-21600': ('Служебные элементы', 'Координаты'),
    '-21400': ('Служебные элементы', 'Координаты'),
    '\u00a0каб-4.9': ('Инженерные сети', 'Прочие кабели')
}



def assign_mapping_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    labels = new_mapping.get(row['Text'].strip())
    if labels:
        return labels
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_mapping_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_5.04.csv', index=False)
print(f"[5.04]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[5.04]  +547  | размечено: 1116861 | осталось: 1952819


In [193]:
new_mapping = {
    '\u00a0каб-4.3': ('Инженерные сети', 'Прочие кабели'),
    '\u00a0каб-1.5': ('Инженерные сети', 'Прочие кабели'),
    '\u00a0каб-1.2': ('Инженерные сети', 'Прочие кабели'),
    '\u00a0каб-1.7': ('Инженерные сети', 'Прочие кабели'),
    '\u00a0каб-0.7': ('Инженерные сети', 'Прочие кабели'),
    '\u00a0каб-1.4': ('Инженерные сети', 'Прочие кабели'),
    '\u00a0люк гтс': ('Инженерные сети', 'Сети связи'),
    '\u00a0кип охр каб': ('Инженерные сети', 'Прочие кабели'),
    '\u00a0l': ('Неизвестно', 'Неизвестно'),
    'Затоп.^Jдо воды-1.83': ('Инженерные сети', 'Инженерная инфраструктура'),
    'К.ливн.^Jниз-3.52^Jлот-3.66': ('Инженерные сети', 'Ливневая канализация'),
    'Труб.СТ|200|-2.13': ('Инженерные сети', 'Прочие трубопроводы'),
    'К.ливн + Труба^Jниз-4.41': ('Инженерные сети', 'Ливневая канализация'),
    'Труб.СТ.400|-3.59': ('Инженерные сети', 'Прочие трубопроводы'),
    'закрыт^Jпо косвенным признакам\\PК.ливн. низ -4.79': ('Инженерные сети', 'Ливневая канализация'),
    '0.35': ('Служебные элементы', 'Размеры'),
    'вышка связи': ('Инженерные сети', 'Сети связи'),
    'баннер': ('МАФ', 'Информационная конструкция'),
    'Резин.': ('Твердые покрытия', 'Резиновая крошка'),
    '"Чайхана Душанбе"': ('Здания и сооружения', 'Неизвестно'),
    'боярышник': ('Озеленение', 'Кустарник'),
    'кн': ('Здания и сооружения', 'Неизвестно'),
    'мн': ('Здания и сооружения', 'Неизвестно'),
    'Закрыт': ('Служебные элементы', 'Прочее'),
    'МУП "Водоканал"': ('Инженерные сети', 'Водопровод'),
    'АО "Мособлэнерго" Подольское ПО': ('Инженерные сети', 'Электроснабжение'),
    'цветы': ('Озеленение', 'Цветник'),
    'ТЦ "Полет"': ('Здания и сооружения', 'Неизвестно'),
    '5 КЖ': ('Здания и сооружения', 'Неизвестно'),
    '174,45 ПЛ': ('Служебные элементы', 'Прочее'),
    'маг.': ('Здания и сооружения', 'Неизвестно'),
    'б.пр.': ('Инженерные сети', 'Прочие кабели'),
    'Ц пл-ка': ('Твердые покрытия', 'Бетон'),
    'яма -1,6': ('Рельеф', 'Откос'),
    'агит.': ('МАФ', 'Информационная конструкция'),
    'ст-700': ('Инженерные сети', 'Прочие трубопроводы'),
    'н.н.': ('Инженерные сети', 'Электроснабжение'),
    'а.ц. 200': ('Инженерные сети', 'Прочие трубопроводы'),
    'ориентир.': ('Служебные элементы', 'Прочее'),
    'а.ц. 300': ('Инженерные сети', 'Прочие трубопроводы'),
    'пл-ка': ('Служебные элементы', 'Прочее'),
    '3 К': ('Здания и сооружения', 'Неизвестно'),
    'закр.': ('Служебные элементы', 'Прочее'),
    '2 КЖ': ('Здания и сооружения', 'Неизвестно'),
    'А разр.': ('Твердые покрытия', 'Асфальт'),
    'рек.': ('Служебные элементы', 'Прочее'),
    'Ц плитка': ('Твердые покрытия', 'Плитка'),
    'Касса ФОК "Салют"': ('Здания и сооружения', 'Неизвестно'),
    '10КЖ': ('Здания и сооружения', 'Неизвестно'),
    'счетчик': ('Инженерные сети', 'Инженерная инфраструктура'),
    'ОУ': ('Инженерные сети', 'Наружное освещение'),
    'пл. Собина': ('Твердые покрытия', 'Плитка'),
    'территория': ('Служебные элементы', 'Прочее'),
    'Преображенской Церкви ': ('Здания и сооружения', 'Неизвестно'),
    'витрина': ('МАФ', 'Информационная конструкция'),
    '178,44 ПЛ': ('Служебные элементы', 'Прочее'),
    'молодая посадка': ('Озеленение', 'Кустарник'),
    'строительная площадка': ('Служебные элементы', 'Прочее'),
    'Ростелеком': ('Инженерные сети', 'Сети связи'),
    'вышка': ('Инженерные сети', 'Столбы и опоры'),
    'сот.связи': ('Инженерные сети', 'Сети связи'),
    'стр. пл.': ('Служебные элементы', 'Прочее'),
    '23КЖ': ('Здания и сооружения', 'Неизвестно'),
    'панд.': ('Здания и сооружения', 'Лестницы и пандусы'),
    'не действ.': ('Служебные элементы', 'Прочее'),
    'Коммуникации': ('Инженерные сети', 'Инженерная инфраструктура'),
    '\\pxi-3,l4,t4;Водопровод': ('Инженерные сети', 'Водопровод'),
    '\\pxi-3,l4,t4;Канализация': ('Инженерные сети', 'Канализация бытовая'),
    '\\pxi-3,l4,t4;Газопровод': ('Инженерные сети', 'Газоснабжение'),
    '\\pxi-3,l4,t4;Теплотрасса': ('Инженерные сети', 'Теплосеть'),
    '\\pxi-3,l4,t4;Ливневая канализация': ('Инженерные сети', 'Ливневая канализация'),
    '\\pxi-3,l4,t4;Напорная канализация': ('Инженерные сети', 'Канализация бытовая'),
    '4 КЖ': ('Здания и сооружения', 'Неизвестно'),
    '2 КН': ('Здания и сооружения', 'Неизвестно'),
    '17 КЖ': ('Здания и сооружения', 'Неизвестно'),
    '12 КЖ': ('Здания и сооружения', 'Неизвестно'),
    'АО "Мособлэнерго"': ('Инженерные сети', 'Электроснабжение'),
    'Ориентировочно': ('Служебные элементы', 'Прочее'),
    'МУП "БКС"': ('Инженерные сети', 'Инженерная инфраструктура'),
    'собстенник не установлен': ('Служебные элементы', 'Прочее'),
    'ВОУ': ('Инженерные сети', 'Водопровод'),
    'Стойка КЛЗ': ('МАФ', 'Прочее'),
    'АО "Мособлгаз" "Восток" ': ('Инженерные сети', 'Газоснабжение'),
    'канализационный дюкер': ('Инженерные сети', 'Канализация бытовая'),
    'Бет.': ('Твердые покрытия', 'Бетон'),
    'ДИП': ('МАФ', 'Игровое оборудование'),
    'озеро Травинское': ('Водные объекты', 'Неизвестно'),
    'Бет.пл.': ('Твердые покрытия', 'Бетон'),
    'А ': ('Твердые покрытия', 'Асфальт'),
    'МН ': ('Здания и сооружения', 'Неизвестно'),
    'Б': ('Твердые покрытия', 'Бетон'),
    'Бет.лоток': ('Инженерные сети', 'Ливневая канализация'),
    'нет доступа': ('Служебные элементы', 'Прочее'),
    'Мет.': ('Твердые покрытия', 'Металлический настил'),
    '0.08': ('Служебные элементы', 'Размеры')
}



def assign_mapping_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    labels = new_mapping.get(row['Text'].strip())
    if labels:
        return labels
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_mapping_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_5.05.csv', index=False)
print(f"[5.05]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[5.05]  +1555  | размечено: 1118416 | осталось: 1951264


In [194]:
new_mapping = {
    'бет.плиты': ('Твердые покрытия', 'Бетон'),
    'Пушкино-Тяговая-Пушкино': ('Инженерные сети', 'Электроснабжение'),
    'МУП "МЩВ"': ('Инженерные сети', 'Водопровод'),
    'Крош.': ('Твердые покрытия', 'Резиновая крошка'),
    'СН': ('Здания и сооружения', 'Неизвестно'),
    'голуб.': ('Здания и сооружения', 'Неизвестно'),
    'Московская область, г.о. Власиха, пос. Власиха': ('Служебные элементы', 'Штамп'),
    '108-18-ИЗ': ('Служебные элементы', 'Штамп'),
    '10-18г.': ('Служебные элементы', 'Штамп'),
    '4отв.': ('Инженерные сети', 'Прочие кабели'),
    'Территория ВЗУ': ('Инженерные сети', 'Водопровод'),
    'АО"Оборонэнерго"': ('Инженерные сети', 'Электроснабжение'),
    'Одинцовская РЭС': ('Инженерные сети', 'Электроснабжение'),
    'ГТУ г.Одинцово': ('Служебные элементы', 'Прочее'),
    'МУП"Благоустройство и развитие"': ('Служебные элементы', 'Прочее'),
    'стадии согласования с в/ч 95306. ': ('Служебные элементы', 'Прочее'),
    'Б.пл.': ('Твердые покрытия', 'Бетон'),
    'Фонтан': ('Здания и сооружения', 'Фонтаны'),
    'затопл.': ('Инженерные сети', 'Инженерная инфраструктура'),
    'h-9,25м': ('Служебные элементы', 'Размеры'),
    'h-12м': ('Служебные элементы', 'Размеры'),
    'временная теплосеть': ('Инженерные сети', 'Теплосеть'),
    'truba': ('Инженерные сети', 'Прочие трубопроводы'),
    'line': ('Служебные элементы', 'Контуры'),
    'gaz': ('Инженерные сети', 'Газоснабжение'),
    'dz': ('МАФ', 'Дорожные знаки'),
    'prud': ('Водные объекты', 'Неизвестно'),
    'hi': ('Служебные элементы', 'Прочее'),
    'ог.': ('МАФ', 'Ограждение'),
    '\\pi1.77333;МН\\P\\pi-0.75532;(голубятня)': ('Здания и сооружения', 'Неизвестно'),
    '\\pi-0.5214;Заболочено': ('Водные объекты', 'Неизвестно'),
    '\\pi1.77333;МН': ('Здания и сооружения', 'Неизвестно'),
    'б.пл.': ('Твердые покрытия', 'Бетон'),
    '.': ('Неизвестно', 'Неизвестно'),
    '6 КН ': ('Здания и сооружения', 'Неизвестно'),
    '9КЖ ': ('Здания и сооружения', 'Неизвестно'),
    'мн ': ('Здания и сооружения', 'Неизвестно'),
    'ось дороги': ('Твердые покрытия', 'Асфальт'),
    '2КЖ ': ('Здания и сооружения', 'Неизвестно'),
    '3КЖ ': ('Здания и сооружения', 'Неизвестно'),
    '4КН ': ('Здания и сооружения', 'Неизвестно'),
    'ТЦ': ('Здания и сооружения', 'Неизвестно'),
    'Промышленый. пер': ('Твердые покрытия', 'Асфальт'),
    'КН ': ('Здания и сооружения', 'Неизвестно'),
    'р. Пахра': ('Водные объекты', 'Неизвестно'),
    '2ДЖ ': ('Здания и сооружения', 'Неизвестно'),
    '5КЖ ': ('Здания и сооружения', 'Неизвестно'),
    '4КЖ ': ('Здания и сооружения', 'Неизвестно'),
    'Баннер': ('МАФ', 'Информационная конструкция'),
    'Ось дороги': ('Твердые покрытия', 'Асфальт'),
    'Доска почета': ('МАФ', 'Информационная конструкция'),
    'Вышка связи': ('Инженерные сети', 'Сети связи'),
    '1-й Деловой пр-д': ('Твердые покрытия', 'Асфальт'),
    'ТЦ "Капитолий"': ('Здания и сооружения', 'Неизвестно'),
    'Автомойка': ('Здания и сооружения', 'Неизвестно'),
    '1 тр. св.': ('Инженерные сети', 'Сети связи'),
    '3 тр.': ('Инженерные сети', 'Прочие трубопроводы'),
    '24тр.': ('Инженерные сети', 'Прочие трубопроводы'),
    '24 тр.': ('Инженерные сети', 'Прочие трубопроводы'),
    '4 тр.': ('Инженерные сети', 'Прочие трубопроводы'),
    '2тр. 172.24': ('Инженерные сети', 'Прочие трубопроводы'),
    '2 тр.': ('Инженерные сети', 'Прочие трубопроводы'),
    'зысыпан': ('Инженерные сети', 'Инженерная инфраструктура'),
    '6тр.': ('Инженерные сети', 'Прочие трубопроводы'),
    '12тр.': ('Инженерные сети', 'Прочие трубопроводы'),
    'НД80': ('Инженерные сети', 'Прочие трубопроводы'),
    '4 КН ': ('Здания и сооружения', 'Неизвестно'),
    '6КЖ ': ('Здания и сооружения', 'Неизвестно'),
    '14КЖ ': ('Здания и сооружения', 'Неизвестно'),
    '17КЖ ': ('Здания и сооружения', 'Неизвестно'),
    'Октябрьский пр-т': ('Твердые покрытия', 'Асфальт'),
    'Ленинградский пр-д': ('Твердые покрытия', 'Асфальт'),
    'Генератор': ('Инженерные сети', 'Электроснабжение'),
    'КПП': ('Здания и сооружения', 'Неизвестно'),
    '14КЖ': ('Здания и сооружения', 'Неизвестно'),
    'Переход': ('Элементы благоустройства', 'Дорожная разметка'),
    'Октябрьский.просп': ('Твердые покрытия', 'Асфальт'),
    'Парковка': ('Твердые покрытия', 'Асфальт'),
    'Ленин В.И.': ('МАФ', 'Памятники'),
    'МОУ Лицей №1': ('Служебные элементы', 'Прочее'),
    '3 КН': ('Здания и сооружения', 'Неизвестно'),
    'Мет.ящик': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Подольск': ('Служебные элементы', 'Прочее'),
    'Хоккейное поле': ('МАФ', 'Спортивное оборудование'),
    'Б.р': ('Твердые покрытия', 'Брусчатка'),
    '8 тр.': ('Инженерные сети', 'Прочие трубопроводы'),
    '6 тр.': ('Инженерные сети', 'Прочие трубопроводы'),
    '6 тр': ('Инженерные сети', 'Прочие трубопроводы'),
    '1 тр.': ('Инженерные сети', 'Прочие трубопроводы'),
    '12 тр.': ('Инженерные сети', 'Прочие трубопроводы'),
    '\u00a01 тр 1 каб': ('Инженерные сети', 'Прочие трубопроводы'),
    '2 тр': ('Инженерные сети', 'Прочие трубопроводы'),
    '4 тр': ('Инженерные сети', 'Прочие трубопроводы'),
    '16КЖ': ('Здания и сооружения', 'Неизвестно'),
    'АО "МособлгаЗ"': ('Инженерные сети', 'Газоснабжение')
}



def assign_mapping_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    labels = new_mapping.get(row['Text'].strip())
    if labels:
        return labels
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_mapping_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_5.06.csv', index=False)
print(f"[5.06]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[5.06]  +1047  | размечено: 1119463 | осталось: 1950217


In [195]:
new_mapping = {
    'СЗПГ': ('Инженерные сети', 'Газоснабжение'),
    'АЗ': ('Инженерные сети', 'Газоснабжение'),
    'Ку': ('Инженерные сети', 'Инженерная инфраструктура'),
    'КУ газ СКЗ\\P ': ('Инженерные сети', 'Газоснабжение'),
    'СКЗ (СЗПГ)\\P ': ('Инженерные сети', 'Газоснабжение'),
    'н/н\\P ': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Ку газ': ('Инженерные сети', 'Газоснабжение'),
    'КУ газ СКЗ\\PСЗПГ\\P ': ('Инженерные сети', 'Газоснабжение'),
    'не действующая': ('Инженерные сети', 'Инженерная инфраструктура'),
    'ПАО МТС': ('Инженерные сети', 'Сети связи'),
    'АО "РиалКом"': ('Инженерные сети', 'Сети связи'),
    'ТК': ('Инженерные сети', 'Инженерная инфраструктура'),
    '000 "Спецэксплуатация"': ('Служебные элементы', 'Прочее'),
    'ПС-791 ф17': ('Инженерные сети', 'Электроснабжение'),
    'РП-12 ф2.7': ('Инженерные сети', 'Электроснабжение'),
    '47-223': ('Служебные элементы', 'Нумерация'),
    'д.56': ('Здания и сооружения', 'Неизвестно'),
    'д.58': ('Здания и сооружения', 'Неизвестно'),
    'д.54': ('Здания и сооружения', 'Неизвестно'),
    'д.52': ('Здания и сооружения', 'Неизвестно'),
    'д.50': ('Здания и сооружения', 'Неизвестно'),
    'ОА "Мособлэнерго"': ('Инженерные сети', 'Электроснабжение'),
    'ГАИ': ('Здания и сооружения', 'Неизвестно'),
    '47-246': ('Служебные элементы', 'Прочее'),
    '418-406': ('Служебные элементы', 'Прочее'),
    'ПС-791': ('Инженерные сети', 'Электроснабжение'),
    'РП-3': ('Инженерные сети', 'Электроснабжение'),
    'РП-17': ('Инженерные сети', 'Электроснабжение'),
    'РП-36': ('Инженерные сети', 'Электроснабжение'),
    'РП-9': ('Инженерные сети', 'Электроснабжение'),
    'РП-29': ('Инженерные сети', 'Электроснабжение'),
    'рынок': ('Здания и сооружения', 'Неизвестно'),
    'опора': ('Инженерные сети', 'Столбы и опоры'),
    'РП-19': ('Инженерные сети', 'Электроснабжение'),
    'РП-140': ('Инженерные сети', 'Электроснабжение'),
    'ж.д.N40': ('Здания и сооружения', 'Неизвестно'),
    '197-60': ('Служебные элементы', 'Прочее'),
    '197-140 65-109': ('Служебные элементы', 'Прочее'),
    '109-140': ('Служебные элементы', 'Прочее'),
    'РП-31': ('Инженерные сети', 'Электроснабжение'),
    'ж.д.N13': ('Здания и сооружения', 'Неизвестно'),
    'ж.д.N11': ('Здания и сооружения', 'Неизвестно'),
    'РП-31 АО "Мособлэнерго"': ('Инженерные сети', 'Электроснабжение'),
    'ПС-480': ('Инженерные сети', 'Электроснабжение'),
    'РП-14': ('Инженерные сети', 'Электроснабжение'),
    'РП-14 АО "Мособлэнерго"': ('Инженерные сети', 'Электроснабжение'),
    'П-11 АО "Мособлэнерго"': ('Инженерные сети', 'Электроснабжение'),
    'ПС-480 АО "Мособлэнерго"': ('Инженерные сети', 'Электроснабжение'),
    'РП-13': ('Инженерные сети', 'Электроснабжение'),
    'д.1': ('Здания и сооружения', 'Неизвестно'),
    'РП-14 Ф14,15': ('Инженерные сети', 'Электроснабжение'),
    'РП-11': ('Инженерные сети', 'Электроснабжение'),
    'Ф 8': ('Инженерные сети', 'Электроснабжение'),
    'Ф 3,6,9': ('Инженерные сети', 'Электроснабжение'),
    'РП-14  Ф. 14,15': ('Инженерные сети', 'Электроснабжение'),
    'РП-9 ': ('Инженерные сети', 'Электроснабжение'),
    'н.тр167.56': ('Инженерные сети', 'Прочие трубопроводы'),
    'лот ': ('Инженерные сети', 'Инженерная инфраструктура'),
    'ПГ затоп.': ('Инженерные сети', 'Инженерная инфраструктура'),
    '\u00a02*250': ('Служебные элементы', 'Размеры'),
    'в тр 151.53': ('Инженерные сети', 'Прочие трубопроводы'),
    'корсис 300': ('Инженерные сети', 'Прочие трубопроводы'),
    'а/ц 400': ('Инженерные сети', 'Прочие трубопроводы'),
    'а/ц 200': ('Инженерные сети', 'Прочие трубопроводы'),
    'п/э': ('Инженерные сети', 'Прочие трубопроводы'),
    'нед': ('Служебные элементы', 'Прочее'),
    'КНС': ('Инженерные сети', 'Канализация бытовая'),
    'технич.': ('Служебные элементы', 'Прочее'),
    '3 КЖ': ('Здания и сооружения', 'Неизвестно'),
    'КЖ 16': ('Здания и сооружения', 'Неизвестно'),
    '3КН ': ('Здания и сооружения', 'Неизвестно'),
    'Смотровая площадка': ('Здания и сооружения', 'Неизвестно'),
    'раздевалка': ('Здания и сооружения', 'Неизвестно'),
    'КН подстанция': ('Инженерные сети', 'Электроснабжение'),
    'Частный сектор': ('Служебные элементы', 'Прочее'),
    'Парк': ('Служебные элементы', 'Прочее'),
    'бет.': ('Твердые покрытия', 'Бетон'),
    'Территория кафе "Якорь"': ('Служебные элементы', 'Прочее'),
    'контур СКЗ': ('Служебные элементы', 'Контуры'),
    'к РП-13': ('Инженерные сети', 'Электроснабжение'),
    'к КРУН': ('Инженерные сети', 'Электроснабжение'),
    'к ПС-596': ('Инженерные сети', 'Электроснабжение'),
    'Отстойник': ('Инженерные сети', 'Канализация бытовая'),
    'Очистные': ('Инженерные сети', 'Канализация бытовая'),
    'не действующий': ('Инженерные сети', 'Инженерная инфраструктура'),
    'ТХ1': ('Инженерные сети', 'Инженерная инфраструктура'),
    'ТХ2': ('Инженерные сети', 'Инженерная инфраструктура'),
    '4 отв.': ('Инженерные сети', 'Прочие кабели'),
    'ПС-82': ('Инженерные сети', 'Электроснабжение'),
    'ф-114, 107': ('Инженерные сети', 'Электроснабжение'),
    'РП-1': ('Инженерные сети', 'Электроснабжение'),
    'РП-5': ('Инженерные сети', 'Электроснабжение'),
    'ЗАО "Бецема"': ('Служебные элементы', 'Прочее'),
    'схематично': ('Служебные элементы', 'Прочее'),
    'сцена дер.': ('Водопроницаемые покрытия', 'Деревянный настил')
}




def assign_mapping_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    labels = new_mapping.get(row['Text'].strip())
    if labels:
        return labels
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_mapping_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_5.07.csv', index=False)
print(f"[5.07]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[5.07]  +206  | размечено: 1119669 | осталось: 1950011


In [196]:
new_mapping = {
    'ручей': ('Водные объекты', 'Неизвестно'),
    'ВРУ': ('Инженерные сети', 'Электроснабжение'),
    'сидения': ('МАФ', 'Уличная мебель'),
    'Септик': ('Инженерные сети', 'Канализация бытовая'),
    'верх. 160.38': ('Рельеф', 'Отметки высот'),
    'зем. 159.73': ('Рельеф', 'Отметки высот'),
    'Не действ.': ('Инженерные сети', 'Инженерная инфраструктура'),
    'лот 156.15': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Спец. условные обозначения': ('Служебные элементы', 'Условные обозначения'),
    'Информационный столб': ('МАФ', 'Информационная конструкция'),
    'Эл. шкаф': ('Инженерные сети', 'Электроснабжение'),
    '(западный флигель)': ('Здания и сооружения', 'Неизвестно'),
    '(восточный флигель)': ('Здания и сооружения', 'Неизвестно'),
    'Знаменская Церковь': ('Здания и сооружения', 'Неизвестно'),
    'Фундамент (разр.)': ('Твердые покрытия', 'Бетон'),
    '(флигель)': ('Здания и сооружения', 'Неизвестно'),
    'Кедр': ('Озеленение', 'Дерево'),
    'заилен': ('Инженерные сети', 'Инженерная инфраструктура'),
    'суд': ('Здания и сооружения', 'Неизвестно'),
    'админ.': ('Здания и сооружения', 'Неизвестно'),
    'Цразр.': ('Твердые покрытия', 'Плитка'),
    'стр.пл.': ('Служебные элементы', 'Прочее'),
    'Московская область, г.о. Зарайск, г. Зарайск,': ('Служебные элементы', 'Штамп'),
    '121-19-ИЗ(1)': ('Служебные элементы', 'Штамп'),
    '09-19г.': ('Служебные элементы', 'Штамп'),
    'границах улиц Советская, Октябрьская и ': ('Служебные элементы', 'Штамп'),
    'Комсомольская.': ('Служебные элементы', 'Штамп'),
    'Условные обозначения:': ('Служебные элементы', 'Условные обозначения'),
    'АЦТ100': ('Инженерные сети', 'Прочие трубопроводы'),
    '2АЦТ100': ('Инженерные сети', 'Прочие трубопроводы'),
    '4АЦТ100': ('Инженерные сети', 'Прочие трубопроводы'),
    '6АЦТ100': ('Инженерные сети', 'Прочие трубопроводы'),
    '1АЦТ100': ('Инженерные сети', 'Прочие трубопроводы'),
    '3АЦТ100': ('Инженерные сети', 'Прочие трубопроводы'),
    '12АЦТ100': ('Инженерные сети', 'Прочие трубопроводы'),
    'АЦТ': ('Инженерные сети', 'Прочие трубопроводы'),
    '2АЦТ': ('Инженерные сети', 'Прочие трубопроводы'),
    '3АЦТ': ('Инженерные сети', 'Прочие трубопроводы'),
    '4АЦТ': ('Инженерные сети', 'Прочие трубопроводы'),
    'Зарайская РЭС': ('Инженерные сети', 'Электроснабжение'),
    'на Котельную': ('Инженерные сети', 'Теплосеть'),
    'МУП"ЕСКХ"': ('Служебные элементы', 'Прочее'),
    'влад.не установл.': ('Служебные элементы', 'Прочее'),
    'на гаражи': ('Инженерные сети', 'Инженерная инфраструктура'),
    '0.05': ('Служебные элементы', 'Размеры'),
    '0.07': ('Служебные элементы', 'Размеры'),
    'Дом отдыха "Сенеж"': ('Здания и сооружения', 'Неизвестно'),
    'Московская область, г. Солнечногорск, ': ('Служебные элементы', 'Штамп'),
    '122-19-ИЗ': ('Служебные элементы', 'Штамп'),
    '10-19г.': ('Служебные элементы', 'Штамп'),
    'Яремчук Н.Р.': ('Служебные элементы', 'Штамп'),
    'благоустройство набережной озера Сенеж от': ('Служебные элементы', 'Штамп'),
    'курсов "Выстрел" до дома отдыха "Сенеж".': ('Служебные элементы', 'Штамп'),
    'набережной на территории бывших офицерских': ('Служебные элементы', 'Штамп'),
    'Линия сводки с листом 3': ('Служебные элементы', 'Условные обозначения'),
    'Линия сводки с листом 4': ('Служебные элементы', 'Условные обозначения'),
    'Яремчук': ('Служебные элементы', 'Штамп'),
    'Козин': ('Служебные элементы', 'Штамп'),
    'Безрук': ('Служебные элементы', 'Штамп'),
    'Иошкин': ('Служебные элементы', 'Штамп'),
    'Площадка для выгула собак': ('Водопроницаемые элементы', 'Песок'),
    'Деревянный настил': ('Водопроницаемые покрытия', 'Деревянный настил'),
    'Площадка\\PВоркаут': ('Твердые покрытия', 'Резиновая крошка'),
    '18.08.2022': ('Служебные элементы', 'Штамп'),
    '2dy 400': ('Инженерные сети', 'Прочие трубопроводы'),
    '2dy 250': ('Инженерные сети', 'Прочие трубопроводы'),
    'dy 200': ('Инженерные сети', 'Прочие трубопроводы'),
    'dy 300': ('Инженерные сети', 'Прочие трубопроводы'),
    'dy 150': ('Инженерные сети', 'Прочие трубопроводы'),
    'АО «МСК Энерго»': ('Инженерные сети', 'Электроснабжение'),
    'Гравий': ('Водопроницаемые покрытия', 'Гравий'),
    'Резина': ('Твердые покрытия', 'Резиновая крошка'),
    'инфо щит': ('МАФ', 'Информационная конструкция'),
    '15КЖ': ('Здания и сооружения', 'Неизвестно'),
    'голубятня': ('Здания и сооружения', 'Неизвестно'),
    'Юбилейный пр-кт': ('Твердые покрытия', 'Асфальт'),
    'СКЗ': ('Инженерные сети', 'Газоснабжение'),
    'КУ ГАЗ': ('Инженерные сети', 'Газоснабжение'),
    'АО "Мособлгаз" "Восток"': ('Инженерные сети', 'Газоснабжение'),
    'МУП "Реутовский водоканал"': ('Инженерные сети', 'Водопровод'),
    'Б.пл': ('Твердые покрытия', 'Бетон'),
    '50:09:0020217': ('Служебные элементы', 'Кадастр'),
    'Озеро "Сенеж"': ('Водные объекты', 'Неизвестно'),
    'Флагшток': ('МАФ', 'Флагшток'),
    '(ТП)': ('Инженерные сети', 'Электроснабжение'),
    'АО "КХЗ"': ('Служебные элементы', 'Прочее'),
    'МУП "ККК"': ('Служебные элементы', 'Прочее'),
    'АО "Мособлгаз" "Север" СПРЭС': ('Инженерные сети', 'Газоснабжение'),
    'тропа': ('Водопроницаемые покрытия', 'Грунт'),
    'Проспект Победы': ('Твердые покрытия', 'Асфальт'),
    'Теплосеть МУП ПТО ЖКХ г.о. Ступино': ('Инженерные сети', 'Теплосеть'),
    'Филиал АО "Мособлгаз" "Юг"': ('Инженерные сети', 'Газоснабжение')
}



def assign_mapping_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    labels = new_mapping.get(row['Text'].strip())
    if labels:
        return labels
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_mapping_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_5.08.csv', index=False)
print(f"[5.08]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[5.08]  +505  | размечено: 1120174 | осталось: 1949506


In [197]:
new_mapping = {
    'ВКХ МУП ПТО ЖКХ г.о. Ступино': ('Инженерные сети', 'Водопровод'),
    'Коломенский филиал АО "Мособлэнерго" Ступинское ПО': ('Инженерные сети', 'Электроснабжение'),
    '2СМН': ('Здания и сооружения', 'Неизвестно'),
    'СМН': ('Здания и сооружения', 'Неизвестно'),
    '3СМН': ('Здания и сооружения', 'Неизвестно'),
    'Г.В.': ('Инженерные сети', 'Водопровод'),
    'нав.': ('Здания и сооружения', 'Неизвестно'),
    '0.2': ('Служебные элементы', 'Размеры'),
    'Парковский переулок': ('Твердые покрытия', 'Асфальт'),
    'Пионерский переулок': ('Твердые покрытия', 'Асфальт'),
    'ост': ('МАФ', 'Павильоны'),
    'замур.': ('Инженерные сети', 'Инженерная инфраструктура'),
    'N4А': ('Служебные элементы', 'Нумерация'),
    'тц "Солнечный"': ('Здания и сооружения', 'Неизвестно'),
    'ДС"Сатурн"': ('Здания и сооружения', 'Неизвестно'),
    '12КЖ': ('Здания и сооружения', 'Неизвестно'),
    '"Незнакомка"': ('МАФ', 'Игровое оборудование'),
    'Авт. остановка': ('МАФ', 'Павильоны'),
    'сеп.': ('Инженерные сети', 'Канализация бытовая'),
    'мусорка': ('МАФ', 'Павильоны'),
    'отс.': ('Водопроницаемые покрытия', 'Гранитный отсев'),
    'площадка для': ('Служебные элементы', 'Прочее'),
    'КС': ('Инженерные сети', 'Сети связи'),
    '3каб,': ('Инженерные сети', 'Прочие кабели'),
    'плитка': ('Твердые покрытия', 'Плитка'),
    'тел.щит': ('Инженерные сети', 'Сети связи'),
    'строительный': ('Служебные элементы', 'Прочее'),
    'волейбола': ('МАФ', 'Спортивное оборудование'),
    'скам.': ('МАФ', 'Уличная мебель'),
    'карьер с водой': ('Водные объекты', 'Неизвестно'),
    'плит.': ('Твердые покрытия', 'Плитка'),
    'огр. из покрышек': ('МАФ', 'Ограждение'),
    'ограждение из покрышек': ('МАФ', 'Ограждение'),
    'загл.': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Луговая 63': ('Служебные элементы', 'Прочее'),
    '1к.св.': ('Инженерные сети', 'Сети связи'),
    '2 к.св.': ('Инженерные сети', 'Сети связи'),
    '117.48 лот': ('Инженерные сети', 'Инженерная инфраструктура'),
    'авт.вор.': ('МАФ', 'Ворота'),
    '3каб': ('Инженерные сети', 'Прочие кабели'),
    'кер300': ('Инженерные сети', 'Прочие трубопроводы'),
    'б/п': ('Инженерные сети', 'Прочие кабели'),
    '-2м': ('Служебные элементы', 'Размеры'),
    'б/пр': ('Инженерные сети', 'Прочие кабели'),
    '50:36:0010308': ('Служебные элементы', 'Кадастр'),
    '50:36:0010317': ('Служебные элементы', 'Кадастр'),
    'А.разр': ('Твердые покрытия', 'Асфальт'),
    'пр-д Лесной': ('Здания и сооружения', 'Неизвестно'),
    'ПАО "Ростелеком" Западный ТЦТЭТ': ('Инженерные сети', 'Сети связи'),
    'МУП "ТЕПЛО КОЛОМНЫ"': ('Инженерные сети', 'Теплосеть'),
    'Фундамент разрушен': ('Твердые покрытия', 'Бетон'),
    'разрушенные фундаменты': ('Твердые покрытия', 'Бетон'),
    'лоток 142.61': ('Инженерные сети', 'Инженерная инфраструктура'),
    'ПГ 146.41': ('Инженерные сети', 'Водопровод'),
    'лоток 142.09': ('Инженерные сети', 'Инженерная инфраструктура'),
    'выс. 4.5.м': ('Служебные элементы', 'Размеры'),
    'АО "Воентелеком"': ('Инженерные сети', 'Сети связи'),
    'А кр.': ('Твердые покрытия', 'Асфальт'),
    'черная': ('Озеленение', 'Дерево'),
    '1-ый Москворецкий пер.': ('Здания и сооружения', 'Неизвестно'),
    'Тихий пер.': ('Здания и сооружения', 'Неизвестно'),
    'Суперфосфатный пер.': ('Здания и сооружения', 'Неизвестно'),
    '\u00a0ПАО "Ростелеком"': ('Инженерные сети', 'Сети связи'),
    'АО "Газпром теплоэнерго МО"': ('Инженерные сети', 'Теплосеть'),
    'МУП "Белоозерское ЖКХ"': ('Инженерные сети', 'Инженерная инфраструктура'),
    'напорный канализационный коллектор': ('Инженерные сети', 'Канализация бытовая'),
    'Ф. Э. Дзержинский': ('МАФ', 'Памятники'),
    'остановка': ('МАФ', 'Павильоны'),
    'Плиты': ('Твердые покрытия', 'Бетон'),
    'ПАО "Ростелеком" ЛЦ Сергиев Посад ЦЭСВ МО': ('Инженерные сети', 'Сети связи'),
    'ПАО "Ростелеком" Восточного ТЦТЭТ': ('Инженерные сети', 'Сети связи'),
    'В.И.Ленин\u00a0': ('МАФ', 'Памятники'),
    '\u00a0АО "Мособлэнерго"': ('Инженерные сети', 'Электроснабжение'),
    'ПАО «Россети»': ('Инженерные сети', 'Электроснабжение'),
    'Флагштоки': ('МАФ', 'Флагштоки'),
    'гидрант 125.35': ('Инженерные сети', 'Водопровод'),
    'Столб металлический': ('Инженерные сети', 'Столбы и опоры'),
    'Парк К и От.': ('Служебные элементы', 'Прочее'),
    'Троицкая церковь': ('Здания и сооружения', 'Неизвестно'),
    'с.д.d100': ('Инженерные сети', 'Прочие трубопроводы'),
    'Филиал АО "Мособлгаз"': ('Инженерные сети', 'Газоснабжение'),
    '"Юго-Восток" Озерская РЭС': ('Инженерные сети', 'Электроснабжение'),
    'футб. поле': ('Водопроницаемые покрытия', 'Газон спортивный'),
    '(газст)': ('Инженерные сети', 'Газоснабжение'),
    '(вода)': ('Инженерные сети', 'Водопровод'),
    '(мн)': ('Здания и сооружения', 'Неизвестно'),
    '(612пер)': ('Служебные элементы', 'Прочее'),
    '(кн)': ('Здания и сооружения', 'Неизвестно'),
    '(кж)': ('Здания и сооружения', 'Неизвестно'),
    '(отм)': ('Рельеф', 'Отметки высот')
}



def assign_mapping_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    labels = new_mapping.get(row['Text'].strip())
    if labels:
        return labels
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_mapping_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_5.09.csv', index=False)
print(f"[5.09]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[5.09]  +689  | размечено: 1120863 | осталось: 1948817


In [199]:
new_mapping = {
    '(тп)': ('Инженерные сети', 'Электроснабжение'),
    '(тбо)': ('МАФ', 'Павильоны'),
    '(теплогаз)': ('Инженерные сети', 'Инженерная инфраструктура'),
    '(ов)': ('Инженерные сети', 'Инженерная инфраструктура'),
    '(он)': ('Инженерные сети', 'Инженерная инфраструктура'),
    '251Б': ('Служебные элементы', 'Нумерация'),
    'тр.': ('Инженерные сети', 'Прочие трубопроводы'),
    'платформа Балашиха': ('Здания и сооружения', 'Неизвестно'),
    'авт.': ('Здания и сооружения', 'Неизвестно'),
    '3к.': ('Инженерные сети', 'Прочие кабели'),
    'автосервис': ('Здания и сооружения', 'Неизвестно'),
    'р.Пехорка': ('Водные объекты', 'Неизвестно'),
    'пеш.': ('Элементы благоустройства', 'Дорожная разметка'),
    'ящ.': ('Инженерные сети', 'Инженерная инфраструктура'),
    'А зазр.': ('Твердые покрытия', 'Асфальт'),
    '2.0': ('Служебные элементы', 'Размеры'),
    '0.7': ('Служебные элементы', 'Размеры'),
    'автостоянка': ('Твердые покрытия', 'Асфальт'),
    '0.6': ('Служебные элементы', 'Размеры'),
    '0.8': ('Служебные элементы', 'Размеры'),
    'в землю': ('Инженерные сети', 'Инженерная инфраструктура'),
    'из земли': ('Инженерные сети', 'Инженерная инфраструктура'),
    'зам.': ('Служебные элементы', 'Прочее'),
    '1к. ': ('Инженерные сети', 'Прочие кабели'),
    'ТД"Текстиль"': ('Здания и сооружения', 'Неизвестно'),
    'МСТ-5': ('Инженерные сети', 'Сети связи'),
    'РП-70': ('Инженерные сети', 'Электроснабжение'),
    'РП-45': ('Инженерные сети', 'Электроснабжение'),
    'на РП-35': ('Инженерные сети', 'Электроснабжение'),
    'РП-74': ('Инженерные сети', 'Электроснабжение'),
    'РП-35': ('Инженерные сети', 'Электроснабжение'),
    'напорн.': ('Инженерные сети', 'Прочие трубопроводы'),
    'ст700': ('Инженерные сети', 'Прочие трубопроводы'),
    '+0.7': ('Рельеф', 'Отметки высот'),
    '+1.2': ('Рельеф', 'Отметки высот'),
    '+1.0': ('Рельеф', 'Отметки высот'),
    '+0.9': ('Рельеф', 'Отметки высот'),
    'опоры разрушенного забора': ('МАФ', 'Ограждение'),
    'РП-460': ('Инженерные сети', 'Электроснабжение'),
    '-0.7': ('Рельеф', 'Отметки высот'),
    'АЗС"Лукойл"': ('Здания и сооружения', 'Неизвестно'),
    'Болото': ('Водные объекты', 'Неизвестно'),
    '160.': ('Служебные элементы', 'Размеры'),
    'ст2*150;89;50;40': ('Инженерные сети', 'Прочие трубопроводы'),
    'ПАО "Ростелеком"ЛКС СЦ Подольск': ('Инженерные сети', 'Сети связи'),
    'к РП 11  ': ('Инженерные сети', 'Электроснабжение'),
    'к РП 45  ': ('Инженерные сети', 'Электроснабжение'),
    'к РП 9 ф8  ': ('Инженерные сети', 'Электроснабжение'),
    'ф9, ф3,6  ': ('Инженерные сети', 'Электроснабжение'),
    'ф20  ': ('Инженерные сети', 'Электроснабжение'),
    'к РП31  ': ('Инженерные сети', 'Электроснабжение'),
    'к РП 13  ': ('Инженерные сети', 'Электроснабжение'),
    'рщ': ('Инженерные сети', 'Электроснабжение'),
    'МУП "Подольская теплосеть"': ('Инженерные сети', 'Теплосеть'),
    'н 259': ('Служебные элементы', 'Нумерация'),
    'н 260': ('Служебные элементы', 'Нумерация'),
    'н 261': ('Служебные элементы', 'Нумерация'),
    'н 262': ('Служебные элементы', 'Нумерация'),
    'н 263': ('Служебные элементы', 'Нумерация'),
    'н 264': ('Служебные элементы', 'Нумерация'),
    'н 265': ('Служебные элементы', 'Нумерация'),
    'н 1': ('Служебные элементы', 'Нумерация'),
    'н 2': ('Служебные элементы', 'Нумерация'),
    'н 3': ('Служебные элементы', 'Нумерация'),
    'н 4': ('Служебные элементы', 'Нумерация'),
    'н 5': ('Служебные элементы', 'Нумерация'),
    'н 6': ('Служебные элементы', 'Нумерация'),
    'н 7': ('Служебные элементы', 'Нумерация'),
    'н 8': ('Служебные элементы', 'Нумерация'),
    'н 9': ('Служебные элементы', 'Нумерация'),
    'н 10': ('Служебные элементы', 'Нумерация'),
    'н 11': ('Служебные элементы', 'Нумерация'),
    'н 12': ('Служебные элементы', 'Нумерация'),
    'н 13': ('Служебные элементы', 'Нумерация'),
    'н 14': ('Служебные элементы', 'Нумерация'),
    'н 15': ('Служебные элементы', 'Нумерация'),
    'н 16': ('Служебные элементы', 'Нумерация'),
    'н 17': ('Служебные элементы', 'Нумерация'),
    'н 18': ('Служебные элементы', 'Нумерация'),
    'н 19': ('Служебные элементы', 'Нумерация'),
    'н 20': ('Служебные элементы', 'Нумерация'),
    'н 21': ('Служебные элементы', 'Нумерация'),
    'лот 186.34': ('Инженерные сети', 'Инженерная инфраструктура'),
    'лот 187.52': ('Инженерные сети', 'Инженерная инфраструктура'),
    'лот 190.77': ('Инженерные сети', 'Инженерная инфраструктура'),
    'лот 193.74': ('Инженерные сети', 'Инженерная инфраструктура'),
    'лот 194.04': ('Инженерные сети', 'Инженерная инфраструктура'),
    'лот 195.10': ('Инженерные сети', 'Инженерная инфраструктура'),
    'лот 195.35': ('Инженерные сети', 'Инженерная инфраструктура'),
    'лот 196.92': ('Инженерные сети', 'Инженерная инфраструктура'),
    'опоры бетонные': ('Инженерные сети', 'Столбы и опоры'),
    'н.д': ('Служебные элементы', 'Прочее'),
    'лоток': ('Инженерные сети', 'Инженерная инфраструктура'),
    '{\\Q0;\\W1;155}': ('Служебные элементы', 'Размеры')
}



def assign_mapping_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    labels = new_mapping.get(row['Text'].strip())
    if labels:
        return labels
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_mapping_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_5.10.csv', index=False)
print(f"[5.10]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[5.10]  +583  | размечено: 1121446 | осталось: 1948234


In [200]:
new_mapping = {
    '{\\Q0;\\W1;154}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W1;153}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W1;152}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W1;151}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W1;150}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W1;149}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W1;148}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W1;147}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W1;144}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W1;143}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W1;158}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W1;157}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W1;156}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W1;159}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W1;146}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W1;145}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W1;140.5}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W1;140}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W1;141}': ('Рельеф', 'Отметки высот'),
    '(инфщ)': ('МАФ', 'Информационная конструкция'),
    '(лив)': ('Инженерные сети', 'Ливневая канализация'),
    '(ступ)': ('Здания и сооружения', 'Лестницы и пандусы'),
    '(619пл)': ('Твердые покрытия', 'Плитка'),
    '(крест на камне)': ('МАФ', 'Памятники'),
    '(стол)': ('МАФ', 'Уличная мебель'),
    '(мн кофейня)': ('Здания и сооружения', 'Неизвестно'),
    '(рщ)': ('Инженерные сети', 'Электроснабжение'),
    '(619плит)': ('Твердые покрытия', 'Плитка'),
    '(пруд)': ('Водные объекты', 'Неизвестно'),
    '(борт)': ('Элементы благоустройства', 'Бортовой камень'),
    '(автодром)': ('Твердые покрытия', 'Асфальт'),
    'Волейбольная площадка': ('Твердые покрытия', 'Резиновая крошка'),
    'Пруд': ('Водные объекты', 'Неизвестно'),
    '(скейт площ)': ('Твердые покрытия', 'Асфальт'),
    '(панд)': ('Здания и сооружения', 'Лестницы и пандусы'),
    '(ост)': ('МАФ', 'Павильоны'),
    '(2кн)': ('Здания и сооружения', 'Неизвестно'),
    '(светоф)': ('Инженерные сети', 'Инженерная инфраструктура'),
    '(баннер)': ('МАФ', 'Информационная конструкция'),
    '(козырек)': ('Здания и сооружения', 'Неизвестно'),
    '(веранда)': ('Здания и сооружения', 'Неизвестно'),
    '(4кн)': ('Здания и сооружения', 'Неизвестно'),
    '(приям)': ('Инженерные сети', 'Инженерная инфраструктура'),
    '(панд2)': ('Здания и сооружения', 'Лестницы и пандусы'),
    '(2т Теп наз d50)': ('Инженерные сети', 'Теплосеть'),
    '(2т теп наз d50)': ('Инженерные сети', 'Теплосеть'),
    '(мет столб со звонком)': ('Инженерные сети', 'Столбы и опоры'),
    '(619резин)': ('Твердые покрытия', 'Резиновая крошка'),
    '(ищ)': ('МАФ', 'Информационная конструкция'),
    '(закладной камень)': ('МАФ', 'Памятники'),
    '(лоток)': ('Инженерные сети', 'Инженерная инфраструктура'),
    '(308светоф)': ('Инженерные сети', 'Инженерная инфраструктура'),
    '(Хип хоп)': ('МАФ', 'Игровое оборудование'),
    '(Днастил)': ('Водопроницаемые покрытия', 'Деревянный настил'),
    '(сцена)': ('Водопроницаемые покрытия', 'Деревянный настил'),
    '(мн засценное)': ('Здания и сооружения', 'Неизвестно'),
    '(мн арьерсцена)': ('Здания и сооружения', 'Неизвестно'),
    '(резин)': ('Твердые покрытия', 'Резиновая крошка'),
    'проспект Юбилейный': ('Твердые покрытия', 'Асфальт'),
    'пр-д Солнечный': ('Твердые покрытия', 'Асфальт'),
    'Ресторан "Дюна"': ('Здания и сооружения', 'Неизвестно'),
    'Скейт-площадка': ('Твердые покрытия', 'Асфальт'),
    'Казанская церковь': ('Здания и сооружения', 'Неизвестно'),
    'Арьерсцена': ('Здания и сооружения', 'Неизвестно'),
    'Кафе': ('Здания и сооружения', 'Неизвестно'),
    '"Городок"': ('МАФ', 'Игровое оборудование'),
    'летняя веранда': ('Здания и сооружения', 'Неизвестно'),
    'теннис': ('Твердые покрытия', 'Резиновая крошка'),
    '"Хип-хоп"': ('МАФ', 'Игровое оборудование'),
    'Бассейн': ('Водные объекты', 'Неизвестно'),
    'Дер': ('Озеленение', 'Дерево'),
    'Рез.': ('Твердые покрытия', 'Резиновая крошка'),
    '"Тир"': ('Здания и сооружения', 'Неизвестно'),
    '"Вальс"': ('МАФ', 'Игровое оборудование'),
    '"Сапоги': ('МАФ', 'Игровое оборудование'),
    'насос': ('Инженерные сети', 'Инженерная инфраструктура'),
    'трансформатор': ('Инженерные сети', 'Электроснабжение'),
    'непроходной канал': ('Инженерные сети', 'Инженерная инфраструктура'),
    'ООО "РСК"': ('Служебные элементы', 'Прочее'),
    'АО "Реутовский Водоканал"': ('Инженерные сети', 'Водопровод'),
    'Бпл.': ('Твердые покрытия', 'Бетон'),
    'лот 163.64': ('Инженерные сети', 'Инженерная инфраструктура'),
    'лот 166.62': ('Инженерные сети', 'Инженерная инфраструктура'),
    'тепло 1.67 2.34': ('Инженерные сети', 'Теплосеть')
}



def assign_mapping_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    labels = new_mapping.get(row['Text'].strip())
    if labels:
        return labels
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_mapping_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_5.11.csv', index=False)
print(f"[5.11]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[5.11]  +1153  | размечено: 1122599 | осталось: 1947081


In [201]:
new_mapping = {
    '6.V': ('Служебные элементы', 'Прочее'),
    '50:55:0030803': ('Служебные элементы', 'Кадастр'),
    '50.27.2.14': ('Служебные элементы', 'Кадастр'),
    'KН': ('Здания и сооружения', 'Неизвестно'),
    'н\\д': ('Служебные элементы', 'Прочее'),
    'к РП 3': ('Инженерные сети', 'Электроснабжение'),
    'к РП 11': ('Инженерные сети', 'Электроснабжение'),
    'к РП 5': ('Инженерные сети', 'Электроснабжение'),
    'к РП 45': ('Инженерные сети', 'Электроснабжение'),
    'к РП 19': ('Инженерные сети', 'Электроснабжение'),
    'к д. 6А': ('Инженерные сети', 'Инженерная инфраструктура'),
    'на ЗАГС': ('Инженерные сети', 'Инженерная инфраструктура'),
    'к д. 54': ('Инженерные сети', 'Инженерная инфраструктура'),
    '\u00a0МУП "Водоканал"': ('Инженерные сети', 'Водопровод'),
    '\u00a0МУП "Подольская теплосеть"': ('Инженерные сети', 'Теплосеть'),
    'Гран.': ('Твердые покрытия', 'Гранит'),
    'Бет.бор.': ('Элементы благоустройства', 'Бортовой камень'),
    'A': ('Твердые покрытия', 'Асфальт'),
    'ПАО "Ростелеком"Восточный ТЦТЭТ': ('Инженерные сети', 'Сети связи'),
    'ТЦ "ГАГАРИН"': ('Здания и сооружения', 'Неизвестно'),
    'Гагарин': ('МАФ', 'Памятники'),
    'д/с': ('Здания и сооружения', 'Неизвестно'),
    'ООО "Водоканал"': ('Инженерные сети', 'Водопровод'),
    '(100м)': ('Служебные элементы', 'Размеры'),
    '(100и)': ('Неизвестно', 'Неизвестно'),
    '(5борт)': ('Элементы благоустройства', 'Бортовой камень'),
    '(5исп)': ('Неизвестно', 'Неизвестно'),
    '(останоака)': ('МАФ', 'Павильон'),
    '(кн4)': ('Здания и сооружения', 'Неизвестно'),
    '(кн2)': ('Здания и сооружения', 'Неизвестно'),
    'фонтан': ('МАФ', 'Фонтаны'),
    'элящик': ('Инженерные сети', 'Электроснабжение'),
    '50 ливн': ('Инженерные сети', 'Ливневая канализация'),
    '100мет': ('Инженерные сети', 'Прочие трубопроводы'),
    'фонарь': ('Инженерные сети', 'Столбы и опоры'),
    'лавка': ('МАФ', 'Уличная мебель'),
    'вентшахты': ('Инженерные сети', 'Инженерная инфраструктура'),
    'светофор': ('Инженерные сети', 'Инженерная инфраструктура'),
    'знак': ('МАФ', 'Дорожные знаки'),
    'Остановка\u00a0': ('МАФ', 'Павильон'),
    'Городские часы\u00a0': ('МАФ', 'Прочее'),
    '2Ас1': ('Здания и сооружения', 'Неизвестно'),
    '(опор)': ('Инженерные сети', 'Столбы и опоры'),
    'Медучилище': ('Здания и сооружения', 'Неизвестно'),
    '(рамка)': ('Служебные элементы', 'Штамп'),
    'Арка': ('МАФ', 'Уличная мебель'),
    'Стол': ('МАФ', 'Уличная мебель'),
    '(коз)': ('Здания и сооружения', 'Неизвестно'),
    'Эл.ящ.': ('Инженерные сети', 'Электроснабжение'),
    '(светофор)': ('Инженерные сети', 'Инженерная инфраструктура'),
    'АО "Мособлэнерго"\u00a0': ('Инженерные сети', 'Электроснабжение'),
    'АО "ПТЭК"': ('Инженерные сети', 'Теплосеть'),
    'ТЦУМС-22': ('Инженерные сети', 'Сети связи'),
    '18КЖ': ('Здания и сооружения', 'Неизвестно'),
    'РУ-6': ('Инженерные сети', 'Электроснабжение'),
    'оборонные заграждения': ('МАФ', 'Ограждение'),
    'Луговая-Шереметьево': ('Служебные элементы', 'Прочее'),
    'Луговая-Котуар': ('Служебные элементы', 'Прочее'),
    'окоп': ('Рельеф', 'Откос'),
    'емкость': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Бет. пл.': ('Твердые покрытия', 'Бетон'),
    'Бет.\u00a0': ('Твердые покрытия', 'Бетон'),
    'тр-р': ('Инженерные сети', 'Электроснабжение'),
    'ров': ('Рельеф', 'Откос'),
    'капонир': ('Рельеф', 'Откос'),
    'бетонные \\Pколья': ('МАФ', 'Ограждение'),
    '0,6 мПА': ('Инженерные сети', 'Газоснабжение'),
    'ООО "ТЭК"': ('Инженерные сети', 'Теплосеть'),
    'Бет. кольцо': ('Инженерные сети', 'Инженерная инфраструктура'),
    'рез.': ('Твердые покрытия', 'Резиновая крошка'),
    'октябрьский проспект': ('Твердые покрытия', 'Асфальт'),
    'ру 10 кВ': ('Инженерные сети', 'Электроснабжение'),
    'Бес.': ('Здания и сооружения', 'Неизвестно'),
    'бетонные плиты': ('Твердые покрытия', 'Бетон'),
    'строящаяся': ('Служебные элементы', 'Прочее'),
    'АО "Люберецкий Водоканал"': ('Инженерные сети', 'Водопровод'),
    'АО "Люберецкая Теплосеть"': ('Инженерные сети', 'Теплосеть'),
    'Вытяжка': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Билборд': ('МАФ', 'Информационная конструкция'),
    'Рекламный щит': ('МАФ', 'Информационная конструкция'),
    'Кормушка для белок': ('МАФ', 'Уличная мебель'),
    'вывеска': ('МАФ', 'Информационная конструкция'),
    'кип': ('Инженерные сети', 'Инженерная инфраструктура'),
    'КУТ': ('Инженерные сети', 'Инженерная инфраструктура'),
    'плита': ('Твердые покрытия', 'Бетон'),
    '50:11:0030206': ('Служебные элементы', 'Кадастр')
}



def assign_mapping_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    labels = new_mapping.get(row['Text'].strip())
    if labels:
        return labels
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_mapping_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_5.12.csv', index=False)
print(f"[5.12]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[5.12]  +433  | размечено: 1123032 | осталось: 1946648


In [202]:
new_mapping = {
    '(газстолб)': ('Инженерные сети', 'Газоснабжение'),
    '(колодец)': ('Инженерные сети', 'Инженерная инфраструктура'),
    '(метзаборштакет)': ('МАФ', 'Ограждение'),
    '(отмостка)': ('Твердые покрытия', 'Бетон'),
    '(отмостка2кжзабор)': ('Твердые покрытия', 'Бетон'),
    '(2КЖ)': ('Здания и сооружения', 'Неизвестно'),
    '(СТУП)': ('Здания и сооружения', 'Лестницы и пандусы'),
    '(МН)': ('Здания и сооружения', 'Неизвестно'),
    '(дорзнак)': ('МАФ', 'Дорожные знаки'),
    '(ливреш)': ('Инженерные сети', 'Ливневая канализация'),
    '(а/у плитка)': ('Твердые покрытия', 'Плитка'),
    '(ЛЭП10)': ('Инженерные сети', 'Электроснабжение'),
    '(ЛЭП10ОТТЯЖ)': ('Инженерные сети', 'Электроснабжение'),
    '(ЛЭП10БЕТ)': ('Инженерные сети', 'Электроснабжение'),
    '(метстолбогр)': ('МАФ', 'Ограждение'),
    '(метогр)': ('МАФ', 'Ограждение'),
    '(пластстолбогр)': ('МАФ', 'Ограждение'),
    '(МННАПР)': ('Здания и сооружения', 'Неизвестно'),
    '(гравий)': ('Водопроницаемые покрытия', 'Гравий'),
    '(метогргаз)': ('МАФ', 'Ограждение'),
    '(бет)': ('Твердые покрытия', 'Бетон'),
    '(а/ц плит)': ('Твердые покрытия', 'Плитка'),
    '(ЭЛ.ЩИТ)': ('Инженерные сети', 'Электроснабжение'),
    '(табличказапрещено)': ('МАФ', 'Информационная конструкция'),
    '(ЛЭП10БЕТНАПР)': ('Инженерные сети', 'Электроснабжение'),
    '(колодецКУ КАЗ)': ('Инженерные сети', 'Инженерная инфраструктура'),
    '(табличка КУ КАЗ)': ('МАФ', 'Информационная конструкция'),
    '(ПОДПОРСТЕН)': ('Здания и сооружения', 'Подпорная стенка'),
    '(ПОДПОРСТЕНВЕРХ)': ('Здания и сооружения', 'Подпорная стенка'),
    '(БЕТПЛИТА)': ('Твердые покрытия', 'Бетон'),
    '(ОТКОСВЕРХ)': ('Рельеф', 'Откос'),
    '(ТРАНШЕЯВЕРХ)': ('Рельеф', 'Откос'),
    '(ТРАНШЕЯНИЗ)': ('Рельеф', 'Откос'),
    '(ТРУБАМЕТ300)': ('Инженерные сети', 'Прочие трубопроводы'),
    '(ТРУБАМЕТ100)': ('Инженерные сети', 'Прочие трубопроводы'),
    '(ОТКОСНИЗ)': ('Рельеф', 'Откос'),
    '(ВОДА)': ('Водные объекты', 'Неизвестно'),
    '(ОТКОСНИЗВОДА)': ('Рельеф', 'Откос'),
    '(ТРУБАГОФРА100)': ('Инженерные сети', 'Прочие трубопроводы'),
    '(МЕТЗАБОРПРОФ)': ('МАФ', 'Ограждение'),
    '(МЕТЗАБОРПРОФВОРОТА)': ('МАФ', 'Ворота'),
    '(ВОРОТА)': ('МАФ', 'Ворота'),
    '(МЕТЗАБОРПРОФРАБИЦА)': ('МАФ', 'Ограждение'),
    '(МЕТЗАБОРШТАКРАБИЦА)': ('МАФ', 'Ограждение'),
    '(МЕТРАБИЦА)': ('МАФ', 'Ограждение'),
    '(МЕТРАБИЦАДЕР)': ('МАФ', 'Ограждение'),
    '(МЕТЗАБОРПРОФДЕР)': ('МАФ', 'Ограждение'),
    '(ДЕРЗАБОРРАБИЦА)': ('МАФ', 'Ограждение'),
    '(РАБИЦА)': ('МАФ', 'Ограждение'),
    '(ДЕРЗАБОР)': ('МАФ', 'Ограждение'),
    '(БУГОРНИЗ)': ('Рельеф', 'Откос'),
    '(БУГОРВЕРХ)': ('Рельеф', 'Откос'),
    '(ДЕРЗАБОРГАРАЖ)': ('МАФ', 'Ограждение'),
    '(МЕТЗАБОРПРОФГАРАЖ)': ('МАФ', 'Ограждение'),
    '(ДЕРЗАБОРМЕТВОРОТА)': ('МАФ', 'Ворота'),
    '(РАБИЦАГАРАЖ)': ('МАФ', 'Ограждение'),
    '(ГАРАЖ)': ('Здания и сооружения', 'Неизвестно'),
    '(ГАРАЖМЕТШТАК)': ('Здания и сооружения', 'Неизвестно'),
    '(МЕТЗАБОРШТАК)': ('МАФ', 'Ограждение'),
    '(МЕТЗАБОРШТАКВОРОТА)': ('МАФ', 'Ворота'),
    '(МЕТЗАБОРШТАКПРОФ)': ('МАФ', 'Ограждение'),
    '(ГАРАЖМЕТ)': ('Здания и сооружения', 'Неизвестно'),
    '(МЕТЗАБОРПРОФКИРПИЧ)': ('МАФ', 'Ограждение'),
    '(ЗАБОРКИРПИЧ)': ('МАФ', 'Ограждение'),
    '(ГАРАЖВОРОТА)': ('МАФ', 'Ворота'),
    '(ГАРАЖЗАБОРКИРПИЧ)': ('МАФ', 'Ограждение'),
    '(КОЛОДЕЦПИТЬЕВОЙ)': ('Инженерные сети', 'Водопровод'),
    '(ЭЛЩИТ)': ('Инженерные сети', 'Электроснабжение'),
    '(РАСПАЯЧНЫЕ КОРОБКИ)': ('Инженерные сети', 'Электроснабжение'),
    '(РАБИЦАКАМЗАБОР)': ('МАФ', 'Ограждение'),
    '(КАМЗАБОР)': ('МАФ', 'Ограждение'),
    'озеро Торфянка': ('Водные объекты', 'Неизвестно'),
    'Гравий.': ('Водопроницаемые покрытия', 'Гравий'),
    'А.разр.': ('Твердые покрытия', 'Асфальт'),
    'кат. защита': ('Инженерные сети', 'Инженерная инфраструктура'),
    'ПАО "Росстелеком"': ('Инженерные сети', 'Сети связи'),
    'ПГ': ('Инженерные сети', 'Водопровод'),
    'Футбольная площадка': ('Водопроницаемые покрытия', 'Газон спортивный'),
    'Вышка спасат.': ('Здания и сооружения', 'Неизвестно'),
    'душ': ('Здания и сооружения', 'Неизвестно'),
    'БН': ('Здания и сооружения', 'Неизвестно'),
    'Сад н/д': ('Озеленение', 'Дерево'),
    'ВЛС': ('Инженерные сети', 'Сети связи'),
    'K': ('Инженерные сети', 'Канализация бытовая'),
    '3 пр.': ('Инженерные сети', 'Прочие кабели'),
    '4 пр.': ('Инженерные сети', 'Прочие кабели'),
    'б': ('Твердые покрытия', 'Брусчатка'),
    'футбольная площадка': ('Водопроницаемые покрытия', 'Газон спортивный'),
    'емк.': ('Инженерные сети', 'Инженерная инфраструктура'),
    '198.82-втр.': ('Инженерные сети', 'Прочие трубопроводы'),
    'канава-0.6м': ('Рельеф', 'Откос'),
    'а/ц 300': ('Инженерные сети', 'Прочие трубопроводы'),
    'канава-0.4м': ('Рельеф', 'Откос'),
    'газ.ук.': ('Инженерные сети', 'Газоснабжение'),
    'Территория ВЗУ №2': ('Инженерные сети', 'Водопровод'),
}



def assign_mapping_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    labels = new_mapping.get(row['Text'].strip())
    if labels:
        return labels
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_mapping_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_5.13.csv', index=False)
print(f"[5.13]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[5.13]  +837  | размечено: 1123869 | осталось: 1945811


In [203]:
new_mapping = {
    '"автобау"': ('Здания и сооружения', 'Неизвестно'),
    'Территория ЛОС №1': ('Инженерные сети', 'Канализация бытовая'),
    '112.45цоколь': ('Рельеф', 'Отметки высот'),
    'т-1': ('Инженерные сети', 'Теплосеть'),
    'т-8': ('Инженерные сети', 'Теплосеть'),
    'т-5': ('Инженерные сети', 'Теплосеть'),
    'т-4': ('Инженерные сети', 'Теплосеть'),
    'т-7': ('Инженерные сети', 'Теплосеть'),
    'т-2': ('Инженерные сети', 'Теплосеть'),
    'т-3': ('Инженерные сети', 'Теплосеть'),
    'т-9': ('Инженерные сети', 'Теплосеть'),
    'в.': ('Инженерные сети', 'Водопровод'),
    'н': ('Инженерные сети', 'Наружное освещение'),
    'ГОСТИННИЦА': ('Здания и сооружения', 'Неизвестно'),
    'выход из. зем.': ('Инженерные сети', 'Инженерная инфраструктура'),
    'шк': ('Здания и сооружения', 'Неизвестно'),
    'Музей': ('Здания и сооружения', 'Неизвестно'),
    '159cт н/д': ('Инженерные сети', 'Прочие трубопроводы'),
    'б.пр': ('Инженерные сети', 'Прочие кабели'),
    'Стадион': ('Водопроницаемые покрытия', 'Газон спортивный'),
    'ту.': ('Здания и сооружения', 'Неизвестно'),
    'ф. поле': ('Водопроницаемые покрытия', 'Газон спортивный'),
    'Пож.': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Резерв.': ('Инженерные сети', 'Инженерная инфраструктура'),
    '200кер': ('Инженерные сети', 'Прочие трубопроводы'),
    'грав.': ('Водопроницаемые покрытия', 'Гравий'),
    'зем.': ('Водопроницаемые покрытия', 'Грунт'),
    'гл.-0.7': ('Служебные элементы', 'Размеры'),
    '50ст': ('Инженерные сети', 'Прочие трубопроводы'),
    '150кер': ('Инженерные сети', 'Прочие трубопроводы'),
    'гл.-1.8': ('Служебные элементы', 'Размеры'),
    'н.бар.': ('Элементы благоустройства', 'Бортовой камень'),
    'в.бар.': ('Элементы благоустройства', 'Бортовой камень'),
    '100ст': ('Инженерные сети', 'Прочие трубопроводы'),
    'гл-1.10': ('Служебные элементы', 'Размеры'),
    'т-6': ('Инженерные сети', 'Теплосеть'),
    'т-13': ('Инженерные сети', 'Теплосеть'),
    'SEV': ('Неизвестно', 'Неизвестно'),
    '129.72ц': ('Рельеф', 'Отметки высот'),
    'Туполевское ш.': ('Твердые покрытия', 'Асфальт'),
    'ВАГОН': ('Здания и сооружения', 'Неизвестно'),
    'канава -0.3': ('Водные объекты', 'Неизвестно'),
    'НАВАЛ ГР.': ('Водопроницаемые покрытия', 'Грунт'),
    '+1.7': ('Рельеф', 'Отметки высот'),
    '+0.5': ('Рельеф', 'Отметки высот'),
    'канава -0.4': ('Водные объекты', 'Неизвестно'),
    'УСЛОВНЫЕ ОБОЗНАЧЕНИЯ:': ('Служебные элементы', 'Условные обозначения'),
    '- газопровод': ('Служебные элементы', 'Условные обозначения'),
    '- кадастровая граница': ('Служебные элементы', 'Условные обозначения'),
    'Коммуникации нанесены ориентировочно': ('Служебные элементы', 'Прочее'),
    'Кузнецова 18': ('Служебные элементы', 'Прочее'),
    '2 пр.': ('Инженерные сети', 'Прочие кабели'),
    'Кузнецова 14': ('Служебные элементы', 'Прочее'),
    'Ген.директ.': ('Служебные элементы', 'Штамп'),
    'Изм.': ('Служебные элементы', 'Штамп'),
    'Сумской М.Н.': ('Служебные элементы', 'Штамп'),
    'Зятиков Р.А.': ('Служебные элементы', 'Штамп'),
    'Геодезист': ('Служебные элементы', 'Штамп'),
    'Подпись': ('Служебные элементы', 'Штамп'),
    'Дата': ('Служебные элементы', 'Штамп'),
    'К.уч': ('Служебные элементы', 'Кадастр'),
    '№ док': ('Служебные элементы', 'Штамп'),
    'Мурманской обл."': ('Служебные элементы', 'Штамп'),
    '{\\fBook Antiqua|b0|i0|c204|p18;ООО\\fBook Antiqua|b0|i0|c0|p18; \\fBook Antiqua|b0|i0|c204|p18;«НОРДГЕО»}': ('Служебные элементы', 'Штамп'),
    'П': ('Инженерные сети', 'Прочие трубопроводы'),
    '{\\f@Arial Unicode MS|b0|i1|c204|p34;газ.ст}': ('Инженерные сети', 'Газоснабжение'),
    '{\\fArial|b0|i0|c204|p34;А}': ('Твердые покрытия', 'Асфальт'),
    '{\\fArial|b0|i1|c204|p34;д. Воронино}': ('Служебные элементы', 'Прочее'),
    '{\\fArial|b0|i1|c204|p34;П Р У Д}': ('Водные объекты', 'Неизвестно'),
    '{\\fArial|b0|i0|c204|p34;щ}': ('Водопроницаемые покрытия', 'Щебень'),
    '{\\f@Arial Unicode MS|b0|i1|c204|p34;газ.ков.}': ('Инженерные сети', 'Газоснабжение'),
    '{\\fArial|b0|i0|c204|p34;\\C4;Г}': ('Инженерные сети', 'Газоснабжение'),
    '{\\f@Arial Unicode MS|b0|i1|c204|p34;п \\f@Arial Unicode MS|b0|i1|c0|p34;d\\f@Arial Unicode MS|b0|i1|c204|p34;89}': ('Инженерные сети', 'Прочие трубопроводы'),
    '{\\fArial|b0|i0|c204|p34;\\C3;В}': ('Инженерные сети', 'Водопровод'),
    '{\\fArial|b0|i0|c204|p34;К}': ('Инженерные сети', 'Канализация бытовая'),
    'АГИТ': ('МАФ', 'Информационная конструкция'),
    'ТАРТАН': ('Твердые покрытия', 'Резиновая крошка'),
    'Почта': ('Здания и сооружения', 'Неизвестно'),
    '+0,1': ('Рельеф', 'Отметки высот'),
    '+0,2': ('Рельеф', 'Отметки высот'),
    'ИНФ.': ('МАФ', 'Информационная конструкция'),
    'ГАЗ': ('Инженерные сети', 'Газоснабжение'),
    'территория стройки': ('Служебные элементы', 'Прочее'),
    '{\\Feskd1|c204;А}': ('Твердые покрытия', 'Асфальт'),
    'Ж/Д Вокзал': ('Здания и сооружения', 'Неизвестно'),
    '{\\fBm431|c1|b0|i1;\\l\\C0;В}': ('Инженерные сети', 'Водопровод'),
    '{\\fBm431|c1|b0|i1;\\l\\C0;К}': ('Инженерные сети', 'Канализация бытовая'),
    '{\\fBm431|c1|b0|i1;\\l\\C0;Г}': ('Инженерные сети', 'Газоснабжение'),
    '{\\fBm431|c1|b0|i1;\\l\\C0;W}': ('Инженерные сети', 'Сети связи'),
    '{\\fBm431|c1|b0|i1;\\l\\C0;2T}': ('Инженерные сети', 'Теплосеть'),
    'Рожнова И.А.': ('Служебные элементы', 'Штамп'),
    'Аладышев А.Н.': ('Служебные элементы', 'Штамп'),
    'ФИО': ('Служебные элементы', 'Штамп'),
    'Наименование объекта:М.О. ': ('Служебные элементы', 'Штамп')
}



def assign_mapping_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    labels = new_mapping.get(row['Text'].strip())
    if labels:
        return labels
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_mapping_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_5.14.csv', index=False)
print(f"[5.14]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[5.14]  +341  | размечено: 1124210 | осталось: 1945470


In [204]:
new_mapping = {
    '{\\fBm431|b0|i1|c204|p18;\\C0;ориент.}': ('Служебные элементы', 'Прочее'),
    'Московский проспект': ('Твердые покрытия', 'Асфальт'),
    'афиша': ('МАФ', 'Информационная конструкция'),
    'пень': ('Озеленение', 'Дерево'),
    'выт.': ('Инженерные сети', 'Инженерная инфраструктура'),
    'зеркало': ('МАФ', 'Дорожные знаки'),
    'дор. ук.': ('МАФ', 'Дорожные знаки'),
    'Мельница': ('МАФ', 'Игровое оборудование'),
    'ЗАГС': ('Здания и сооружения', 'Неизвестно'),
    'КУ А.З.': ('Инженерные сети', 'Газоснабжение'),
    'МУП "Межрайонный Щелковский Водоканал"': ('Инженерные сети', 'Водопровод'),
    '"Водоканал Пушкинского района"': ('Инженерные сети', 'Водопровод'),
    '"Мытищимежрайгаз"': ('Инженерные сети', 'Газоснабжение'),
    'Пушкинская РЭС': ('Инженерные сети', 'Электроснабжение'),
    'КУ ': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Пушкинский Филиал': ('Служебные элементы', 'Прочее'),
    '12отв.': ('Инженерные сети', 'Прочие кабели'),
    '10отв.': ('Инженерные сети', 'Прочие кабели'),
    'ЛТЦ Пушкино': ('Инженерные сети', 'Сети связи'),
    'МБУ "Пушкинское городское хоз-во"': ('Служебные элементы', 'Прочее'),
    'Восточного ТЦТЭТ': ('Инженерные сети', 'Сети связи'),
    'АО"Вымпелком"': ('Инженерные сети', 'Сети связи'),
    '№15': ('Служебные элементы', 'Нумерация'),
    'озеро Сенеж': ('Водные объекты', 'Неизвестно'),
    '2.5': ('Служебные элементы', 'Размеры'),
    '1.5': ('Служебные элементы', 'Размеры'),
    'пирс': ('Здания и сооружения', 'Неизвестно'),
    'ЛИНИЯ СОПРЯЖЕНИЯ С ЛИСТОМ 2': ('Служебные элементы', 'Условные обозначения'),
    'ЛИНИЯ СОПРЯЖЕНИЯ С ЛИСТОМ 3': ('Служебные элементы', 'Условные обозначения'),
    'ЛИНИЯ СОПРЯЖЕНИЯ С ЛИСТОМ 1': ('Служебные элементы', 'Условные обозначения'),
    'ЛИНИЯ СОПРЯЖЕНИЯ С ЛИСТОМ 4': ('Служебные элементы', 'Условные обозначения'),
    'ЛИНИЯ СОПРЯЖЕНИЯ С ЛИСТОМ 5': ('Служебные элементы', 'Условные обозначения'),
    'ЛИНИЯ СОПРЯЖЕНИЯ С ЛИСТОМ 6': ('Служебные элементы', 'Условные обозначения'),
    'гильза': ('Инженерные сети', 'Инженерная инфраструктура'),
    'дор.зн.': ('МАФ', 'Дорожные знаки'),
    'инф.щит': ('МАФ', 'Информационная конструкция'),
    'резина': ('Твердые покрытия', 'Резиновая крошка'),
    'саженец': ('Озеленение', 'Дерево'),
    'Кн': ('Здания и сооружения', 'Неизвестно'),
    'ПИО 160': ('Инженерные сети', 'Прочие трубопроводы'),
    'ТК Шереметьево': ('Здания и сооружения', 'Неизвестно'),
    '{\\fArial|c1|b0|i0;\\l\\C0;А}': ('Твердые покрытия', 'Асфальт'),
    '{\\fArial|c1|b0|i1;\\l\\C0;пл.}': ('Служебные элементы', 'Прочее'),
    '{\\fArial|c1|b0|i0;\\l\\C0;2}': ('Служебные элементы', 'Нумерация'),
    '{\\fArial|c1|b0|i0;\\l\\C0;МН}': ('Здания и сооружения', 'Неизвестно'),
    '{\\fArial|c1|b0|i1;\\l\\C0;ул Менделеева}': ('Служебные элементы', 'Прочее'),
    '{\\fArial|c1|b0|i0;\\l\\C0;КН}': ('Здания и сооружения', 'Неизвестно'),
    'м.я.': ('Инженерные сети', 'Инженерная инфраструктура'),
    '"Комарик"': ('МАФ', 'Игровое оборудование'),
    'скейтплощадка': ('Твердые покрытия', 'Асфальт'),
    'поле': ('МАФ', 'Спортивное оборудование'),
    'баскет.кольцо': ('МАФ', 'Спортивное оборудование'),
    'шлаг.': ('МАФ', 'Шлагбаум'),
    'тенистый корт закрыт': ('Твердые покрытия', 'Резиновая крошка'),
    'ПИРС': ('Здания и сооружения', 'Неизвестно'),
    'ОЗЕРО': ('Водные объекты', 'Неизвестно'),
    'бурелом': ('Озеленение', 'Дерево'),
    '{\\Q0;\\W0.8;152}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W0.8;150}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W0.8;148}': ('Рельеф', 'Отметки высот'),
    'связь': ('Инженерные сети', 'Сети связи'),
    'бег.дорожка': ('Твердые покрытия', 'Резиновая крошка'),
    '-1.8': ('Рельеф', 'Отметки высот'),
    'выгр.': ('Инженерные сети', 'Канализация бытовая'),
    'резерв.': ('Инженерные сети', 'Инженерная инфраструктура'),
    'изыскания': ('Служебные элементы', 'Штамп'),
    'Кол.уч': ('Служебные элементы', 'Штамп'),
    'N док.': ('Служебные элементы', 'Штамп'),
    '34-2021-ИГДИ': ('Служебные элементы', 'Штамп'),
    'Каширин С.С.': ('Служебные элементы', 'Штамп'),
    'Шмелев С.А.': ('Служебные элементы', 'Штамп'),
    'Подп.': ('Служебные элементы', 'Штамп'),
    '0.00': ('Рельеф', 'Отметки высот'),
    'руч. Копнинка': ('Водные объекты', 'Неизвестно'),
    '14.04.2017.': ('Служебные элементы', 'Штамп'),
    '11.04.2017.': ('Служебные элементы', 'Штамп'),
    'территория СНТ': ('Служебные элементы', 'Прочее'),
    '№92': ('Служебные элементы', 'Нумерация'),
    '№93': ('Служебные элементы', 'Нумерация'),
    '№94': ('Служебные элементы', 'Нумерация'),
    '№95': ('Служебные элементы', 'Нумерация'),
    'болото 1.5': ('Водные объекты', 'Неизвестно'),
    'Hн.пр.=207.03': ('Рельеф', 'Отметки высот'),
    'Hоп.=208.02': ('Рельеф', 'Отметки высот'),
    'Hн.пр.=208.34': ('Рельеф', 'Отметки высот'),
    'Hоп.=207.87': ('Рельеф', 'Отметки высот'),
    'Hн.пр.=209.31': ('Рельеф', 'Отметки высот'),
    'Hоп.=208.72': ('Рельеф', 'Отметки высот'),
    'Hн.пр.=209.63': ('Рельеф', 'Отметки высот'),
    'Hоп.=209.09': ('Рельеф', 'Отметки высот')
}


def assign_mapping_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    labels = new_mapping.get(row['Text'].strip())
    if labels:
        return labels
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_mapping_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_5.15.csv', index=False)
print(f"[5.15]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[5.15]  +769  | размечено: 1124979 | осталось: 1944701


In [205]:
new_mapping = {
    'ТК-21': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Ленинского Комсомола 14': ('Твердые покрытия', 'Асфальт'),
    'Ленинского Комсомола 12': ('Твердые покрытия', 'Асфальт'),
    'КБ': ('Здания и сооружения', 'Неизвестно'),
    'уч. 7': ('Служебные элементы', 'Условные обозначения'),
    'уч. 5': ('Служебные элементы', 'Условные обозначения'),
    'р. Осетр': ('Водные объекты', 'Неизвестно'),
    'парковка': ('Твердые покрытия', 'Асфальт'),
    'волейбол. пл.': ('Твердые покрытия', 'Резиновая крошка'),
    'Лежаки': ('МАФ', 'Уличная мебель'),
    'асб. 150': ('Инженерные сети', 'Прочие трубопроводы'),
    'Московская область, г.Зарайск, благоустройство': ('Служебные элементы', 'Штамп'),
    '119-19-ИЗ': ('Служебные элементы', 'Штамп'),
    'Мигутина М.В.': ('Служебные элементы', 'Штамп'),
    'Козлов И.С.': ('Служебные элементы', 'Штамп'),
    'участка набережной на правом берегу': ('Служебные элементы', 'Штамп'),
    'р. Осетр до лодочной станции.': ('Служебные элементы', 'Штамп'),
    'р. Монастырка': ('Водные объекты', 'Неизвестно'),
    'разруш.': ('Служебные элементы', 'Прочее'),
    '0.06': ('Служебные элементы', 'Размеры'),
    'волейбол.': ('Твердые покрытия', 'Резиновая крошка'),
    'Кремлевский пер.': ('Твердые покрытия', 'Асфальт'),
    '0.12': ('Служебные элементы', 'Размеры'),
    'Схема расположения листов': ('Служебные элементы', 'Условные обозначения'),
    'АО Мособлгаз': ('Инженерные сети', 'Газоснабжение'),
    '"Мособлгаз"': ('Инженерные сети', 'Газоснабжение'),
    '"Ступиномежрайгаз"': ('Инженерные сети', 'Газоснабжение'),
    'ВОК ПАО "Ростелеком" Восточного ТЦТЭТ': ('Инженерные сети', 'Сети связи'),
    'ПАО "Ростелеком" Зарайский ЛТЦ': ('Инженерные сети', 'Сети связи'),
    'ZARAISK3': ('Служебные элементы', 'Прочее'),
    '2-6': ('Служебные элементы', 'Нумерация'),
    'задв. ': ('Инженерные сети', 'Инженерная инфраструктура'),
    'ведутся земляные работы': ('Служебные элементы', 'Прочее'),
    'газ в дом': ('Инженерные сети', 'Газоснабжение'),
    'Сквер им. В.Н. Леонова, Пл. Урицкого, ': ('Служебные элементы', 'Штамп'),
    '121(2)-19-ИЗ': ('Служебные элементы', 'Штамп'),
    'Козлов И.': ('Служебные элементы', 'Штамп'),
    'Московская область, г. о. Зарайск, г. Зарайск,': ('Служебные элементы', 'Штамп'),
    'ZARAISK4': ('Служебные элементы', 'Прочее'),
    'ZARAISK5': ('Служебные элементы', 'Прочее'),
    'ZARAISK6': ('Служебные элементы', 'Прочее'),
    'ZARAISK7': ('Служебные элементы', 'Прочее'),
    'ZARAISK8': ('Служебные элементы', 'Прочее'),
    'Тарасенков Ю.М.': ('Служебные элементы', 'Штамп'),
    'на БАМК': ('Инженерные сети', 'Инженерная инфраструктура'),
    'г. Зарайск': ('Служебные элементы', 'Штамп'),
    'АЦТ 100': ('Инженерные сети', 'Прочие трубопроводы'),
    'гофр. 50': ('Инженерные сети', 'Прочие трубопроводы'),
    'гл. -0.5;-0.7': ('Служебные элементы', 'Размеры'),
    'гл. -1.5': ('Служебные элементы', 'Размеры'),
    'на банк': ('Служебные элементы', 'Прочее'),
    'МУП "ЕСКХ"': ('Служебные элементы', 'Прочее'),
    'стелла': ('МАФ', 'Памятники'),
    'ПАО "МОЭСК"': ('Инженерные сети', 'Электроснабжение'),
    'т.': ('Инженерные сети', 'Теплосеть'),
    'Ворота': ('МАФ', 'Ворота'),
    'Дорога': ('Твердые покрытия', 'Асфальт'),
    'верх отк': ('Рельеф', 'Откос'),
    'дор': ('Твердые покрытия', 'Асфальт'),
    'дор бордюр': ('Элементы благоустройства', 'Бортовой камень'),
    'дор бордюр кон': ('Элементы благоустройства', 'Бортовой камень'),
    'дор бордюр нач': ('Элементы благоустройства', 'Бортовой камень'),
    'дор знак': ('МАФ', 'Дорожные знаки'),
    'забор': ('МАФ', 'Ограждение'),
    'здан': ('Здания и сооружения', 'Неизвестно'),
    'мет фонарь': ('Инженерные сети', 'Столбы и опоры'),
    'низ откос': ('Рельеф', 'Откос'),
    'рел': ('Рельеф', 'Отметки высот'),
    'руч': ('Водные объекты', 'Неизвестно'),
    'руч нач': ('Водные объекты', 'Неизвестно'),
    'сух25': ('Озеленение', 'Дерево'),
    'сух45': ('Озеленение', 'Дерево'),
    'Подпорка': ('Здания и сооружения', 'Подпорная стенка'),
    'Столб': ('Инженерные сети', 'Столбы и опоры'),
    'угол дома': ('Здания и сооружения', 'Неизвестно'),
    'дор.указ.': ('МАФ', 'Дорожные знаки'),
    '8пр.': ('Инженерные сети', 'Прочие кабели'),
    'дор.уаз.': ('МАФ', 'Дорожные знаки'),
    '2ДЖ': ('Здания и сооружения', 'Неизвестно'),
    'др.указ.': ('МАФ', 'Дорожные знаки'),
    'СМЖ': ('Здания и сооружения', 'Неизвестно'),
    'м.я': ('Инженерные сети', 'Инженерная инфраструктура'),
    'септик': ('Инженерные сети', 'Канализация бытовая'),
    'Наугольная башня': ('Здания и сооружения', 'Неизвестно'),
    'A6-1': ('Служебные элементы', 'Нумерация'),
    'территория заправки': ('Служебные элементы', 'Прочее'),
    'колонка': ('Инженерные сети', 'Водопровод'),
    'мойка': ('Здания и сооружения', 'Неизвестно'),
    'СМ': ('Здания и сооружения', 'Неизвестно'),
    'автозапчасти': ('Здания и сооружения', 'Неизвестно'),
    'п а ш н я ': ('Водопроницаемые покрытия', 'Грунт'),
    '0.15': ('Служебные элементы', 'Размеры'),
    '8(495)996-90-47': ('Служебные элементы', 'Штамп'),
    '8(495)996-90-64': ('Служебные элементы', 'Штамп'),
    'уч-к 26': ('Служебные элементы', 'Условные обозначения')
}


def assign_mapping_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    labels = new_mapping.get(row['Text'].strip())
    if labels:
        return labels
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_mapping_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_5.16.csv', index=False)
print(f"[5.16]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[5.16]  +852  | размечено: 1125831 | осталось: 1943849


In [206]:
new_mapping = {
    'р.ОСЕТР': ('Водные объекты', 'Неизвестно'),
    'По факту 2 этапа было доснято 11,6 га': ('Служебные элементы', 'Прочее'),
    'ст.150': ('Инженерные сети', 'Прочие трубопроводы'),
    'ор.': ('Служебные элементы', 'Прочее'),
    '-1.75': ('Служебные элементы', 'Прочее'),
    '-2.10': ('Служебные элементы', 'Прочее'),
    'ПАО "Ростлеком"': ('Инженерные сети', 'Сети связи'),
    'конденсат.': ('Инженерные сети', 'Инженерная инфраструктура'),
    'сбор.': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Кузнецкий спуск': ('Твердые покрытия', 'Асфальт'),
    'ОАО "Ростелеком" Восточное ТЦТЭТ': ('Инженерные сети', 'Сети связи'),
    '{\\Q0;\\W0.8;122}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W0.8;124}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W0.8;126}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W0.8;128}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W0.8;130}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W0.8;132}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W0.8;136}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W0.8;134}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W0.8;140}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W0.8;138}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W0.8;142}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W0.8;144}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W0.8;146}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W0.8;120}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W0.8;118}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W0.8;116}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W0.8;114}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W0.8;112}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W0.8;154}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W0.8;156}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W0.8;158}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W0.8;160}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W0.8;162}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W0.8;0}': ('Рельеф', 'Отметки высот'),
    'ЛКС СЦ г. Солнечногорск': ('Инженерные сети', 'Сети связи'),
    'Закладной камень ': ('МАФ', 'Памятники'),
    'р. Ока': ('Водные объекты', 'Неизвестно'),
    'Площадка для выгула собак ': ('Водопроницаемые покрытия', 'Песок'),
    '\u00a0 \u00a0 \u00a0 \u00a0Условные обозначения:\u00a0': ('Служебные элементы', 'Условные обозначения'),
    'граница кадастрового участка': ('Служебные элементы', 'Кадастр'),
    'канализация ': ('Инженерные сети', 'Канализация бытовая'),
    'газопровод': ('Инженерные сети', 'Газоснабжение'),
    'водопровод подземный': ('Инженерные сети', 'Водопровод'),
    'граница участка работ': ('Служебные элементы', 'Границы'),
    'теплотрасса подземная': ('Инженерные сети', 'Теплосеть'),
    'АО "Мособлгаз" "Юго-Восток" Коломенская РЭС': ('Инженерные сети', 'Газоснабжение'),
    'АО "Мособлэнерго" Коломенское ПО': ('Инженерные сети', 'Электроснабжение'),
    'Барьер ': ('МАФ', 'Ограждение'),
    'Пляжные раздевалки': ('МАФ', 'Павильоны'),
    'Футбольное поле': ('Водопроницаемые покрытия', 'Газон спортивный'),
    'Скейт площадка ': ('Твердые покрытия', 'Асфальт'),
    'Баскетбольная площадка': ('Твердые покрытия', 'Резиновая крошка'),
    'в.д. 80': ('Инженерные сети', 'Газоснабжение'),
    'в.д. 100': ('Инженерные сети', 'Газоснабжение'),
    'в.д. 350': ('Инженерные сети', 'Газоснабжение'),
    'ориент': ('Служебные элементы', 'Прочее'),
    'корс 160': ('Инженерные сети', 'Прочие трубопроводы'),
    '\u00a0ПАО "Ростелеком" СЦ г. Подольск ориентировочно ': ('Инженерные сети', 'Сети связи'),
    'КРП 60': ('Инженерные сети', 'Газоснабжение'),
    'не действ': ('Служебные элементы', 'Прочее'),
    'абс 150': ('Инженерные сети', 'Прочие трубопроводы'),
    'катодная защита': ('Инженерные сети', 'Инженерная инфраструктура'),
    'затоплин': ('Инженерные сети', 'Инженерная инфраструктура'),
    'МУП ПТО "Теплосеть"': ('Инженерные сети', 'Теплосеть'),
    'Коломенские электросети Ступинское ПО': ('Инженерные сети', 'Электроснабжение'),
    '(камни)': ('Твердые покрытия', 'Камень'),
    '(плитка)': ('Твердые покрытия', 'Плитка'),
    '(стелла)': ('МАФ', 'Памятники'),
    '(ступбр)': ('Здания и сооружения', 'Лестницы и пандусы'),
    'Информационный щит': ('МАФ', 'Информационная конструкция'),
    '19КЖ': ('Здания и сооружения', 'Неизвестно'),
    'ПАО Ростелеком': ('Инженерные сети', 'Сети связи'),
    'каток': ('Твердые покрытия', 'Асфальт'),
    'в.землю': ('Инженерные сети', 'Инженерная инфраструктура'),
    'погреб': ('Здания и сооружения', 'Неизвестно'),
    'кам.': ('Твердые покрытия', 'Камень'),
    '2КН ': ('Здания и сооружения', 'Неизвестно'),
    'газ 150': ('Инженерные сети', 'Газоснабжение'),
    'наземный': ('Инженерные сети', 'Инженерная инфраструктура'),
    'аварийное N17': ('Озеленение', 'Дерево'),
    'аварийное N15': ('Озеленение', 'Дерево'),
    'рельеф нарушен': ('Водопроницаемые покрытия', 'Грунт')
}


def assign_mapping_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    labels = new_mapping.get(row['Text'].strip())
    if labels:
        return labels
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_mapping_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_5.17.csv', index=False)
print(f"[5.17]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[5.17]  +587  | размечено: 1126418 | осталось: 1943262


In [207]:
new_mapping = {
    'стр. матер.': ('Служебные элементы', 'Прочее'),
    'ресторан': ('Здания и сооружения', 'Неизвестно'),
    'веранда': ('Здания и сооружения', 'Неизвестно'),
    '1пр': ('Инженерные сети', 'Прочие кабели'),
    '2пр': ('Инженерные сети', 'Прочие кабели'),
    '4пр': ('Инженерные сети', 'Прочие кабели'),
    '3пр': ('Инженерные сети', 'Прочие кабели'),
    '7пр': ('Инженерные сети', 'Прочие кабели'),
    '5пр': ('Инженерные сети', 'Прочие кабели'),
    '2камеры': ('Инженерные сети', 'Инженерная инфраструктура'),
    '8пр': ('Инженерные сети', 'Прочие кабели'),
    '10пр': ('Инженерные сети', 'Прочие кабели'),
    '6пр': ('Инженерные сети', 'Прочие кабели'),
    'указ. парка': ('МАФ', 'Информационная конструкция'),
    'WIFI': ('Инженерные сети', 'Сети связи'),
    'территория пейнтбола': ('Служебные элементы', 'Прочее'),
    '3кам.': ('Инженерные сети', 'Инженерная инфраструктура'),
    'сушилка': ('МАФ', 'Уличная мебель'),
    'CН': ('Здания и сооружения', 'Неизвестно'),
    '250 а/ц': ('Инженерные сети', 'Прочие трубопроводы'),
    'КH': ('Здания и сооружения', 'Неизвестно'),
    'Эл. табло': ('МАФ', 'Информационная конструкция'),
    'Футбольное \\Pполе': ('Водопроницаемые покрытия', 'Газон спортивный'),
    'резервуар\\Pдрен.вод': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Л': ('Неизвестно', 'Неизвестно'),
    'SOL1.1': ('Неизвестно', 'Неизвестно'),
    'SOL2.1': ('Неизвестно', 'Неизвестно'),
    'МУП "ИК ЖКХ"': ('Служебные элементы', 'Прочее'),
    'ВОЛС АО"ФОРТЕКС" в ТК ПАО"Ростелеком"': ('Инженерные сети', 'Сети связи'),
    'хоз.инвентарь': ('МАФ', 'Павильоны'),
    'пож.шкаф.': ('Инженерные сети', 'Инженерная инфраструктура'),
    'АО "СМЗ"': ('Служебные элементы', 'Прочее'),
    'ПАО"МОЭСК" Сев.сети': ('Инженерные сети', 'Электроснабжение'),
    'АО"СМЗ"': ('Служебные элементы', 'Прочее'),
    'Б. пл.': ('Твердые покрытия', 'Бетон'),
    'навал ': ('Водопроницаемые покрытия', 'Грунт'),
    ' гр.': ('Водопроницаемые покрытия', 'Грунт'),
    'МУП"ИК ЖКХ"': ('Служебные элементы', 'Прочее'),
    '126а': ('Здания и сооружения', 'Неизвестно'),
    'МОЭСК': ('Инженерные сети', 'Электроснабжение'),
    'адм. парка': ('Здания и сооружения', 'Неизвестно'),
    'Клинский филиал': ('Служебные элементы', 'Прочее'),
    '"Красногорскмежрайгаз"': ('Инженерные сети', 'Газоснабжение'),
    'СРЭС': ('Инженерные сети', 'Электроснабжение'),
    'в г.Кандалакша, Мурманской обл."': ('Служебные элементы', 'Штамп'),
    'Стадион "Локомотив"': ('Здания и сооружения', 'Неизвестно'),
    '\\P 165': ('Рельеф', 'Отметки высот'),
    '1пр. связи': ('Инженерные сети', 'Сети связи'),
    '2 громк-ля': ('Инженерные сети', 'Сети связи'),
    '4 ящ.': ('Инженерные сети', 'Инженерная инфраструктура'),
    '2 камеры': ('Инженерные сети', 'Инженерная инфраструктура'),
    '2 троса ': ('Инженерные сети', 'Прочие кабели'),
    'трос': ('Инженерные сети', 'Прочие кабели'),
    'зам. ': ('Служебные элементы', 'Прочее'),
    'тр.пл.': ('Твердые покрытия', 'Плитка'),
    'флагш.': ('МАФ', 'Флагштоки'),
    'Коломенским': ('Служебные элементы', 'Прочее'),
    'Жертвам': ('МАФ', 'Памятники'),
    'Радиационных ': ('МАФ', 'Памятники'),
    'Катастроф ': ('МАФ', 'Памятники'),
    'Репрессированным ': ('МАФ', 'Памятники'),
    'Мать': ('МАФ', 'Памятники'),
    'ракеты': ('МАФ', 'Памятники'),
    'Мужеству ': ('МАФ', 'Памятники'),
    'Артеллеристов': ('МАФ', 'Памятники'),
    'Музей боевой славы': ('Здания и сооружения', 'Неизвестно'),
    'Петропавловкая церковь': ('Здания и сооружения', 'Неизвестно'),
    'Сергиевская часовня': ('Здания и сооружения', 'Неизвестно'),
    'лоток 0.2': ('Инженерные сети', 'Инженерная инфраструктура'),
    'стела': ('МАФ', 'Памятники'),
    '1пр.св.': ('Инженерные сети', 'Сети связи'),
    '2пр.св.': ('Инженерные сети', 'Сети связи'),
    'курсантам': ('МАФ', 'Памятники'),
    'ОУ №301': ('Инженерные сети', 'Наружное освещение'),
    'в локальных': ('МАФ', 'Памятники'),
    'войнах': ('МАФ', 'Памятники'),
    'Листопад И.Н. 8(916)809-09-99': ('Служебные элементы', 'Штамп'),
    '(СПГЗ)': ('Инженерные сети', 'Газоснабжение'),
    'контактный провод 600В ': ('Инженерные сети', 'Электроснабжение'),
    'МУП "Коломенский трамвай"': ('Элементы благоустройства', 'Трамвайные пути'),
    'МБУ Коломенская Электросеть': ('Инженерные сети', 'Электроснабжение'),
    'собственник не установлен ': ('Служебные элементы', 'Прочее'),
    'поливочный водопровод': ('Инженерные сети', 'Водопровод'),
    'р.щ.': ('Инженерные сети', 'Электроснабжение'),
    'ступени': ('Здания и сооружения', 'Лестницы и пандусы'),
    'ог': ('МАФ', 'Ограждение'),
    'замощен': ('Твердые покрытия', 'Брусчатка'),
    'колодцы': ('Инженерные сети', 'Инженерная инфраструктура'),
}



def assign_mapping_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    labels = new_mapping.get(row['Text'].strip())
    if labels:
        return labels
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_mapping_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_5.18.csv', index=False)
print(f"[5.18]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[5.18]  +350  | размечено: 1126768 | осталось: 1942912


In [208]:
new_mapping = {
    'б.п': ('Твердые покрытия', 'Бетон'),
    'р.щит': ('Инженерные сети', 'Электроснабжение'),
    'мет.желоб': ('Инженерные сети', 'Ливневая канализация'),
    'декор.': ('Озеленение', 'Цветник'),
    'декор. сад': ('Озеленение', 'Цветник'),
    '?': ('Неизвестно', 'Неизвестно'),
    'шк.': ('Здания и сооружения', 'Неизвестно'),
    'А(разр.)': ('Твердые покрытия', 'Асфальт'),
    'напорн': ('Инженерные сети', 'Прочие трубопроводы'),
    'асб.': ('Инженерные сети', 'Прочие трубопроводы'),
    'ст': ('Инженерные сети', 'Прочие трубопроводы'),
    '1м.': ('Служебные элементы', 'Размеры'),
    'выгреб': ('Инженерные сети', 'Канализация бытовая'),
    'газ. защита': ('Инженерные сети', 'Газоснабжение'),
    'рекл': ('МАФ', 'Информационная конструкция'),
    'стр.пл': ('Служебные элементы', 'Прочее'),
    'Пушкинский пер.': ('Твердые покрытия', 'Асфальт'),
    'площадь Володарского': ('Твердые покрытия', 'Асфальт'),
    'стр. пл': ('Служебные элементы', 'Прочее'),
    'столбы мет.': ('Инженерные сети', 'Столбы и опоры'),
    'Сопряжение с фрагментом 2 ': ('Служебные элементы', 'Условные обозначения'),
    'Сопряжение с фрагментом 1 ': ('Служебные элементы', 'Условные обозначения'),
    'Сопряжение с фрагментом 3': ('Служебные элементы', 'Условные обозначения'),
    'Сопряжение с фрагментом 3 ': ('Служебные элементы', 'Условные обозначения'),
    'Сопряжение с фрагментом 2': ('Служебные элементы', 'Условные обозначения'),
    'мет. 2200': ('Инженерные сети', 'Прочие трубопроводы'),
    'Западный объезд г. Сергиев Посад': ('Служебные элементы', 'Прочее'),
    'р.Коптинка': ('Водные объекты', 'Неизвестно'),
    'оз. Загорское море': ('Водные объекты', 'Неизвестно'),
    'СНТ "Росток"': ('Служебные элементы', 'Прочее'),
    'Род.': ('Водные объекты', 'Неизвестно'),
    '{\\Q15;\\W0.8;АО «Специализированный застройщик Берендей»}': ('Служебные элементы', 'Штамп'),
    '{\\Q0;\\W0.8;202}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W0.8;200}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W0.8;198}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W0.8;204}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W0.8;206}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W0.8;208}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W0.8;196}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W0.8;210}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W0.8;212}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W0.8;214}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W0.8;216}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W0.8;218}': ('Рельеф', 'Отметки высот'),
    'Возможно воентелеком': ('Инженерные сети', 'Сети связи'),
    'лэп': ('Инженерные сети', 'Электроснабжение'),
    'территория автостоянки': ('Твердые покрытия', 'Асфальт'),
    'А3': ('Служебные элементы', 'Штамп'),
    '{\\fTimes New Roman|b0|i1|c204|p18;бер.}': ('Озеленение', 'Дерево'),
    'вых.скалы': ('Рельеф', 'Откос'),
    'кромка воды на момент снятия дна реки': ('Водные объекты', 'Неизвестно'),
    'отметки дна реки': ('Рельеф', 'Отметки высот'),
    'род.': ('Водные объекты', 'Неизвестно'),
    'Автостоянка': ('Твердые покрытия', 'Асфальт'),
    'арт.вода': ('Инженерные сети', 'Водопровод'),
    '0.36': ('Служебные элементы', 'Размеры'),
    'АСБ1000': ('Инженерные сети', 'Прочие трубопроводы'),
    'привоз.': ('Водопроницаемые покрытия', 'Грунт'),
    'р. Пехорка': ('Водные объекты', 'Неизвестно'),
    'Изрыто': ('Водопроницаемые покрытия', 'Грунт'),
    '125-19-ИЗ': ('Служебные элементы', 'Штамп'),
    'Кутилина В.Н.': ('Служебные элементы', 'Штамп'),
    'Соседов М.М.': ('Служебные элементы', 'Штамп'),
    'эстакада': ('Твердые покрытия', 'Асфальт'),
    'домов 3,9,15,12,6 участок, расположенный с вос.': ('Служебные элементы', 'Штамп'),
    '1,8,5,6 в мкр. Керамик и зап. стороны от домов ': ('Служебные элементы', 'Штамп'),
    '17КЖ': ('Здания и сооружения', 'Неизвестно'),
    '\u00a0пл.': ('Твердые покрытия', 'Плитка'),
    '\u00a0территория кафе': ('Служебные элементы', 'Прочее'),
    'Балашихамежрайгаз': ('Инженерные сети', 'Газоснабжение'),
    'Реутовская РЭС': ('Инженерные сети', 'Электроснабжение'),
    'ВЭС ПАО "МОЭСК"': ('Инженерные сети', 'Электроснабжение'),
    '20отв.': ('Инженерные сети', 'Прочие кабели'),
    'ВТЦТЭТ ОАО"Ростелеком"': ('Инженерные сети', 'Сети связи'),
    'охр.зона 2м': ('Служебные элементы', 'Кадастр'),
    '24отв.': ('Инженерные сети', 'Прочие кабели'),
    'пустой': ('Инженерные сети', 'Прочие кабели'),
    '4отв. ориент.': ('Инженерные сети', 'Прочие кабели'),
    'ПАО "Ростелеком" ЛТЦ г.Железнодорожный': ('Инженерные сети', 'Сети связи'),
    'ЛТЦ г.Железнодорожный': ('Инженерные сети', 'Сети связи'),
    'МУП "Балашихинский ': ('Инженерные сети', 'Водопровод'),
    'Водоканал"': ('Инженерные сети', 'Водопровод'),
    'ООО"Бэлс-Энергосервис"': ('Инженерные сети', 'Электроснабжение'),
    '2хd2000': ('Инженерные сети', 'Прочие трубопроводы'),
    'ЗАО"ЭЛЭКС"': ('Инженерные сети', 'Электроснабжение'),
    'ООО"ГрадИнвест"': ('Служебные элементы', 'Прочее'),
    'МУП"ТЕПЛОСЕТЬ г. Желзнодорожный М.О."': ('Инженерные сети', 'Теплосеть'),
    'МБУ"Благоустройство Балашиха"': ('Служебные элементы', 'Прочее'),
    'Балашиха"': ('Служебные элементы', 'Прочее'),
    'МБУ"Благоустройство': ('Служебные элементы', 'Прочее'),
    'МБУ "Благоустройство-Балашиха"': ('Служебные элементы', 'Прочее'),
    'Водозаборный узел N3': ('Инженерные сети', 'Водопровод'),
    'МУП "Балашихинский водоканал"': ('Инженерные сети', 'Водопровод')
}



def assign_mapping_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    labels = new_mapping.get(row['Text'].strip())
    if labels:
        return labels
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_mapping_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_5.19.csv', index=False)
print(f"[5.19]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[5.19]  +422  | размечено: 1127190 | осталось: 1942490


In [209]:
new_mapping = {
    'МУП"ТЕПЛОСЕТЬ': ('Инженерные сети', 'Теплосеть'),
    'г. Желзнодорожный М.О."': ('Служебные элементы', 'Штамп'),
    'владелец': ('Служебные элементы', 'Прочее'),
    'не установ.': ('Служебные элементы', 'Прочее'),
    '{\\fTimes New Roman|b0|i1|c204|p18;4}': ('Неизвестно', 'Неизвестно'),
    '9\\P{\\O0,19}': ('Служебные элементы', 'Размеры дерева'),
    'Мира 35': ('Твердые покрытия', 'Асфальт'),
    'бер.\\Pряб.': ('Озеленение', 'Дерево'),
    '{\\fTimes New Roman|b0|i1|c204|p18;12}': ('Служебные элементы', 'Размеры дерева'),
    '{\\fTimes New Roman|b0|i1|c204|p18;0.14}': ('Служебные элементы', 'Размеры дерева'),
    '{\\fTimes New Roman|b0|i1|c204|p18;3}': ('Служебные элементы', 'Размеры дерева'),
    'место стройки 2008 г.': ('Служебные элементы', 'Прочее'),
    'строй городок': ('Служебные элементы', 'Прочее'),
    'к': ('Инженерные сети', 'Канализация бытовая'),
    'г': ('Инженерные сети', 'Газоснабжение'),
    'т': ('Инженерные сети', 'Теплосеть'),
    'в': ('Инженерные сети', 'Водопровод'),
    'БАЛАШИХА- АРЕНА': ('Здания и сооружения', 'Неизвестно'),
    'ХК МВД- ЧЕМПИОН': ('МАФ', 'Информационная конструкция'),
    'зав.': ('Инженерные сети', 'Инженерная инфраструктура'),
    'рп-400': ('Инженерные сети', 'Электроснабжение'),
    'г а р а ж и': ('Здания и сооружения', 'Неизвестно'),
    'З': ('Служебные элементы', 'Прочее'),
    'Е': ('Служебные элементы', 'Прочее'),
    'провис': ('Инженерные сети', 'Электроснабжение'),
    'бет': ('Твердые покрытия', 'Бетон'),
    'к - стр': ('Инженерные сети', 'Канализация бытовая'),
    'линия    застройки': ('Служебные элементы', 'Границы'),
    'бассейн': ('Водные объекты', 'Неизвестно'),
    '16кж': ('Здания и сооружения', 'Неизвестно'),
    'зав': ('Инженерные сети', 'Инженерная инфраструктура'),
    'ЛЭП': ('Инженерные сети', 'Электроснабжение'),
    '0.20 - 0.40': ('Служебные элементы', 'Размеры дерева'),
    '15 - 30': ('Служебные элементы', 'Размеры дерева'),
    'волновод': ('Инженерные сети', 'Сети связи'),
    'Б.ПР.': ('Инженерные сети', 'Прочие кабели'),
    'А.': ('Твердые покрытия', 'Асфальт'),
    'тропинка': ('Водопроницаемые покрытия', 'Грунт'),
    'к РП 35': ('Инженерные сети', 'Электроснабжение'),
    'к РП 20': ('Инженерные сети', 'Электроснабжение'),
    'шахта проходки': ('Инженерные сети', 'Инженерная инфраструктура'),
    'усл.': ('Служебные элементы', 'Условные обозначения'),
    'л е с': ('Озеленение', 'Дерево'),
    'на РП-400': ('Инженерные сети', 'Электроснабжение'),
    'о г о р о д': ('Водопроницаемые покрытия', 'Грунт'),
    'п у с т ы р ь': ('Водопроницаемые покрытия', 'Грунт'),
    'напор.': ('Инженерные сети', 'Прочие трубопроводы'),
    'Л Е С': ('Озеленение', 'Дерево'),
    '10 - 20': ('Служебные элементы', 'Размеры дерева'),
    '0.15 - 0.20': ('Служебные элементы', 'Размеры дерева'),
    'напор': ('Инженерные сети', 'Прочие трубопроводы'),
    'ГСК " Вера"': ('Здания и сооружения', 'Неизвестно'),
    'л.': ('Служебные элементы', 'Прочее'),
    'проспект Ленина': ('Служебные элементы', 'Прочее'),
    'ту-6': ('Инженерные сети', 'Инженерная инфраструктура'),
    'БУЭС': ('Инженерные сети', 'Сети связи'),
    '152.86 в.к.': ('Инженерные сети', 'Прочие кабели'),
    '151.91 в.к.': ('Инженерные сети', 'Прочие кабели'),
    '152.51.в.к': ('Инженерные сети', 'Прочие кабели'),
    'причал': ('Здания и сооружения', 'Неизвестно'),
    'б.п.': ('Твердые покрытия', 'Бетон'),
    '25-30': ('Служебные элементы', 'Размеры дерева'),
    '0.25-0.45': ('Служебные элементы', 'Размеры дерева'),
    '25-35': ('Служебные элементы', 'Размеры дерева'),
    'пляж': ('Водопроницаемые покрытия', 'Песок'),
    'газон': ('Озеленение', 'Газон'),
    'канава  -0.50': ('Водные объекты', 'Неизвестно'),
    'Территория стройки': ('Служебные элементы', 'Прочее'),
    'яма -2.50': ('Рельеф', 'Откос'),
    ' 20-30': ('Служебные элементы', 'Размеры дерева'),
    ' 0.20-0.30': ('Служебные элементы', 'Размеры дерева'),
    ' 3-5': ('Служебные элементы', 'Размеры дерева'),
    'канализация ливневая': ('Инженерные сети', 'Ливневая канализация'),
    'теплотрасса наземная': ('Инженерные сети', 'Теплосеть'),
    'газопровод подземный': ('Инженерные сети', 'Газоснабжение'),
    'Парадный пр-д': ('Твердые покрытия', 'Асфальт'),
    '14.10.2022': ('Служебные элементы', 'Штамп'),
    '\\pt1;Бет.': ('Твердые покрытия', 'Бетон'),
    'Кадастровая граница': ('Служебные элементы', 'Кадастр'),
    'Вышка ЛЭП': ('Инженерные сети', 'Столбы и опоры'),
    '\\pt1;МН': ('Здания и сооружения', 'Неизвестно'),
    'Резервуар': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Нет доступа ': ('Служебные элементы', 'Прочее'),
    '№': ('Служебные элементы', 'Нумерация'),
    'МВД': ('Здания и сооружения', 'Неизвестно'),
    'пл.Ленина': ('Твердые покрытия', 'Плитка'),
    'ВЛ0.4 кВт': ('Инженерные сети', 'Электроснабжение'),
    'ВЛ0.4 квт': ('Инженерные сети', 'Электроснабжение'),
    'ср. д.': ('Инженерные сети', 'Газоснабжение'),
    'Ковер': ('Инженерные сети', 'Газоснабжение'),
}




def assign_mapping_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    labels = new_mapping.get(row['Text'].strip())
    if labels:
        return labels
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_mapping_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_5.20.csv', index=False)
print(f"[5.20]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[5.20]  +466  | размечено: 1127656 | осталось: 1942024


In [210]:
new_mapping = {
    'БП': ('Твердые покрытия', 'Бетон'),
    'пл.. напорн.': ('Инженерные сети', 'Прочие трубопроводы'),
    'кер': ('Инженерные сети', 'Прочие трубопроводы'),
    'з. 142,61': ('Рельеф', 'Отметки высот'),
    'ор.зам.': ('Служебные элементы', 'Прочее'),
    'з. 135,07': ('Рельеф', 'Отметки высот'),
    'з. 135,38': ('Рельеф', 'Отметки высот'),
    'з. 140,55': ('Рельеф', 'Отметки высот'),
    'з. 148,66': ('Рельеф', 'Отметки высот'),
    'з. 148,28': ('Рельеф', 'Отметки высот'),
    'з. 122,7': ('Рельеф', 'Отметки высот'),
    'фундамент': ('Твердые покрытия', 'Бетон'),
    '-0,6': ('Рельеф', 'Отметки высот'),
    'А Т С': ('Здания и сооружения', 'Неизвестно'),
    'пожарное депо': ('Здания и сооружения', 'Неизвестно'),
    'универмаг': ('Здания и сооружения', 'Неизвестно'),
    '1ДЖ': ('Здания и сооружения', 'Неизвестно'),
    'УВД города и района': ('Здания и сооружения', 'Неизвестно'),
    'КАФЕ РУСЬ': ('Здания и сооружения', 'Неизвестно'),
    'Подольский почтамп': ('Здания и сооружения', 'Неизвестно'),
    '6 корп.3': ('Здания и сооружения', 'Неизвестно'),
    '6 корп.2': ('Здания и сооружения', 'Неизвестно'),
    '6 корп.1': ('Здания и сооружения', 'Неизвестно'),
    '6 корп.5': ('Здания и сооружения', 'Неизвестно'),
    '6 корп.4': ('Здания и сооружения', 'Неизвестно'),
    '6 стр.1': ('Здания и сооружения', 'Неизвестно'),
    '2СМ': ('Здания и сооружения', 'Неизвестно'),
    'МГУПИ': ('Здания и сооружения', 'Неизвестно'),
    'МЖ': ('Здания и сооружения', 'Неизвестно'),
    'медицинское училище': ('Здания и сооружения', 'Неизвестно'),
    'Разрушенная колокольня': ('Здания и сооружения', 'Неизвестно'),
    'треб. уточн.': ('Служебные элементы', 'Прочее'),
    'Русский лес': ('Служебные элементы', 'Прочее'),
    'аптека': ('Здания и сооружения', 'Неизвестно'),
    'парн.': ('Здания и сооружения', 'Неизвестно'),
    'теплица': ('Здания и сооружения', 'Неизвестно'),
    'тепл': ('Инженерные сети', 'Теплосеть'),
    'беседка': ('Здания и сооружения', 'Неизвестно'),
    'Труба': ('Инженерные сети', 'Прочие трубопроводы'),
    'парник': ('Здания и сооружения', 'Неизвестно'),
    'не допущен': ('Служебные элементы', 'Прочее'),
    'фунд. +0,8': ('Здания и сооружения', 'Неизвестно'),
    'Ленина площадь': ('Твердые покрытия', 'Плитка'),
    'Плитка': ('Твердые покрытия', 'Плитка'),
    '3,4': ('Служебные элементы', 'Размеры'),
    'УТ-т. С': ('Инженерные сети', 'Инженерная инфраструктура'),
    'ТК-7': ('Инженерные сети', 'Инженерная инфраструктура'),
    'ТК-6': ('Инженерные сети', 'Инженерная инфраструктура'),
    'ТК-6"смотр': ('Инженерные сети', 'Инженерная инфраструктура'),
    'ТК-6\'смотр': ('Инженерные сети', 'Инженерная инфраструктура'),
    'ТК-14': ('Инженерные сети', 'Инженерная инфраструктура'),
    'УТ-1': ('Инженерные сети', 'Инженерная инфраструктура'),
    'УТ-2': ('Инженерные сети', 'Инженерная инфраструктура'),
    'пустырь': ('Водопроницаемые покрытия', 'Грунт'),
    'изр.': ('Водопроницаемые покрытия', 'Грунт'),
    'свалка': ('Служебные элементы', 'Прочее'),
    'мусоро': ('Служебные элементы', 'Прочее'),
    'сборник': ('Служебные элементы', 'Прочее'),
    'двор': ('Служебные элементы', 'Прочее'),
    'Стр.площадка': ('Служебные элементы', 'Прочее'),
    'H = 0,2': ('Служебные элементы', 'Размеры дерева'),
    'з. 139,04': ('Рельеф', 'Отметки высот'),
    'з. 137,55': ('Рельеф', 'Отметки высот'),
    'з. 133,56': ('Рельеф', 'Отметки высот'),
    'з. 142,65': ('Рельеф', 'Отметки высот'),
    '17 а': ('Здания и сооружения', 'Неизвестно'),
    '4 а': ('Здания и сооружения', 'Неизвестно'),
    '18 а': ('Здания и сооружения', 'Неизвестно'),
    'з. 141,46 ': ('Рельеф', 'Отметки высот'),
    'ГПР': ('Инженерные сети', 'Газоснабжение'),
    'н. д.': ('Служебные элементы', 'Прочее'),
    'ф.с.': ('Служебные элементы', 'Штамп'),
    '0,15': ('Служебные элементы', 'Размеры'),
    '{\\fTimes New Roman|b0|i1|c204|p18;116.53 19.09.2020}': ('Служебные элементы', 'Прочее'),
    'эл.щ.': ('Инженерные сети', 'Электроснабжение'),
    'скейт': ('Твердые покрытия', 'Асфальт'),
    'территория ФОК': ('Здания и сооружения', 'Неизвестно'),
    'Инф.': ('МАФ', 'Информационная конструкция'),
    'мет.труба': ('Инженерные сети', 'Прочие трубопроводы'),
    'Арка ': ('МАФ', 'Уличная мебель'),
    'Кафе "Сказка"': ('Здания и сооружения', 'Неизвестно'),
    'Кафе "Василек"': ('Здания и сооружения', 'Неизвестно'),
    'Арка декор': ('МАФ', 'Уличная мебель'),
    'Беседка': ('МАФ', 'Уличная мебель'),
    'разр. фунд.': ('Твердые покрытия', 'Бетон'),
    '"Мини Зоопарк"': ('МАФ', 'Игровое оборудование'),
    'Фунд.разр.': ('Твердые покрытия', 'Бетон'),
    'Кафе-автомойка "Н2О"': ('Здания и сооружения', 'Неизвестно'),
    'Мособлэнерго': ('Инженерные сети', 'Электроснабжение'),
    'Площадь военной техники': ('Твердые покрытия', 'Асфальт'),
    'Бассейн "Сатурн"': ('Водные объекты', 'Неизвестно'),
    'родник': ('Водные объекты', 'Неизвестно'),
    'декоративная арка': ('МАФ', 'Уличная мебель'),
}


def assign_mapping_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    labels = new_mapping.get(row['Text'].strip())
    if labels:
        return labels
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_mapping_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_5.21.csv', index=False)
print(f"[5.21]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[5.21]  +238  | размечено: 1127894 | осталось: 1941786


In [211]:
new_mapping = {
    'оз.Борисоглебское': ('Водные объекты', 'Неизвестно'),
    'дер.сцена': ('Здания и сооружения', 'Неизвестно'),
    'Раменское ЛТЦ': ('Инженерные сети', 'Сети связи'),
    'бет. пл.': ('Твердые покрытия', 'Бетон'),
    'А.разрушен': ('Твердые покрытия', 'Асфальт'),
    'н./д.': ('Инженерные сети', 'Инженерная инфраструктура'),
    'кж9': ('Здания и сооружения', 'Неизвестно'),
    'Вент. выход': ('Инженерные сети', 'Инженерная инфраструктура'),
    '(тр.заглушена)': ('Инженерные сети', 'Инженерная инфраструктура'),
    '(обочина)': ('Водопроницаемые покрытия', 'Грунт'),
    '(бетон)': ('Твердые покрытия', 'Бетон'),
    '(100бет)': ('Твердые покрытия', 'Бетон'),
    '(бетонплитка)': ('Твердые покрытия', 'Плитка'),
    '(тропинка)': ('Водопроницаемые покрытия', 'Грунт'),
    '(тропинкабет)': ('Твердые покрытия', 'Бетон'),
    '(ковер)': ('Инженерные сети', 'Газоснабжение'),
    '(кабстолб)': ('Инженерные сети', 'Столбы и опоры'),
    '(платформа)': ('Здания и сооружения', 'Неизвестно'),
    '(инф щит)': ('МАФ', 'Информационная конструкция'),
    '(врем 715)': ('Здания и сооружения', 'Неизвестно'),
    '(619б)': ('Твердые покрытия', 'Брусчатка'),
    '(119ц)': ('Твердые покрытия', 'Бетон'),
    '(перила)': ('МАФ', 'Ограждение'),
    '(714кон)': ('Здания и сооружения', 'Неизвестно'),
    '(nadpis)': ('МАФ', 'Информационная конструкция'),
    '(опора)': ('Инженерные сети', 'Столбы и опоры'),
    '(ирис)': ('Озеленение', 'Кустарник'),
    ' pt': ('Служебные элементы', 'Прочее'),
    ' дуб': ('Озеленение', 'Дерево'),
    ' бепеза': ('Озеленение', 'Дерево'),
    ' бепеза1': ('Озеленение', 'Дерево'),
    'Б Пл': ('Твердые покрытия', 'Бетон'),
    'лот 128.80': ('Инженерные сети', 'Инженерная инфраструктура'),
    'лот 128.89': ('Инженерные сети', 'Инженерная инфраструктура'),
    'лот 128.40': ('Инженерные сети', 'Инженерная инфраструктура'),
    'лот 128.53': ('Инженерные сети', 'Инженерная инфраструктура'),
    'лот 128.99': ('Инженерные сети', 'Инженерная инфраструктура'),
    'лот 127.66': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Пешеходный \\Pпереход': ('Элементы благоустройства', 'Дорожная разметка'),
    'Т3': ('Инженерные сети', 'Теплосеть'),
    'Т5': ('Инженерные сети', 'Теплосеть'),
    'Т4': ('Инженерные сети', 'Теплосеть'),
    'Т2': ('Инженерные сети', 'Теплосеть'),
    'Т7': ('Инженерные сети', 'Теплосеть'),
    'каб': ('Инженерные сети', 'Прочие кабели'),
    'конд.': ('Инженерные сети', 'Инженерная инфраструктура'),
    'ст100': ('Инженерные сети', 'Прочие трубопроводы'),
    'п/э300': ('Инженерные сети', 'Прочие трубопроводы'),
    'бет500': ('Инженерные сети', 'Прочие трубопроводы'),
    'водопровод перекачки': ('Инженерные сети', 'Водопровод'),
    'озеро': ('Водные объекты', 'Неизвестно'),
    '0.50': ('Служебные элементы', 'Размеры'),
    '0.60': ('Служебные элементы', 'Размеры'),
    'а/ц100': ('Инженерные сети', 'Прочие трубопроводы'),
    'ст400': ('Инженерные сети', 'Прочие трубопроводы'),
    'ст219': ('Инженерные сети', 'Прочие трубопроводы'),
    '0.90': ('Служебные элементы', 'Размеры'),
    'а/ц200': ('Инженерные сети', 'Прочие трубопроводы'),
    '0.16': ('Служебные элементы', 'Размеры'),
    '1.00': ('Служебные элементы', 'Размеры'),
    '0.41': ('Служебные элементы', 'Размеры'),
    'р.Москва': ('Водные объекты', 'Неизвестно'),
    'Т6': ('Инженерные сети', 'Теплосеть'),
    'Т101': ('Инженерные сети', 'Теплосеть'),
    '114.90лот': ('Рельеф', 'Отметки высот'),
    '111.09лот': ('Рельеф', 'Отметки высот'),
    '113.94в .тр.': ('Рельеф', 'Отметки высот'),
    '110.29лот': ('Рельеф', 'Отметки высот'),
    '113.58 в. тр.': ('Рельеф', 'Отметки высот'),
    '115.68в. тр.': ('Рельеф', 'Отметки высот'),
    '113.58в .тр.': ('Рельеф', 'Отметки высот'),
    'ст500': ('Инженерные сети', 'Прочие трубопроводы'),
    '{\\fTahoma|b0|i0|c204|p34;1}': ('Служебные элементы', 'Нумерация'),
    '{\\fTahoma|b0|i0|c204|p34;2}': ('Служебные элементы', 'Нумерация'),
    '{\\fTahoma|b0|i0|c204|p34;3}': ('Служебные элементы', 'Нумерация'),
    '{\\fTahoma|b0|i0|c204|p34;\\C5; НПУ 114,35}': ('Рельеф', 'Отметки высот'),
    '{\\fTahoma|b0|i0|c204|p34;4}': ('Служебные элементы', 'Нумерация'),
    '{\\fTahoma|b0|i0|c204|p34;\\C0;1:2,5}': ('Служебные элементы', 'Нумерация'),
    '{\\fTahoma|b0|i0|c204|p34;5}': ('Служебные элементы', 'Нумерация'),
    '{\\fTahoma|b0|i0|c204|p34;6}': ('Служебные элементы', 'Нумерация'),
    '{\\fTahoma|b0|i0|c204|p34;\\C5; НПУ 112,90}': ('Рельеф', 'Отметки высот'),
    '{\\fTahoma|b0|i0|c204|p34;7}': ('Служебные элементы', 'Нумерация'),
    '{\\fTahoma|b0|i0|c204|p34;8}': ('Служебные элементы', 'Нумерация'),
    '{\\fTahoma|b0|i0|c204|p34;9}': ('Служебные элементы', 'Нумерация'),
    '{\\fTimes New Roman|b1|i0|c204|p18;ВД}': ('Инженерные сети', 'Ливневая канализация'),
    '{\\fTimes New Roman|b1|i0|c204|p18;Вд-2}': ('Инженерные сети', 'Ливневая канализация'),
    '{\\fTimes New Roman|b1|i0|c204|p18;Вд-3}': ('Инженерные сети', 'Ливневая канализация'),
    '{\\fTimes New Roman|b1|i0|c204|p18;Вд-1}': ('Инженерные сети', 'Ливневая канализация'),
    '{\\fTimes New Roman|b1|i0|c204|p18;Сб-1}': ('Инженерные сети', 'Инженерная инфраструктура'),
    '{\\fTimes New Roman|b1|i0|c204|p18;Сб-3}': ('Инженерные сети', 'Инженерная инфраструктура'),
    '{\\fTimes New Roman|b1|i0|c204|p18;Сб-2}': ('Инженерные сети', 'Инженерная инфраструктура'),
    '{\\fTimes New Roman|b1|i0|c204|p18;В-1}': ('Инженерные сети', 'Водопровод'),
    '{\\fTahoma|b0|i0|c204|p34;\\C5; НПУ 110,70}': ('Рельеф', 'Отметки высот'),
}


def assign_mapping_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    labels = new_mapping.get(row['Text'].strip())
    if labels:
        return labels
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_mapping_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_5.22.csv', index=False)
print(f"[5.22]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[5.22]  +427  | размечено: 1128321 | осталось: 1941359


In [212]:
new_mapping = {
    '\\pxqc;{\\fTahoma|b0|i0|c204|p34;Площадь зеркала при НПУ, м2}': ('Служебные элементы', 'Прочее'),
    '\\pxqc;{\\fTahoma|b0|i0|c204|p34;Объем воды при НПУ, м3}': ('Служебные элементы', 'Прочее'),
    '\\pxqc;{\\fTahoma|b0|i0|c204|p34;Средняя глубина при НПУ, м}': ('Служебные элементы', 'Прочее'),
    '\\pxqc;{\\fTahoma|b0|i0|c204|p34;Заложение откосов подводн.}': ('Служебные элементы', 'Прочее'),
    '{\\fTahoma|b0|i0|c204|p34;Характеристика пруда № 1}': ('Служебные элементы', 'Прочее'),
    '{\\fTahoma|b0|i0|c204|p34;2015/150}': ('Служебные элементы', 'Прочее'),
    '{\\fTahoma|b0|i0|c204|p34;3035}': ('Служебные элементы', 'Прочее'),
    '\\pxqc;{\\fTahoma|b0|i0|c204|p34;2,35-2,45}': ('Служебные элементы', 'Размеры'),
    '{\\fTahoma|b0|i0|c204|p34;1:2,5}': ('Служебные элементы', 'Прочее'),
    '{\\fTahoma|b0|i0|c204|p34;5405}': ('Служебные элементы', 'Размеры'),
    '{\\fTahoma|b0|i0|c204|p34;Характеристика пруда № 2}': ('Служебные элементы', 'Прочее'),
    '{\\fTahoma|b0|i0|c204|p34;4195/225}': ('Служебные элементы', 'Прочее'),
    '{\\fTahoma|b0|i0|c204|p34;5110}': ('Служебные элементы', 'Прочее'),
    '\\pxqc;{\\fTahoma|b0|i0|c204|p34;2,30-2,40}': ('Служебные элементы', 'Прочее'),
    '{\\fTahoma|b0|i0|c204|p34;9950}': ('Служебные элементы', 'Прочее'),
    '\\pxqc;{\\fTahoma|b0|i0|c204|p34;\\C30;объем - 20 м3}': ('Служебные элементы', 'Прочее'),
    '\\pxqc;{\\fTahoma|b0|i0|c204|p34;\\C30;110,30}': ('Рельеф', 'Отметки высот'),
    '\\pxqc;{\\fTimes New Roman|b0|i0|c204|p18;\\W0.9;\\C30;112,75}': ('Рельеф', 'Отметки высот'),
    '\\pxqc;{\\fTahoma|b0|i0|c204|p34;\\C30;объем - 170 м3}': ('Служебные элементы', 'Прочее'),
    'РП-1544': ('Инженерные сети', 'Электроснабжение'),
    'подставка ': ('МАФ', 'Уличная мебель'),
    'под цветы': ('МАФ', 'Уличная мебель'),
    'эл. ящ.': ('Инженерные сети', 'Электроснабжение'),
    'атракц.': ('МАФ', 'Игровое оборудование'),
    'на дер.': ('Служебные элементы', 'Прочее'),
    'мет. ящ.': ('Инженерные сети', 'Инженерная инфраструктура'),
    'контактный зоопарк': ('Служебные элементы', 'Прочее'),
    'дор.ук.': ('МАФ', 'Дорожные знаки'),
    'полив.': ('Инженерные сети', 'Водопровод'),
    'на дом': ('Служебные элементы', 'Прочее'),
    'на ТП': ('Служебные элементы', 'Прочее'),
    'бассейн ': ('Водные объекты', 'Неизвестно'),
    'разборный': ('Служебные элементы', 'Прочее'),
    'PUK-3': ('Инженерные сети', 'Инженерная инфраструктура'),
    'мет. лоток': ('Инженерные сети', 'Инженерная инфраструктура'),
    'фунд. заб. разв.': ('Твердые покрытия', 'Бетон'),
    'шиповник 2.5': ('Озеленение', 'Кустарник'),
    'спец. оборудование': ('Инженерные сети', 'Инженерная инфраструктура'),
    'скворечник': ('МАФ', 'Уличная мебель'),
    'шумозащитный экран': ('МАФ', 'Ограждение'),
    'мкр. Дзержинец 26': ('Служебные элементы', 'Прочее'),
    'мкр. Дзержинец 24': ('Служебные элементы', 'Прочее'),
    'мкр. Дзержинец9': ('Служебные элементы', 'Прочее'),
    'мкр. Дзержинец 14': ('Служебные элементы', 'Прочее'),
    'р. Серебрянка': ('Водные объекты', 'Неизвестно'),
    '24КЖ': ('Здания и сооружения', 'Неизвестно'),
    '-0.15': ('Рельеф', 'Отметки высот'),
    'ТЦ "Вит"': ('Здания и сооружения', 'Неизвестно'),
    'терминал': ('Инженерные сети', 'Инженерная инфраструктура'),
    'въезд в': ('Твердые покрытия', 'Асфальт'),
    'подземный': ('Служебные элементы', 'Прочее'),
    'паркинг': ('Твердые покрытия', 'Асфальт'),
    'башня': ('Здания и сооружения', 'Неизвестно'),
    '1 с2': ('Здания и сооружения', 'Неизвестно'),
    '1 с1': ('Здания и сооружения', 'Неизвестно'),
    'мкр. Дзержинец 23': ('Служебные элементы', 'Прочее'),
    'мкр. Дзержинец 19': ('Служебные элементы', 'Прочее'),
    'мкр. Дзержинец 21': ('Служебные элементы', 'Прочее'),
    '1 к3': ('Здания и сооружения', 'Неизвестно'),
    '1 к2': ('Здания и сооружения', 'Неизвестно'),
    'автостанция Пушкино': ('Здания и сооружения', 'Неизвестно'),
    'зилен': ('Водные объекты', 'Неизвестно'),
    'ведутся стр. работы': ('Служебные элементы', 'Прочее'),
    'АО "РЭСС"': ('Инженерные сети', 'Электроснабжение'),
    'ООО "РЭСС"': ('Инженерные сети', 'Электроснабжение'),
    'ООО "НВП"ДИАМЕТ"': ('Служебные элементы', 'Прочее'),
    '5отв.': ('Инженерные сети', 'Прочие кабели'),
    '26отв.': ('Инженерные сети', 'Прочие кабели'),
    '11отв.': ('Инженерные сети', 'Прочие кабели'),
    '28отв.': ('Инженерные сети', 'Прочие кабели'),
    '18отв.': ('Инженерные сети', 'Прочие кабели'),
    '22отв.': ('Инженерные сети', 'Прочие кабели'),
    '30отв.': ('Инженерные сети', 'Прочие кабели'),
    'прагма 200': ('Инженерные сети', 'Прочие трубопроводы'),
    'прагма 500': ('Инженерные сети', 'Прочие трубопроводы'),
    'АО "Мострансавто"': ('Служебные элементы', 'Прочее'),
    'Автоколонна N789': ('Служебные элементы', 'Прочее'),
    'ЭЧ-4': ('Инженерные сети', 'Электроснабжение'),
    'РЦС-2': ('Инженерные сети', 'Сети связи'),
    'септ.': ('Инженерные сети', 'Канализация бытовая'),
    'ф-л ОАО "РЖД" ЦДТВ': ('Служебные элементы', 'Прочее'),
    'ф-л ОАО "РЖД"': ('Служебные элементы', 'Прочее'),
    'на рельеф': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Провис провода': ('Инженерные сети', 'Электроснабжение'),
    '{\\fTimes New Roman|c1|b0|i0;\\l\\C0; }': ('Неизвестно', 'Неизвестно'),
    '{\\fP131|c1|b1|i0;\\l\\C32;172}': ('Рельеф', 'Отметки высот'),
    '{\\fP131|c1|b1|i0;\\l\\C32;164}': ('Рельеф', 'Отметки высот'),
    '{\\fP131|c1|b1|i0;\\l\\C32;174}': ('Рельеф', 'Отметки высот'),
    '{\\fP131|c1|b1|i0;\\l\\C32;180}': ('Рельеф', 'Отметки высот'),
    '{\\fP131|c1|b1|i0;\\l\\C32;178}': ('Рельеф', 'Отметки высот'),
    '{\\fBm431|c1|b0|i1;\\l\\C5;чуг.}': ('Служебные элементы', 'Условные обозначения'),
    '{\\fBm431|c1|b0|i1;\\l\\C5;300}': ('Служебные элементы', 'Размеры'),
    '{\\fBm431|c1|b0|i1;\\l\\C5;В}': ('Инженерные сети', 'Водопровод'),
    '{\\fArial|c1|b0|i1;\\l\\C0;А}': ('Твердые покрытия', 'Асфальт'),
    '{\\fArial|c1|b0|i1;\\l\\C0;вент.}': ('Инженерные сети', 'Инженерная инфраструктура'),
    '{\\fArial|c1|b0|i1;\\l\\C0;вх.2}': ('Здания и сооружения', 'Неизвестно'),
    '{\\fArial|c1|b0|i1;\\l\\C0;стелла}': ('МАФ', 'Памятники'),
    '{\\fArial|c1|b0|i1;\\l\\C0;фонтан}': ('Водные объекты', 'Неизвестно'),
    '{\\fArial|c1|b0|i1;\\l\\C0;вх.1}': ('Здания и сооружения', 'Неизвестно')
}


def assign_mapping_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    labels = new_mapping.get(row['Text'].strip())
    if labels:
        return labels
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_mapping_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_5.23.csv', index=False)
print(f"[5.23]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[5.23]  +189  | размечено: 1128510 | осталось: 1941170


In [213]:
new_mapping = {
    '{\\fBm431|c1|b0|i1;\\l\\C14;К}': ('Инженерные сети', 'Канализация бытовая'),
    '{\\fArial|c1|b0|i1;\\l\\C0;+1,02м}': ('Рельеф', 'Отметки высот'),
    '{\\fArial|c1|b0|i1;\\l\\C0;+1,14м}': ('Рельеф', 'Отметки высот'),
    '{\\fArial|c1|b0|i1;\\l\\C0;+0,61м}': ('Рельеф', 'Отметки высот'),
    '{\\fArial|c1|b0|i1;\\l\\C0;+0,87м}': ('Рельеф', 'Отметки высот'),
    '{\\fArial|c1|b0|i1;\\l\\C0;+1,08м}': ('Рельеф', 'Отметки высот'),
    '{\\fArial|c1|b0|i1;\\l\\C0;+0,68м}': ('Рельеф', 'Отметки высот'),
    '{\\fArial|c1|b0|i1;\\l\\C0;+0,0м}': ('Рельеф', 'Отметки высот'),
    '{\\fArial|c1|b0|i1;\\l\\C0;вент. канал}': ('Инженерные сети', 'Инженерная инфраструктура'),
    '{\\fArial|c1|b0|i1;\\l\\C0;2194350}': ('Служебные элементы', 'Координаты'),
    '{\\fArial|c1|b0|i1;\\l\\C0;366700}': ('Служебные элементы', 'Координаты'),
    '{\\fArial|c1|b0|i1;\\l\\C0;366800}': ('Служебные элементы', 'Координаты'),
    '{\\fArial|c1|b0|i1;\\l\\C0;2194200}': ('Служебные элементы', 'Координаты'),
    '{\\fArial|c1|b0|i1;\\l\\C0;366600}': ('Служебные элементы', 'Координаты'),
    '{\\fArial|c1|b0|i1;\\l\\C0;2194150}': ('Служебные элементы', 'Координаты'),
    'ливневка': ('Инженерные сети', 'Ливневая канализация'),
    '\u00a0знак': ('МАФ', 'Дорожные знаки'),
    '182.41з.': ('Рельеф', 'Отметки высот'),
    '7\\P{\\O0,18}': ('Служебные элементы', 'Прочее'),
    'Гвардейская 24А': ('Здания и сооружения', 'Неизвестно'),
    'ТЦ "Авиатор"': ('Здания и сооружения', 'Неизвестно'),
    '(спланировано)': ('Служебные элементы', 'Прочее'),
    'Бр. (разр)': ('Твердые покрытия', 'Брусчатка'),
    '\u00a0АО Мособлгаз "Север" СПРЭС': ('Инженерные сети', 'Газоснабжение'),
    'А.З.': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Сергиев Посад': ('Служебные элементы', 'Прочее'),
    'Иной собственник': ('Служебные элементы', 'Прочее'),
    'частные участки': ('Служебные элементы', 'Прочее'),
    'асб160': ('Инженерные сети', 'Прочие трубопроводы'),
    'спец. оборуд.': ('Инженерные сети', 'Инженерная инфраструктура'),
    'МУП"РСО го Серебряные пруды"': ('Служебные элементы', 'Прочее'),
    'АО"МОСОБЛЭНЕРГО"': ('Инженерные сети', 'Электроснабжение'),
    'на РП-14': ('Инженерные сети', 'Электроснабжение'),
    'ливневый желоб': ('Инженерные сети', 'Ливневая канализация'),
    'на КНС': ('Инженерные сети', 'Канализация бытовая'),
    'на ТЦ': ('Инженерные сети', 'Прочие кабели'),
    'на церковь': ('Инженерные сети', 'Прочие кабели'),
    'ПАО"МОЭСК" южные эл.сети': ('Инженерные сети', 'Электроснабжение'),
    'на 12а': ('Инженерные сети', 'Прочие кабели'),
    'абонент': ('Инженерные сети', 'Прочие кабели'),
    'мет. тр.': ('Инженерные сети', 'Прочие трубопроводы'),
    'ТЦ"Серебряный"': ('Здания и сооружения', 'Неизвестно'),
    'выход газа': ('Инженерные сети', 'Газоснабжение'),
    'на пожарную часть': ('Инженерные сети', 'Прочие кабели'),
    'Гл.А.З.': ('Инженерные сети', 'Инженерная инфраструктура'),
    'МУП"РСО" г. Сергиев Посад': ('Служебные элементы', 'Прочее'),
    'МУП"РСО" ': ('Служебные элементы', 'Прочее'),
    'г. Сергиев Посад': ('Служебные элементы', 'Прочее'),
    'Серебряно-Посадская РЭС': ('Инженерные сети', 'Электроснабжение'),
    '4-5 КЖ': ('Здания и сооружения', 'Неизвестно'),
    'Бирюкова 3': ('Служебные элементы', 'Прочее'),
    'Флотская 12': ('Служебные элементы', 'Прочее'),
    'к дому №75': ('Служебные элементы', 'Прочее'),
    'к дому №59': ('Служебные элементы', 'Прочее'),
    'абонентская': ('Служебные элементы', 'Прочее'),
    'эл.шкаф': ('Инженерные сети', 'Электроснабжение'),
    'к дому №4': ('Служебные элементы', 'Прочее'),
    'ПТИ': ('Инженерные сети', 'Инженерная инфраструктура'),
    'киоск': ('МАФ', 'Павильоны'),
    'сквер': ('Служебные элементы', 'Прочее'),
    'ТЦ "Рогожский"': ('Здания и сооружения', 'Неизвестно'),
    'Россельхозбанк': ('Здания и сооружения', 'Неизвестно'),
    'ТЦ "Ногинский"': ('Здания и сооружения', 'Неизвестно'),
    'банк': ('Здания и сооружения', 'Неизвестно'),
    'пивной бар': ('Здания и сооружения', 'Неизвестно'),
    'недоступен': ('Служебные элементы', 'Прочее'),
    'ЛТЦ г. Ногинск': ('Инженерные сети', 'Сети связи'),
    'ВЭС ПАО МОЭСК': ('Инженерные сети', 'Электроснабжение'),
    '4-5КЖ': ('Здания и сооружения', 'Неизвестно'),
    'пульт': ('Инженерные сети', 'Инженерная инфраструктура'),
    'ООО"НТК"': ('Инженерные сети', 'Сети связи'),
    'КП': ('Инженерные сети', 'Инженерная инфраструктура'),
    'ЛТЦ г.Ногинск': ('Инженерные сети', 'Сети связи'),
    'ЛТЦ': ('Инженерные сети', 'Сети связи'),
    'АО "БЭС"': ('Инженерные сети', 'Электроснабжение'),
    'РУ': ('Инженерные сети', 'Электроснабжение'),
    'вдп.': ('Инженерные сети', 'Водопровод'),
    'бас.': ('Водные объекты', 'Неизвестно'),
    'ООО"БКС"': ('Служебные элементы', 'Прочее'),
    'ООО"Тепловодосервис"': ('Инженерные сети', 'Инженерная инфраструктура'),
    'ЛТЦ Ногинск': ('Инженерные сети', 'Сети связи'),
    'h=-0.65': ('Рельеф', 'Отметки высот'),
    'училище №37': ('Здания и сооружения', 'Неизвестно'),
    'срезан': ('Служебные элементы', 'Прочее'),
    'SKAM': ('МАФ', 'Уличная мебель'),
    'PEN': ('Озеленение', 'Дерево'),
}



def assign_mapping_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    labels = new_mapping.get(row['Text'].strip())
    if labels:
        return labels
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_mapping_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_5.24.csv', index=False)
print(f"[5.24]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[5.24]  +344  | размечено: 1128854 | осталось: 1940826


In [214]:
new_mapping = {
    'STD NVS': ('Инженерные сети', 'Столбы и опоры'),
    'STD NVS S': ('Инженерные сети', 'Столбы и опоры'),
    'TVP E VERX': ('Инженерные сети', 'Теплосеть'),
    'TVP S 600 VERX': ('Инженерные сети', 'Теплосеть'),
    '8*6': ('Служебные элементы', 'Размеры'),
    '3,5': ('Служебные элементы', 'Размеры'),
    '2,5': ('Служебные элементы', 'Размеры'),
    '1,5': ('Служебные элементы', 'Размеры'),
    'BOL': ('Водные объекты', 'Неизвестно'),
    'POVELENO': ('Служебные элементы', 'Прочее'),
    'огороды': ('Водопроницаемые покрытия', 'Грунт'),
    'горелый лес': ('Озеленение', 'Дерево'),
    '0902-KNV-372.1': ('Рельеф', 'Откос'),
    'ZN': ('МАФ', 'Дорожные знаки'),
    'A LEP12 1KAB': ('Инженерные сети', 'Электроснабжение'),
    'STD 4PR': ('Инженерные сети', 'Столбы и опоры'),
    'PL E BRD12': ('Твердые покрытия', 'Плитка'),
    'PL E LST S LST1 S': ('Здания и сооружения', 'Лестницы и пандусы'),
    'PL E OGRM': ('МАФ', 'Ограждение'),
    'STD KAB VZM 4PR': ('Инженерные сети', 'Столбы и опоры'),
    'OP': ('Инженерные сети', 'Столбы и опоры'),
    'CNT CR BRUS\'YA': ('МАФ', 'Спортивное оборудование'),
    'CNT BRUS\'YA': ('МАФ', 'Спортивное оборудование'),
    'CNT S BRUS\'YA': ('МАФ', 'Спортивное оборудование'),
    'OP CNT E TURNIK': ('МАФ', 'Спортивное оборудование'),
    'OP CNT S TURNIK': ('МАФ', 'Спортивное оборудование'),
    'DNO': ('Рельеф', 'Откос'),
    'TN12 2 X 200': ('Инженерные сети', 'Теплосеть'),
    'TN VERX 3 X 200': ('Инженерные сети', 'Теплосеть'),
    'цем. пл.': ('Твердые покрытия', 'Бетон'),
    '2 Д': ('Здания и сооружения', 'Неизвестно'),
    'Екатерининская церковь': ('Здания и сооружения', 'Неизвестно'),
    '0.04': ('Служебные элементы', 'Размеры'),
    '0.01': ('Служебные элементы', 'Размеры'),
    '0.02': ('Служебные элементы', 'Размеры'),
    '0.03': ('Служебные элементы', 'Размеры'),
    'по мосту': ('Служебные элементы', 'Прочее'),
    '12 отв.': ('Инженерные сети', 'Прочие кабели'),
    'Яма\u00a0': ('Рельеф', 'Откос'),
    'OPB': ('Инженерные сети', 'Столбы и опоры'),
    'STD AGIT': ('МАФ', 'Информационная конструкция'),
    'VODA': ('Инженерные сети', 'Водопровод'),
    'NA LDU': ('Служебные элементы', 'Прочее'),
    '-1.0 м': ('Рельеф', 'Отметки высот'),
    'тепл.': ('Инженерные сети', 'Теплосеть'),
    '4,5': ('Служебные элементы', 'Размеры'),
    'VISHNYA': ('Озеленение', 'Дерево'),
    'волейбольная': ('МАФ', 'Спортивное оборудование'),
    'пересох.': ('Водные объекты', 'Неизвестно'),
    'CMT S PLITA': ('Твердые покрытия', 'Бетон'),
    'ШРП': ('Инженерные сети', 'Газоснабжение'),
    'Прибрежный п-к': ('Твердые покрытия', 'Асфальт'),
    '7-й Прибрежный п-к': ('Твердые покрытия', 'Асфальт'),
    '-0,4': ('Рельеф', 'Отметки высот'),
    'оз.Сенеж': ('Водные объекты', 'Неизвестно'),
    'мет.решетка': ('Инженерные сети', 'Ливневая канализация'),
    'Часовня': ('Здания и сооружения', 'Неизвестно'),
    '\\P 176': ('Рельеф', 'Отметки высот'),
    '\\P 179': ('Рельеф', 'Отметки высот'),
    '\u00a0t': ('Неизвестно', 'Неизвестно'),
    '\\P 174': ('Рельеф', 'Отметки высот'),
    'Ц разр.': ('Твердые покрытия', 'Плитка'),
    'м ящ': ('Инженерные сети', 'Инженерная инфраструктура'),
    'канава-0,5': ('Водные объекты', 'Неизвестно'),
    'канава-0,9': ('Водные объекты', 'Неизвестно'),
    'канава-0,6': ('Водные объекты', 'Неизвестно'),
    'А разр': ('Твердые покрытия', 'Асфальт'),
    'Ц\u00a0': ('Твердые покрытия', 'Плитка'),
    'ц фунд.разр.': ('Твердые покрытия', 'Бетон'),
    'ТАРТАР': ('Твердые покрытия', 'Резиновая крошка'),
    'тенисный стол': ('МАФ', 'Спортивное оборудование'),
    'рампа': ('МАФ', 'Спортивное оборудование'),
    'А кр': ('Твердые покрытия', 'Асфальт'),
    'железная дорога': ('Элементы благоустройства', 'ЖД пути'),
    'лежаки': ('МАФ', 'Уличная мебель'),
    'СНТ Машиностроитель 1': ('Служебные элементы', 'Прочее'),
    'канава-0,3': ('Водные объекты', 'Неизвестно'),
    'канава-0,2': ('Водные объекты', 'Неизвестно'),
    'дер.настил': ('Водопроницаемые покрытия', 'Деревянный настил'),
    'КНразр.': ('Здания и сооружения', 'Неизвестно'),
    'Административное': ('Здания и сооружения', 'Неизвестно'),
    'агит': ('МАФ', 'Информационная конструкция'),
    'заболоченость': ('Водные объекты', 'Неизвестно'),
    'канава-0.4': ('Водные объекты', 'Неизвестно'),
    'Поле для регби': ('Водопроницаемые покрытия', 'Газон спортивный'),
}


def assign_mapping_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    labels = new_mapping.get(row['Text'].strip())
    if labels:
        return labels
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_mapping_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_5.25.csv', index=False)
print(f"[5.25]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[5.25]  +189  | размечено: 1129043 | осталось: 1940637


In [215]:
new_mapping = {
    'футбольное поле': ('Водопроницаемые покрытия', 'Газон спортивный'),
    'СУ-25': ('МАФ', 'Памятники'),
    '3к1': ('Здания и сооружения', 'Неизвестно'),
    'фгагшт.': ('МАФ', 'Флагштоки'),
    'ЦРПЛ': ('Инженерные сети', 'Электроснабжение'),
    '10кВx2': ('Инженерные сети', 'Прочие кабели'),
    'а/ц150': ('Инженерные сети', 'Прочие трубопроводы'),
    'а/ц300': ('Инженерные сети', 'Прочие трубопроводы'),
    'а/ц250': ('Инженерные сети', 'Прочие трубопроводы'),
    '-1.6': ('Рельеф', 'Отметки высот'),
    'гильзы': ('Инженерные сети', 'Инженерная инфраструктура'),
    'линия сводки': ('Служебные элементы', 'Границы'),
    '\u00a0клён 15': ('Озеленение', 'Дерево'),
    '\\P 143': ('Рельеф', 'Отметки высот'),
    '\\P 144': ('Рельеф', 'Отметки высот'),
    '\u00a0мет труба': ('Инженерные сети', 'Прочие трубопроводы'),
    '\u00a0мет столб': ('Инженерные сети', 'Столбы и опоры'),
    '\u00a0флагшток': ('МАФ', 'Флагштоки'),
    '\u00a0дорога': ('Твердые покрытия', 'Асфальт'),
    '\u00a0бордюр': ('Элементы благоустройства', 'Бортовой камень'),
    '\u00a0забор мет': ('МАФ', 'Ограждение'),
    '\u00a0сторожевой знак': ('МАФ', 'Информационная конструкция'),
    '\u00a0газ': ('Инженерные сети', 'Газоснабжение'),
    '\u00a0знак дор': ('МАФ', 'Дорожные знаки'),
    'столы для тенниса': ('МАФ', 'Спортивное оборудование'),
    'желоб': ('Инженерные сети', 'Ливневая канализация'),
    'фон бет': ('Инженерные сети', 'Столбы и опоры'),
    'ц. пл.': ('Твердые покрытия', 'Плитка'),
    'АО "Мытищинская теплосеть"': ('Инженерные сети', 'Теплосеть'),
    'труба мет 30': ('Инженерные сети', 'Прочие трубопроводы'),
    'вход в землю': ('Инженерные сети', 'Инженерная инфраструктура'),
    'гтс^Jзатоп^Jдо воды -1.47': ('Инженерные сети', 'Сети связи'),
    'гтс^Jзатоп.^Jдо воды -1.83': ('Инженерные сети', 'Сети связи'),
    '^Jн.к.-1.18': ('Инженерные сети', 'Канализация бытовая'),
    '-1': ('Рельеф', 'Отметки высот'),
    '-0.8': ('Рельеф', 'Отметки высот'),
    'дек.': ('Озеленение', 'Цветник'),
    'вх.в зем.': ('Инженерные сети', 'Инженерная инфраструктура'),
    'дер.наст.': ('Водопроницаемые покрытия', 'Деревянный настил'),
    'Акведук': ('Здания и сооружения', 'Неизвестно'),
    '5к.': ('Инженерные сети', 'Прочие кабели'),
    'Решетка': ('Инженерные сети', 'Ливневая канализация'),
    'Сцена дер.': ('Водопроницаемые покрытия', 'Деревянный настил'),
    'Газонная решетка': ('Озеленение', 'Газонная решетка'),
    'ц.пл.': ('Твердые покрытия', 'Плитка'),
    'Игровое поле': ('Твердые покрытия', 'Резиновая крошка'),
    'ТКО': ('МАФ', 'Павильон'),
    'Иск.газон': ('Водопроницаемые покрытия', 'Газон спортивный'),
    'Вход труб': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Разр.': ('Твердые покрытия', 'Асфальт'),
    'Дер.пантон': ('Здания и сооружения', 'Неизвестно'),
    'р.Яуза': ('Водные объекты', 'Неизвестно'),
    'канава-0.7м.': ('Водные объекты', 'Неизвестно'),
    'V.2025': ('Служебные элементы', 'Прочее'),
    'Полив': ('Инженерные сети', 'Водопровод'),
    'до воды154.30': ('Рельеф', 'Отметки высот'),
    'низ 147.73': ('Рельеф', 'Отметки высот'),
    'низ147.63': ('Рельеф', 'Отметки высот'),
    'низ 147.88': ('Рельеф', 'Отметки высот'),
    'низ 147.27': ('Рельеф', 'Отметки высот'),
    'низ 146.51': ('Рельеф', 'Отметки высот'),
    'затоп.': ('Инженерные сети', 'Инженерная инфраструктура'),
    'низ 147.69': ('Рельеф', 'Отметки высот'),
    'до воды 146.80': ('Рельеф', 'Отметки высот'),
    'до воды 146.35': ('Рельеф', 'Отметки высот'),
    'до воды147.87': ('Рельеф', 'Отметки высот'),
    'до воды144.61': ('Рельеф', 'Отметки высот'),
    'низ148.29': ('Рельеф', 'Отметки высот'),
    'до воды145.78': ('Рельеф', 'Отметки высот'),
    'до воды144.74': ('Рельеф', 'Отметки высот'),
    'Луховня': ('Служебные элементы', 'Прочее'),
    'Дубровицы': ('Служебные элементы', 'Прочее'),
    'канава -1,0': ('Водные объекты', 'Неизвестно'),
    'канава-1.0': ('Водные объекты', 'Неизвестно'),
    'канава-1.5': ('Водные объекты', 'Неизвестно'),
    'канава-1.2': ('Водные объекты', 'Неизвестно'),
    'RP1 156.33': ('Рельеф', 'Отметки высот'),
    'RP3 162.51': ('Рельеф', 'Отметки высот'),
    'RP2 172.23': ('Рельеф', 'Отметки высот'),
    'врем.не действ.': ('Служебные элементы', 'Прочее'),
    'Воздух': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Связь': ('Инженерные сети', 'Сети связи'),
    'Тен.стол': ('МАФ', 'Спортивное оборудование'),
    'труба': ('Инженерные сети', 'Прочие трубопроводы'),
    '"Магнит"': ('Здания и сооружения', 'Неизвестно'),
    'Вентиляционная\u00a0': ('Инженерные сети', 'Инженерная инфраструктура'),
    'система': ('Инженерные сети', 'Инженерная инфраструктура'),
    'флагштог': ('МАФ', 'Флагштоки'),
    'зас.': ('Служебные элементы', 'Прочее'),
    'зак.': ('Служебные элементы', 'Прочее'),
    'зазем..': ('Инженерные сети', 'Электроснабжение'),
    'зак..': ('Служебные элементы', 'Прочее'),
    'ТП N99': ('Инженерные сети', 'Электроснабжение'),
    'РЩ': ('Инженерные сети', 'Электроснабжение'),
    'канава -0,5': ('Водные объекты', 'Неизвестно'),
    'цем.лоток 0,5': ('Инженерные сети', 'Ливневая канализация'),
    'канава -0.5': ('Водные объекты', 'Неизвестно'),
}


def assign_mapping_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    labels = new_mapping.get(row['Text'].strip())
    if labels:
        return labels
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_mapping_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_5.26.csv', index=False)
print(f"[5.26]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[5.26]  +369  | размечено: 1129412 | осталось: 1940268


In [216]:
new_mapping = {
    'канава -0,3': ('Водные объекты', 'Неизвестно'),
    'Газ': ('Инженерные сети', 'Газоснабжение'),
    '225.96-ПГ': ('Инженерные сети', 'Водопровод'),
    'кат.защ.': ('Инженерные сети', 'Инженерная инфраструктура'),
    'анодная': ('Инженерные сети', 'Инженерная инфраструктура'),
    'защита': ('Инженерные сети', 'Инженерная инфраструктура'),
    '225.35-з.': ('Рельеф', 'Отметки высот'),
    '224.43-з.': ('Рельеф', 'Отметки высот'),
    'АО "Мособлэнерго" ориент.': ('Инженерные сети', 'Электроснабжение'),
    'ПС-105': ('Инженерные сети', 'Электроснабжение'),
    'нанесен ориентировочно': ('Служебные элементы', 'Прочее'),
    'АО "Мособлэнерго" нанесен ориент.': ('Инженерные сети', 'Электроснабжение'),
    'п/э 250 ': ('Инженерные сети', 'Прочие трубопроводы'),
    'на д.N 205-2': ('Инженерные сети', 'Прочие трубопроводы'),
    '228.95-ПГ': ('Рельеф', 'Отметки высот'),
    'иной собственник': ('Служебные элементы', 'Прочее'),
    'Строймусор': ('Служебные элементы', 'Прочее'),
    'Стройка': ('Служебные элементы', 'Прочее'),
    'лоток-0.05': ('Инженерные сети', 'Инженерная инфраструктура'),
    'до воды223.41': ('Рельеф', 'Отметки высот'),
    'до воды223.97': ('Рельеф', 'Отметки высот'),
    'зотоп.': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Профсоюзный комитет': ('Здания и сооружения', 'Неизвестно'),
    'Парикмахерская': ('Здания и сооружения', 'Неизвестно'),
    'Корпорация тактического ракетного вооружения': ('Здания и сооружения', 'Неизвестно'),
    'мет.столбики': ('МАФ', 'Ограждение'),
    'Пруд Костино': ('Водные объекты', 'Неизвестно'),
    'Араз.': ('Твердые покрытия', 'Асфальт'),
    'Оранжерея': ('Здания и сооружения', 'Неизвестно'),
    'ваза': ('МАФ', 'Вазоны'),
    'Церковь': ('Здания и сооружения', 'Неизвестно'),
    '160.45-ПГ': ('Рельеф', 'Отметки высот'),
    'колодец гашения': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Столовая': ('Здания и сооружения', 'Неизвестно'),
    'полив': ('Инженерные сети', 'Водопровод'),
    'заброшен': ('Служебные элементы', 'Прочее'),
    'АО "МСК-Энерго"': ('Инженерные сети', 'Электроснабжение'),
    'АО "МС-Энерго"': ('Инженерные сети', 'Электроснабжение'),
    'футляр 500': ('Инженерные сети', 'Инженерная инфраструктура'),
    ' G7 new': ('Служебные элементы', 'Прочее'),
    ' G7 old': ('Служебные элементы', 'Прочее'),
    '8к.': ('Инженерные сети', 'Прочие кабели'),
    '7к.': ('Инженерные сети', 'Прочие кабели'),
    'канава-0.5': ('Водные объекты', 'Неизвестно'),
    'Огород': ('Водопроницаемые покрытия', 'Грунт'),
    'канава-0.8': ('Водные объекты', 'Неизвестно'),
    'канава-0.2': ('Водные объекты', 'Неизвестно'),
    'выход из земли': ('Инженерные сети', 'Инженерная инфраструктура'),
    ' троп': ('Водопроницаемые покрытия', 'Грунт'),
    ' троп(плав)': ('Водопроницаемые покрытия', 'Грунт'),
    ' tp': ('Служебные элементы', 'Прочее'),
    ' Заб раб': ('МАФ', 'Ограждение'),
    ' откос': ('Рельеф', 'Откос'),
    ' Заб мет': ('МАФ', 'Ограждение'),
    ' Заб дер': ('МАФ', 'Ограждение'),
    ' конт мет': ('Здания и сооружения', 'Неизвестно'),
    ' гар мет без выс ': ('Здания и сооружения', 'Неизвестно'),
    ' бет плит': ('Твердые покрытия', 'Бетон'),
    ' rp 22.03.03': ('Служебные элементы', 'Прочее'),
    ' cтолб бет': ('Инженерные сети', 'Столбы и опоры'),
    ' cтолб мет': ('Инженерные сети', 'Столбы и опоры'),
    ' забор дер': ('МАФ', 'Ограждение'),
    ' забор дер стык': ('МАФ', 'Ограждение'),
    ' фонарь бет': ('Инженерные сети', 'Столбы и опоры'),
    ' Заб раб стык': ('МАФ', 'Ограждение'),
    ' Заб бет': ('МАФ', 'Ограждение'),
    ' дом плит бет': ('Здания и сооружения', 'Неизвестно'),
    ' Забраб ': ('МАФ', 'Ограждение'),
    ' Заб раб ': ('МАФ', 'Ограждение'),
    ' Заб раб стык ': ('МАФ', 'Ограждение'),
    ' дор бет пллит ': ('Твердые покрытия', 'Бетон'),
    ' Заб мет стык': ('МАФ', 'Ограждение'),
    ' ворота': ('МАФ', 'Ворота'),
    ' Заб бет стык': ('МАФ', 'Ограждение'),
    ' фонарь мет': ('Инженерные сети', 'Столбы и опоры'),
    '\\P 188': ('Рельеф', 'Отметки высот'),
    ' тротуар': ('Твердые покрытия', 'Асфальт'),
    ' парковка': ('Твердые покрытия', 'Асфальт'),
    ' ящик мет': ('Инженерные сети', 'Инженерная инфраструктура'),
    ' кабель': ('Инженерные сети', 'Прочие кабели'),
    ' тп': ('Инженерные сети', 'Электроснабжение'),
    ' мусорка': ('МАФ', 'Павильон'),
    ' тротуар бет плит': ('Твердые покрытия', 'Бетон'),
    ' агит': ('МАФ', 'Информационная конструкция'),
    '\\P 189': ('Рельеф', 'Отметки высот'),
    ' фон бет': ('Инженерные сети', 'Столбы и опоры'),
    ' т': ('Инженерные сети', 'Теплосеть'),
    '\\P 191': ('Рельеф', 'Отметки высот'),
    '\\P 192': ('Рельеф', 'Отметки высот'),
    'Лобня г, Окружная ул, 13': ('Служебные элементы', 'Прочее'),
    'Лобня г, Окружная ул, 11к2': ('Служебные элементы', 'Прочее'),
    'Лобня г, Окружная ул, 11к4': ('Служебные элементы', 'Прочее'),
    'Лобня г, Окружная ул, 11к3': ('Служебные элементы', 'Прочее')
}


def assign_mapping_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    labels = new_mapping.get(row['Text'].strip())
    if labels:
        return labels
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_mapping_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_5.27.csv', index=False)
print(f"[5.27]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[5.27]  +100  | размечено: 1129512 | осталось: 1940168


In [217]:
new_mapping = {
    'Лобня г, Окружная ул, 11к1': ('Служебные элементы', 'Прочее'),
    'Лобня г, Калинина ул, 32': ('Служебные элементы', 'Прочее'),
    'Лобня г, Калинина ул, 21': ('Служебные элементы', 'Прочее'),
    'Лобня г, Окружная ул, 1': ('Служебные элементы', 'Прочее'),
    ' сторж каб': ('Инженерные сети', 'Прочие кабели'),
    ' t другая': ('Служебные элементы', 'Прочее'),
    ' t ': ('Служебные элементы', 'Прочее'),
    ' свалка снега рельеф': ('Служебные элементы', 'Прочее'),
    ' свалка снега релье': ('Служебные элементы', 'Прочее'),
    ' свалка снега рельеф_': ('Служебные элементы', 'Прочее'),
    '\\P 187': ('Рельеф', 'Отметки высот'),
    '\\P 184': ('Рельеф', 'Отметки высот'),
    '\\P 186': ('Рельеф', 'Отметки высот'),
    ' yas 15x2': ('Озеленение', 'Дерево'),
    '\\P 190': ('Рельеф', 'Отметки высот'),
    ' тахик': ('Служебные элементы', 'Прочее'),
    '{\\Q0;\\W0.8;190}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W0.8;192}': ('Рельеф', 'Отметки высот'),
    '{\\Q0;\\W0.8;188}': ('Рельеф', 'Отметки высот'),
    '20ст': ('Инженерные сети', 'Прочие трубопроводы'),
    '150ст': ('Инженерные сети', 'Прочие трубопроводы'),
    'ориенир.': ('Служебные элементы', 'Прочее'),
    'УП-4': ('Инженерные сети', 'Инженерная инфраструктура'),
    'L=43м': ('Служебные элементы', 'Размеры'),
    'УП-5': ('Инженерные сети', 'Инженерная инфраструктура'),
    ' ц пл': ('Твердые покрытия', 'Плитка'),
    ' заб мет/ворота': ('МАФ', 'Ограждение'),
    ' борд': ('Элементы благоустройства', 'Бортовой камень'),
    ' борд/ц пл': ('Элементы благоустройства', 'Бортовой камень'),
    ' зеак дор': ('МАФ', 'Дорожные знаки'),
    ' ворота/заб бет': ('МАФ', 'Ворота'),
    ' звб бет': ('МАФ', 'Ограждение'),
    ' заб бет': ('МАФ', 'Ограждение'),
    ' ц пл/заб бет/заб мет': ('Твердые покрытия', 'Плитка'),
    ' заб мет/кн': ('МАФ', 'Ограждение'),
    ' пл': ('Твердые покрытия', 'Плитка'),
    ' кн/заб мет': ('Здания и сооружения', 'Неизвестно'),
    ' юорд': ('Элементы благоустройства', 'Бортовой камень'),
    ' кип газ': ('Инженерные сети', 'Газоснабжение'),
    ' мн': ('Здания и сооружения', 'Неизвестно'),
    ' ц': ('Твердые покрытия', 'Плитка'),
    ' фон мет': ('Инженерные сети', 'Столбы и опоры'),
    ' ц пл/спец/заб мет': ('Твердые покрытия', 'Плитка'),
    ' ц пл/борд': ('Твердые покрытия', 'Плитка'),
    ' смешанное покрытие': ('Водопроницаемые покрытия', 'Щебень'),
    ' щнб': ('Водопроницаемые покрытия', 'Щебень'),
    ' спец/заб мет': ('Здания и сооружения', 'Неизвестно'),
    ' лавояка': ('МАФ', 'Уличная мебель'),
    ' мсорка': ('МАФ', 'Уличная мебель'),
    ' дек': ('Озеленение', 'Цветник'),
    ' ц разр': ('Твердые покрытия', 'Плитка'),
    ' лестн': ('Здания и сооружения', 'Лестницы и пандусы'),
    ' ступень': ('Здания и сооружения', 'Лестницы и пандусы'),
    ' дер настил': ('Водопроницаемые покрытия', 'Деревянный настил'),
    ' дер настил/бет': ('Водопроницаемые покрытия', 'Деревянный настил'),
    ' бет/дер настил': ('Твердые покрытия', 'Бетон'),
    ' дер настил/заб мет': ('Водопроницаемые покрытия', 'Деревянный настил'),
    ' дер настил/Заб мет': ('Водопроницаемые покрытия', 'Деревянный настил'),
    ' дер наст/заб мет': ('Водопроницаемые покрытия', 'Деревянный настил'),
    'фон мет': ('Инженерные сети', 'Столбы и опоры'),
    'заб мет': ('МАФ', 'Ограждение'),
    'ц пл/газон': ('Твердые покрытия', 'Плитка'),
    'фон мет с мя': ('Инженерные сети', 'Столбы и опоры'),
    'ц пл/бет': ('Твердые покрытия', 'Плитка'),
    'ц пл': ('Твердые покрытия', 'Плитка'),
    'ц пл/1кн': ('Твердые покрытия', 'Плитка'),
    'мя': ('Инженерные сети', 'Инженерная инфраструктура'),
    'столб мет': ('Инженерные сети', 'Столбы и опоры'),
    'желоб/бордюр z': ('Инженерные сети', 'Ливневая канализация'),
    'желоб/ц пл z': ('Инженерные сети', 'Ливневая канализация'),
    '2мн z': ('Здания и сооружения', 'Неизвестно'),
    'желоб/ц пл': ('Инженерные сети', 'Ливневая канализация'),
    'желоб/бордюр': ('Инженерные сети', 'Ливневая канализация'),
    'низ отк': ('Рельеф', 'Откос'),
    'бордюр': ('Элементы благоустройства', 'Бортовой камень'),
    'тко мет': ('МАФ', 'Павильон'),
    'бордюр ': ('Элементы благоустройства', 'Бортовой камень'),
    'площадка ': ('Твердые покрытия', 'Асфальт'),
    'р': ('Водные объекты', 'Неизвестно'),
    'заб раб': ('МАФ', 'Ограждение'),
    'провода выход из земли': ('Инженерные сети', 'Электроснабжение'),
    'бордюр/желоб': ('Элементы благоустройства', 'Бортовой камень'),
    'решетка': ('Инженерные сети', 'Ливневая канализация'),
    'дер наст': ('Водопроницаемые покрытия', 'Деревянный настил'),
    'Ц пл': ('Твердые покрытия', 'Плитка'),
    'игр. инвент.': ('МАФ', 'Игровое оборудование'),
    'игр. инвен.': ('МАФ', 'Игровое оборудование'),
    'игр инвен': ('МАФ', 'Игровое оборудование'),
    'ц. разр.': ('Твердые покрытия', 'Плитка'),
    'ц.разр.': ('Твердые покрытия', 'Плитка'),
    'спец. соор.': ('Здания и сооружения', 'Неизвестно'),
    'C': ('Инженерные сети', 'Дренаж'),
    'rp 2': ('Рельеф', 'Отметки высот'),
    ' -2.79': ('Рельеф', 'Отметки высот'),
    'затоп|до воды-1.34': ('Рельеф', 'Отметки высот'),
    'затоп|до воды -2.75': ('Рельеф', 'Отметки высот'),
    '-2.95': ('Рельеф', 'Отметки высот'),
    '-1.14': ('Служебные элементы', 'Прочее'),
}



def assign_mapping_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    labels = new_mapping.get(row['Text'].strip())
    if labels:
        return labels
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_mapping_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_5.28.csv', index=False)
print(f"[5.28]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[5.28]  +417  | размечено: 1129929 | осталось: 1939751


In [218]:
new_mapping = {
    '-1.20': ('Рельеф', 'Отметки высот'),
    'затоп.|до воды -0.88': ('Рельеф', 'Отметки высот'),
    'дек': ('Озеленение', 'Цветник'),
    'цистерна': ('Инженерные сети', 'Инженерная инфраструктура'),
    'закоп.': ('Инженерные сети', 'Инженерная инфраструктура'),
    '13к.': ('Инженерные сети', 'Прочие кабели'),
    'газон и плитка': ('Твердые покрытия', 'Плитка'),
    'вход.из земли': ('Инженерные сети', 'Инженерная инфраструктура'),
    'пруд Москвич': ('Водные объекты', 'Неизвестно'),
    'охрана': ('Здания и сооружения', 'Неизвестно'),
    'ДУ': ('МАФ', 'Дорожные знаки'),
    'ДЗ': ('МАФ', 'Дорожные знаки'),
    'авт. ост.': ('МАФ', 'Павильон'),
    'шиномонтаж': ('Здания и сооружения', 'Неизвестно'),
    'прокат': ('Здания и сооружения', 'Неизвестно'),
    'флаг': ('МАФ', 'Флагштоки'),
    'N 37А': ('Служебные элементы', 'Нумерация'),
    'флаги': ('МАФ', 'Флагштоки'),
    'ВЗУ': ('Инженерные сети', 'Водопровод'),
    'ур.': ('МАФ', 'Уличная мебель'),
    'спасательный': ('Служебные элементы', 'Прочее'),
    'пост': ('Здания и сооружения', 'Неизвестно'),
    'стол': ('МАФ', 'Уличная мебель'),
    'р. Банька': ('Водные объекты', 'Неизвестно'),
    'р. Синичка': ('Водные объекты', 'Неизвестно'),
    'МЕТ.': ('Твердые покрытия', 'Металлический настил'),
    'пож.щ.': ('Инженерные сети', 'Инженерная инфраструктура'),
    '161.60в.рт.': ('Инженерные сети', 'Прочие кабели'),
    'ПАО "Россети МР"': ('Инженерные сети', 'Электроснабжение'),
    'АО"Мосгаз"': ('Инженерные сети', 'Газоснабжение'),
    'h-1.2 2d50': ('Инженерные сети', 'Прочие трубопроводы'),
    'СНТ "Садовод-сад-3"': ('Служебные элементы', 'Прочее'),
    'Q': ('Служебные элементы', 'Прочее'),
    'B': ('Служебные элементы', 'Прочее'),
    'D': ('Служебные элементы', 'Прочее'),
    'R': ('Служебные элементы', 'Прочее'),
    '\'': ('Служебные элементы', 'Прочее'),
    '+': ('Служебные элементы', 'Прочее'),
    'купель': ('Водные объекты', 'Неизвестно'),
    'замур': ('Инженерные сети', 'Инженерная инфраструктура'),
    'забит': ('Инженерные сети', 'Инженерная инфраструктура'),
    'асб300': ('Инженерные сети', 'Прочие трубопроводы'),
    'асб100': ('Инженерные сети', 'Прочие трубопроводы'),
    'рез.п.': ('Твердые покрытия', 'Резиновая крошка'),
    'h опоры 9,03': ('Инженерные сети', 'Столбы и опоры'),
    'h опоры 9,06': ('Инженерные сети', 'Столбы и опоры'),
    'h опоры 9,01': ('Инженерные сети', 'Столбы и опоры'),
    'h опоры 8,97': ('Инженерные сети', 'Столбы и опоры'),
    'h опоры 9,02': ('Инженерные сети', 'Столбы и опоры'),
    'h опоры 8,94': ('Инженерные сети', 'Столбы и опоры'),
    'h опоры 8,87': ('Инженерные сети', 'Столбы и опоры'),
    'h опоры 8,86': ('Инженерные сети', 'Столбы и опоры'),
    'h опоры 8,91': ('Инженерные сети', 'Столбы и опоры'),
    'h опоры 8,99': ('Инженерные сети', 'Столбы и опоры'),
    'h опоры 9,09': ('Инженерные сети', 'Столбы и опоры'),
    'времен.': ('Служебные элементы', 'Прочее'),
    'строительные работы': ('Служебные элементы', 'Прочее'),
    'строительные \\Pработы': ('Служебные элементы', 'Прочее'),
    'работает эвакуатор': ('МАФ', 'Дорожные знаки'),
    '{\\fArial Narrow|b1|i0|c204|p34;20}': ('Служебные элементы', 'Прочее'),
    '{\\fArial Narrow|b1|i0|c204|p34;40}': ('Служебные элементы', 'Прочее'),
    'искусств.': ('Служебные элементы', 'Прочее'),
    'камень': ('Твердые покрытия', 'Камень'),
    'гр.пл.': ('Твердые покрытия', 'Гранит'),
    'эл.ящик': ('Инженерные сети', 'Электроснабжение'),
    'авт.ост.': ('МАФ', 'Павильон'),
    'участок 32': ('Служебные элементы', 'Условные обозначение'),
    'участок 27а': ('Служебные элементы', 'Условные обозначение'),
    'участок 25': ('Служебные элементы', 'Условные обозначение'),
    'забор разр.': ('МАФ', 'Ограждение'),
    'навис.': ('Здания и сооружения', 'Неизвестно'),
    'участок 10а': ('Служебные элементы', 'Условные обозначение'),
    'молниеотвод': ('Инженерные сети', 'Инженерная инфраструктура'),
    'участок 4': ('Служебные элементы', 'Условные обозначение'),
    'участок 6а': ('Служебные элементы', 'Условные обозначение'),
    'врем.': ('Служебные элементы', 'Прочее'),
    '1-ая Советская 13': ('Служебные элементы', 'Прочее'),
    'заб.разр.': ('МАФ', 'Ограждение'),
    'наст.': ('Водопроницаемые покрытия', 'Деревянный настил'),
    'стоянка': ('Твердые покрытия', 'Асфальт'),
    'заб.строящ.': ('МАФ', 'Ограждение'),
    'искусств.камень': ('Твердые покрытия', 'Камень'),
    'теплотрасса': ('Инженерные сети', 'Теплосеть'),
    '4СН': ('Здания и сооружения', 'Неизвестно'),
    'ОАО "Русский районный\nспецкомбинат"': ('Служебные элементы', 'Прочее'),
    'машинка': ('МАФ', 'Игровое оборудование'),
    'теремок': ('МАФ', 'Игровое оборудование'),
    'домик утки': ('МАФ', 'Игровое оборудование'),
    'ГРП N1': ('Инженерные сети', 'Газоснабжение'),
    'КНП': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Мед. училище': ('Здания и сооружения', 'Неизвестно'),
    'Волоколамское ш.': ('Служебные элементы', 'Прочее'),
    'пер. Урицкого': ('Твердые покрытия', 'Асфальт'),
    'Демократический пер.': ('Твердые покрытия', 'Асфальт'),
    'ст.4*50': ('Инженерные сети', 'Прочие трубопроводы'),
    'ГКУ Пожарная Часть N 312': ('Служебные элементы', 'Прочее'),
    '-2.70': ('Рельеф', 'Отметки высот')
}




def assign_mapping_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    labels = new_mapping.get(row['Text'].strip())
    if labels:
        return labels
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_mapping_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_5.29.csv', index=False)
print(f"[5.29]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[5.29]  +233  | размечено: 1130162 | осталось: 1939518


In [219]:
new_mapping = {
    '50:40:0010311': ('Служебные элементы', 'Кадастр'),
    'р. Десна': ('Водные объекты', 'Неизвестно'),
    'МУП Водоканал': ('Инженерные сети', 'Водопровод'),
    'а\\д 46Н-13981': ('Служебные элементы', 'Прочее'),
    'на д. Луковня': ('Служебные элементы', 'Прочее'),
    'на Подольск': ('Служебные элементы', 'Прочее'),
    'Бетонные плиты': ('Твердые покрытия', 'Бетон'),
    'Камень М. Цветаевой': ('МАФ', 'Памятники'),
    'растительность высокотравная': ('Озеленение', 'Газон'),
    'растительность травяная, луговая': ('Озеленение', 'Газон'),
    'лесопосадки молодые': ('Озеленение', 'Дерево'),
    'заросли камыша': ('Озеленение', 'Газон'),
    '0.40': ('Служебные элементы', 'Размеры'),
    'откосы неукрепленные': ('Озеленение', 'Газон'),
    'ограды мет. высотой менее 1м': ('МАФ', 'Ограждение'),
    'ограды мет. высотой 1м и более': ('МАФ', 'Ограждение'),
    'полосы древ. насаждений выс.более 4м шириной до 1м': ('Озеленение', 'Дерево'),
    'изгороди,плетни': ('МАФ', 'Ограждение'),
    'заборы дер.с кап.опорами': ('МАФ', 'Ограждение'),
    'проволока': ('МАФ', 'Ограждение'),
    'полосы древ. насаждений выс.до 4м шир.до 1м': ('Озеленение', 'Кустарник'),
    'ограждения из гладк.проволоки': ('МАФ', 'Ограждение'),
    'ограды кам. и жб высотой менее 1м': ('МАФ', 'Ограждение'),
    'ограды кам.и жб высотой 1м и более': ('МАФ', 'Ограждение'),
    'ограждения из гладк.проволоки с бет. опорами': ('МАФ', 'Ограждение'),
    'просеки': ('Озеленение', 'Газон'),
    'бордюры кам.': ('Элементы благоустройства', 'Бортовой камень'),
    'заборы дер.решетчатые': ('МАФ', 'Ограждение'),
    'контур': ('Служебные элементы', 'Контуры'),
    'отмостка': ('Твердые покрытия', 'Бетон'),
    'дороги': ('Твердые покрытия', 'Асфальт'),
    'железные дороги': ('Элементы благоустройства', 'ЖД пути'),
    'канализация подземная': ('Инженерные сети', 'Канализация бытовая'),
    'линии связи': ('Инженерные сети', 'Сети связи'),
    'ливневая канализация подземная': ('Инженерные сети', 'Ливневая канализация'),
    'дренаж подземный': ('Инженерные сети', 'Инженерная инфраструктура'),
    'напорная канализация': ('Инженерные сети', 'Канализация бытовая'),
    'заборы сетчатые': ('МАФ', 'Ограждение'),
    'заборы дер.сплошные': ('МАФ', 'Ограждение'),
    'заборы из колючей проволоки': ('МАФ', 'Ограждение'),
    'осевые линии, граница землеотвода': ('Служебные элементы', 'Границы'),
    'пикеты с отметками высот': ('Рельеф', 'Отметки высот'),
    'вспомогательные точки': ('Служебные элементы', 'Прочее'),
    '(формируются программой автоматически, внимание на них можно не обращать)': ('Служебные элементы', 'Прочее')
}





def assign_mapping_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    labels = new_mapping.get(row['Text'].strip())
    if labels:
        return labels
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_mapping_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_5.30.csv', index=False)
print(f"[5.30]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[5.30]  +60  | размечено: 1130222 | осталось: 1939458


In [70]:
new_mapping = {
    'Зона с особыми условиями ': ('Служебные элементы', 'Условные обозначения'),
    'использования территорий': ('Служебные элементы', 'Условные обозначения'),
    'летняя': ('Здания и сооружения', 'Неизвестно'),
    'кухня': ('Здания и сооружения', 'Неизвестно'),
    'МУП "ПТО ЖКХ" ': ('Служебные элементы', 'Прочее'),
    'мет.': ('Элементы благоустройства', 'Металлический настил'),
    'мрамор': ('Твердые покрытия', 'Камень'),
    'площадка': ('Служебные элементы', 'Прочее'),
    'ДН': ('Здания и сооружения', 'Неизвестно'),
    '10.III': ('Водные объекты', 'Неизвестно'),
    'Ж': ('Здания и сооружения', 'Неизвестно'),
    'ПУ': ('Инженерные сети', 'Инженерная инфраструктура'),
    'территория музея милиции': ('Здания и сооружения', 'Неизвестно'),
    '01.XII': ('Водные объекты', 'Неизвестно'),
    'Зона ТБО': ('МАФ', 'Павильон'),
    'СХОДНЯ': ('Служебные элементы', 'Прочее'),
    'М.Я.': ('Инженерные сети', 'Инженерная инфраструктура'),
    'СТР.':  ('Инженерные сети', 'Инженерная инфраструктура'),
    'РАЗР.': ('Водопроницаемые покрытия', 'Грунт'),
    'ТР.': ('Инженерные сети', 'Инженерная инфраструктура'),
    'XI-22г.': ('Водные объекты', 'Неизвестно'),
    ' люк': ('Инженерные сети', 'Инженерная инфраструктура'),
    ' знак дор большой': ('МАФ', 'Дорожные знаки'),
    ' воздух': ('Инженерные сети', 'Инженерная инфраструктура'),
    ' заб мет/заб раб': ('МАФ', 'Ограждение'),
    ' столб мет камера': ('Инженерные сети', 'Столбы и опоры'),
    ' кип каб': ('Инженерные сети', 'Инженерная инфраструктура'),
    ' столб мет кам': ('Инженерные сети', 'Столбы и опоры'),
    ' выход труб 1000 ст': ('Инженерные сети', 'Прочие трубопроводы'),
    ' люк квадр': ('Инженерные сети', 'Инженерная инфраструктура'),
    ' ящ мет': ('Инженерные сети', 'Инженерная инфраструктура'),
    ' опора труб': ('Инженерные сети', 'Инженерная инфраструктура'),
    ' поворот труб': ('Инженерные сети', 'Инженерная инфраструктура'),
    ' труб верх': ('Инженерные сети', 'Инженерная инфраструктура'),
    ' труб поворот': ('Инженерные сети', 'Инженерная инфраструктура'),
    ' ет': ('Служебные элементы', 'Прочее'),
    ' выход труб': ('Инженерные сети', 'Инженерная инфраструктура'),
    ' люк квадрат': ('Инженерные сети', 'Инженерная инфраструктура'),
    ' труб бет 800 верх': ('Инженерные сети', 'Прочие трубопроводы'),
    ' труб бет 800 низ': ('Инженерные сети', 'Прочие трубопроводы'),
    ' r': ('Служебные элементы', 'Прочее'),
    ' верх': ('Служебные элементы', 'Прочее'),
    ' урез': ('Служебные элементы', 'Прочее'),
    ' низ': ('Служебные элементы', 'Прочее'),
    ' болото': ('Водные объекты', 'Неизвестно'),
    ' верх откоса': ('Служебные элементы', 'Прочее'),
    ' ель35': ('Озеленение', 'Дерево'),
    ' ель15': ('Озеленение', 'Дерево'),
    ' ель30': ('Озеленение', 'Дерево'),
    ' ель50': ('Озеленение', 'Дерево'),
    ' насыпь': ('Служебные элементы', 'Прочее'),
    ' насыпь верх': ('Служебные элементы', 'Прочее'),
    ' канава верх': ('Служебные элементы', 'Прочее'),
    ' канава низ': ('Служебные элементы', 'Прочее'),
    ' заболоченность': ('Водные объекты', 'Неизвестно'),
    ' овраг верх': ('Служебные элементы', 'Прочее'),
    ' овраг низ': ('Служебные элементы', 'Прочее'),
    ' верх канавы': ('Служебные элементы', 'Прочее'),
    ' рельеф': ('Рельеф', 'Отметки высот'),
    ' труб мет 130': ('Инженерные сети', 'Прочие трубопроводы'),
    ' труб мет 130 низ': ('Инженерные сети', 'Прочие трубопроводы'),
    ' бет разр': ('Твердые покрытия', 'Бетон'),
    ' труб мет': ('Инженерные сети', 'Прочие трубопроводы'),
    'С': ('Инженерные сети', 'Дренаж'),
    ' каб-4.9': ('Инженерные сети', 'Прочие кабели'),
    ' каб-4.3': ('Инженерные сети', 'Прочие кабели'),
    ' каб-1.5': ('Инженерные сети', 'Прочие кабели'),
    ' каб-1.2': ('Инженерные сети', 'Прочие кабели'),
    ' каб-1.7': ('Инженерные сети', 'Прочие кабели'),
    ' каб-0.7': ('Инженерные сети', 'Прочие кабели'),
    ' каб-1.4': ('Инженерные сети', 'Прочие кабели'),
    ' люк гтс': ('Инженерные сети', 'Сети связи'),
    ' кип охр каб': ('Служебные элементы', 'Прочее'),
    ' l': ('Служебные элементы', 'Прочее'),
    'Дер.': ('Водопроницаемые покрытия', 'Деревянный настил'),
    'Территория кафе': ('Служебные элементы', 'Прочее'),
    'М': ('Здания и сооружения', 'Неизвестно'),
    'территория ФОК "Салют"': ('Служебные элементы', 'Прочее'),
    'Преображенской Церкви ': ('Здания и сооружения', 'Неизвестно'),
    'под А': ('Твердые покрытия', 'Асфальт'),
    'А ': ('Твердые покрытия', 'Асфальт'),
    'н/о': ('Инженерные сети', 'Инженерная инфраструктура'),
    'замус.': ('Служебные элементы', 'Прочее'),
    'стадии согласования с в/ч 95306. ': ('Служебные элементы', 'Прочее'),
    '6 КН ': ('Здания и сооружения', 'Неизвестно'),
    '9КЖ ': ('Здания и сооружения', 'Неизвестно'),
    '2КЖ ': ('Здания и сооружения', 'Неизвестно'),
    '3КЖ ': ('Здания и сооружения', 'Неизвестно'),
    '4КН ': ('Здания и сооружения', 'Неизвестно'),
    '31.X': ('Водные объекты', 'Неизвестно'),
    '20.X': ('Водные объекты', 'Неизвестно'),
    '5КЖ ': ('Здания и сооружения', 'Неизвестно'),
    '4КЖ ': ('Здания и сооружения', 'Неизвестно'),
    '4 КН ': ('Здания и сооружения', 'Неизвестно'),
    '6КЖ ': ('Здания и сооружения', 'Неизвестно'),
    'Территория лицея': ('Служебные элементы', 'Прочее'),
    ' 1 тр 1 каб': ('Инженерные сети', 'Инженерная инфраструктура'),
    'КУ газ СКЗ\\P ': ('Инженерные сети', 'Газоснабжение'),
    'СКЗ (СЗПГ)\\P ': ('Инженерные сети', 'Газоснабжение'),
    'н/н\\P ': ('Инженерные сети', 'Инженерная инфраструктура')
}

total_changed = int(df['l1'].notna().sum())
total_nan     = int(df['l1'].isna().sum())

def assign_mapping_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    labels = new_mapping.get(row['Text'].strip())
    if labels:
        return labels
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_mapping_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_5.31.csv', index=False)
print(f"[5.31]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[5.31]  +0  | размечено: 1130940 | осталось: 1938740


In [103]:
new_mapping = {
    'КУ газ СКЗ\\PСЗПГ\\P ': ('Инженерные сети', 'Газоснабжение'),
    'н/в 47-ГАИ': ('Инженерные сети', 'Инженерная инфраструктура'),
    '2 4/6': ('Инженерные сети', 'Инженерная инфраструктура'),
    'лот ': ('Инженерные сети', 'Инженерная инфраструктура'),
    ' 2*250': ('Инженерные сети', 'Прочие трубопроводы'),
    'о/р': ('Служебные элементы', 'Прочее'),
    'ор': ('Служебные элементы', 'Прочее'),
    '3КН ': ('Здания и сооружения', 'Неизвестно'),
    'Золотая рыбка': ('МАФ', 'Игровое оборудование'),
    'бес.': ('Здания и сооружения', 'Неизвестно'),
    'тир.': ('Здания и сооружения', 'Неизвестно'),
    'Усадьба Знаменское-Губайлово': ('Служебные элементы', 'Прочее'),
    'Акр.': ('Твердые покрытия', 'Асфальт'),
    'границах улиц Советская, Октябрьская и ': ('Служебные элементы', 'Границы'),
    'о. Сенеж': ('Водные объекты', 'Неизвестно'),
    'Московская область, г. Солнечногорск, ': ('Служебные элементы', 'Штамп'),
    'огород': ('Водопроницаемые покрытия', 'Грунт'),
    'Площадка отдыха': ('Твердые покрытия', 'Асфальт'),
    'Театральная': ('Твердые покрытия', 'Асфальт'),
    'тент': ('Здания и сооружения', 'Неизвестно'),
    'X': ('Водные объекты', 'Неизвестно'),
    '20.XI.20': ('Водные объекты', 'Неизвестно'),
    '07-12.III.20': ('Водные объекты', 'Неизвестно'),
    '25.III': ('Водные объекты', 'Неизвестно'),
    'Стр. бой': ('Твердые покрытия', 'Бетон'),
    'стр. мусор': ('Твердые покрытия', 'Бетон'),
    'навоз': ('Водопроницаемые покрытия', 'Грунт'),
    'Стр. мусор': ('Твердые покрытия', 'Бетон'),
    '19.III': ('Водные объекты', 'Неизвестно'),
    '(619бр)': ('Твердые покрытия', 'Брусчатка'),
    '15.XII': ('Водные объекты', 'Неизвестно'),
    '16.XII': ('Водные объекты', 'Неизвестно'),
    '15.VI': ('Водные объекты', 'Неизвестно'),
    'к РП 9 ф8  ': ('Инженерные сети', 'Электроснабжение'),
    'ф9, ф3,6  ': ('Инженерные сети', 'Электроснабжение'),
    'ф20  ': ('Инженерные сети', 'Электроснабжение'),
    'к РП31  ': ('Инженерные сети', 'Электроснабжение'),
    'к РП 13  ': ('Инженерные сети', 'Электроснабжение'),
    '15.X': ('Служебные элементы', 'Прочее'),
    '10.ХI': ('Служебные элементы', 'Прочее'),
    '(715врем)': ('Служебные элементы', 'Прочее'),
    '(619рез)': ('Служебные элементы', 'Прочее'),
    '(619абр)': ('Служебные элементы', 'Прочее'),
    '(619к)': ('Служебные элементы', 'Прочее'),
    '(714врем)': ('Служебные элементы', 'Прочее'),
    '(stpark)': ('Служебные элементы', 'Прочее'),
    '(358б)': ('Служебные элементы', 'Прочее'),
    '(619а)': ('Служебные элементы', 'Прочее'),
    '(715к)': ('Служебные элементы', 'Прочее'),
    '(614а)': ('Служебные элементы', 'Прочее'),
    '(зк)': ('Служебные элементы', 'Прочее'),
    '(дн)': ('Служебные элементы', 'Прочее'),
    '(бр)': ('Служебные элементы', 'Прочее'),
    '26.VIII': ('Водные объекты', 'Неизвестно'),
    'Ат': ('МАФ', 'Игровое оборудование'),
    'Ат.': ('МАФ', 'Игровое оборудование'),
    '(100бр)': ('Твердые покрытия', 'Брусчатка'),
    '(5бр)': ('Служебные элементы', 'Прочее'),
    'горчасы': ('Служебные элементы', 'Прочее'),
    'дргр40': ('Служебные элементы', 'Прочее'),
    'Городские часы ': ('МАФ', 'Уличная мебель'),
    '(st)': ('Служебные элементы', 'Прочее'),
    'ДОТ': ('Служебные элементы', 'Прочее'),
    'блиндаж': ('Здания и сооружения', 'Неизвестно'),
    'территория АЗС "ОТРК"': ('Служебные элементы', 'Прочее'),
    'А.кр.': ('Твердые покрытия', 'Асфальт'),
    '25.X': ('Водные объекты', 'Неизвестно'),
    'ООО "ЧАЙКА"': ('Служебные элементы', 'Прочее'),
    'стр. материалы': ('Твердые покрытия', 'Бетон'),
    'ст.': ('Служебные элементы', 'Прочее'),
    'правила дорожного движения': ('Служебные элементы', 'Прочее'),
    '(ФБС)': ('Служебные элементы', 'Прочее'),
    '01.XI': ('Водные объекты', 'Неизвестно'),
    '3.VII': ('Водные объекты', 'Неизвестно'),
    'уч. 18а': ('Служебные элементы', 'Кадастр'),
    'уч. 18': ('Служебные элементы', 'Кадастр'),
    'эл.шк.': ('Инженерные сети', 'Электроснабжение'),
    'пол': ('Служебные элементы', 'Прочее'),
    'КАЗ': ('Инженерные сети', 'Инженерная инфраструктура'),
    'гл. 1.0': ('Служебные элементы', 'Размеры'),
    'XI.20': ('Водные объекты', 'Неизвестно'),
    'в.д.': ('Инженерные сети', 'Газоснабжение'),
    'Наименование объекта:М.О. ': ('Служебные элементы', 'Штамп'),
    'растяжка': ('Служебные элементы', 'Прочее'),
    'территория  серф-кайт клуба': ('Служебные элементы', 'Прочее'),
    'бер.': ('Озеленение', 'Дерево'),
    'территория лодочной станции': ('Служебные элементы', 'Прочее'),
    'III.2020': ('Водные объекты', 'Неизвестно'),
    'рампа разр.': ('Служебные элементы', 'Прочее'),
    '19.V': ('Водные объекты', 'Неизвестно'),
    '16.VI': ('Водные объекты', 'Неизвестно'),
    'территория школы N82': ('Служебные элементы', 'Прочее'),
    'Корчилова 5': ('Здания и сооружения', 'Неизвестно'),
    'zadw. ': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Пл. Урицкого': ('Твердые покрытия', 'Асфальт'),
    'тер. котельной': ('Служебные элементы', 'Прочее'),
    'Сквер им. В.Н. Леонова, Пл. Урицкого, ': ('Служебные элементы', 'Штамп'),
    'территория парка': ('Служебные элементы', 'Прочее'),
    'декор': ('Озеленение', 'Цветник'),
    'п а ш н я ': ('Водопроницаемые покрытия', 'Грунт')
}


def assign_mapping_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    labels = new_mapping.get(row['Text'].strip())
    if labels:
        return labels
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_mapping_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_5.32.csv', index=False)
print(f"[5.32]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[5.32]  +3896  | размечено: 1134836 | осталось: 1934844


In [129]:
new_mapping = {
    'фр.': ('Озеленение', 'Дерево'),
    'территория Водозаборного узла': ('Инженерные сети', 'Водопровод'),
    'Кремлевский спуск': ('Служебные элементы', 'Прочее'),
    'Строительный мусор': ('Твердые покрытия', 'Бетон'),
    'Закладной камень ': ('МАФ', 'Памятники'),
    '01.II': ('Водные объекты', 'Неизвестно'),
    'Морской знак': ('МАФ', 'Дорожные знаки'),
    'канализация ': ('Инженерные сети', 'Канализация бытовая'),
    '(100р)': ('Служебные элементы', 'Прочее'),
    ':28965': ('Служебные элементы', 'Кадастр'),
    ':28965/1': ('Служебные элементы', 'Кадастр'),
    ':314': ('Служебные элементы', 'Кадастр'),
    ':28959/1': ('Служебные элементы', 'Кадастр'),
    ':28959/2': ('Служебные элементы', 'Кадастр'),
    'Барьер ': ('МАФ', 'Ограждение'),
    '04.VII': ('Водные объекты', 'Неизвестно'),
    '29.VI': ('Водные объекты', 'Неизвестно'),
    '25.VI': ('Водные объекты', 'Неизвестно'),
    'Скейт площадка ': ('Твердые покрытия', 'Асфальт'),
    '(фп)': ('Служебные элементы', 'Прочее'),
    ' ПАО "Ростелеком" СЦ г. Подольск ориентировочно ': ('Инженерные сети', 'Сети связи'),
    '(мет подст)': ('Служебные элементы', 'Прочее'),
    '2КН ': ('Здания и сооружения', 'Неизвестно'),
    'ОМВД РФ по Солнечногорскому району': ('Здания и сооружения', 'Неизвестно'),
    'стр. городок': ('Здания и сооружения', 'Неизвестно'),
    'мусор.': ('Твердые покрытия', 'Бетон'),
    'Крупская, 1': ('Здания и сооружения', 'Неизвестно'),
    'Крупская, 3': ('Здания и сооружения', 'Неизвестно'),
    '3шт.': ('Служебные элементы', 'Прочее'),
    '8шт.': ('Служебные элементы', 'Прочее'),
    '6шт.': ('Служебные элементы', 'Прочее'),
    '10шт.': ('Служебные элементы', 'Прочее'),
    '9шт.': ('Служебные элементы', 'Прочее'),
    'навал ': ('Водопроницаемые покрытия', 'Грунт'),
    ' гр.': ('Водопроницаемые покрытия', 'Грунт'),
    '2 троса ': ('Инженерные сети', 'Прочие кабели'),
    'Радиационных ': ('МАФ', 'Памятники'),
    'Катастроф ': ('МАФ', 'Памятники'),
    'Репрессированным ': ('МАФ', 'Памятники'),
    'Мужеству ': ('МАФ', 'Памятники'),
    'контактный провод 600В ': ('Инженерные сети', 'Электроснабжение'),
    'собственник не установлен ': ('Служебные элементы', 'Прочее'),
    'отст': ('Водопроницаемые покрытия', 'Грунт'),
    'стр. площадка': ('Твердые покрытия', 'Асфальт'),
    'закл.': ('МАФ', 'Памятники'),
    'гар.': ('Здания и сооружения', 'Неизвестно'),
    'Сопряжение с фрагментом 1 ': ('Служебные элементы', 'Условные обозначения'),
    '18.IV': ('Водные объекты', 'Неизвестно'),
    'Школьный 2': ('Здания и сооружения', 'Неизвестно'),
    '6\\P{\\O0,08}': ('Служебные элементы', 'Прочее'),
    '17.IV': ('Служебные элементы', 'Прочее'),
    '1,8,5,6 в мкр. Керамик и зап. стороны от домов ': ('Служебные элементы', 'Штамп'),
    '+1,5м': ('Рельеф', 'Отметки высот'),
    ' пл.': ('Твердые покрытия', 'Плитка'),
    ' территория кафе': ('Здания и сооружения', 'Неизвестно'),
    'МУП "Балашихинский ': ('Инженерные сети', 'Инженерная инфраструктура'),
    'территория стадиона': ('Служебные элементы', 'Прочее'),
    'мус.': ('МАФ', 'Павильон'),
    'Ферсмана 15': ('Здания и сооружения', 'Неизвестно'),
    'Ферсмана 17': ('Здания и сооружения', 'Неизвестно'),
    'ГК 24 I': ('Служебные элементы', 'Прочее'),
    'ГК 25 I': ('Служебные элементы', 'Прочее'),
    'им.Г.В. Свиридова': ('Служебные элементы', 'Прочее'),
    'к-стр': ('Служебные элементы', 'Условные обозначения'),
    ' 20-30': ('Служебные элементы', 'Размеры дерева'),
    ' 0.20-0.30': ('Служебные элементы', 'Размеры дерева'),
    ' 3-5': ('Служебные элементы', 'Размеры дерева'),
    '14.Х': ('Водные объекты', 'Неизвестно'),
    '27.I': ('Водные объекты', 'Неизвестно'),
    'Нет доступа ': ('Служебные элементы', 'Прочее'),
    'ц 146': ('Рельеф', 'Отметки высот'),
    'ц 141': ('Рельеф', 'Отметки высот'),
    'а/ц': ('Инженерные сети', 'Прочие трубопроводы'),
    'з. 141,46 ': ('Рельеф', 'Отметки высот'),
    'V': ('Водные объекты', 'Неизвестно'),
    'площадка для гольфа': ('Озеленение', 'Газон'),
    'Площадка "Лира"': ('Служебные элементы', 'Прочее'),
    '26.II': ('Служебные элементы', 'Прочее'),
    '(ост100)': ('Служебные элементы', 'Прочее'),
    '(619ц)': ('Служебные элементы', 'Прочее'),
    '(надпись)': ('Служебные элементы', 'Прочее'),
    ' pt': ('Служебные элементы', 'Прочее'),
    ' дуб': ('Озеленение', 'Дерево'),
    ' бепеза': ('Озеленение', 'Дерево'),
    ' бепеза1': ('Озеленение', 'Дерево'),
    'VII.18': ('Служебные элементы', 'Прочее'),
    'бук': ('Озеленение', 'Дерево'),
    'I': ('Водные объекты', 'Неизвестно'),
    'XIII': ('Водные объекты', 'Неизвестно'),
    'XIV': ('Водные объекты', 'Неизвестно'),
    'подставка ': ('Служебные элементы', 'Прочее'),
    'территория школы N9': ('Служебные элементы', 'Прочее'),
    ' АО Мособлгаз "Север" СПРЭС': ('Инженерные сети', 'Газоснабжение'),
    'сушки': ('МАФ', 'Игровое оборудование'),
    'территория пожарной части': ('Служебные элементы', 'Прочее'),
    '"Молодежный"': ('Здания и сооружения', 'Неизвестно'),
    'территория Знаменской церкви': ('Здания и сооружения', 'Неизвестно'),
    'МУП"РСО" ': ('Служебные элементы', 'Прочее'),
    'Р': ('Элементы благоустройства"', 'Парковка'),
    'территория школы №2': ('Служебные элементы', 'Прочее')
}



def assign_mapping_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    labels = new_mapping.get(row['Text'].strip())
    if labels:
        return labels
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_mapping_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_5.33.csv', index=False)
print(f"[5.33]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[5.33]  +361  | размечено: 1135197 | осталось: 1934483


In [145]:
new_mapping = {
    'территория училища №37': ('Служебные элементы', 'Прочее'),
    'Площадь фонтанов': ('Служебные элементы', 'Прочее'),
    'OB': ('Служебные элементы', 'Прочее'),
    'KNV E': ('Служебные элементы', 'Прочее'),
    'KNV': ('Служебные элементы', 'Прочее'),
    'BR C': ('Служебные элементы', 'Прочее'),
    'BR': ('Служебные элементы', 'Прочее'),
    'BR S': ('Служебные элементы', 'Прочее'),
    'MOGILA': ('Служебные элементы', 'Прочее'),
    '"леж. "': ('Служебные элементы', 'Прочее'),
    'Яма\u00a0': ('Озеленение', 'Газон'),
    '13.II': ('Водные объекты', 'Неизвестно'),
    'POSL': ('Служебные элементы', 'Прочее'),
    '-0.5': ('Рельеф', 'Отметки высот'),
    'TRP E': ('Служебные элементы', 'Прочее'),
    'TRP': ('Служебные элементы', 'Прочее'),
    'TRP S': ('Служебные элементы', 'Прочее'),
    'TRP TRP12': ('Служебные элементы', 'Прочее'),
    'TRP E B=1.2 PEREKR': ('Служебные элементы', 'Прочее'),
    'TRP B=1.2': ('Служебные элементы', 'Прочее'),
    'TRP S B=1.2': ('Служебные элементы', 'Прочее'),
    'место для купания': ('Служебные элементы', 'Прочее'),
    'IV.21': ('Водные объекты', 'Неизвестно'),
    '\u00a0t': ('Инженерные сети', 'Теплосеть'),
    'с разр.': ('Служебные элементы', 'Прочее'),
    'Ц\u00a0': ('Твердые покрытия', 'Плитка'),
    '"Восход"': ('Служебные элементы', 'Прочее'),
    '\u00a0клён 15': ('Озеленение', 'Дерево'),
    '\u00a0мет труба': ('Инженерные сети', 'Прочие трубопроводы'),
    '\u00a0мет столб': ('Инженерные сети', 'Столбы и опоры'),
    '\u00a0дорога': ('Служебные элементы', 'Прочее'),
    '\u00a0забор мет': ('МАФ', 'Ограждение'),
    '\u00a0сторожевой знак': ('МАФ', 'Информационная конструкция'),
    '\u00a0знак дор': ('МАФ', 'Дорожные знаки'),
    'спец.': ('Служебные элементы', 'Прочее'),
    'Вентиляционная\u00a0': ('Инженерные сети', 'Инженерная инфраструктура'),
    '207Б':('Здания и сооружения', 'Неизвестно'),
    'Территория стадиона': ('Служебные элементы', 'Прочее'),
    'п/э 250\u00a0': ('Инженерные сети', 'Прочие трубопроводы'),
    '205Б': ('Здания и сооружения', 'Неизвестно'),
    'VIII.21': ('Водные объекты', 'Неизвестно'),
    '\u00a0G7 new': ('Служебные элементы', 'Прочее'),
    '\u00a0G7 old': ('Служебные элементы', 'Прочее'),
    '\u00a0а1': ('Служебные элементы', 'Прочее'),
    '\u00a0троп': ('Водопроницаемые покрытия', 'Грунт'),
    '\u00a0троп(плав)': ('Водопроницаемые покрытия', 'Грунт'),
    '\u00a0tp': ('Инженерные сети', 'Электроснабжение'),
    '\u00a0Заб раб': ('МАФ', 'Ограждение'),
    '\u00a0откос': ('Рельеф', 'Откос'),
    '\u00a0Заб мет': ('МАФ', 'Ограждение'),
    '\u00a0Заб дер': ('МАФ', 'Ограждение'),
    '\u00a0конт мет': ('Здания и сооружения', 'Неизвестно'),
    '\u00a0гар мет без выс\u00a0': ('Здания и сооружения', 'Неизвестно'),
    '\u00a0 бет плит': ('Твердые покрытия', 'Бетон'),
    '\u00a0бет плит': ('Твердые покрытия', 'Бетон'),
    '\u00a0rp 22.03.03': ('Служебные элементы', 'Прочее'),
    '\u00a0cтолб бет': ('Инженерные сети', 'Столбы и опоры'),
    '\u00a0cтолб мет': ('Инженерные сети', 'Столбы и опоры'),
    '\u00a0столб бет': ('Инженерные сети', 'Столбы и опоры'),
    '\u00a0забор дер': ('МАФ', 'Ограждение'),
    '\u00a0забор дер стык': ('МАФ', 'Ограждение'),
    '\u00a0фонарь бет': ('Инженерные сети', 'Столбы и опоры'),
    '\u00a0Заб раб стык': ('МАФ', 'Ограждение'),
    '\u00a0Заб бет': ('МАФ', 'Ограждение'),
    '\u00a0дом плит бет': ('Здания и сооружения', 'Неизвестно'),
    '\u00a0Забраб\u00a0': ('МАФ', 'Ограждение'),
    '\u00a0Заб раб\u00a0': ('МАФ', 'Ограждение'),
    '\u00a0Заб раб стык\u00a0': ('МАФ', 'Ограждение'),
    '\u00a0дор бет пллит\u00a0': ('Твердые покрытия', 'Бетон'),
    '\u00a0Заб мет стык': ('МАФ', 'Ограждение'),
    '\u00a0Заб дер стык': ('МАФ', 'Ограждение'),
    '\u00a0Заб бет стык': ('МАФ', 'Ограждение'),
    '\u00a0фонарь мет': ('Инженерные сети', 'Столбы и опоры'),
    '\u00a0тротуар': ('Твердые покрытия', 'Асфальт'),
    '\u00a0ящик мет': ('Инженерные сети', 'Инженерная инфраструктура'),
    '\u00a0кабель': ('Инженерные сети', 'Прочие кабели'),
    '\u00a0тп': ('Инженерные сети', 'Электроснабжение'),
    '\u00a0тротуар бет плит': ('Твердые покрытия', 'Бетон'),
    '\u00a0сторж каб': ('Инженерные сети', 'Прочие кабели'),
    '\u00a0t другая': ('Инженерные сети', 'Теплосеть'),
    '\u00a0t\u00a0': ('Инженерные сети', 'Теплосеть'),
    '\u00a0свалка снега рельеф': ('Служебные элементы', 'Прочее'),
    '\u00a0свалка снега релье': ('Служебные элементы', 'Прочее'),
    '\u00a0свалка снега рельеф_': ('Служебные элементы', 'Прочее'),
    '\u00a0yas 15x2': ('Озеленение', 'Дерево'),
    '\u00a0тахик': ('Служебные элементы', 'Прочее'),
    '\u00a0заб мет/ворота': ('МАФ', 'Ворота'),
    '\u00a0борд': ('Элементы благоустройства', 'Бортовой камень'),
    '\u00a0борд/ц пл': ('Твердые покрытия', 'Плитка'),
    '\u00a0зеак дор': ('МАФ', 'Дорожные знаки'),
    '\u00a0ворота/заб бет': ('МАФ', 'Ворота'),
    '\u00a0звб бет': ('МАФ', 'Ограждение'),
    '\u00a0заб бет': ('МАФ', 'Ограждение'),
    '\u00a0ц пл/заб бет/заб мет': ('Твердые покрытия', 'Плитка'),
    '\u00a0заб мет/кн': ('МАФ', 'Ограждение'),
    '\u00a0пл': ('Твердые покрытия', 'Бетон'),
    '\u00a0кн/заб мет': ('Здания и сооружения', 'Неизвестно'),
    '\u00a0юорд': ('Элементы благоустройства', 'Бортовой камень'),
    '\u00a0кип газ': ('Инженерные сети', 'Газоснабжение'),
    '\u00a0ц': ('Твердые покрытия', 'Плитка'),
    '\u00a0ц пл/спец/заб мет': ('Твердые покрытия', 'Плитка'),
    '\u00a0ц пл/борд': ('Твердые покрытия', 'Плитка'),
    '\u00a0смешанное покрытие': ('Водопроницаемые покрытия', 'Щебень'),
    '\u00a0щнб': ('Водопроницаемые покрытия', 'Щебень'),
    '\u00a0спец/заб мет': ('Здания и сооружения', 'Неизвестно'),
    '\u00a0лавояка': ('МАФ', 'Уличная мебель'),
    '\u00a0мсорка': ('МАФ', 'Уличная мебель'),
    '\u00a0ц разр': ('Твердые покрытия', 'Плитка'),
    '\u00a0лестн': ('Здания и сооружения', 'Лестницы и пандусы'),
    '\u00a0ступень': ('Здания и сооружения', 'Лестницы и пандусы'),
    '\u00a0дер настил': ('Водопроницаемые покрытия', 'Деревянный настил'),
    '\u00a0дер настил/бет': ('Водопроницаемые покрытия', 'Деревянный настил'),
    '\u00a0бет/дер настил': ('Твердые покрытия', 'Бетон'),
    '\u00a0дер настил/заб мет': ('Водопроницаемые покрытия', 'Деревянный настил'),
    '\u00a0дер настил/Заб мет': ('Водопроницаемые покрытия', 'Деревянный настил'),
    '\u00a0дер наст/заб мет': ('Водопроницаемые покрытия', 'Деревянный настил'),
    'урез': ('Служебные элементы', 'Прочее'),
    'площадка/1кн': ('Здания и сооружения', 'Неизвестно'),
    'площадка\u00a0': ('Служебные элементы', 'Прочее'),
    '\u00a0-2.79': ('Рельеф', 'Отметки высот'),
    'территория больницы': ('Служебные элементы', 'Прочее'),
    'ОАО "Русский районный\nспецкомбинат"': ('Служебные элементы', 'Прочее'),
    'территория Мособлпожспас': ('Служебные элементы', 'Прочее'),
    'а\\\\д 46Н-13981': ('Служебные элементы', 'Прочее'),
    'прогон': ('Водопроницаемые покрытия', 'Грунт'),
}



def assign_mapping_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    labels = new_mapping.get(row['Text'].strip())
    if labels:
        return labels
    return row['l1'], row['l2']

_before = df['l1'].notna().sum()
df[['l1', 'l2']] = df.apply(assign_mapping_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - _before)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_5.34.csv', index=False)
print(f"[5.34]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[5.34]  +221  | размечено: 1135418 | осталось: 1934262


In [10]:
new_mapping = {
    'Зона с особыми условиями\xa0': ('Служебные элементы', 'Границы'),
    'МУП "ПТО ЖКХ"\xa0': ('Служебные элементы', 'Прочее'),
    '\xa0люк': ('Инженерные сети', 'Инженерная инфраструктура'),
    '\xa0знак дор большой': ('МАФ', 'Дорожные знаки'),
    '\xa0воздух': ('Инженерные сети', 'Инженерная инфраструктура'),
    '\xa0заб мет/заб раб': ('МАФ', 'Ограждение'),
    '\xa0столб мет камера': ('Инженерные сети', 'Столбы и опоры'),
    '\xa0кип каб': ('Инженерные сети', 'Инженерная инфраструктура'),
    '\xa0столб мет кам': ('Инженерные сети', 'Столбы и опоры'),
    '\xa0выход труб 1000 ст': ('Инженерные сети', 'Прочие трубопроводы'),
    '\xa0люк квадр': ('Инженерные сети', 'Инженерная инфраструктура'),
    '\xa0ящ мет': ('Инженерные сети', 'Инженерная инфраструктура'),
    '\xa0опора труб': ('Инженерные сети', 'Инженерная инфраструктура'),
    '\xa0поворот труб': ('Инженерные сети', 'Инженерная инфраструктура'),
    '\xa0труб верх': ('Инженерные сети', 'Инженерная инфраструктура'),
    '\xa0труб поворот': ('Инженерные сети', 'Инженерная инфраструктура'),
    '\xa0 ет': ('Инженерные сети', 'Электроснабжение'),
    '\xa0выход труб': ('Инженерные сети', 'Инженерная инфраструктура'),
    '\xa0люк квадрат': ('Инженерные сети', 'Инженерная инфраструктура'),
    '\xa0труб бет 800 верх': ('Инженерные сети', 'Прочие трубопроводы'),
    '\xa0труб бет 800 низ': ('Инженерные сети', 'Прочие трубопроводы'),
    '\xa0r': ('Служебные элементы', 'Прочее'),
    '\xa0верх': ('Рельеф', 'Откос'),
    '\xa0низ': ('Рельеф', 'Откос'),
    '\xa0болото': ('Водные объекты', 'Неизвестно'),
    '\xa0верх откоса': ('Рельеф', 'Откос'),
    '\xa0ель35': ('Озеленение', 'Дерево'),
    '\xa0ель15': ('Озеленение', 'Дерево'),
    '\xa0ель30': ('Озеленение', 'Дерево'),
    '\xa0ель50': ('Озеленение', 'Дерево'),
    '\xa0насыпь': ('Рельеф', 'Откос'),
    '\xa0насыпь верх': ('Рельеф', 'Откос'),
    '\xa0канава верх': ('Рельеф', 'Откос'),
    '\xa0канава низ': ('Рельеф', 'Откос'),
    '\xa0заболоченность': ('Водные объекты', 'Неизвестно'),
    '\xa0овраг верх': ('Рельеф', 'Откос'),
    '\xa0овраг низ': ('Рельеф', 'Откос'),
    '\xa0верх канавы': ('Рельеф', 'Откос'),
    '\xa0рельеф': ('Рельеф', 'Отметки высот'),
    '\xa0труб мет 130': ('Инженерные сети', 'Прочие трубопроводы'),
    '\xa0труб мет 130 низ': ('Инженерные сети', 'Прочие трубопроводы'),
    '\xa0бет разр': ('Твердые покрытия', 'Бетон'),
    '\xa0труб мет': ('Инженерные сети', 'Прочие трубопроводы'),
    '\xa0каб-4.9': ('Инженерные сети', 'Прочие кабели'),
    '\xa0каб-4.3': ('Инженерные сети', 'Прочие кабели'),
    '\xa0каб-1.5': ('Инженерные сети', 'Прочие кабели'),
    '\xa0каб-1.2': ('Инженерные сети', 'Прочие кабели'),
    '\xa0каб-1.7': ('Инженерные сети', 'Прочие кабели'),
    '\xa0каб-0.7': ('Инженерные сети', 'Прочие кабели'),
    '\xa0каб-1.4': ('Инженерные сети', 'Прочие кабели'),
    '\xa0люк гтс': ('Инженерные сети', 'Сети связи'),
    '\xa0кип охр каб': ('Инженерные сети', 'Инженерная инфраструктура'),
    '\xa0l': ('Служебные элементы', 'Размеры'),
    'Преображенской Церкви\xa0': ('Здания и сооружения', 'Неизвестно'),
    'А\xa0': ('Твердые покрытия', 'Асфальт'),
    'стадии согласования с в/ч 95306.\xa0': ('Служебные элементы', 'Прочее'),
    '6 КН\xa0': ('Здания и сооружения', 'Неизвестно'),
    '9КЖ\xa0': ('Здания и сооружения', 'Неизвестно'),
    '2КЖ\xa0': ('Здания и сооружения', 'Неизвестно'),
    '3КЖ\xa0': ('Здания и сооружения', 'Неизвестно'),
    '4КН\xa0': ('Здания и сооружения', 'Неизвестно'),
    '5КЖ\xa0': ('Здания и сооружения', 'Неизвестно'),
    '4КЖ\xa0': ('Здания и сооружения', 'Неизвестно'),
    '4 КН\xa0': ('Здания и сооружения', 'Неизвестно'),
    '6КЖ\xa0': ('Здания и сооружения', 'Неизвестно'),
    '\xa01 тр 1 каб': ('Инженерные сети', 'Инженерная инфраструктура'),
    'КУ газ СКЗ\\P\xa0': ('Инженерные сети', 'Газоснабжение'),
    'СКЗ (СЗПГ)\\P\xa0': ('Инженерные сети', 'Газоснабжение'),
    'н/н\\P\xa0': ('Инженерные сети', 'Инженерная инфраструктура'),
    'КУ газ СКЗ\\PСЗПГ\\P\xa0': ('Инженерные сети', 'Газоснабжение'),
    'лот\xa0': ('Инженерные сети', 'Инженерная инфраструктура'),
    '\xa02*250': ('Инженерные сети', 'Прочие трубопроводы'),
    '3КН\xa0': ('Здания и сооружения', 'Неизвестно'),
    'границах улиц Советская, Октябрьская и\xa0': ('Служебные элементы', 'Границы'),
    'Московская область, г. Солнечногорск,\xa0': ('Служебные элементы', 'Штамп'),
    'к РП 9 ф8\xa0\xa0': ('Инженерные сети', 'Электроснабжение'),
    'ф9, ф3,6\xa0\xa0': ('Инженерные сети', 'Электроснабжение'),
    'ф20\xa0\xa0': ('Инженерные сети', 'Электроснабжение'),
    'к РП31\xa0\xa0': ('Инженерные сети', 'Электроснабжение'),
    'к РП 13\xa0\xa0': ('Инженерные сети', 'Электроснабжение'),
    '(стпарк)': ('Служебные элементы', 'Прочее'),
    'Городские часы\xa0': ('МАФ', 'Уличная мебель'),
    '(ст)': ('Служебные элементы', 'Прочее'),
    'Наименование объекта:М.О.\xa0': ('Служебные элементы', 'Штамп'),
    'задв.\xa0': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Сквер им. В.Н. Леонова, Пл. Урицкого,\xa0': ('Служебные элементы', 'Штамп'),
    'п а ш н я\xa0': ('Водопроницаемые покрытия', 'Грунт'),
    'Закладной камень\xa0': ('МАФ', 'Памятники'),
    'канализация\xa0': ('Инженерные сети', 'Канализация бытовая'),
    'Барьер\xa0': ('МАФ', 'Ограждение'),
    'Скейт площадка\xa0': ('Твердые покрытия', 'Асфальт'),
    '\xa0ПАО "Ростелеком" СЦ г. Подольск ориентировочно\xa0': ('Инженерные сети', 'Сети связи'),
    '2КН\xa0': ('Здания и сооружения', 'Неизвестно'),
    'навал\xa0': ('Водопроницаемые покрытия', 'Грунт'),
    '\xa0гр.': ('Водопроницаемые покрытия', 'Грунт'),
    '2 троса\xa0': ('Инженерные сети', 'Прочие кабели'),
    'Радиационных\xa0': ('МАФ', 'Памятники'),
    'Катастроф\xa0': ('МАФ', 'Памятники'),
    'Репрессированным\xa0': ('МАФ', 'Памятники'),
    'Мужеству\xa0': ('МАФ', 'Памятники'),
    'контактный провод 600В\xa0': ('Инженерные сети', 'Электроснабжение'),
    'собственник не установлен\xa0': ('Служебные элементы', 'Прочее'),
    'Сопряжение с фрагментом 1\xa0': ('Служебные элементы', 'Условные обозначения'),
    '1,8,5,6 в мкр. Керамик и зап. стороны от домов\xa0': ('Служебные элементы', 'Штамп'),
    '\xa0пл.': ('Твердые покрытия', 'Плитка'),
    '\xa0территория кафе': ('Здания и сооружения', 'Неизвестно'),
    'МУП "Балашихинский\xa0': ('Инженерные сети', 'Инженерная инфраструктура'),
    '\xa020-30': ('Служебные элементы', 'Размеры дерева'),
    '\xa00.20-0.30': ('Служебные элементы', 'Размеры дерева'),
    '\xa03-5': ('Служебные элементы', 'Размеры дерева'),
    'Нет доступа\xa0': ('Служебные элементы', 'Прочее'),
    'з. 141,46\xa0': ('Рельеф', 'Отметки высот'),
    '\xa0pt': ('Служебные элементы', 'Прочее'),
    '\xa0дуб': ('Озеленение', 'Дерево'),
    '\xa0бепеза': ('Озеленение', 'Дерево'),
    '\xa0бепеза1': ('Озеленение', 'Дерево'),
    'подставка\xa0': ('Служебные элементы', 'Прочее'),
    '\xa0АО Мособлгаз "Север" СПРЭС': ('Инженерные сети', 'Газоснабжение'),
    'МУП"РСО"\xa0': ('Служебные элементы', 'Прочее'),
    'Яма\xa0': ('Озеленение', 'Газон'),
    '\xa0t': ('Инженерные сети', 'Теплосеть'),
    'Ц\xa0': ('Твердые покрытия', 'Плитка'),
    '\xa0клён 15': ('Озеленение', 'Дерево'),
    '\xa0мет труба': ('Инженерные сети', 'Прочие трубопроводы'),
    '\xa0мет столб': ('Инженерные сети', 'Столбы и опоры'),
    '\xa0дорога': ('Служебные элементы', 'Прочее'),
    '\xa0забор мет': ('МАФ', 'Ограждение'),
    '\xa0сторожевой знак': ('МАФ', 'Информационная конструкция'),
    '\xa0знак дор': ('МАФ', 'Дорожные знаки'),
    'Вентиляционная\xa0': ('Инженерные сети', 'Инженерная инфраструктура'),
    'п/э 250\xa0': ('Инженерные сети', 'Прочие трубопроводы'),
    '\xa0G7 new': ('Служебные элементы', 'Прочее'),
    '\xa0G7 old': ('Служебные элементы', 'Прочее'),
    '\xa0а1': ('Служебные элементы', 'Прочее'),
    '\xa0троп': ('Водопроницаемые покрытия', 'Грунт'),
    '\xa0троп(плав)': ('Водопроницаемые покрытия', 'Грунт'),
    '\xa0tp': ('Инженерные сети', 'Электроснабжение'),
    '\xa0Заб раб': ('МАФ', 'Ограждение'),
    '\xa0откос': ('Рельеф', 'Откос'),
    '\xa0Заб мет': ('МАФ', 'Ограждение'),
    '\xa0Заб дер': ('МАФ', 'Ограждение'),
    '\xa0конт мет': ('Здания и сооружения', 'Неизвестно'),
    '\xa0гар мет без выс\xa0': ('Здания и сооружения', 'Неизвестно'),
    '\xa0 бет плит': ('Твердые покрытия', 'Бетон'),
    '\xa0бет плит': ('Твердые покрытия', 'Бетон'),
    '\xa0rp 22.03.03': ('Служебные элементы', 'Прочее'),
    '\xa0cтолб бет': ('Инженерные сети', 'Столбы и опоры'),
    '\xa0cтолб мет': ('Инженерные сети', 'Столбы и опоры'),
    '\xa0столб бет': ('Инженерные сети', 'Столбы и опоры'),
    '\xa0забор дер': ('МАФ', 'Ограждение'),
    '\xa0забор дер стык': ('МАФ', 'Ограждение'),
    '\xa0фонарь бет': ('Инженерные сети', 'Столбы и опоры'),
    '\xa0Заб раб стык': ('МАФ', 'Ограждение'),
    '\xa0Заб бет': ('МАФ', 'Ограждение'),
    '\xa0дом плит бет': ('Здания и сооружения', 'Неизвестно'),
    '\xa0Забраб\xa0': ('МАФ', 'Ограждение'),
    '\xa0Заб раб\xa0': ('МАФ', 'Ограждение'),
    '\xa0Заб раб стык\xa0': ('МАФ', 'Ограждение'),
    '\xa0дор бет пллит\xa0': ('Твердые покрытия', 'Бетон'),
    '\xa0Заб мет стык': ('МАФ', 'Ограждение'),
    '\xa0Заб дер стык': ('МАФ', 'Ограждение'),
    '\xa0Заб бет стык': ('МАФ', 'Ограждение'),
    '\xa0фонарь мет': ('Инженерные сети', 'Столбы и опоры'),
    '\xa0тротуар': ('Твердые покрытия', 'Асфальт'),
    '\xa0ящик мет': ('Инженерные сети', 'Инженерная инфраструктура'),
    '\xa0кабель': ('Инженерные сети', 'Прочие кабели'),
    '\xa0тп': ('Инженерные сети', 'Электроснабжение'),
    '\xa0тротуар бет плит': ('Твердые покрытия', 'Бетон'),
    '\xa0сторж каб': ('Инженерные сети', 'Прочие кабели'),
    '\xa0t другая': ('Инженерные сети', 'Теплосеть'),
    '\xa0t\xa0': ('Инженерные сети', 'Теплосеть'),
    '\xa0свалка снега рельеф': ('Служебные элементы', 'Прочее'),
    '\xa0свалка снега релье': ('Служебные элементы', 'Прочее'),
    '\xa0свалка снега рельеф_': ('Служебные элементы', 'Прочее'),
    '\xa0yas 15x2': ('Озеленение', 'Дерево'),
    '\xa0тахик': ('Служебные элементы', 'Прочее'),
    '\xa0заб мет/ворота': ('МАФ', 'Ворота'),
    '\xa0борд': ('Элементы благоустройства', 'Бортовой камень'),
    '\xa0борд/ц пл': ('Твердые покрытия', 'Плитка'),
    '\xa0зеак дор': ('МАФ', 'Дорожные знаки'),
    '\xa0ворота/заб бет': ('МАФ', 'Ворота'),
    '\xa0звб бет': ('МАФ', 'Ограждение'),
    '\xa0заб бет': ('МАФ', 'Ограждение'),
    '\xa0ц пл/заб бет/заб мет': ('Твердые покрытия', 'Плитка'),
    '\xa0заб мет/кн': ('МАФ', 'Ограждение'),
    '\xa0пл': ('Твердые покрытия', 'Бетон'),
    '\xa0кн/заб мет': ('Здания и сооружения', 'Неизвестно'),
    '\xa0юорд': ('Элементы благоустройства', 'Бортовой камень'),
    '\xa0кип газ': ('Инженерные сети', 'Газоснабжение'),
    '\xa0ц': ('Твердые покрытия', 'Плитка'),
    '\xa0ц пл/спец/заб мет': ('Твердые покрытия', 'Плитка'),
    '\xa0ц пл/борд': ('Твердые покрытия', 'Плитка'),
    '\xa0смешанное покрытие': ('Водопроницаемые покрытия', 'Щебень'),
    '\xa0щнб': ('Водопроницаемые покрытия', 'Щебень'),
    '\xa0спец/заб мет': ('Здания и сооружения', 'Неизвестно'),
    '\xa0лавояка': ('МАФ', 'Уличная мебель'),
    '\xa0мсорка': ('МАФ', 'Уличная мебель'),
    '\xa0ц разр': ('Твердые покрытия', 'Плитка'),
    '\xa0лестн': ('Здания и сооружения', 'Лестницы и пандусы'),
    '\xa0ступень': ('Здания и сооружения', 'Лестницы и пандусы'),
    '\xa0дер настил': ('Водопроницаемые покрытия', 'Деревянный настил'),
    '\xa0дер настил/бет': ('Водопроницаемые покрытия', 'Деревянный настил'),
    '\xa0бет/дер настил': ('Твердые покрытия', 'Бетон'),
    '\xa0дер настил/заб мет': ('Водопроницаемые покрытия', 'Деревянный настил'),
    '\xa0дер настил/Заб мет': ('Водопроницаемые покрытия', 'Деревянный настил'),
    '\xa0дер наст/заб мет': ('Водопроницаемые покрытия', 'Деревянный настил'),
    '\xa0-2.79': ('Рельеф', 'Отметки высот'),
    'ОАО "Русский районный\nспецкомбинат"': ('Служебные элементы', 'Прочее'),
    'территория училища №37': ('Служебные элементы', 'Прочее'),
    'Площадь фонтанов': ('Служебные элементы', 'Прочее'),
    'OB': ('Служебные элементы', 'Прочее'),
    'KNV E': ('Служебные элементы', 'Прочее'),
    'KNV': ('Служебные элементы', 'Прочее'),
    'BR C': ('Служебные элементы', 'Прочее'),
    'BR': ('Служебные элементы', 'Прочее'),
    'BR S': ('Служебные элементы', 'Прочее'),
    'MOGILA': ('Служебные элементы', 'Прочее'),
    '"леж. "': ('Служебные элементы', 'Прочее'),
    '13.II': ('Водные объекты', 'Неизвестно'),
    'POSL': ('Служебные элементы', 'Прочее'),
    '-0.5': ('Рельеф', 'Отметки высот'),
    'TRP E': ('Служебные элементы', 'Прочее'),
    'TRP': ('Служебные элементы', 'Прочее'),
    'TRP S': ('Служебные элементы', 'Прочее'),
    'TRP TRP12': ('Служебные элементы', 'Прочее'),
    'TRP E B=1.2 PEREKR': ('Служебные элементы', 'Прочее'),
    'TRP B=1.2': ('Служебные элементы', 'Прочее'),
    'TRP S B=1.2': ('Служебные элементы', 'Прочее'),
    'место для купания': ('Служебные элементы', 'Прочее'),
    'IV.21': ('Водные объекты', 'Неизвестно'),
    'с разр.': ('Служебные элементы', 'Прочее'),
    '"Восход"': ('Служебные элементы', 'Прочее'),
    'спец.': ('Служебные элементы', 'Прочее'),
    '207Б': ('Здания и сооружения', 'Неизвестно'),
    'Территория стадиона': ('Служебные элементы', 'Прочее'),
    '205Б': ('Здания и сооружения', 'Неизвестно'),
    'VIII.21': ('Водные объекты', 'Неизвестно'),
    'урез': ('Служебные элементы', 'Прочее'),
    'площадка/1кн': ('Здания и сооружения', 'Неизвестно'),
    'площадка\xa0': ('Служебные элементы', 'Прочее'),
    'территория больницы': ('Служебные элементы', 'Прочее'),
    'территория Мособлпожспас': ('Служебные элементы', 'Прочее'),
    'а\\\\д 46Н-13981': ('Служебные элементы', 'Прочее'),
    'прогон': ('Водопроницаемые покрытия', 'Грунт'),
    'фр.': ('Озеленение', 'Дерево'),
    'территория Водозаборного узла': ('Инженерные сети', 'Водопровод'),
    'Кремлевский спуск': ('Служебные элементы', 'Прочее'),
    'Строительный мусор': ('Твердые покрытия', 'Бетон'),
    'Закладной камень ': ('МАФ', 'Памятники'),
    '01.II': ('Водные объекты', 'Неизвестно'),
    'Морской знак': ('МАФ', 'Дорожные знаки'),
    'канализация ': ('Инженерные сети', 'Канализация бытовая'),
    '(100р)': ('Служебные элементы', 'Прочее'),
    ':28965': ('Служебные элементы', 'Кадастр'),
    ':28965/1': ('Служебные элементы', 'Кадастр'),
    ':314': ('Служебные элементы', 'Кадастр'),
    ':28959/1': ('Служебные элементы', 'Кадастр'),
    ':28959/2': ('Служебные элементы', 'Кадастр'),
    'Барьер ': ('МАФ', 'Ограждение'),
    '04.VII': ('Водные объекты', 'Неизвестно'),
    '29.VI': ('Водные объекты', 'Неизвестно'),
    '25.VI': ('Водные объекты', 'Неизвестно'),
    'Скейт площадка ': ('Твердые покрытия', 'Асфальт'),
    '(фп)': ('Служебные элементы', 'Прочее'),
    ' ПАО "Ростелеком" СЦ г. Подольск ориентировочно ': ('Инженерные сети', 'Сети связи'),
    '(мет подст)': ('Служебные элементы', 'Прочее'),
    'ОМВД РФ по Солнечногорскому району': ('Здания и сооружения', 'Неизвестно'),
    'стр. городок': ('Здания и сооружения', 'Неизвестно'),
    'мусор.': ('Твердые покрытия', 'Бетон'),
    'Крупская, 1': ('Здания и сооружения', 'Неизвестно'),
    'Крупская, 3': ('Здания и сооружения', 'Неизвестно'),
    '3шт.': ('Служебные элементы', 'Прочее'),
    '8шт.': ('Служебные элементы', 'Прочее'),
    '6шт.': ('Служебные элементы', 'Прочее'),
    '10шт.': ('Служебные элементы', 'Прочее'),
    '9шт.': ('Служебные элементы', 'Прочее'),
    'навал ': ('Водопроницаемые покрытия', 'Грунт'),
    ' гр.': ('Водопроницаемые покрытия', 'Грунт'),
    '2 троса ': ('Инженерные сети', 'Прочие кабели'),
    'Радиационных ': ('МАФ', 'Памятники'),
    'Катастроф ': ('МАФ', 'Памятники'),
    'Репрессированным ': ('МАФ', 'Памятники'),
    'Мужеству ': ('МАФ', 'Памятники'),
    'контактный провод 600В ': ('Инженерные сети', 'Электроснабжение'),
    'собственник не установлен ': ('Служебные элементы', 'Прочее'),
    'отст': ('Водопроницаемые покрытия', 'Грунт'),
    'стр. площадка': ('Твердые покрытия', 'Асфальт'),
    'закл.': ('МАФ', 'Памятники'),
    'гар.': ('Здания и сооружения', 'Неизвестно'),
    'Сопряжение с фрагментом 1 ': ('Служебные элементы', 'Условные обозначения'),
    '18.IV': ('Водные объекты', 'Неизвестно'),
    'Школьный 2': ('Здания и сооружения', 'Неизвестно'),
    '6\\P{\\O0,08}': ('Служебные элементы', 'Прочее'),
    '17.IV': ('Служебные элементы', 'Прочее'),
    '1,8,5,6 в мкр. Керамик и зап. стороны от домов ': ('Служебные элементы', 'Штамп'),
    '+1,5м': ('Рельеф', 'Отметки высот'),
    ' пл.': ('Твердые покрытия', 'Плитка'),
    ' территория кафе': ('Здания и сооружения', 'Неизвестно'),
    'МУП "Балашихинский ': ('Инженерные сети', 'Инженерная инфраструктура'),
    'мус.': ('МАФ', 'Павильон'),
    'Ферсмана 15': ('Здания и сооружения', 'Неизвестно'),
    'Ферсмана 17': ('Здания и сооружения', 'Неизвестно'),
    'ГК 24 I': ('Служебные элементы', 'Прочее'),
    'ГК 25 I': ('Служебные элементы', 'Прочее'),
    'им.Г.В. Свиридова': ('Служебные элементы', 'Прочее'),
    'к-стр': ('Служебные элементы', 'Условные обозначения'),
    ' 20-30': ('Служебные элементы', 'Размеры дерева'),
    ' 0.20-0.30': ('Служебные элементы', 'Размеры дерева'),
    ' 3-5': ('Служебные элементы', 'Размеры дерева'),
    '14.Х': ('Водные объекты', 'Неизвестно'),
    '27.I': ('Водные объекты', 'Неизвестно'),
    'Нет доступа ': ('Служебные элементы', 'Прочее'),
    'ц 146': ('Рельеф', 'Отметки высот'),
    'ц 141': ('Рельеф', 'Отметки высот'),
    'а/ц': ('Инженерные сети', 'Прочие трубопроводы'),
    'з. 141,46 ': ('Рельеф', 'Отметки высот'),
    'V': ('Водные объекты', 'Неизвестно'),
    'площадка для гольфа': ('Озеленение', 'Газон'),
    'Площадка "Лира"': ('Служебные элементы', 'Прочее'),
    '26.II': ('Водные объекты', 'Неизвестно'),
    '(ост100)': ('Служебные элементы', 'Прочее'),
    '(619ц)': ('Служебные элементы', 'Прочее'),
    '(надпись)': ('Служебные элементы', 'Прочее'),
    ' pt': ('Служебные элементы', 'Прочее'),
    ' дуб': ('Озеленение', 'Дерево'),
    ' бепеза': ('Озеленение', 'Дерево'),
    ' бепеза1': ('Озеленение', 'Дерево'),
    'VII.18': ('Водные объекты', 'Неизвестно'),
    'бук': ('Озеленение', 'Дерево'),
    'I': ('Водные объекты', 'Неизвестно'),
    'XIII': ('Водные объекты', 'Неизвестно'),
    'XIV': ('Водные объекты', 'Неизвестно'),
    'подставка ': ('Служебные элементы', 'Прочее'),
    'территория школы N9': ('Служебные элементы', 'Прочее'),
    ' АО Мособлгаз "Север" СПРЭС': ('Инженерные сети', 'Газоснабжение'),
    'сушки': ('МАФ', 'Игровое оборудование'),
    'территория пожарной части': ('Служебные элементы', 'Прочее'),
    '"Молодежный"': ('Здания и сооружения', 'Неизвестно'),
    'территория Знаменской церкви': ('Здания и сооружения', 'Неизвестно'),
    'МУП"РСО" ': ('Служебные элементы', 'Прочее'),
    'Р': ('Элементы благоустройства', 'Парковка'),
    'территория школы №2': ('Служебные элементы', 'Прочее'),
    'н/в 47-ГАИ': ('Инженерные сети', 'Инженерная инфраструктура'),
    '2 4/6': ('Инженерные сети', 'Инженерная инфраструктура'),
    'лот ': ('Инженерные сети', 'Инженерная инфраструктура'),
    ' 2*250': ('Инженерные сети', 'Прочие трубопроводы'),
    'о/р': ('Служебные элементы', 'Прочее'),
    'ор': ('Служебные элементы', 'Прочее'),
    'Золотая рыбка': ('МАФ', 'Игровое оборудование'),
    'бес.': ('Здания и сооружения', 'Неизвестно'),
    'тир.': ('Здания и сооружения', 'Неизвестно'),
    'Усадьба Знаменское-Губайлово': ('Служебные элементы', 'Прочее'),
    'Акр.': ('Твердые покрытия', 'Асфальт'),
    'границах улиц Советская, Октябрьская и ': ('Служебные элементы', 'Границы'),
    'о. Сенеж': ('Водные объекты', 'Неизвестно'),
    'Московская область, г. Солнечногорск, ': ('Служебные элементы', 'Штамп'),
    'огород': ('Водопроницаемые покрытия', 'Грунт'),
    'Площадка отдыха': ('Твердые покрытия', 'Асфальт'),
    'Театральная': ('Твердые покрытия', 'Асфальт'),
    'тент': ('Здания и сооружения', 'Неизвестно'),
    'X': ('Водные объекты', 'Неизвестно'),
    '20.XI.20': ('Водные объекты', 'Неизвестно'),
    '07-12.III.20': ('Водные объекты', 'Неизвестно'),
    '25.III': ('Водные объекты', 'Неизвестно'),
    'Стр. бой': ('Твердые покрытия', 'Бетон'),
    'стр. мусор': ('Твердые покрытия', 'Бетон'),
    'навоз': ('Водопроницаемые покрытия', 'Грунт'),
    'Стр. мусор': ('Твердые покрытия', 'Бетон'),
    '19.III': ('Водные объекты', 'Неизвестно'),
    '(619бр)': ('Твердые покрытия', 'Брусчатка'),
    '15.XII': ('Водные объекты', 'Неизвестно'),
    '16.XII': ('Водные объекты', 'Неизвестно'),
    '15.VI': ('Водные объекты', 'Неизвестно'),
    '15.X': ('Водные объекты', 'Неизвестно'),
    '10.ХI': ('Водные объекты', 'Неизвестно'),
    '(715врем)': ('Служебные элементы', 'Прочее'),
    '(619рез)': ('Служебные элементы', 'Прочее'),
    '(619абр)': ('Служебные элементы', 'Прочее'),
    '(619к)': ('Служебные элементы', 'Прочее'),
    '(714врем)': ('Служебные элементы', 'Прочее'),
    '(stpark)': ('Служебные элементы', 'Прочее'),
    '(358б)': ('Служебные элементы', 'Прочее'),
    '(619а)': ('Служебные элементы', 'Прочее'),
    '(715к)': ('Служебные элементы', 'Прочее'),
    '(614а)': ('Служебные элементы', 'Прочее'),
    '(зк)': ('Служебные элементы', 'Прочее'),
    '(дн)': ('Служебные элементы', 'Прочее'),
    '(бр)': ('Служебные элементы', 'Прочее'),
    '26.VIII': ('Водные объекты', 'Неизвестно'),
    'Ат': ('МАФ', 'Игровое оборудование'),
    'Ат.': ('МАФ', 'Игровое оборудование'),
    '(100бр)': ('Твердые покрытия', 'Брусчатка'),
    '(5бр)': ('Служебные элементы', 'Прочее'),
    'горчасы': ('Служебные элементы', 'Прочее'),
    'дргр40': ('Служебные элементы', 'Прочее'),
    'Городские часы ': ('МАФ', 'Уличная мебель'),
    '(ст)': ('Служебные элементы', 'Прочее'),
    'ДОТ': ('Служебные элементы', 'Прочее'),
    'блиндаж': ('Здания и сооружения', 'Неизвестно'),
    'территория АЗС "ОТРК"': ('Служебные элементы', 'Прочее'),
    'А.кр.': ('Твердые покрытия', 'Асфальт'),
    '25.X': ('Водные объекты', 'Неизвестно'),
    'ООО "ЧАЙКА"': ('Служебные элементы', 'Прочее'),
    'стр. материалы': ('Твердые покрытия', 'Бетон'),
    'ст.': ('Служебные элементы', 'Прочее'),
    'правила дорожного движения': ('Служебные элементы', 'Прочее'),
    '(ФБС)': ('Служебные элементы', 'Прочее'),
    '01.XI': ('Водные объекты', 'Неизвестно'),
    '3.VII': ('Водные объекты', 'Неизвестно'),
    'уч. 18а': ('Служебные элементы', 'Кадастр'),
    'уч. 18': ('Служебные элементы', 'Кадастр'),
    'эл.шк.': ('Инженерные сети', 'Электроснабжение'),
    'пол': ('Служебные элементы', 'Прочее'),
    'КАЗ': ('Инженерные сети', 'Инженерная инфраструктура'),
    'гл. 1.0': ('Служебные элементы', 'Размеры'),
    'XI.20': ('Водные объекты', 'Неизвестно'),
    'в.д.': ('Инженерные сети', 'Газоснабжение'),
    'Наименование объекта:М.О. ': ('Служебные элементы', 'Штамп'),
    'растяжка': ('Служебные элементы', 'Прочее'),
    'территория \xa0серф-кайт клуба': ('Служебные элементы', 'Прочее'),
    'бер.': ('Озеленение', 'Дерево'),
    'территория лодочной станции': ('Служебные элементы', 'Прочее'),
    'III.2020': ('Водные объекты', 'Неизвестно'),
    'рампа разр.': ('Служебные элементы', 'Прочее'),
    '19.V': ('Водные объекты', 'Неизвестно'),
    '16.VI': ('Водные объекты', 'Неизвестно'),
    'территория школы N82': ('Служебные элементы', 'Прочее'),
    'Корчилова 5': ('Здания и сооружения', 'Неизвестно'),
    'zadw. ': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Пл. Урицкого': ('Твердые покрытия', 'Асфальт'),
    'тер. котельной': ('Служебные элементы', 'Прочее'),
    'Сквер им. В.Н. Леонова, Пл. Урицкого, ': ('Служебные элементы', 'Штамп'),
    'территория парка': ('Служебные элементы', 'Прочее'),
    'декор': ('Озеленение', 'Цветник'),
    'п а ш н я ': ('Водопроницаемые покрытия', 'Грунт')
}


total_changed = int(df['l1'].notna().sum())
total_nan     = int(df['l1'].isna().sum())



new_mapping = {
    k.replace('\xa0', ' ').strip(): v
    for k, v in new_mapping.items()
}

def assign_mapping_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    if not isinstance(row['Text'], str):
        return row['l1'], row['l2']
    text = row['Text'].replace('\xa0', ' ').strip()   # ← заменяем \xa0 на обычный пробел
    labels = new_mapping.get(text)
    if labels:
        return labels
    return row['l1'], row['l2']

df[['l1', 'l2']] = df.apply(assign_mapping_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - total_changed)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_5.35.csv', index=False)
print(f"[5.35]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[5.35]  +1261  | размечено: 1136684 | осталось: 1932996


In [20]:
df = pd.read_csv('NEW_REGIONS_in_progress_5.35.csv', low_memory=False)

In [21]:
df[df['l1'].isna()]['Layer'].value_counts()

0                   478267
Растительность      217550
растительность      104769
TREE SURV SYMBOL     76753
Ограждения           60324
                     ...  
0113                     1
торговые ряды            1
51472700                 1
6 Брусья                 1
T-Zdanie                 1
Name: Layer, Length: 1061, dtype: int64

In [4]:
pd.DataFrame(df[df['l1'].isna()]['Text'].unique()).to_excel('TEXTS_left.xlsx')

## 6.0 Blocks mapping (from file with manual labeling)

In [14]:
df[(df['BlockName'].notna()) & (df['l1'].isna())].shape[0]

820968

In [23]:
df_before = df.copy()

In [24]:
blocks_df = pd.read_excel('blocks_dictionary.xlsx')
blocks_df.columns = blocks_df.columns.str.strip()
blocks_df['BlockName'] = blocks_df['BlockName'].astype(str).str.strip()

blocks_df = blocks_df[
    blocks_df['BlockName'].notna() &
    (blocks_df['BlockName'] != '') &
    (blocks_df['BlockName'] != 'nan')
]

block_mapping = (
    blocks_df
    .set_index('BlockName')[['l1', 'l2']]
    .apply(lambda row: (row['l1'], row['l2']), axis=1)
    .to_dict()
)

df['BlockName'] = df['BlockName'].astype(str).str.strip()
df.loc[df['BlockName'] == 'nan', 'BlockName'] = pd.NA

mapped = df['BlockName'].map(block_mapping)
valid_mask = mapped.notna()
mapped_df = pd.DataFrame(
    mapped[valid_mask].tolist(),
    index=mapped[valid_mask].index,
    columns=['l1_new', 'l2_new']
)


df.loc[
    valid_mask & df['l1'].isna(),
    'l1'
] = mapped_df['l1_new']


df.loc[
    valid_mask & df['l2'].isna(),
    'l2'
] = mapped_df['l2_new']

mapped_count = valid_mask.sum()
unmapped_count = df[
    (df['BlockName'].notna()) &
    (~df['BlockName'].isin(block_mapping.keys()))
].shape[0]

print(f"Успешно размечено блоков: {mapped_count}")
print(f"Осталось неразмеченных блоков: {unmapped_count}")

unmapped_blocks = df.loc[
    (df['BlockName'].notna()) &
    (~df['BlockName'].isin(block_mapping.keys())),
    'BlockName'
].unique()

print(f"Уникальных неизвестных блоков: {len(unmapped_blocks)}")
print("Примеры:", unmapped_blocks[:20])

c:\ProgramData\anaconda3\lib\site-packages\openpyxl\worksheet\_read_only.py:79: UserWarning: Data Validation extension is not supported and will be removed
  for idx, row in parser.parse():


Успешно размечено блоков: 820968
Осталось неразмеченных блоков: 0
Уникальных неизвестных блоков: 0
Примеры: []


In [27]:
df[(df['BlockName'].notna()) & (df['l1'].isna())].shape[0]

0

In [ ]:
df.to_csv('NEW_REGIONS_in_progress_5.40.csv')

In [31]:
df[(df['l1'].isna()) & (df['Type'].isin([
    'LWPOLYLINE', 
    'LINE', 
    'ARC', 
    'INSERT', 
    'CIRCLE', 
    'SPLINE', 
    'POLYLINE',
    'MLINE',
    'ELLIPSE',
    'HATCH'
]))].shape[0]

1073514

In [ ]:
valid_types = [
    'LWPOLYLINE', 
    'LINE', 
    'ARC', 
    'INSERT', 
    'CIRCLE', 
    'SPLINE', 
    'POLYLINE',
    'MLINE',
    'ELLIPSE',
    'HATCH'
]

filtered_df = df[
    (df['l1'].isna()) &
    (df['Type'].isin(valid_types))
]

layers_stats = (
    filtered_df
    .groupby('Layer')
    .size()
    .reset_index(name='count')
    .sort_values(by='count', ascending=False)
)

layers_stats.to_excel('lines_layers.xlsx', index=False)

## 7.0. lines, polylines, etc. classification

In [76]:
new_mapping = {
    '0': ('Неизвестно', 'Неизвестно'),
    'Ограждения': ('МАФ', 'Ограждение'),
    'SIT_LDEFAULT': ('Неизвестно', 'Неизвестно'),
    'РЕЛЬЕФ-ГОРИЗОНТАЛ': ('Рельеф', 'Горизонтали'),
    'Откосы': ('Рельеф', 'Откос'),
    'LAND': ('Озеленение', 'Газон'),
    '1184v(otk)': ('Рельеф', 'Откос'),
    'TREE SURV SYMBOL': ('Озеленение', 'Дерево'),
    'Горизонтали--MAJ': ('Рельеф', 'Горизонтали'),
    'ЭЛЕКТРОСЕТ-КЛ_РАСПРЕД': ('Инженерные сети', 'Электроснабжение'),
    'ОГРАЖДЕНИЯ-ОГРАЖДЕНИЯ': ('МАФ', 'Ограждение'),
    'ЭЛЕКТРОСЕТ-ВЛ_РАСПРЕД': ('Инженерные сети', 'Электроснабжение'),
    'дороги': ('Твердые покрытия', 'Асфальт'),
    'Рельеф': ('Рельеф', 'Откосы'),
    'Горизонтали': ('Рельеф', 'Горизонтали'),
    'C-TOPO-MAJR': ('Рельеф', 'Горизонтали'),
    'Горизонтали--MIN': ('Рельеф', 'Горизонтали'),
    'Гидрография': ('Водные объекты', 'Неизвестно'),
    'AUTOMAG_20': ('Твердые покрытия', 'Асфальт'),
    'C-TOPO-MINR': ('Рельеф', 'Горизонтали'),
    'OTKOSОТКОС': ('Рельеф', 'Откос'),
    'Отметки высот': ('Рельеф', 'Отметки высот'),
    '03115n(3n 3nr)': ('Инженерные сети', 'Электроснабжение'),
    'SETKR': ('Служебные элементы', 'Координатные кресты'),
    'Растительность': ('Озеленение', 'Газон'),
    '00p': ('Неизвестно', 'Неизвестно'),
    'Автом_и_грунт_дороги__тропы': ('Элементы благоустройства', 'Бортовой камень'),
    'Автодорожное хозяйство': ('Служебные элементы', 'Контуры'),
    '0113(1n)': ('Здания и сооружения', 'Неизвестно'),
    'ТЕПЛОСЕТИ-КОТЕЛЬНЫЕ': ('Инженерные сети', 'Теплосеть'),
    'Слой1': ('Служебные элементы', 'Прочее'),
    'Горизантали': ('Рельеф', 'Горизонтали'),
    '12330': ('Рельеф', 'Отметки высот'),
    'SIT_L0': ('Служебные элементы', 'Контуры'),
    '2пикет': ('Рельеф', 'Отметки высот'),
    '05187-1(dr)': ('Элементы благоустройства', 'Бортовой камень'),
    '12_Рельеф': ('Рельеф', 'Откос'),
    'РАСТИТЕЛЬН-ЛЕСА_И_ПОЛ': ('Озеленение', 'Дерево'),
    'Откос': ('Рельеф', 'Откос'),
    'RASTIT_27': ('Озеленение', 'Газон'),
    'ЭЛЕКТРОСЕТ-КЛ_ПИТАЮЩА': ('Инженерные сети', 'Электроснабжение'),
    'ДОРОГИ-ТРОТУАРЫ_И': ('Элементы благоустройства', 'Бортовой камень'),
    'границы зданий': ('Здания и сооружения', 'Неизвестно'),
    'RELIEF_1': ('Рельеф', 'Откос'),
    'Сетка': ('Служебные элементы', 'Координатные кресты'),
    'откосы': ('Рельеф', 'Откос'),
    'GORIZDEFAULT': ('Рельеф', 'Горизонтали'),
    '05189-1(dr)': ('Элементы благоустройства', 'Бортовой камень'),
    'Доп.отметки': ('Рельеф', 'Отметки высот'),
    'Координатная': ('Служебные элементы', 'Координатные кресты'),
    'R_point_T': ('Рельеф', 'Отметки высот'),
    'PROCHIE': ('Служебные элементы', 'Прочее'),
    'Инженерные сооружения': ('Инженерные сети', 'Инженерная инфраструктура'),
    'ДОРОГИ-ДОРОГИ': ('Твердые покрытия', 'Асфальт'),
    'Здания и строения': ('Здания и сооружения', 'Неизвестно'),
    'ВОДООТВЕДЕ-КАНАЛИЗАЦИ': ('Инженерные сети', 'Канализация бытовая'),
    '05189-2(dr)': ('Служебные элементы', 'Контуры'),
    'PI_STDEFAULT': ('Служебные элементы', 'Прочее'),
    'Строения__здания_и_их_части': ('Здания и сооружения', 'Неизвестно'),
    '03122(3n)': ('Инженерные сети', 'Прочие кабели'),
    'ВОДОПРОВОД-ВОДОПРОВОД': ('Инженерные сети', 'Водопровод'),
    'ГАЗОПРОВОД-ГАЗОПРОВОД': ('Инженерные сети', 'Газоснабжение'),
    'МАФ': ('МАФ', 'Прочее'),
    'Границы покрытий и угодий': ('Служебные элементы', 'Границы'),
    '11329-2(gor)': ('Рельеф', 'Горизонтали'),
    '10474-1(zb)': ('МАФ', 'Ограждение'),
    '_скелетная модель': ('Служебные элементы', 'Прочее'),
    'GILYO_3': ('Здания и сооружения', 'Неизвестно'),
    'CABEL': ('Инженерные сети', 'Электроснабжение'),
    'сетка': ('Служебные элементы', 'Координатные кресты'),
    'Отметки_точки': ('Рельеф', 'Отметки высот'),
    '5_ГИДРОГРАФИЯ': ('Водные объекты', 'Неизвестно'),
    'OTKOS0': ('Рельеф', 'Откос'),
    'WODOPROV_32': ('Инженерные сети', 'Водопровод'),
    'GEO_GAZ_PIPES': ('Инженерные сети', 'Газоснабжение'),
    'канализация': ('Инженерные сети', 'Канализация бытовая'),
    'горизонтали целые': ('Рельеф', 'Горизонтали'),
    '71314100': ('Служебные элементы', 'Прочее'),
    'Дороги': ('Твердые покрытия', 'Асфальт'),
    'ТРОТУАРЫ': ('Твердые покрытия', 'Асфальт'),
    'KANAL_33': ('Инженерные сети', 'Канализация бытовая'),
    'SVYAZ_38': ('Инженерные сети', 'Сети связи'),
    'Основные горизонтали': ('Рельеф', 'Горизонтали'),
    'пикеты': ('Рельеф', 'Отметки высот'),
    'ВОДООТВЕДЕ-КОЛОДЦЫ_И_': ('Инженерные сети', 'Канализация бытовая'),
    'ЭЛЕКТРОСЕТ-СТОЛБЫ__ФО': ('Инженерные сети', 'Наружное освещение'),
    'СВЯЗЬ-КАБЕЛИ_СВЯ': ('Инженерные сети', 'Сети связи'),
    'ЛЭП': ('Инженерные сети', 'Электроснабжение'),
    '1184(otk)': ('Рельеф', 'Откос'),
    'ограждения': ('МАФ', 'Ограждение'),
    'Водопровод': ('Инженерные сети', 'Водопровод'),
    'ВОДОПРОВОД-КОЛОДЦЫ__К': ('Инженерные сети', 'Водопровод'),
    'OTKOS': ('Рельеф', 'Откос'),
    '00r': ('Рельеф', 'Прочее'),
    'Подземные_кабели_связи_и_электрические': ('Инженерные сети', 'Прочие кабели'),
    'OGRADA_31': ('МАФ', 'Ограждение'),
    'GOR_BDEFAULT': ('Рельеф', 'Горизонтали'),
    'ДОРОГИ-ТРУБЫ_ПОД_': ('Инженерные сети', 'Ливневая канализация'),
    'номер подеревки': ('Служебные элементы', 'Номер дерева')
}

def assign_layer_only_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    labels = new_mapping.get(str(row['Layer']).strip())
    if labels:
        return labels
    return row['l1'], row['l2']

df[['l1', 'l2']] = df.apply(assign_layer_only_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - total_changed)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_7.00.csv', index=False)
print(f"[7.00]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[7.00]  +1847536  | размечено: 2984220 | осталось: 85460


In [93]:
new_mapping = {
    '11329-1(gor)': ('Рельеф', 'Горизонтали'),
    'SVAZ': ('Инженерные сети', 'Сети связи'),
    'SIT_LДОРОГИ': ('Элементы благоустройства', 'Бортовой камень'),
    'РЕЛЬЕФ-ВЫСОТНЫЕ_О': ('Рельеф', 'Отметки высот'),
    'ELEKT_KAB_37': ('Инженерные сети', 'Электроснабжение'),
    'канализация ливневая': ('Инженерные сети', 'Ливневая канализация'),
    'PI_STD-Рома': ('Служебные элементы', 'Прочее'),
    'Ситуация': ('Служебные элементы', 'Прочее'),
    '6 Урны': ('МАФ', 'Уличная мебель'),
    'ROADS': ('Служебные элементы', 'Контуры'),
    'Кабели связи': ('Инженерные сети', 'Сети связи'),
    'Кабели электрические': ('Инженерные сети', 'Прочие кабели'),
    'М горизонталь': ('Рельеф', 'Горизонтали'),
    'Здания_сооружения': ('Здания и сооружения', 'Неизвестно'),
    'PI_STD-Комков2': ('Служебные элементы', 'Прочее'),
    'РАСТИТЕЛЬН-КУСТАРНИК': ('Озеленение', 'Кустарник'),
    'крест_координат': ('Служебные элементы', 'Координатные кресты'),
    'LAND_HORYZONTAL': ('Рельеф', 'Горизонтали'),
    'ИИ_ОТКОС_025': ('Рельеф', 'Откос'),
    'Границы зданий': ('Здания и сооружения', 'Неизвестно'),
    'TG_DOROGA': ('Служебные элементы', 'Контуры'),
    'ЗДАНИЯ_И_С-ЗДАНИЯ': ('Здания и сооружения', 'Неизвестно'),
    '10472(zb)': ('МАФ', 'Ограждение'),
    'B': ('Служебные элементы', 'Прочее'),
    'P': ('Служебные элементы', 'Прочее'),
    'Кадастр': ('Служебные элементы', 'Кадастр'),
    'Объекты электропередачи': ('Инженерные сети', 'Электроснабжение'),
    'растительность': ('Озеленение', 'Газон'),
    'OTKOSDEFAULT': ('Рельеф', 'Откос'),
    '08_Поребрики': ('Элементы благоустройства', 'Бортовой камень'),
    'РАСТИТЕЛЬН-ШТРИХОВЫЕ_': ('Озеленение', 'Прочее'),
    '03b-1': ('Инженерные сети', 'Инженерная инфраструктура'),
    'теплотрасса': ('Инженерные сети', 'Теплосеть'),
    'ТЕПЛОСЕТИ-ТЕПЛОСЕТИ': ('Инженерные сети', 'Теплосеть'),
    '03114n(3n 3nr)': ('Инженерные сети', 'Электроснабжение'),
    'DOROGA': ('Элементы благоустройства', 'Бортовой камень'),
    'ЛЭП низкого напряжения': ('Инженерные сети', 'Электроснабжение'),
    'C3D-Поверхность_метки': ('Рельеф', 'Отметки высот'),
    'Канализации': ('Инженерные сети', 'Канализация бытовая'),
    'ПРОЧИЕ_ОБЪ-СТЕНКИ_ПОД': ('Здания и сооружения', 'Подпорные стенки'),
    'ПРОЧИЕ_СЕТ-КОЛОДЦЫ__К': ('Инженерные сети', 'Инженерная инфраструктура'),
    '10_Границы покрытий и угодий': ('Служебные элементы', 'Границы'),
    'ЛЭП__ЛЭС_и_воздушные_каб_линии': ('Инженерные сети', 'Электроснабжение'),
    '08397-1(pkd)': ('Озеленение', 'Кустарник'),
    'T': ('Инженерные сети', 'Теплосеть'),
    'ЗДАНИЯ И СООРУЖЕНИЯ': ('Здания и сооружения', 'Неизвестно'),
    'Лавочки': ('МАФ', 'Уличная мебель'),
    '2 Лавочки': ('МАФ', 'Уличная мебель'),
    'ЗДАНИЯ_ЖИЛЫЕ': ('Здания и сооружения', 'Неизвестно'),
    'знаки': ('МАФ', 'Дорожные знаки'),
    'Крышки колодцев': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Prochee': ('Служебные элементы', 'Прочее'),
    'Строение': ('Здания и сооружения', 'Неизвестно'),
    'Водопроводы': ('Инженерные сети', 'Водопровод'),
    'ЗДАНИЯ_И_С-СООРУЖЕНИЯ': ('Здания и сооружения', 'Неизвестно'),
    '06_Инженерно-технические сооружения': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Газопроводы': ('Инженерные сети', 'Газоснабжение'),
    'Теплопроводы': ('Инженерные сети', 'Теплосеть'),
    '03119v(3n)': ('Инженерные сети', 'Прочие кабели'),
    'Горизонтали--LBL': ('Рельеф', 'Горизонтали'),
    'Коммуникации': ('Инженерные сети', 'Инженерная инфраструктура'),
    'РАСТИТЕЛЬН-ГАЗОНЫ_И_П': ('Озеленение', 'Газон'),
    'TG_DOM': ('Здания и сооружения', 'Неизвестно'),
    '03b-5': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Рельеф__грунты_и_микроформы': ('Рельеф', 'Прочее'),
    '03119n(3n)': ('Инженерные сети', 'Электроснабжение'),
    '1185n(otk)': ('Рельеф', 'Откос'),
    '9 Урны': ('МАФ', 'Уличная мебель'),
    'ЗДАНИЯ_И_С-ДОПОЛНЕНИЯ': ('Здания и сооружения', 'Неизвестно'),
    'газопровод': ('Инженерные сети', 'Газоснабжение'),
    'Блоки деревья': ('Озеленение', 'Дерево'),
    'ДОРОГИ-ВЫСОТНЫЕ_О': ('Рельеф', 'Отметки высот'),
    'Газо__нефте__и_продуктопроводы': ('Инженерные сети', 'Газоснабжение'),
    'SETK_NAD': ('Служебные элементы', 'Координатные кресты'),
    'GAZ_35': ('Инженерные сети', 'Газоснабжение'),
    '10474-3(zb)': ('МАФ', 'Ограждение'),
    'РАСТИТЕЛЬН-КУЛЬТУРНАЯ': ('Озеленение', 'Прочее'),
    '03_Здания и строения': ('Здания и сооружения', 'Неизвестно'),
    'контур_здан_строящ': ('Здания и сооружения', 'Неизвестно'),
    '03b-4': ('Инженерные сети', 'Инженерная инфраструктура'),
    '30_Канализация': ('Инженерные сети', 'Канализация бытовая'),
    'ЗДАН_ОБЩЕСТ': ('Здания и сооружения', 'Неизвестно'),
    'OGRADA': ('МАФ', 'Ограждение'),
    'DOM': ('Здания и сооружения', 'Неизвестно'),
    'PLANT': ('Озеленение', 'Газон'),
    'Кабели низкого напряжения': ('Инженерные сети', 'Прочие кабели'),
    'Бордюр садовый': ('Элементы благоустройства', 'Бортовой камень'),
    'PI_TTDEFAULT': ('Служебные элементы', 'Прочее'),
    '10475-1(zb)': ('МАФ', 'Ограждение'),
    'РАСТРОВЫЕ_-КРЕСТЫ': ('Служебные элементы', 'Координатные кресты'),
    'Номера деревьев': ('Служебные элементы', 'Номер дерева'),
    'Земельные участки КПТ': ('Служебные элементы', 'Кадастр'),
    '19_Горизонтали': ('Рельеф', 'Горизонтали'),
    'Подерёвка': ('Служебные элементы', 'Номер дерева'),
    'Растительные_объекты': ('Озеленение', 'Прочее'),
    'ИИ_ДОРОГА_025': ('Твердые покрытия', 'Асфальт'),
    'ИИ_РЕЛЬЕФ_025': ('Рельеф', 'Прочее'),
    'PI_STD-Бонд': ('Служебные элементы', 'Прочее'),
    '5 Лавочки': ('МАФ', 'Уличная мебель'),
    '06b': ('Служебные элементы', 'Прочее'),
    '08b': ('Служебные элементы', 'Прочее')
}


def assign_layer_only_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    labels = new_mapping.get(str(row['Layer']).strip())
    if labels:
        return labels
    return row['l1'], row['l2']

df[['l1', 'l2']] = df.apply(assign_layer_only_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - total_changed)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_7.01.csv', index=False)
print(f"[7.01]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[7.01]  +52697  | размечено: 3036917 | осталось: 32763


In [106]:
new_mapping = {
    'Б горизонталь': ('Рельеф', 'Горизонтали'),
    'ПРОЧИЕ_ПЛО-ЗЕМЛИ_ОБЩ_': ('Служебные элементы', 'Границы'),
    'C_Рельеф': ('Рельеф', 'Прочее'),
    'LEP_17': ('Инженерные сети', 'Электроснабжение'),
    'TG_KANAL': ('Инженерные сети', 'Канализация бытовая'),
    'КПТ _Земельные участки': ('Служебные элементы', 'Кадастр'),
    'Съемка 28 03 23': ('Служебные элементы', 'Прочее'),
    '12492': ('Служебные элементы', 'Прочее'),
    '10476-2(zb)': ('МАФ', 'Ограждение'),
    'CONSTRACTION': ('Здания и сооружения', 'Неизвестно'),
    'СВЯЗЬ-КОЛОДЦЫ_И_': ('Инженерные сети', 'Сети связи'),
    '03133(3n)': ('Инженерные сети', 'Сети связи'),
    'ДОРОГИ-ДОРОЖНЫЕ_З': ('МАФ', 'Дорожные знаки'),
    '11_Сооружения': ('Здания и сооружения', 'Неизвестно'),
    '03129(3n)': ('Инженерные сети', 'Ливневая канализация'),
    'TG_VODA': ('Инженерные сети', 'Водопровод'),
    '11b': ('Служебные элементы', 'Прочее'),
    '10_РАМКА': ('Служебные элементы', 'Штамп'),
    'R_contorlines': ('Рельеф', 'Горизонтали'),
    '14_Ограждения': ('МАФ', 'Ограждение'),
    '06211(gid)': ('Водные объекты', 'Неизвестно'),
    'Строения': ('Здания и сооружения', 'Неизвестно'),
    'ЗД_ТОРГОВЫЕ': ('Здания и сооружения', 'Неизвестно'),
    'Бортовой камень': ('Элементы благоустройства', 'Бортовой камень'),
    'Дорога': ('Твердые покрытия', 'Асфальт'),
    'TG_OGRADA': ('МАФ', 'Ограждение'),
    '1185v(otk)': ('Рельеф', 'Откос'),
    '44_Крышки колодцев': ('Инженерные сети', 'Инженерная инфраструктура'),
    'кадастры': ('Служебные элементы', 'Кадастр'),
    'SIT_L31': ('Служебные элементы', 'Прочее'),
    'строения': ('Здания и сооружения', 'Неизвестно'),
    'SIT_LДОРОГИ бр': ('Твердые покрытия', 'Брусчатка'),
    'p': ('Служебные элементы', 'Прочее'),
    'Кабели высокого напряжения': ('Инженерные сети', 'Электроснабжение'),
    'гидрография': ('Водные объекты', 'Неизвестно'),
    'ПРОЧИЕ_ПЛО-ГРУНТЫ__ПУ': ('Водопроницаемые покрытия', 'Грунт'),
    'водопровод': ('Инженерные сети', 'Водопровод'),
    '05186(dr)': ('Твердые покрытия', 'Асфальт'),
    'КАНАЛИЗАЦИ-КАНАЛИЗАЦИ': ('Инженерные сети', 'Канализация бытовая'),
    'Граница покрытий': ('Служебные элементы', 'Контуры'),
    'Новые_номера_деревьев': ('Служебные элементы', 'Номер дерева'),
    '19_Горизонтали тонкие': ('Рельеф', 'Горизонтали'),
    'Лестницы': ('Здания и сооружения', 'Лестницы и пандусы'),
    'Дренажи': ('Инженерные сети', 'Дренаж'),
    'TG_SVAZ': ('Инженерные сети', 'Сети связи'),
    'GORIZ0': ('Рельеф', 'Горизонтали'),
    'Измерения тахеометрии': ('Служебные элементы', 'Прочее'),
    '2 Лавочки дер.': ('МАФ', 'Уличная мебель'),
    'Подписи точек': ('Рельеф', 'Отметки высот'),
    '07_Объекты электропередачи': ('Инженерные сети', 'Электроснабжение'),
    'Ограждение': ('МАФ', 'Ограждение'),
    'Водостоки': ('Инженерные сети', 'Ливневая канализация'),
    'Здания и сооружения': ('Здания и сооружения', 'Неизвестно'),
    '0137(1n)': ('Здания и сооружения', 'Лестницы и пандусы'),
    'BUILDINGS': ('Здания и сооружения', 'Неизвестно'),
    '13_Растительность': ('Озеленение', 'Газон'),
    'ZABOR': ('МАФ', 'Ограждение'),
    '08_Бордюрный камень': ('Элементы благоустройства', 'Бортовой камень'),
    '0133(1n)': ('Здания и сооружения', 'Неизвестно'),
    'кабели': ('Инженерные сети', 'Прочие кабели'),
    'SIT_L34': ('Служебные элементы', 'Прочее'),
    '2 Пикет': ('Рельеф', 'Отметки высот'),
    'Электрозащита': ('Инженерные сети', 'Электроснабжение'),
    'горизонтали не подрезанные': ('Рельеф', 'Горизонтали'),
    'водостоки': ('Инженерные сети', 'Ливневая канализация'),
    'ОГРАЖДЕНИЯ': ('МАФ', 'Ограждение'),
    'Растительность общая': ('Озеленение', 'Прочее'),
    'Зоны КПТ': ('Служебные элементы', 'Кадастр'),
    'Z': ('Рельеф', 'Отметки высот'),
    '05_Элементы зданий': ('Здания и сооружения', 'Неизвестно'),
    'ганицы зданий': ('Здания и сооружения', 'Неизвестно'),
    'GEO_WATER_PIPE': ('Инженерные сети', 'Водопровод'),
    'трубы под дорогой': ('Инженерные сети', 'Ливневая канализация'),
    'Водопровод_и_канализация': ('Инженерные сети', 'Водопровод'),
    'дорога': ('Твердые покрытия', 'Асфальт'),
    'Ограждение_металическое': ('МАФ', 'Ограждение'),
    '10280(par)': ('Здания и сооружения', 'Подпорная стенка'),
    'MOST_26': ('Здания и сооружения', 'Мост'),
    'GEO_POINTS': ('Рельеф', 'Отметки высот'),
    'неизвестно': ('Неизвестно', 'Неизвестно'),
    '9 Урна каменная': ('МАФ', 'Уличная мебель'),
    'SIT_LЗДАНИЯ': ('Здания и сооружения', 'Неизвестно'),
    'ИИ_ОГРАДА_025': ('МАФ', 'Ограждение'),
    '07320': ('Инженерные сети', 'Прочие трубопроводы'),
    'PROIZV_12': ('Служебные элементы', 'Прочее'),
    'КАНАЛИЗАЦИЯ': ('Инженерные сети', 'Канализация бытовая'),
    'Горизонтали целые': ('Рельеф', 'Горизонтали'),
    'C_Горизонтали': ('Рельеф', 'Горизонтали'),
    'горизонтали': ('Рельеф', 'Горизонтали'),
    'SIT_LДОРОГИ асф': ('Твердые покрытия', 'Асфальт'),
    'ИИ_РЕЛЬЕФ_050': ('Рельеф', 'Прочее'),
    '03106(3n)': ('Инженерные сети', 'Столбы и опоры'),
    '0124(1n)': ('Здания и сооружения', 'Неизвестно'),
    'РАСТИТЕЛЬНОС': ('Озеленение', 'Газон'),
    'SIT_L37': ('Служебные элементы', 'Прочее'),
    'Вопросы': ('Служебные элементы', 'Прочее'),
    '003 Мусорка мет': ('МАФ', 'Павильон'),
    'GEO_LVL': ('Рельеф', 'Отметки высот'),
    'Объекты_пром__комм_и_с_х_пр_ва': ('Здания и сооружения', 'Неизвестно'),
    'ТЕПЛОФИКАЦИЯ': ('Инженерные сети', 'Теплосеть'),
    '03116': ('Инженерные сети', 'Электроснабжение'),
    '07203(lst)': ('Здания и сооружения', 'Лестницы и пандусы'),
    'Горизонтали-О-MIN': ('Рельеф', 'Горизонтали'),
    'GEO_SEWAGE_PIPE': ('Инженерные сети', 'Канализация бытовая'),
    'ВОДОПРОВОД': ('Инженерные сети', 'Водопровод'),
    'GEO_PIPE_UNDER_ROADS_L': ('Инженерные сети', 'Ливневая канализация'),
    'лавочки и урны': ('МАФ', 'Уличная мебель'),
    'Сдания и сооружения': ('Здания и сооружения', 'Неизвестно'),
    'Теплосети': ('Инженерные сети', 'Теплосеть'),
    '04155(gd)': ('Элементы благоустройства', 'ЖД пути'),
    'текст': ('Служебные элементы', 'Прочее'),
    'KAN_LIVN_34': ('Инженерные сети', 'Ливневая канализация'),
    'ЭЛЕКТ_КАБЕЛИ': ('Инженерные сети', 'Прочие кабели'),
    'Кабели радио': ('Инженерные сети', 'Сети связи'),
    'TG_GAS': ('Инженерные сети', 'Газоснабжение'),
    'OPORA_16': ('Инженерные сети', 'Столбы и опоры'),
    '0147(1n)': ('Здания и сооружения', 'Неизвестно'),
    'GOR_B0': ('Рельеф', 'Горизонтали'),
    'CodeLine_1': ('Служебные элементы', 'Прочее'),
    'SIT_LДОРОГИ тропы': ('Водопроницаемые покрытия', 'Грунт'),
    'СВЯЗЬ': ('Инженерные сети', 'Сети связи'),
    '31_Водопровод': ('Инженерные сети', 'Водопровод'),
    '05206': ('МАФ', 'Информационная конструкция'),
    '11_ЗемельныеУчастки': ('Служебные элементы', 'Кадастр'),
    'Характеристики покрытий и дорог': ('Служебные элементы', 'Прочее'),
    'Водослив': ('Инженерные сети', 'Ливневая канализация'),
    'GIDROTEH_25': ('Водные объекты', 'Неизвестно'),
    'резиновое покрытие': ('Твердые покрытия', 'Резиновая крошка'),
    'Горизонтали-6-MAJ': ('Рельеф', 'Горизонтали'),
    'Сопряжение_поверхностей': ('Рельеф', 'Откос'),
    'Ограды сетка': ('МАФ', 'Ограждение'),
    'SYMBOL': ('Служебные элементы', 'Условные обозначения'),
    'Теплосети_Опоры Назем': ('Инженерные сети', 'Теплосеть'),
    'Линии совмещения листов': ('Служебные элементы', 'Штамп'),
    'Воздухопроводы': ('Инженерные сети', 'Инженерная инфраструктура'),
    'электрика': ('Инженерные сети', 'Электроснабжение'),
    '03121(3n)': ('Инженерные сети', 'Прочие трубопроводы'),
    '001 Лавочки мет.': ('МАФ', 'Уличная мебель'),
    '22_Опоры': ('Инженерные сети', 'Столбы и опоры'),
    'маф': ('МАФ', 'Прочее'),
    '03123(3n)': ('Инженерные сети', 'Прочие трубопроводы'),
    'Dorstroy_23': ('Твердые покрытия', 'Асфальт'),
    'Подпорная стенка': ('Здания и сооружения', 'Подпорные стенки'),
    'TG_CABEL': ('Инженерные сети', 'Электроснабжение'),
    'КАНАЛИЗАЦИ-РЕШЕТКИ_И_': ('Инженерные сети', 'Ливневая канализация'),
    'Дренаж': ('Инженерные сети', 'Дренаж'),
    'КАНАЛ_ЛИВНЕВ': ('Инженерные сети', 'Ливневая канализация'),
    '64_Отметка высоты крышки колодца': ('Рельеф', 'Отметки высот'),
    '11_Гидрография': ('Водные объекты', 'Неизвестно')
}



def assign_layer_only_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    labels = new_mapping.get(str(row['Layer']).strip())
    if labels:
        return labels
    return row['l1'], row['l2']

df[['l1', 'l2']] = df.apply(assign_layer_only_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - total_changed)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_7.02.csv', index=False)
print(f"[7.02]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[7.02]  +18867  | размечено: 3055784 | осталось: 13896


In [123]:
new_mapping = {
    '05191(dr)': ('Твердые покрытия', 'Асфальт'),
    '10475-2(zb)': ('МАФ', 'Ограждение'),
    '09_Путевое хозяйство': ('Элементы благоустройства', 'ЖД пути'),
    '35_Телефон': ('Инженерные сети', 'Сети связи'),
    'Цветники': ('Озеленение', 'Цветник'),
    'Урны': ('МАФ', 'Уличная мебель'),
    'Горизонтали-О-MAJ': ('Рельеф', 'Горизонтали'),
    '08386-1(pkd)': ('Озеленение', 'Кустарник'),
    'Коммуникации_и_выходы_коммуникаций': ('Инженерные сети', 'Инженерная инфраструктура'),
    '12494': ('Служебные элементы', 'Прочее'),
    'SIT_L12': ('Служебные элементы', 'Прочее'),
    '10366(zb)': ('МАФ', 'Ограждение'),
    '32_Теплосеть': ('Инженерные сети', 'Теплосеть'),
    '002 Лавочки дер.': ('МАФ', 'Уличная мебель'),
    'C': ('Служебные элементы', 'Прочее'),
    'Столбы__фермы_и_сооружения_на_них': ('Инженерные сети', 'Столбы и опоры'),
    'ГАЗОПРОВОД-КОЛОДЦЫ_И_': ('Инженерные сети', 'Газоснабжение'),
    'Отмостка': ('Здания и сооружения', 'Неизвестно'),
    'Объекты электро передачи': ('Инженерные сети', 'Электроснабжение'),
    'Зем. участки': ('Служебные элементы', 'Кадастр'),
    'клумба': ('Озеленение', 'Цветник'),
    '0139(1n)': ('Здания и сооружения', 'Лестницы и пандусы'),
    'сооружения': ('Здания и сооружения', 'Неизвестно'),
    'D': ('Служебные элементы', 'Прочее'),
    '06466(bol)': ('Водные объекты', 'Неизвестно'),
    'Строение и сооружение': ('Здания и сооружения', 'Неизвестно'),
    'Высоты': ('Рельеф', 'Отметки высот'),
    'Опоры-столбы': ('Инженерные сети', 'Столбы и опоры'),
    '1деревья': ('Озеленение', 'Дерево'),
    'Горизонтали_00': ('Рельеф', 'Горизонтали'),
    'Номер дерева': ('Служебные элементы', 'Номер дерева'),
    'd': ('Служебные элементы', 'Прочее'),
    '38_Кабели высокого напряжения': ('Инженерные сети', 'Электроснабжение'),
    'Тропы': ('Водопроницаемые покрытия', 'Грунт'),
    'ИИ_КАБЕЛЬ_025': ('Инженерные сети', 'Прочие кабели'),
    'GEO_LVC': ('Инженерные сети', 'Электроснабжение'),
    '11 Качели перевесы': ('МАФ', 'Игровое оборудование'),
    'GEO_HVC': ('Инженерные сети', 'Электроснабжение'),
    'канализация напорная': ('Инженерные сети', 'Канализация бытовая'),
    'TROPA_1': ('Водопроницаемые покрытия', 'Грунт'),
    'TG_LKANAL': ('Инженерные сети', 'Ливневая канализация'),
    '33_Газопровод': ('Инженерные сети', 'Газоснабжение'),
    'GEO_TEPLO_PIPE': ('Инженерные сети', 'Теплосеть'),
    '08b подерёвка': ('Озеленение', 'Номер дерева'),
    'Ливневая канализация': ('Инженерные сети', 'Ливневая канализация'),
    'TROPA_1_0m': ('Водопроницаемые покрытия', 'Грунт'),
    'номдер': ('Служебные элементы', 'Номер дерева'),
    'SVIAZ': ('Инженерные сети', 'Сети связи'),
    'Напорный коллектор': ('Инженерные сети', 'Канализация бытовая'),
    'ЭЛЕКТРОСЕТ-МУФТЫ_РАСП': ('Инженерные сети', 'Электроснабжение'),
    'TG_TEPLO': ('Инженерные сети', 'Теплосеть'),
    'трава': ('Озеленение', 'Газон'),
    '20_Горизонтали утолщенные': ('Рельеф', 'Горизонтали'),
    '21 Горка': ('МАФ', 'Игровое оборудование'),
    'земельные участки': ('Служебные элементы', 'Кадастр'),
    'теплотрасса надземная': ('Инженерные сети', 'Теплосеть'),
    '36 Вазы для цветов': ('МАФ', 'Вазоны'),
    'GEO_STORM_SEWAGE_PIPE': ('Инженерные сети', 'Ливневая канализация'),
    'номера деревьев': ('Служебные элементы', 'Номер дерева'),
    'здание': ('Здания и сооружения', 'Неизвестно'),
    'Оси': ('Служебные элементы', 'Прочее'),
    'Ливневые канализации': ('Инженерные сети', 'Ливневая канализация'),
    'Стрелка': ('Служебные элементы', 'Прочее'),
    'канализация2': ('Инженерные сети', 'Канализация бытовая'),
    '10473(zb)': ('МАФ', 'Ограждение'),
    'контуры кустарников': ('Озеленение', 'Кустарник'),
    'TG_OTMETKA': ('Рельеф', 'Отметки высот'),
    'TG_HYDROGRAPHY': ('Водные объекты', 'Неизвестно'),
    'SEWAGE': ('Инженерные сети', 'Канализация бытовая'),
    'ramka': ('Служебные элементы', 'Штамп'),
    'Сооружения КПТ': ('Служебные элементы', 'Кадастр'),
    'SIT_L16': ('Служебные элементы', 'Прочее'),
    'Отметки': ('Рельеф', 'Отметки высот'),
    'Канализация': ('Инженерные сети', 'Канализация бытовая'),
    'Откос_верх': ('Рельеф', 'Откос'),
    'ПРОЧИЕ_ОБЪ-ФОНТАНЫ': ('Водные объекты', 'Неизвестно'),
    'Сетка координат': ('Служебные элементы', 'Координатные кресты'),
    'ТЕПЛОСЕТИ-КОЛОДЦЫ_И_': ('Инженерные сети', 'Теплосеть'),
    '019 табличка': ('МАФ', 'Информационная конструкция'),
    'Бетонный лоток': ('Инженерные сети', 'Ливневая канализация'),
    'Козырёк': ('Здания и сооружения', 'Неизвестно'),
    'Остановка': ('МАФ', 'Павильон'),
    'Ограды деревянные': ('МАФ', 'Ограждение'),
    'TEL_CABLE': ('Инженерные сети', 'Сети связи'),
    'T-Rastit': ('Озеленение', 'Газон'),
    'Водопровод2': ('Инженерные сети', 'Водопровод'),
    '20_Социальные_объекты': ('Здания и сооружения', 'Неизвестно'),
    'T-Ograda': ('МАФ', 'Ограждение'),
    'Строение и сооружения': ('Здания и сооружения', 'Неизвестно'),
    'TG_RAST': ('Озеленение', 'Газон'),
    '07319': ('Водные объекты', 'Неизвестно'),
    'ПРОЧИЕ_ОБЪ-АБСТРАКТНЫ': ('Служебные элементы', 'Прочее'),
    '11_Земельные участки': ('Служебные элементы', 'Кадастр'),
    '015 Клумба': ('МАФ', 'Вазоны'),
    '0154(1n)': ('Здания и сооружения', 'Неизвестно'),
    '27Тренажеры': ('МАФ', 'Спортивное оборудование'),
    'канавы': ('Водные объекты', 'Неизвестно'),
    '37_Кабели низкого напряжения': ('Инженерные сети', 'Электроснабжение'),
    'ИИ_ЗДАНИЕ_025': ('Здания и сооружения', 'Неизвестно'),
    'ПАМЯТНИКИ': ('МАФ', 'Памятники'),
    'ДОРОГИ-ОСИ_И_НАЗВ': ('Твердые покрытия', 'Асфальт'),
    'GAZ_PIPE': ('Инженерные сети', 'Газоснабжение'),
    '07313': ('Здания и сооружения', 'Мост'),
    'WATER_PIPE': ('Инженерные сети', 'Водопровод'),
    '0138(1n)': ('Здания и сооружения', 'Лестницы и пандусы'),
    '37_Кабель низкого напряжения': ('Инженерные сети', 'Электроснабжение'),
    'ТЕКСТ': ('Служебные элементы', 'Прочее'),
    'ЗемельныеУчастки': ('Служебные элементы', 'Кадастр'),
    'лестница': ('Здания и сооружения', 'Лестницы и пандусы'),
    'GEO_KLS_LINE': ('Инженерные сети', 'Сети связи'),
    '07323о': ('Инженерные сети', 'Прочие трубопроводы'),
    'ELEKTRO': ('Инженерные сети', 'Электроснабжение'),
    'Футляры и каналы': ('Инженерные сети', 'Инженерная инфраструктура'),
    'здания и сооружения': ('Здания и сооружения', 'Неизвестно'),
    '15 Клумба': ('МАФ', 'Вазоны'),
    'Топонимика': ('Служебные элементы', 'Прочее'),
    'SKLAD_8': ('Здания и сооружения', 'Неизвестно'),
    'STREET_LAMP': ('Инженерные сети', 'Столбы и опоры'),
    'ГАЗОПРОВОД': ('Инженерные сети', 'Газоснабжение'),
    'Линиии совмещения листов': ('Служебные элементы', 'Условные обозначения'),
    '3номер дерева': ('Служебные элементы', 'Номер дерева'),
    '11_Здания': ('Здания и сооружения', 'Неизвестно'),
    '22 Турник': ('МАФ', 'Спортивное оборудование'),
    '0121(1n)': ('Здания и сооружения', 'Неизвестно'),
    'ОПОРЫ__СТО-ОПОРЫ__СТО': ('Инженерные сети', 'Столбы и опоры'),
    'SIT_L33': ('Служебные элементы', 'Прочее'),
    'Границы участков': ('Служебные элементы', 'Границы'),
    'TROPA_2.5': ('Водопроницаемые покрытия', 'Грунт'),
    'Клумбы': ('Озеленение', 'Вазоны'),
    'ЛИНИИ': ('Служебные элементы', 'Прочее'),
    'Граница съемки': ('Служебные элементы', 'Границы'),
    'МАФы': ('МАФ', 'Прочее'),
    'ТЕПЛОСЕТИ-ПОТРЕБИТЕЛ': ('Инженерные сети', 'Теплосеть'),
    '01_Геодезические пункты': ('Рельеф', 'Отметки высот'),
    'Снаряд спортивный': ('МАФ', 'Спортивное оборудование'),
    'Деревья': ('Озеленение', 'Дерево'),
    '08386-2(pkd)': ('Озеленение', 'Кустарник'),
    'вопросики': ('Служебные элементы', 'Прочее'),
    'КПТ_Здания': ('Служебные элементы', 'Кадастр'),
    'здания КПТ': ('Служебные элементы', 'Кадастр'),
    'SIT_L32': ('Служебные элементы', 'Прочее'),
    'деревья_доп': ('Озеленение', 'Дерево'),
    '26 Велопарковки': ('МАФ', 'Уличная мебель'),
    'Горизантали_пруда 2022': ('Рельеф', 'Горизонтали'),
    '06212(gid)': ('Водные объекты', 'Неизвестно'),
    '06248(gid)': ('Водные объекты', 'Неизвестно'),
    'тропы': ('Водопроницаемые покрытия', 'Грунт'),
    '12 Кораблик': ('МАФ', 'Игровое оборудование'),
    'Поросль': ('Озеленение', 'Кустарник'),
    '0148(1n)': ('Здания и сооружения', 'Неизвестно'),
    'ПРОЧИЕ_ОБЪ-ПАМЯТНИКИ_': ('МАФ', 'Памятники')
}




def assign_layer_only_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    labels = new_mapping.get(str(row['Layer']).strip())
    if labels:
        return labels
    return row['l1'], row['l2']

df[['l1', 'l2']] = df.apply(assign_layer_only_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - total_changed)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_7.03.csv', index=False)
print(f"[7.03]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[7.03]  +4806  | размечено: 3060590 | осталось: 9090


In [138]:
new_mapping = {
    'инф стенд': ('МАФ', 'Информационная конструкция'),
    'AUTODOR_22': ('Твердые покрытия', 'Асфальт'),
    'Пруд': ('Водные объекты', 'Неизвестно'),
    'TEPLO_36': ('Инженерные сети', 'Теплосеть'),
    '614': ('Служебные элементы', 'Прочее'),
    'NAD_MDEFAULT': ('Служебные элементы', 'Прочее'),
    'граница съемки': ('Служебные элементы', 'Границы'),
    'line': ('Служебные элементы', 'Прочее'),
    'Ограждение_деревянное': ('МАФ', 'Ограждение'),
    'Боллард': ('МАФ', 'Ограждение'),
    'ОПОРЫ': ('Инженерные сети', 'Столбы и опоры'),
    '715': ('Служебные элементы', 'Прочее'),
    'Реклама': ('МАФ', 'Информационная конструкция'),
    'ИИ_ТРУБ_Канализация_025': ('Инженерные сети', 'Канализация бытовая'),
    'вопросы': ('Служебные элементы', 'Прочее'),
    '05195(dr)': ('Водопроницаемые покрытия', 'Грунт'),
    '61_Отметки высоты поверхности': ('Рельеф', 'Отметки высот'),
    '0158(1n)': ('МАФ', 'Павильон'),
    'Граница работ': ('Служебные элементы', 'Границы'),
    'Кабель связи': ('Инженерные сети', 'Сети связи'),
    'Ливневой канал': ('Инженерные сети', 'Ливневая канализация'),
    'Элементы благоустройства': ('МАФ', 'Прочее'),
    'плавающие': ('Служебные элементы', 'Прочее'),
    'HEAT_PIPE': ('Инженерные сети', 'Теплосеть'),
    'T-Dor': ('Твердые покрытия', 'Асфальт'),
    'Газопровод': ('Инженерные сети', 'Газоснабжение'),
    'Откос_низ': ('Рельеф', 'Откос'),
    '10280v(par)': ('Здания и сооружения', 'Подпорная стенка'),
    '03121-3(3n)': ('Инженерные сети', 'Прочие трубопроводы'),
    'TG_DRENAZ': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Трубопроводы спецназначения': ('Инженерные сети', 'Прочие трубопроводы'),
    'GEO_GAZ_PIPE': ('Инженерные сети', 'Газоснабжение'),
    '7 Шведская стенка': ('МАФ', 'Спортивное оборудование'),
    'b': ('Служебные элементы', 'Прочее'),
    'PI_NUDEFAULT': ('Служебные элементы', 'Прочее'),
    '1': ('Служебные элементы', 'Прочее'),
    '3 Мусорка мет': ('МАФ', 'Уличная мебель'),
    'ИИ_ТРУБ_ТС_025': ('Инженерные сети', 'Теплосеть'),
    '03115v(3n 3nr)': ('Инженерные сети', 'Электроснабжение'),
    'TG_STOLBY': ('Инженерные сети', 'Столбы и опоры'),
    'ИИ_ЛЭП_025': ('Инженерные сети', 'Электроснабжение'),
    '4 Песочница': ('МАФ', 'Игровое оборудование'),
    'TG_USL_ZN': ('Служебные элементы', 'Условные обозначения'),
    'Участки КПТ 50_57_0000000': ('Служебные элементы', 'Кадастр'),
    'ASFALT': ('Твердые покрытия', 'Асфальт'),
    't': ('Служебные элементы', 'Прочее'),
    'Сдания и сооружение': ('Здания и сооружения', 'Неизвестно'),
    'зоны': ('Служебные элементы', 'Границы'),
    '3 Карусель': ('МАФ', 'Игровое оборудование'),
    'Контура растительности': ('Озеленение', 'Газон'),
    'здания': ('Здания и сооружения', 'Неизвестно'),
    'Пикеты': ('Рельеф', 'Отметки высот'),
    'СООРУЖЕНИЯ': ('Здания и сооружения', 'Неизвестно'),
    'разбивка на листы': ('Служебные элементы', 'Штамп'),
    '015 Горка': ('МАФ', 'Игровое оборудование'),
    '15 Горка': ('МАФ', 'Игровое оборудование'),
    'TG_RELIEF': ('Рельеф', 'Прочее'),
    '+Канава': ('Водные объекты', 'Неизвестно'),
    'CK_ЛЭП_высоковольтная': ('Инженерные сети', 'Электроснабжение'),
    'ЭЛЕКТРОСЕТ-Т_П': ('Инженерные сети', 'Электроснабжение'),
    '0_граница новая': ('Служебные элементы', 'Границы'),
    'SIT_L21': ('Служебные элементы', 'Прочее'),
    'ИИ_ТРУБ_Вода_025': ('Инженерные сети', 'Водопровод'),
    'Стол для пинг-понга': ('МАФ', 'Спортивное оборудование'),
    'Отметка_дата': ('Служебные элементы', 'Прочее'),
    'вопр': ('Служебные элементы', 'Прочее'),
    'C_Границы покрытий': ('Служебные элементы', 'Контуры'),
    'ТЕПЛОСЕТИ-АРМАТУРА': ('Инженерные сети', 'Теплосеть'),
    'T-Otkos': ('Рельеф', 'Откос'),
    '0144(1n)': ('Здания и сооружения', 'Неизвестно'),
    'Металлические клубы': ('МАФ', 'Вазоны'),
    '11_Зоны': ('Служебные элементы', 'Границы'),
    'ЗДАНИЯ_И_С-ЗДАНИЯ_СТР': ('Здания и сооружения', 'Неизвестно'),
    'TROPA_1.5m': ('Водопроницаемые покрытия', 'Грунт'),
    'РАМКА': ('Служебные элементы', 'Штамп'),
    'Граница': ('Служебные элементы', 'Границы'),
    '0299': ('Здания и сооружения', 'Неизвестно'),
    'Качели металл Н 2.5м': ('МАФ', 'Игровое оборудование'),
    'граница': ('Служебные элементы', 'Границы'),
    'Навесы': ('Здания и сооружения', 'Неизвестно'),
    'Электрокабели': ('Инженерные сети', 'Электроснабжение'),
    '0_Граница проектирования нов': ('Служебные элементы', 'Границы'),
    'ELECTROCABLE': ('Инженерные сети', 'Электроснабжение'),
    'Сопряжения с фрагментами': ('Служебные элементы', 'Условные обозначения'),
    'Измерения ПВО': ('Служебные элементы', 'Прочее'),
    'TG_ZAKAZ': ('Служебные элементы', 'Прочее'),
    'СВЯЗЬ-ЛИНИИ_СВЯЗ': ('Инженерные сети', 'Сети связи'),
    '8 Спорткомплекс': ('МАФ', 'Спортивное оборудование'),
    '12 Шведская стенка': ('МАФ', 'Спортивное оборудование'),
    'Инженерно-технические_объекты': ('Инженерные сети', 'Инженерная инфраструктура'),
    '12 Качели на пружине': ('МАФ', 'Игровое оборудование'),
    'канавы_сплош': ('Водные объекты', 'Неизвестно'),
    'ГП2 ограждение': ('МАФ', 'Ограждение'),
    'Основные горизонтали _пруда 2022': ('Рельеф', 'Горизонтали'),
    '1 Лавочки мет.': ('МАФ', 'Уличная мебель'),
    '1 Лежаки': ('МАФ', 'Уличная мебель'),
    'РАСТИТЕЛЬНОСТЬ': ('Озеленение', 'Газон'),
    'стол': ('МАФ', 'Уличная мебель'),
    'Баскетбольное кольцо': ('МАФ', 'Спортивное оборудование'),
    'ПРОЧИЕ_ПЛО-АВТОСТОЯНК': ('Элементы благоустройства', 'Парковка'),
    'Разметка': ('Элементы благоустройства', 'Дорожная разметка'),
    'ИЛОПРОВОД': ('Инженерные сети', 'Прочие трубопроводы'),
    '16 Карусель': ('МАФ', 'Игровое оборудование'),
    'Бетон': ('Твердые покрытия', 'Бетон'),
    'SVYAZ': ('Инженерные сети', 'Сети связи'),
    'рельеф': ('Рельеф', 'Горизонтали'),
    'Ограды металлические ниже 1 м': ('МАФ', 'Ограждение'),
    '10476-1(zb)': ('МАФ', 'Ограждение'),
    '35_Кабели связи': ('Инженерные сети', 'Сети связи'),
    'Канава': ('Водные объекты', 'Неизвестно'),
    'граница полевиков': ('Служебные элементы', 'Границы'),
    '18 Качели подвесные': ('МАФ', 'Игровое оборудование'),
    'WATER_DRAIN': ('Инженерные сети', 'Ливневая канализация'),
    'РАСТИТЕЛЬН-ХАРАКТЕРИС': ('Озеленение', 'Дерево'),
    'брусья': ('МАФ', 'Спортивное оборудование'),
    'ВОДОПРОВОД-ПОВРЕЖДЕНИ': ('Инженерные сети', 'Водопровод'),
    'СКЛАДСКИЕ': ('Здания и сооружения', 'Неизвестно'),
    'Связь': ('Инженерные сети', 'Сети связи'),
    'LEP': ('Инженерные сети', 'Электроснабжение'),
    'Бытовая канализация': ('Инженерные сети', 'Канализация бытовая'),
    'PAMYAT_11': ('МАФ', 'Памятники'),
    'дренаж': ('Инженерные сети', 'Дренаж'),
    'Телефонная канализация': ('Инженерные сети', 'Сети связи'),
    'RAMKA': ('Служебные элементы', 'Штамп'),
    'TG_KOLOD': ('Инженерные сети', 'Инженерная инфраструктура'),
    'z_Условные знаки': ('Служебные элементы', 'Условные обозначения'),
    '004 Качели': ('МАФ', 'Игровое оборудование'),
    'Турник': ('МАФ', 'Спортивное оборудование'),
    'Участки КПТ 50_57_0020419': ('Служебные элементы', 'Кадастр'),
    'c': ('Служебные элементы', 'Прочее'),
    'PI_STДОРОГИ': ('Твердые покрытия', 'Асфальт'),
    '1 Дома': ('Здания и сооружения', 'Неизвестно'),
    '!граница': ('Служебные элементы', 'Границы'),
    'ОБЪЕКТ_ПРОИЗ': ('Здания и сооружения', 'Неизвестно'),
    'ПРОЧИЕ_ОБЪ-ОБЪЕКТЫ_ПР': ('Служебные элементы', 'Прочее'),
    'Ограды металлические выше 1 м': ('МАФ', 'Ограждение'),
    'Газ': ('Инженерные сети', 'Газоснабжение'),
    'Змейка': ('МАФ', 'Игровое оборудование'),
    '12117': ('Рельеф', 'Отметки высот'),
    'ПРОЧИЕ_ПЛО-ДЕТСКИЕ_ПЛ': ('МАФ', 'Игровое оборудование'),
    'Информационные и рекламные щиты': ('МАФ', 'Информационная конструкция'),
    'Качели подвесные': ('МАФ', 'Игровое оборудование'),
    'Default': ('Служебные элементы', 'Прочее'),
    "Из КВЗУ на '50_57_0000000_28959' (из файла 'C_-Users-Людмила-AppData-Local-Temp-b4ba6c70-e025-4580-966b-c339fe6e14ed.xml')": ('Служебные элементы', 'Кадастр'),
    'Канализация ливневая': ('Инженерные сети', 'Ливневая канализация'),
    '51311200': ('Служебные элементы', 'Прочее'),
    'высоты деревьев': ('Служебные элементы', 'Размеры дерева'),
    'Defpoints': ('Служебные элементы', 'Прочее'),
    '10Домик с горкой': ('МАФ', 'Игровое оборудование')
}





def assign_layer_only_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    labels = new_mapping.get(str(row['Layer']).strip())
    if labels:
        return labels
    return row['l1'], row['l2']

df[['l1', 'l2']] = df.apply(assign_layer_only_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - total_changed)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_7.04.csv', index=False)
print(f"[7.04]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[7.04]  +1004  | размечено: 3061594 | осталось: 8086


In [153]:
new_mapping = {
    'ворота': ('МАФ', 'Ворота'),
    'Качели на пружинах': ('МАФ', 'Игровое оборудование'),
    'ЭЛЕКТРОСЕТ-ТР_РЫ': ('Инженерные сети', 'Электроснабжение'),
    'ЛЭП__ЛЭС_И_ВОЗДУ': ('Инженерные сети', 'Электроснабжение'),
    'Лесница': ('Здания и сооружения', 'Лестницы и пандусы'),
    'RAST_KULT_28': ('Озеленение', 'Газон'),
    'PI_TT0': ('Служебные элементы', 'Прочее'),
    'граница участка работ': ('Служебные элементы', 'Границы'),
    '4 Качели': ('МАФ', 'Игровое оборудование'),
    'C_Растительность': ('Озеленение', 'Газон'),
    'КОНТУРЫ КУСТАРНИКОВ': ('Озеленение', 'Кустарник'),
    'футбольные ворота': ('МАФ', 'Спортивное оборудование'),
    '8 Мусорные контейнеры': ('МАФ', 'Уличная мебель'),
    '710': ('Служебные элементы', 'Прочее'),
    'CK_Водопровод': ('Инженерные сети', 'Водопровод'),
    'рстительность': ('Озеленение', 'Газон'),
    'рельеф вспом': ('Рельеф', 'Горизонтали'),
    'подпорные стенки': ('Здания и сооружения', 'Подпорные стенки'),
    'ИИ_КОНТУР_025': ('Служебные элементы', 'Контуры'),
    '011 Кормушки для птиц': ('МАФ', 'Уличная мебель'),
    'навис ч строений': ('Здания и сооружения', 'Неизвестно'),
    'лэп_низ_напр_сплош': ('Инженерные сети', 'Электроснабжение'),
    '012 Шведская стенка': ('МАФ', 'Спортивное оборудование'),
    'канализация напорная2': ('Инженерные сети', 'Канализация бытовая'),
    '013 Качели перевес': ('МАФ', 'Игровое оборудование'),
    '014 Песочница': ('МАФ', 'Игровое оборудование'),
    'Ограды каменные ниже 1 м': ('МАФ', 'Ограждение'),
    'Путевое хозяйство': ('Элементы благоустройства', 'ЖД пути'),
    '13 Резиновая фигура': ('МАФ', 'Игровое оборудование'),
    'СТРОЕНИЯ__ЗДАНИЯ': ('Здания и сооружения', 'Неизвестно'),
    'Kom_Kab_EL_hi': ('Инженерные сети', 'Электроснабжение'),
    'Подеревка': ('Служебные элементы', 'Номер дерева'),
    'OBSCHEST_4': ('Здания и сооружения', 'Неизвестно'),
    'Вершины 3d полилинии': ('Служебные элементы', 'Прочее'),
    'Подпорная': ('Здания и сооружения', 'Подпорные стенки'),
    'ПРОЧИЕ_СЕТ-СЕТИ': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Дорожные знаки и дорожная инфраструктура': ('МАФ', 'Дорожные знаки'),
    'KLMN': ('Служебные элементы', 'Прочее'),
    "Из КВЗУ на '50_57_0000000_28965' (из файла 'C_-Users-Людмила-AppData-Local-Temp-7e20b6d1-1faf-4adb-9827-59238b396e0a.xml')": ('Служебные элементы', 'Кадастр'),
    'Оформление': ('Служебные элементы', 'Прочее'),
    "Из КВЗУ на '50_57_0080301_314' (из файла 'C_-Users-Людмила-AppData-Local-Temp-552e16f3-61d6-43b3-8524-f14e6136c55f.xml')": ('Служебные элементы', 'Кадастр'),
    '50_Воздушные_линии_связи': ('Инженерные сети', 'Сети связи'),
    'СЂР°СЃС‚РёС‚РµР»СЊРЅРѕСЃС‚СЊ': ('Служебные элементы', 'Прочее'),
    '015 Деревянные фигурки': ('МАФ', 'Игровое оборудование'),
    'Kom_GAZ': ('Инженерные сети', 'Газоснабжение'),
    'Канавы': ('Водные объекты', 'Неизвестно'),
    'СПЕЦЭКСПЛУАТАЦИЯ': ('Служебные элементы', 'Прочее'),
    'Канал.ливн': ('Инженерные сети', 'Ливневая канализация'),
    'Высотные отметки': ('Рельеф', 'Отметки высот'),
    'Kom_Kab_EL_low': ('Инженерные сети', 'Электроснабжение'),
    'ПЛОЩАДИ': ('Твердые покрытия', 'Плитка'),
    'дополнительные отметки': ('Рельеф', 'Отметки высот'),
    'дог грунт': ('Водопроницаемые покрытия', 'Грунт'),
    'Дерево для замков': ('Служебные элементы', 'Прочее'),
    'дер на бет осн': ('Служебные элементы', 'Прочее'),
    'КОЛОДЦЫ': ('Инженерные сети', 'Инженерная инфраструктура'),
    'катодная защита': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Канатная дорога': ('Здания и сооружения', 'Неизвестно'),
    'Водопровод подземный': ('Инженерные сети', 'Водопровод'),
    'шлагбаум': ('МАФ', 'Шлагбаум'),
    'ходд': ('Служебные элементы', 'Прочее'),
    '10476-3a(zb)': ('МАФ', 'Ограждение'),
    'Подпорная стенка отвесная': ('Здания и сооружения', 'Подпорные стенки'),
    'топоОтметка': ('Рельеф', 'Отметки высот'),
    'Kom_Kanaliz_1_bit': ('Инженерные сети', 'Канализация бытовая'),
    '+Лестница': ('Здания и сооружения', 'Лестницы и пандусы'),
    'CK_Кабель низкого напряжения': ('Инженерные сети', 'Электроснабжение'),
    'ЗемельныеУчасткиИстория': ('Служебные элементы', 'Кадастр'),
    'скворечник': ('МАФ', 'Уличная мебель'),
    'ИИ_ТРУБ_ГАЗ_025': ('Инженерные сети', 'Газоснабжение'),
    'Kom_Kanaliz_2_Livn': ('Инженерные сети', 'Ливневая канализация'),
    '006 Полка с книгами': ('МАФ', 'Уличная мебель'),
    '009 Урна каменная': ('МАФ', 'Уличная мебель'),
    '1 Турник': ('МАФ', 'Спортивное оборудование'),
    'Survey.td2': ('Рельеф', 'Отметки высот'),
    '011 Паутинка': ('МАФ', 'Игровое оборудование'),
    '1185(otk)': ('Рельеф', 'Откос'),
    'мельница': ('МАФ', 'Игровое оборудование'),
    '6 Полка с книгами': ('МАФ', 'Уличная мебель'),
    'высоты': ('Рельеф', 'Отметки высот'),
    'деревья': ('Озеленение', 'Дерево'),
    '12 Стол для пин понга': ('МАФ', 'Спортивное оборудование'),
    'Теплосеть подземная': ('Инженерные сети', 'Теплосеть'),
    '03114v(3n 3nr)': ('Инженерные сети', 'Электроснабжение'),
    'Шлагбаум': ('МАФ', 'Шлагбаум'),
    'ГРАНИЦА РАБОТ': ('Служебные элементы', 'Границы'),
    'Форсаж': ('Служебные элементы', 'Прочее'),
    'GIDROTEH': ('Водные объекты', 'Неизвестно'),
    'Турники': ('МАФ', 'Спортивное оборудование'),
    'Неизвестно': ('Неизвестно', 'Неизвестно'),
    'ГИДРОГРАФИ-РЕКИ': ('Водные объекты', 'Неизвестно'),
    'Goriz_2': ('Рельеф', 'Горизонтали'),
    '04156(gd)': ('Элементы благоустройства', 'ЖД пути'),
    'KANALIZACIA': ('Инженерные сети', 'Канализация бытовая'),
    '19 Детская беседка': ('МАФ', 'Игровое оборудование'),
    '17 Змейка': ('МАФ', 'Игровое оборудование'),
    '06237(gid)': ('Водные объекты', 'Неизвестно'),
    'ЛЭП и освещение': ('Инженерные сети', 'Электроснабжение'),
    '018 Гимнастическое бревно': ('МАФ', 'Спортивное оборудование'),
    'Качели на пружинке': ('МАФ', 'Игровое оборудование'),
    'Коды пикетов': ('Служебные элементы', 'Прочее'),
    'abris_3': ('Служебные элементы', 'Прочее'),
    '12_ЗемельныеУчастки': ('Служебные элементы', 'Кадастр'),
    'ГАЗОПРОВОД-ГРП': ('Инженерные сети', 'Газоснабжение'),
    'abris dg_5': ('Служебные элементы', 'Прочее'),
    'рамка': ('Служебные элементы', 'Штамп'),
    'Дорожные знаки светофоры': ('МАФ', 'Дорожные знаки'),
    'Теплосеть': ('Инженерные сети', 'Теплосеть'),
    '+Бордюры': ('Элементы благоустройства', 'Бортовой камень'),
    'отметки': ('Рельеф', 'Отметки высот'),
    'ИИ_ГИДРОГРАФИЯ_025': ('Водные объекты', 'Неизвестно'),
    '010Домик с горкой': ('МАФ', 'Игровое оборудование'),
    'Столбы': ('Инженерные сети', 'Столбы и опоры'),
    'МОСТЫ': ('Здания и сооружения', 'Мост'),
    'Труба': ('Инженерные сети', 'Прочие трубопроводы'),
    'номер дерева': ('Служебные элементы', 'Номер дерева'),
    'ИИ_ОПОРА_025': ('Инженерные сети', 'Столбы и опоры'),
    '005 Столик': ('МАФ', 'Уличная мебель'),
    '03b-3': ('Инженерные сети', 'Столбы и опоры'),
    '2': ('Служебные элементы', 'Прочее'),
    '05b': ('МАФ', 'Дорожные знаки'),
    'HOR2': ('Рельеф', 'Горизонтали'),
    'Ограда металлическая на мет. столбе (h до 1м)': ('МАФ', 'Ограждение'),
    '017 Змейка': ('МАФ', 'Игровое оборудование'),
    '10475-3(zb)': ('МАФ', 'Ограждение'),
    'T-Zdanie': ('Здания и сооружения', 'Неизвестно'),
    '016 Карусель': ('МАФ', 'Игровое оборудование'),
    'ВОДООТВЕДЕ-ПОВРЕЖДЕНИ': ('Инженерные сети', 'Канализация бытовая'),
    '20 Теннисные столы': ('МАФ', 'Спортивное оборудование'),
    'Гидротехнические объекты': ('Водные объекты', 'Неизвестно'),
    'Survey.td2_point_desc_Mark': ('Рельеф', 'Отметки высот'),
    'Бревно': ('МАФ', 'Спортивное оборудование'),
    'abris dg_6': ('Служебные элементы', 'Прочее'),
    '0153(1n)': ('Здания и сооружения', 'Неизвестно'),
    'CK_Канализация': ('Инженерные сети', 'Канализация бытовая'),
    '0113': ('Здания и сооружения', 'Неизвестно'),
    'Goriz_05': ('Рельеф', 'Горизонтали'),
    'дер столб': ('Инженерные сети', 'Столбы и опоры'),
    'Дорожные знаки': ('МАФ', 'Дорожные знаки'),
    'кадастр': ('Служебные элементы', 'Кадастр'),
    '014 Велопарковка': ('МАФ', 'Уличная мебель'),
    'ШТАМП': ('Служебные элементы', 'Штамп'),
    'Границы покрытий_(дороги)': ('Служебные элементы', 'Контуры'),
    '03': ('Служебные элементы', 'Прочее'),
    'Kom_Teplo_1': ('Инженерные сети', 'Теплосеть'),
    '24_волейбольная сетка': ('МАФ', 'Спортивное оборудование'),
    'абрис_1': ('Служебные элементы', 'Прочее'),
    'abris dg_2': ('Служебные элементы', 'Прочее'),
    'Эл.кабели': ('Инженерные сети', 'Электроснабжение'),
    'C_Здания и строения': ('Здания и сооружения', 'Неизвестно'),
    'ЭЛЕКТРОСЕТ-СООРУЖЕНИЯ': ('Инженерные сети', 'Электроснабжение'),
    'Граница съемки фактическая': ('Служебные элементы', 'Границы'),
    'Граница_съемки': ('Служебные элементы', 'Границы'),
    'Граница съемки по кадастровому номеру': ('Служебные элементы', 'Границы'),
    'VODA': ('Инженерные сети', 'Водопровод'),
    'ГАЗО__НЕФТЕ__И_П': ('Инженерные сети', 'Газоснабжение'),
    'Граница заказа': ('Служебные элементы', 'Границы'),
    'граница работ': ('Служебные элементы', 'Границы'),
    'ИМЕНА': ('Служебные элементы', 'Прочее'),
    'Линии': ('Служебные элементы', 'Прочее'),
    'ИИ_УГОДЬЯ_025': ('Водопроницаемые покрытия', 'Грунт'),
    'z': ('Рельеф', 'Отметки высот'),
    'Kom_Voda_1': ('Инженерные сети', 'Водопровод'),
    'контур площадки для выгула собак': ('Служебные элементы', 'Контуры'),
    '013 Качалка': ('МАФ', 'Игровое оборудование'),
    'Пикет': ('Рельеф', 'Отметки высот'),
    '6 Брусья': ('МАФ', 'Спортивное оборудование'),
    'Урна': ('МАФ', 'Уличная мебель'),
    'ИИ_ОФОРМЛЕНИЕ_025': ('Служебные элементы', 'Прочее'),
    'Kom_Kab_SV': ('Инженерные сети', 'Сети связи'),
    'коммуникации': ('Инженерные сети', 'Инженерная инфраструктура')
}





def assign_layer_only_labels(row):
    if pd.notna(row['l1']):
        return row['l1'], row['l2']
    labels = new_mapping.get(str(row['Layer']).strip())
    if labels:
        return labels
    return row['l1'], row['l2']

df[['l1', 'l2']] = df.apply(assign_layer_only_labels, axis=1, result_type='expand')
_delta = int(df['l1'].notna().sum() - total_changed)
total_changed += _delta
total_nan     -= _delta
df.to_csv('NEW_REGIONS_in_progress_7.05.csv', index=False)
print(f"[7.05]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[7.05]  +317  | размечено: 3061911 | осталось: 7769


In [155]:
valid_types = [
    'LWPOLYLINE', 
    'LINE', 
    'ARC', 
    'INSERT', 
    'CIRCLE', 
    'SPLINE', 
    'POLYLINE',
    'MLINE',
    'ELLIPSE',
    'HATCH'
]

l = 'ИИ_УГОДЬЯ_025'

filtered_df = df[
    (df['l1'].isna())
]

files_stats = (
    filtered_df
    .groupby('Source_File')
    .size()
    .reset_index(name='count')
    .sort_values(by='count', ascending=False)
)

files_stats.head()

,Source_File,count
16,Топография.dxf,4595
6,Зарайск_итог 10_02_2021_итог.dxf,2456
5,АГО_Геоподоснова.dxf,573
1,04-10-2023-ИГДИ.dxf,37
8,Коломна (Мемориальный парк) - ТОПО.dxf,30


In [157]:
mask_hatches = df['l1'].isna() & (df['Layer'].astype(str).str.strip().str.lower() == 'люки')
df.loc[mask_hatches, ['l1', 'l2']] = ['Инженерные сети', 'Инженерная инфраструктура']

mask_others = df['l1'].isna()
df.loc[mask_others, ['l1', 'l2']] = ['Служебные элементы', 'Прочее']

print(f"Labeled engineering infrastructure: {mask_hatches.sum()}")
print(f"Other objects labeled: {mask_others.sum()}")
print(f"Empty l1 left: {df['l1'].isna().sum()}")

Labeled engineering infrastructure: 37
Other objects labeled: 7732
Empty l1 left: 0


In [5]:
pd.DataFrame(df['l1'].unique()).to_excel('l1_unique.xlsx')
pd.DataFrame(df['l2'].unique()).to_excel('l2_unique.xlsx')

In [7]:
df.loc[df['l1'] == 'Водопроницаемые элементы', 'l1'] = 'Водопроницаемые покрытия'

In [8]:
corrections_l2 = {
    'Номер дерева': 'Номера деревьев',
    'Бортовой камень': 'Бортовые камни',
    'Откос': 'Откосы',
    'Информационная конструкция': 'Информационные конструкции',
    'Кустарник': 'Кустарники',
    'Дерево': 'Деревья',
    'Ограждение': 'Ограждения',
    'Шлагбаум': 'Шлагбаумы',
    'Размеры дерева': 'Размеры деревьев',
    'Деревянный настил': 'Деревянные настилы',
    'Подпорная стенка': 'Подпорные стенки',
    'Флагшток': 'Флагштоки',
    'Мост': 'Мосты',
    'Металлический настил': 'Металлические настилы',
    'Павильон': 'Павильоны',
    'Динамический блок': 'Динамические блоки',
    'Цветник': 'Цветники',
    'Условные обозначение': 'Условные обозначения',
    'Газопровод': 'Газоснабжение' 
}

df['l2'] = df['l2'].replace(corrections_l2)

In [9]:
df.to_csv('REGIONS_FINAL_v1.csv')

In [ ]:
# df = pd.read_csv('REGIONS_FINAL.csv')

C:\Users\artem\AppData\Local\Temp\ipykernel_36588\59399119.py:1: DtypeWarning: Columns (3,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('REGIONS_FINAL.csv')


# Moscow check

In [3]:
msk_df = pd.read_csv('msk_geo_dataset_v2.csv', low_memory=False)

In [7]:
pd.DataFrame(msk_df[msk_df['l1'].isna()]['Text'].unique()).to_excel('msk_texts.xlsx')

## 1.0. Pattern 123.4 / 123.45 - (Рельеф, Отметки высот)

In [ ]:
pattern = r'^\d{3}\.\d{1,2}$'

mask = msk_df['l1'].isna() & msk_df['text_column'].astype(str).str.match(pattern)
msk_df.loc[mask, 'l1'] = 'Рельеф'
msk_df.loc[mask, 'l2'] = 'Отметки высот'

print(f"Labeled points: {mask.sum()}")

In [6]:
total_changed = msk_df['l1'].notna().sum()
total_nan = msk_df['l1'].isna().sum()

pattern = r'^\d{3}\.\d{1,2}$'

mask = msk_df['l1'].isna() & msk_df['Text'].astype(str).str.match(pattern)

msk_df.loc[mask, 'l1'] = 'Рельеф'
msk_df.loc[mask, 'l2'] = 'Отметки высот'

_delta = int(msk_df['l1'].notna().sum() - total_changed)
total_changed += _delta
total_nan -= _delta

msk_df.to_csv('MSK_in_progress_1.00.csv', index=False)

print(f"[1.00]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[1.00]  +111124  | размечено: 111124 | осталось: 3792341


In [11]:
pattern_pipes = r'^\s*\d{3}[.,]\d{1,2}\s*[а-яА-ЯёЁ.]*\s*$'
mask = msk_df['l1'].isna() & msk_df['Text'].astype(str).str.contains(pattern_pipes, regex=True, na=False)

matched_texts = msk_df.loc[mask, 'Text'].unique()
df_check = pd.DataFrame(matched_texts, columns=['matched_text'])
df_check.to_csv('check_pattern_pipes.csv', index=False, encoding='cp1251')

print(f"Unique values: {len(matched_texts)}")
print(f"Total number of rows with such values: {mask.sum()}")


Unique values: 883
Total number of rows with such values: 995


In [13]:
pattern_pipes = r'^\s*\d{3}[.,]\d{1,2}\s*[а-яА-ЯёЁ.]*\s*$'

mask = msk_df['l1'].isna() & msk_df['Text'].astype(str).str.contains(pattern_pipes, regex=True, na=False)

msk_df.loc[mask, 'l1'] = 'Рельеф'
msk_df.loc[mask, 'l2'] = 'Отметки высот'

_delta = int(mask.sum())
total_changed += _delta
total_nan -= _delta

msk_df.to_csv('MSK_in_progress_1.01.csv', index=False, encoding='utf-8-sig')
print(f"[1.01]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[1.01]  +995  | размечено: 112119 | осталось: 3791346


In [14]:
pd.DataFrame(msk_df[msk_df['l1'].isna()]['Text'].unique()).to_excel('msk_texts.xlsx')

In [15]:
text_stats = msk_df[msk_df['l1'].isna()]['Text'].value_counts().reset_index()
text_stats.columns = ['Text', 'Count']
text_stats.to_excel('msk_texts_by_frequency.xlsx', index=False)

## 1.02. "А" - (Твердые покрытия, Асфальт)

In [16]:
mask = (msk_df['Text'] == 'А') & (msk_df['l1'].isna())
msk_df.loc[mask, ['l1', 'l2']] = ['Твердые покрытия', 'Асфальт']

_delta = int(mask.sum())
total_changed += _delta
total_nan -= _delta

msk_df.to_csv('MSK_in_progress_1.02.csv', index=False, encoding='utf-8-sig')
print(f"[1.02]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[1.02]  +6031  | размечено: 118150 | осталось: 3785315


## 1.03 "ГАЗОН" - (Озеленение, Газон)

In [17]:
mask = (msk_df['Text'] == 'ГАЗОН') & (msk_df['l1'].isna())
msk_df.loc[mask, ['l1', 'l2']] = ['Озеленение', 'Газон']

_delta = int(mask.sum())
total_changed += _delta
total_nan -= _delta

msk_df.to_csv('MSK_in_progress_1.03.csv', index=False, encoding='utf-8-sig')
print(f"[1.03]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[1.03]  +2748  | размечено: 120898 | осталось: 3782567


In [25]:
pd.DataFrame(msk_df[(msk_df['l1'].isna()) & (msk_df['Text'].str.contains('щит ', case=False, na=False))]).to_excel('щит_texts.xlsx')

## 1.04. Texts "щит" + слой

In [26]:

shield_layer_map = {
    'Промводопровод': ('Инженерные сети', 'Водопровод'),
    'Водопровод': ('Инженерные сети', 'Водопровод'),
    'Канализация самотёчная': ('Инженерные сети', 'Канализация самотечная'),
    'Канализация напорная': ('Инженерные сети', 'Канализация напорная'),
    'Водосток': ('Инженерные сети', 'Ливневая канализация'),
    'Теплосеть': ('Инженерные сети', 'Теплосеть'),
    'Кабель Мосэнерго': ('Инженерные сети', 'Электроснабжение'),
    'Кабель электрический': ('Инженерные сети', 'Электроснабжение'),
    'Общий коллектор': ('Инженерные сети', 'Инженерная инфраструктура')
}

mask = (msk_df['Text'].str.contains('щит ', case=False, na=False)) & (msk_df['l1'].isna())

def assign_shield_by_layer(row):
    return shield_layer_map.get(row['Layer'], (row['l1'], row['l2']))

results = msk_df.loc[mask].apply(assign_shield_by_layer, axis=1, result_type='expand')
if not results.empty:
    msk_df.loc[mask, ['l1', 'l2']] = results.values


_delta = int(mask.sum())
total_changed += _delta
total_nan -= _delta

msk_df.to_csv('MSK_in_progress_1.04.csv', index=False, encoding='utf-8-sig')
print(f"[1.04]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[1.04]  +172  | размечено: 121070 | осталось: 3782395


## 1.05. "d=" + слой - (Инженерные сети, ...)

In [29]:
pd.DataFrame(msk_df[(msk_df['l1'].isna()) & (msk_df['Text'].str.contains('d=', case=False, na=False))]).to_excel('d=_texts.xlsx')

In [31]:
msk_df[(msk_df['l1'].isna()) & (msk_df['Text'].str.contains('d=', case=False, na=False))]['Layer'].unique()

array(['Водосток', 'Канализация самотёчная', 'Водопровод', 'Дренаж',
       'Теплосеть', 'Газопровод', 'Канализация напорная',
       'Промводопровод', 'Кабель электрический', 'Трубопроводы',
       'Кабель связи', 'Общий коллектор', 'Кабели', 'Кабель Мосэнерго',
       'Водосток проектный', 'Канализация самотёчная проектная',
       'Водопровод проектный'], dtype=object)

In [32]:
d_layer_map = {
    'Водосток': ('Инженерные сети', 'Ливневая канализация'),
    'Водосток проектный': ('Инженерные сети', 'Ливневая канализация'),
    'Канализация самотёчная': ('Инженерные сети', 'Канализация самотечная'),
    'Канализация самотёчная проектная': ('Инженерные сети', 'Канализация самотечная'),
    'Канализация напорная': ('Инженерные сети', 'Канализация напорная'),
    'Водопровод': ('Инженерные сети', 'Водопровод'),
    'Водопровод проектный': ('Инженерные сети', 'Водопровод'),
    'Промводопровод': ('Инженерные сети', 'Водопровод'),
    'Дренаж': ('Инженерные сети', 'Дренаж'),
    'Теплосеть': ('Инженерные сети', 'Теплосеть'),
    'Газопровод': ('Инженерные сети', 'Газоснабжение'),
    'Кабель электрический': ('Инженерные сети', 'Электроснабжение'),
    'Кабель Мосэнерго': ('Инженерные сети', 'Электроснабжение'),
    'Кабель связи': ('Инженерные сети', 'Сети связи'),
    'Кабели': ('Инженерные сети', 'Прочие кабели'),
    'Трубопроводы': ('Инженерные сети', 'Прочие трубопроводы'),
    'Общий коллектор': ('Инженерные сети', 'Инженерная инфраструктура')
}

mask = (msk_df['Text'].str.contains('d=', case=False, na=False)) & (msk_df['l1'].isna())

def assign_d_by_layer(row):
    return d_layer_map.get(row['Layer'], (row['l1'], row['l2']))

results = msk_df.loc[mask].apply(assign_d_by_layer, axis=1, result_type='expand')
if not results.empty:
    msk_df.loc[mask, ['l1', 'l2']] = results.values

new_total_changed = msk_df['l1'].notna().sum()
_delta = int(new_total_changed - total_changed)

total_changed = new_total_changed
total_nan = msk_df['l1'].isna().sum()

msk_df.to_csv('MSK_in_progress_1.05.csv', index=False, encoding='utf-8-sig')
print(f"[1.05]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[1.05]  +5350  | размечено: 126420 | осталось: 3777045


## 1.06. ...

In [10]:
msk_df = pd.read_csv('MSK_in_progress_1.05.csv', low_memory=False)

In [11]:
total_changed = int(msk_df['l1'].notna().sum())
total_nan = int(msk_df['l1'].isna().sum())

In [12]:
greenery_layers = [
    'Отдельно стоящее дерево',
    'Полоса деревьев',
    'Леса и газоны',
    'Граница растительности и грунта',
    'Растительность',
    'Леса и газоны (Construction)'
]

tree_keywords = [
    'БЕРЕЗА', 'БЕРЁЗА', 'СОСНА', 'КЛЕН', 'КЛЁН', 'ЛИПА', 'ДУБ', 'ЕЛЬ', 
    'ОСИНА', 'ТОПОЛЬ', 'ЛИСТВЕННИЦА', 'ИВА', 'ОЛЬХА', 'ЯСЕНЬ', 'ВЯЗ', 
    'ЯБЛОНЯ', 'ТУЯ', 'РЯБИНА', 'ПИХТА'
]

tree_pattern = '|'.join(tree_keywords)
mask_all_greenery = (msk_df['l1'].isna()) & (msk_df['Layer'].isin(greenery_layers))

def classify_greenery(text):
    text_str = str(text).upper()
    if re.search(tree_pattern, text_str):
        return 'Озеленение', 'Дерево'
    else:
        return 'Озеленение', 'Газон'

results = msk_df.loc[mask_all_greenery, 'Text'].apply(classify_greenery)
msk_df.loc[mask_all_greenery, ['l1', 'l2']] = pd.DataFrame(results.tolist(), index=results.index).values

new_total_changed = msk_df['l1'].notna().sum()
_delta = int(new_total_changed - total_changed)
total_changed = new_total_changed
total_nan = msk_df['l1'].isna().sum()

msk_df.to_csv('MSK_in_progress_1.06.csv', index=False, encoding='utf-8-sig')
print(f"[1.06]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[1.06]  +3023851  | размечено: 3150271 | осталось: 753194


In [15]:
# pd.DataFrame(msk_df['Layer'].unique()).to_excel('unique_layers.xlsx')
# msk_df['Layer'].unique()

## 1.07 Text - Здания

In [16]:
building_layers = [
    'Здания',
    'Части зданий',
    'Крыльца',
    'Крыльца (Construction)',
    'Навесы',
    'Внутреннее заполнение',
    'Внутреннее заполнение (Construction)',
    'Номер дома',
    'Отметка высоты пола',
    'Теплицы'
]

mask_buildings = (msk_df['l1'].isna()) & (msk_df['Layer'].isin(building_layers))

if mask_buildings.sum() > 0:
    building_stats = msk_df[mask_buildings]['Text'].value_counts().reset_index()
    building_stats.columns = ['Text', 'Count']
    building_stats.to_excel('msk_buildings_texts.xlsx', index=False)


In [23]:
import pandas as pd

building_layers = [
    'Здания', 'Части зданий', 'Крыльца', 'Крыльца (Construction)',
    'Навесы', 'Внутреннее заполнение', 'Внутреннее заполнение (Construction)',
    'Номер дома', 'Отметка высоты пола', 'Теплицы'
]

# 1. Наш основной паттерн для поиска
pattern_address = r'КОРП|СТР|СООР|С\.|\d+/\d+|^\s*\d+[А-Яа-яЁёA-Za-z]?\s*$'

# 2. Паттерн для исключения (найдет ПОДЗЕМ.СООР., ПОДЗЕМНОЕ СООРУЖЕНИЕ, ПОДЗ.СООР. и т.д.)
pattern_exclude = r'ПОДЗ'

# 3. Обновленная маска
mask = (
    msk_df['l1'].isna() & 
    msk_df['Layer'].isin(building_layers) & 
    msk_df['Text'].str.contains(pattern_address, case=False, regex=True, na=False) &
    ~msk_df['Text'].str.contains(pattern_exclude, case=False, regex=True, na=False) # Знак ~ исключает совпадения
)

# 4. Выгрузка
if mask.sum() > 0:
    address_stats = msk_df[mask]['Text'].value_counts().reset_index()
    address_stats.columns = ['Text', 'Count']
    
    address_stats.to_excel('check_buildings_address.xlsx', index=False)
    
    print(f"Файл check_buildings_address.xlsx готов!")
    print(f"Найдено уникальных текстов: {len(address_stats)}")
    print(f"Всего строк попало под паттерн: {mask.sum()}")
else:
    print("По заданному паттерну не найдено неразмеченных строк.")

Файл check_buildings_address.xlsx готов!
Найдено уникальных текстов: 503
Всего строк попало под паттерн: 1138


In [27]:
building_layers = [
    'Здания', 'Части зданий', 'Крыльца', 'Крыльца (Construction)',
    'Навесы', 'Внутреннее заполнение', 'Внутреннее заполнение (Construction)',
    'Номер дома', 'Отметка высоты пола', 'Теплицы'
]

pattern_exclude = r'ПОДЗ|КАНАВА'

mask = (
    msk_df['l1'].isna() & 
    msk_df['Layer'].isin(building_layers) & 
    ~msk_df['Text'].str.contains(pattern_exclude, case=False, regex=True, na=False)
)


if mask.sum() > 0:
    msk_df.loc[mask, ['l1', 'l2']] = ['Здания и сооружения', 'Неизвестно']
else:
    print("[INFO] Нет строк для разметки.")

new_total_changed = msk_df['l1'].notna().sum()
_delta = int(new_total_changed - total_changed)

total_changed = new_total_changed
total_nan = msk_df['l1'].isna().sum()

msk_df.to_csv('MSK_in_progress_1.07.csv', index=False, encoding='utf-8-sig')
print(f"[1.07]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[1.07]  +37223  | размечено: 3187494 | осталось: 715971


## 1.08 Texts - Улицы, проезды, шосее ... 

In [ ]:
msk_df[(msk_df['Layer'].isin(building_layers)) & (msk_df['Text'] == 'М.Я.')]

In [32]:
import pandas as pd
import re

pattern_streets = r'УЛ\.|УЛИЦА|ПР-Д|ПРОЕЗД|ПЕР\.|ПЕРЕУЛОК|ШОССЕ|Ш\.|ПРОСПЕКТ|ПРОСП\.|БУЛЬВАР|Б-Р|АЛЛЕЯ|ТУПИК|ТУП\.'

raw_exclusions = """РАКУШ.
г. Москва, ЗАО, проезд от 
г. Москва, ЗАО, проезд 
г.Москва, ЦАО, улица 
г. Москва, САО, Проезды от 
Рублевского ш. до торгового центра "Европарк"
г. Москва, ЗАО, улица 
г.Москва, ЦАО , улица 
ул. Академика Павлова до ЦКБ
г. Москва, ЗАО, проезд к ДК 
г. Москва, ЗАО, проезд в 
Медик от ул.Академика Павлова, д.11, корп.1 до ул.Маршала 
транспортная развязка на пересечении ул. Боженко и ул. 
транспортная развязка на пересечении ул.Кубинка и
ул.Горбунова в Можайском районе
бульвар, вл. 10 (2 въезда)
ул.Барклая до роддома №2
г. Москва, ЗАО, проезд к 
жилым домам у дома 10, к. 1 по ул. Толбухина
ул. Дорогобужской до ул. Рябиновой
площадкана ул. Новой Витебской (вдоль нежилого стр.9 по 
ул. Витебская)
ул. Крылатские холмы до 2-ой Крылатской улицы
между нежилыми строениями ДК Сетунь ул. Толбухина д. 10 
и бассейном - ул. Толбухина д. 10, корп. 3
Можайскогошоссе до Сафоновской улицы
г. Москва, ЗАО, проезд к жилым домам у дома 
8, к. 1 по ул. Толбухина"""

exclude_texts = [line.strip() for line in raw_exclusions.split('\n') if line.strip()]
pattern_exclude = '|'.join([re.escape(text) for text in exclude_texts])

mask_streets = (
    msk_df['l1'].isna() & 
    msk_df['Text'].str.contains(pattern_streets, case=False, regex=True, na=False) &
    ~msk_df['Text'].str.contains(pattern_exclude, case=False, regex=True, na=False)
)

if mask_streets.sum() > 0:
    msk_df.loc[mask_streets, ['l1', 'l2']] = ['Твердые покрытия', 'Асфальт']
else:
    print("[INFO] Нет строк для разметки.")


new_total_changed = msk_df['l1'].notna().sum()
_delta = int(new_total_changed - total_changed)

total_changed = new_total_changed
total_nan = msk_df['l1'].isna().sum()

msk_df.to_csv('MSK_in_progress_1.08.csv', index=False, encoding='utf-8-sig')
print(f"[1.08]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[1.08]  +415  | размечено: 3187909 | осталось: 715556


## 1.09 Трубопроводы

In [36]:
pattern_pipes = r'обойма|\d{3}\s*[хxX]\s*\d{3}|тр\.|ДВ\.\d+|гл\.\d+|\d+к\.|отв\.|ЖЕЛОБ|К\.П\.'

network_layer_map = {
    'Кабель электрический': ('Инженерные сети', 'Электроснабжение'),
    'Кабель электрический проектный': ('Инженерные сети', 'Электроснабжение'),
    'Кабель Мосэнерго': ('Инженерные сети', 'Электроснабжение'),
    
    'Кабель связи': ('Инженерные сети', 'Сети связи'),
    'Кабель связи проектный': ('Инженерные сети', 'Сети связи'),
    
    'Кабель защиты': ('Инженерные сети', 'Прочие кабели'),
    'Кабели': ('Инженерные сети', 'Прочие кабели'),
    'Кабели проектные': ('Инженерные сети', 'Прочие кабели'),
    
    'Водопровод': ('Инженерные сети', 'Водопровод'),
    'Промводопровод': ('Инженерные сети', 'Водопровод'),
    
    'Водосток': ('Инженерные сети', 'Ливневая канализация'),
    
    'Канализация самотёчная': ('Инженерные сети', 'Канализация самотечная'),
    'Канализация напорная': ('Инженерные сети', 'Канализация напорная'),
    
    'Теплосеть': ('Инженерные сети', 'Теплосеть'),
    'Газопровод': ('Инженерные сети', 'Газоснабжение'),
    'Дренаж': ('Инженерные сети', 'Дренаж'),
    
    'Общий коллектор': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Трубопроводы': ('Инженерные сети', 'Прочие трубопроводы')
}

mask_pipes = (
    msk_df['l1'].isna() & 
    msk_df['Text'].str.contains(pattern_pipes, case=False, regex=True, na=False) &
    msk_df['Layer'].isin(network_layer_map.keys())
)

def assign_network_class(layer):
    return network_layer_map[layer]

if mask_pipes.sum() > 0:
    results = msk_df.loc[mask_pipes, 'Layer'].apply(assign_network_class)
    msk_df.loc[mask_pipes, ['l1', 'l2']] = pd.DataFrame(results.tolist(), index=results.index).values
else:
    print("[INFO] Нет строк для разметки по данному паттерну на инженерных слоях.")

new_total_changed = msk_df['l1'].notna().sum()
_delta = int(new_total_changed - total_changed)

total_changed = new_total_changed
total_nan = msk_df['l1'].isna().sum()

msk_df.to_csv('MSK_in_progress_1.09.csv', index=False, encoding='utf-8-sig')
print(f"[1.09]  +{_delta}  | размечено: {total_changed} | осталось: {total_nan}")

[1.09]  +11698  | размечено: 3199607 | осталось: 703858


In [38]:
pd.DataFrame(msk_df[msk_df['l1'].isna()]['Text'].unique()).to_excel('msk_texts.xlsx')

## 1.10. Redlines (*.kl files) texts as blocks extraction

In [ ]:
# msk_df = pd.read_csv('MSK_in_progress_1.09.csv')

C:\Users\artem\AppData\Local\Temp\ipykernel_20420\2434188780.py:1: DtypeWarning: Columns (4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  msk_df = pd.read_csv('MSK_in_progress_1.09.csv')


In [76]:
def clean_cad_mtext(text):
    if pd.isna(text) or text == "": return text
    text = str(text)
    text = re.sub(r'%%[cC]', 'Ø', text)
    text = re.sub(r'%%[dD]', '°', text)
    text = re.sub(r'%%[pP]', '±', text)
    text = re.sub(r'\\P', ' ', text)
    text = re.sub(r'\\[A-Za-z0-9~|\-]+[^;]*;', '', text)
    text = re.sub(r'\\[LlOoKkX]', '', text)
    text = re.sub(r'[{}]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Define the relative path to your DXF files
base_path = r'files\msk\output_msk'

# Clean up any trailing/leading spaces in key columns
msk_df['Layer'] = msk_df['Layer'].str.strip()
msk_df['Source_File'] = msk_df['Source_File'].str.strip()

# Create a mask for target blocks in *kl.dxf files
mask_blocks = (
    msk_df['Source_File'].str.endswith('kl.dxf', na=False) & 
    (msk_df['Layer'] == 'Красные линии') & 
    msk_df['BlockName'].str.startswith('msdElementTypeTextNode', na=False)
)

targets = msk_df[mask_blocks].copy()
files_to_process = targets['Source_File'].unique()

print(f"Found {len(targets)} blocks to extract in {len(files_to_process)} files.")

# Process each unique file
for dxf_filename in files_to_process:
    # Construct the full path to the DXF file
    full_path = os.path.join(base_path, dxf_filename)
    
    if not os.path.exists(full_path):
        print(f"File not found: {full_path}")
        continue
        
    try:
        doc = ezdxf.readfile(full_path)
        # Get handles for blocks belonging to the current file
        current_file_handles = targets[targets['Source_File'] == dxf_filename]['Handle'].values
        
        for handle in current_file_handles:
            try:
                entity = doc.entitydb.get(handle)
                all_strings = [] 

                # Extract text based on entity type
                if entity and entity.dxftype() == 'MTEXT':
                    all_strings.append(entity.text)
                
                elif entity and entity.dxftype() == 'INSERT':
                    # Look inside the block definition
                    block_def = doc.blocks.get(entity.dxf.name)
                    for e in block_def:
                        if e.dxftype() == 'TEXT':
                            all_strings.append(e.dxf.text)
                        elif e.dxftype() == 'MTEXT':
                            all_strings.append(e.text)
                
                if all_strings:
                    # Clean and concatenate all found text fragments
                    cleaned_fragments = [clean_cad_mtext(s) for s in all_strings if s.strip()]
                    final_text = " ".join(cleaned_fragments).strip()
                    final_text = re.sub(r'\s+', ' ', final_text)

                    # Update the main dataframe (msk_df) directly using index
                    idx = msk_df[(msk_df['Handle'] == handle) & (msk_df['Source_File'] == dxf_filename)].index
                    if not idx.empty:
                        msk_df.loc[idx, 'Type'] = 'TEXT'
                        msk_df.loc[idx, 'BlockName'] = ""
                        msk_df.loc[idx, 'Text'] = final_text
                    
            except Exception as e:
                print(f"Error processing handle {handle} in {dxf_filename}: {e}")
                
    except Exception as e:
        print(f"Could not read DXF file {full_path}: {e}")

# Export the updated dataframe to CSV
msk_df.to_csv('MSK_in_progress_1.10.csv', index=False, encoding='utf-8-sig')
print("Update complete. Check 'MSK_in_progress_1.10.csv' for results.")

Found 854 blocks to extract in 72 files.
Update complete. Check 'MSK_in_progress_1.10.csv' for results.


## Cleaning CAD system symbols from texts (in other file)

In [78]:
msk_df = pd.read_csv('MSK_in_progress_1.10_TEXT_CLEANED.csv')

C:\Users\artem\AppData\Local\Temp\ipykernel_20420\1396370435.py:1: DtypeWarning: Columns (4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  msk_df = pd.read_csv('MSK_in_progress_1.10_TEXT_CLEANED.csv')


## 1.11 - TEXT - Красные линии

In [80]:
def label_red_lines_strict(text):
    if pd.isna(text) or text == "":
        return None, None
    
    text_upper = str(text).upper().strip()
    if "ПРОЕЗД" in text_upper:
        return "Твердые покрытия", "Асфальт"
    if "ПК" in text_upper:
        return "Служебные элементы", "Зоны ПК"
    if "ООПТ" in text_upper:
        return "Служебные элементы", "Зоны ООПТ"
    if "ОКН" in text_upper:
        return "Служебные элементы", "Зоны ОКН"
    if "БЕРЕГОВАЯ ПОЛОСА" in text_upper:
        return "Служебные элементы", "Зоны Береговая полоса"
    if ("ПО СОСТОЯНИЮ НА" in text_upper or 
        "K=" in text_upper or 
        "R=" in text_upper or 
        re.fullmatch(r'\d+(\.\d+)?', text_upper)):
        return "Служебные элементы", "Прочее"

    return None, None

mask = (msk_df['Layer'] == 'Красные линии') & (msk_df['l1'].isna())
results = msk_df.loc[mask, 'Text'].apply(lambda x: pd.Series(label_red_lines_strict(x)))
msk_df.loc[mask, ['l1', 'l2']] = results.values

print("Classification Summary for 'Красные линии':")
print(msk_df[msk_df['Layer'] == 'Красные линии'][['l1', 'l2']].value_counts())

Classification Summary for 'Красные линии':
l1                  l2                   
Твердые покрытия    Асфальт                  169
Служебные элементы  Прочее                   156
                    Зоны ОКН                  96
                    Зоны ПК                   83
                    Зоны ООПТ                 53
Рельеф              Отметки высот             18
Служебные элементы  Зоны Береговая полоса      5
dtype: int64


In [84]:
# 1. Mapping for Water Protection Zones ("ВОДО")
mask_vodo = (
    (msk_df['Layer'] == 'Красные линии') & 
    (msk_df['Text'].str.contains('ВОДО', case=False, na=False))
)
msk_df.loc[mask_vodo, ['l1', 'l2']] = ['Служебные элементы', 'Зоны Водоохранная зона']

# 2. Mapping for Sanitary Protection Zones ("САНИТАРНО - ЗАЩИТНАЯ ЗОНА")
mask_szz = (
    (msk_df['Layer'] == 'Красные линии') & 
    (msk_df['Text'].str.contains('САНИТАРНО - ЗАЩИТНАЯ ЗОНА', case=False, na=False))
)
msk_df.loc[mask_szz, ['l1', 'l2']] = ['Служебные элементы', 'Зоны СЗЗ']

# Print results for these two rules
print(f"Labeled as Water Protection Zones: {mask_vodo.sum()}")
print(f"Labeled as Sanitary Protection Zones (SZZ): {mask_szz.sum()}")

Labeled as Water Protection Zones: 7
Labeled as Sanitary Protection Zones (SZZ): 36


In [87]:
msk_df.loc[(msk_df['Layer'] == 'Красные линии') & 
    (msk_df['Text'].str.contains('ТЕХНИЧЕСКАЯ ЗОНА ВОДОВОДОВ', case=False, na=False)), 
    ['l1', 'l2']] = ['Служебные элементы', 'Зоны Прочее']

In [89]:
mask_zon = (
    (msk_df['Layer'] == 'Красные линии') & 
    (msk_df['l1'].isna()) & 
    (msk_df['Text'].str.contains('ЗОН', case=False, na=False))
)

# 2. Assign the specific categories
msk_df.loc[mask_zon, 'l1'] = 'Служебные элементы'
msk_df.loc[mask_zon, 'l2'] = 'Зоны Прочее'

# 3. Print the count of newly labeled rows
print(f"Rows labeled using 'ЗОН' logic: {mask_zon.sum()}")

# 4. Final verification of the layer
print(msk_df[msk_df['Layer'] == 'Красные линии']['l2'].value_counts())

Rows labeled using 'ЗОН' logic: 221
Зоны Прочее               223
Асфальт                   169
Прочее                    156
Зоны ОКН                   96
Зоны ПК                    83
Зоны ООПТ                  53
Зоны СЗЗ                   36
Отметки высот              18
Зоны Береговая полоса       5
Зоны Водоохранная зона      5
Name: l2, dtype: int64


In [92]:
# 1. Define a more precise mask
# Target only 'Красные линии' layer where:
# - l1 is still NaN (not labeled yet)
# - Text is NOT empty and NOT NaN (we only label textual descriptions)
mask_remaining_text = (
    (msk_df['Layer'] == 'Красные линии') & 
    (msk_df['l1'].isna()) & 
    (msk_df['Text'].notna()) & 
    (msk_df['Text'].str.strip() != "")
)

# 2. Assign categories only to these textual objects
msk_df.loc[mask_remaining_text, ['l1', 'l2']] = ['Служебные элементы', 'Прочее']

# 3. Print statistics to see how many TEXT objects were labeled
labeled_count = mask_remaining_text.sum()
print(f"Labeled {labeled_count} remaining text objects as ('Служебные элементы', 'Прочее').")

# 4. Check if there are any non-text objects left in this layer without labels
remaining_geometry = (msk_df['Layer'] == 'Красные линии') & (msk_df['l1'].isna())
print(f"Objects remaining without labels in this layer (likely geometry): {remaining_geometry.sum()}")

Labeled 280 remaining text objects as ('Служебные элементы', 'Прочее').
Objects remaining without labels in this layer (likely geometry): 5376


In [93]:
msk_df.to_csv('MSK_in_progress_1.11.csv')

## 1.12 Readlines blocks / lines -> ...

In [99]:
pd.DataFrame(msk_df[(msk_df['BlockName'].notna()) & (msk_df['Layer'] == 'Красные линии')]['BlockName'].unique()).to_excel('readlines_blocknames.xlsx')

In [101]:
msk_df[(msk_df['l1'] == 'Красные линии')]

,Source_File,Handle,Layer,Type,BlockName,Text,l1,l2


In [102]:
# Target: 'Красные линии' layer, specific BlockName pattern
mask_line_blocks = (
    (msk_df['Layer'] == 'Красные линии') & 
    (msk_df['BlockName'].str.startswith('msdElementTypeLine', na=False))
)

# Assign to specific service category for red lines geometry
msk_df.loc[mask_line_blocks, ['l1', 'l2']] = ['Служебные элементы', 'Красные линии']

mask_remaining_blocks = (
    (msk_df['Layer'] == 'Красные линии') & 
    (msk_df['l1'].isna()) & 
    (msk_df['BlockName'].notna()) & 
    (msk_df['BlockName'].str.strip() != "")
)

# Assign remaining blocks to 'Miscellaneous'
msk_df.loc[mask_remaining_blocks, ['l1', 'l2']] = ['Служебные элементы', 'Прочее']

# 2. Diagnostics
labeled_blocks_count = mask_remaining_blocks.sum()
print(f"Labeled {labeled_blocks_count} remaining blocks as ('Служебные элементы', 'Прочее').")

# 3. Check what is left (objects without BlockName and without labels)
still_unlabeled = (msk_df['Layer'] == 'Красные линии') & (msk_df['l1'].isna())
print(f"Objects remaining without labels (non-blocks/primitives): {still_unlabeled.sum()}")

Labeled 243 remaining blocks as ('Служебные элементы', 'Прочее').
Objects remaining without labels (non-blocks/primitives): 3591


In [107]:
# 1. Final sweep for the 'Красные линии' layer
# Any object (geometry or text) that hasn't been labeled yet 
# will now be assigned to ('Служебные элементы', 'Красные линии')
mask_final_sweep = (msk_df['Layer'] == 'Красные линии') & (msk_df['l1'].isna())

# 2. Assign the labels
msk_df.loc[mask_final_sweep, ['l1', 'l2']] = ['Служебные элементы', 'Красные линии']

# 3. Final count and check
final_count = mask_final_sweep.sum()
print(f"Final sweep complete. {final_count} objects were labeled as ('Служебные элементы', 'Красные линии').")

# Verify there are no NaN values left in this layer
remaining_nan = msk_df[msk_df['Layer'] == 'Красные линии']['l1'].isna().sum()
print(f"Remaining unlabeled objects in 'Красные линии': {remaining_nan}")

Final sweep complete. 3591 objects were labeled as ('Служебные элементы', 'Красные линии').
Remaining unlabeled objects in 'Красные линии': 0


In [108]:
msk_df.to_csv('MSK_in_progress_1.12.csv')

In [2]:
msk_df = pd.read_csv('MSK_in_progress_1.12.csv')

C:\Users\artem\AppData\Local\Temp\ipykernel_10960\2465771708.py:1: DtypeWarning: Columns (5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  msk_df = pd.read_csv('MSK_in_progress_1.12.csv')


## 1.13 

In [11]:
# 1. Filter for unlabeled text objects
# We need rows where l1 is NaN and Text is not empty
mask_unlabeled = (
    msk_df['l1'].isna() & 
    msk_df['Text'].notna() & 
    (msk_df['Text'].str.strip() != "")
)

# 2. Select only relevant columns for the analysis
# We keep 'Layer' and 'Text' to understand the context of each unique string
unlabeled_data = msk_df.loc[mask_unlabeled, ['Layer', 'Text']]

# 3. Get unique combinations of Layer and Text
# This drastically reduces the number of rows for review
unique_unlabeled = unlabeled_data.drop_duplicates(subset=['Layer', 'Text'])

# 4. Sort results for better readability in Excel
unique_unlabeled = unique_unlabeled.sort_values(by=['Layer', 'Text'])

# 5. Export to Excel
output_file = 'unique_unlabeled_texts.xlsx'
try:
    unique_unlabeled.to_excel(output_file, index=False)
    print(f"Extraction complete!")
    print(f"Total objects found: {len(unlabeled_data)}")
    print(f"Unique text/layer combinations: {len(unique_unlabeled)}")
    print(f"Saved to: {output_file}")
except Exception as e:
    print(f"Error while saving Excel: {e}")

Extraction complete!
Total objects found: 23038
Unique text/layer combinations: 1207
Saved to: unique_unlabeled_texts.xlsx


In [ ]:
# Apply the strict mappings per your instructions

# 1. Map everything specified in 'Береговая линия' to ('Водные объекты', 'Неизвестно')
mask_coast = (msk_df['Layer'] == 'Береговая линия') & (msk_df['Text'].notna()) & (msk_df['l1'].isna())
msk_df.loc[mask_coast, ['l1', 'l2']] = ['Водные объекты', 'Неизвестно']

# 2. Map specific texts in layer '0' to ('Служебные элементы', 'Штамп')
stamp_texts = ['МОСКОМАРХИТЕКТУРА', 'Масштаб']
mask_stamp = (msk_df['Layer'] == '0') & (msk_df['Text'].isin(stamp_texts)) & (msk_df['l1'].isna())
msk_df.loc[mask_stamp, ['l1', 'l2']] = ['Служебные элементы', 'Штамп']

# 3. Map specific legend/symbol texts in layer '0' to ('Служебные элементы', 'Условные обозначения')
# We can use a pattern to catch all of these since they start with 'границы'
mask_legend = (msk_df['Layer'] == '0') & (msk_df['Text'].str.startswith('границы', na=False)) & (msk_df['l1'].isna())
msk_df.loc[mask_legend, ['l1', 'l2']] = ['Служебные элементы', 'Условные обозначения']

# 4. Map specific flagpoles in layer 'Фонари' to ('МАФ', 'Флагштоки')
flagpole_texts = ['ФЛАГШТ.', 'ФЛАГШТОК', 'ФЛАГШТОКИ']
mask_flagpoles = (msk_df['Layer'] == 'Фонари') & (msk_df['Text'].isin(flagpole_texts)) & (msk_df['l1'].isna())
msk_df.loc[mask_flagpoles, ['l1', 'l2']] = ['МАФ', 'Флагштоки']

# 5. Map the remaining specific lists (Dates, Roman numerals, heights, temps, alphanumeric codes) 
# in layers '0', 'Пояснительные подписи', and 'Подписи рамок' to ('Служебные элементы', 'Прочее')
layers_to_misc = ['0', 'Пояснительные подписи', 'Подписи рамок']

# Define patterns that match your large list
pattern_misc = (
    r'^\(M 1:500\)$|^-$|^\.$|^1:500$|^1к\.ДС$|^\d+$|' # Exact short matches and simple digits
    r'^\d{2}\.\d{2}\.\d{2,4}$|' # Dates like 02.04.24 or 02.04.2024
    r'^3/ДЖКХ.*|' # Strings starting with 3/ДЖКХ
    r'^[A-Z]-[IVX]+-\d{2}-\d{2}.*|' # Complex alphanumeric codes like A-IX-01-04
    r'^h=\d+\.\d+$|' # Heights like h=12.1
    r'^t=.*[CС]$|' # Temps like t=+20^ C
    r'^[IVX]+-\d{4}Г\.$|' # Roman numerals with years like I-2017Г.
    r'^\+?-?\d+\.\d+$|' # Float numbers like +1.00, -0.5
    r'^3 КОРП\.2$|^"ЛОСИНЫЙ ОСТРОВ"$' # Specific single items
)

mask_misc = (
    (msk_df['Layer'].isin(layers_to_misc)) & 
    (msk_df['Text'].str.contains(pattern_misc, regex=True, na=False)) & 
    (msk_df['l1'].isna())
)
msk_df.loc[mask_misc, ['l1', 'l2']] = ['Служебные элементы', 'Прочее']

# 6. Safety check: Catch any remaining single numbers in layer 0 or Подписи рамок
mask_digits = (
    (msk_df['Layer'].isin(['0', 'Подписи рамок'])) & 
    (msk_df['Text'].str.match(r'^\d+$', na=False)) & 
    (msk_df['l1'].isna())
)
msk_df.loc[mask_digits, ['l1', 'l2']] = ['Служебные элементы', 'Прочее']

# 7. Print summary of this operation
print("Labeling complete for the provided lists.")
print("Recent updates distribution:")
print(msk_df[msk_df['Layer'].isin(['0', 'Береговая линия', 'Пояснительные подписи', 'Подписи рамок', 'Фонари'])]['l2'].value_counts())


Labeling complete for the provided lists.
Recent updates distribution:
Условные обозначения    6144
Прочее                  2179
Неизвестно               951
Отметки высот            157
Штамп                    134
Асфальт                   77
Флагштоки                 23
Name: l2, dtype: int64


In [10]:
msk_df.to_csv('MSK_in_progress_1.13.csv')

## 1.14 Инженерные сети

In [ ]:
# 1. Define the mapping dictionary: Key is Layer name, Value is a tuple (l1, l2)
layer_mapping = {
    'Водопровод': ('Инженерные сети', 'Водопровод'),
    'Промводопровод': ('Инженерные сети', 'Водопровод'),
    'Водосток': ('Инженерные сети', 'Ливневая канализация'),
    'Газопровод': ('Инженерные сети', 'Газоснабжение'),
    'Горизонтали': ('Рельеф', 'Горизонтали'),
    'Кабель Мосэнерго': ('Инженерные сети', 'Электроснабжение'),
    'Кабель Мосэнерго проектный': ('Инженерные сети', 'Электроснабжение'),
    'Кабель электрический': ('Инженерные сети', 'Электроснабжение'),
    'Кабель электрический проектный': ('Инженерные сети', 'Электроснабжение'),
    'Кабель связи': ('Инженерные сети', 'Сети связи'),
    'Канализация напорная': ('Инженерные сети', 'Канализация напорная'),
    'Канализация самотёчная': ('Инженерные сети', 'Канализация самотечная'),
    'Колодцы': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Общий коллектор': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Теплосеть': ('Инженерные сети', 'Теплосеть'),
    'Трубопроводы': ('Инженерные сети', 'Прочие трубопроводы')
}

# 2. Apply the mapping using a loop
labeled_total = 0

for layer, (l1, l2) in layer_mapping.items():
    # Mask to target only unlabeled text objects in the specific layer
    mask = (
        (msk_df['Layer'] == layer) & 
        (msk_df['Text'].notna()) & 
        (msk_df['Text'].str.strip() != "") & 
        (msk_df['l1'].isna())
    )
    
    # Apply labels
    msk_df.loc[mask, ['l1', 'l2']] = [l1, l2]
    
    # Count how many were labeled in this iteration
    count = mask.sum()
    labeled_total += count
    if count > 0:
        print(f"Layer '{layer}': labeled {count} text objects.")

# 3. Print final summary
print("-" * 30)
print(f"Total text objects labeled in this batch: {labeled_total}")

Layer 'Водопровод': labeled 135 text objects.
Layer 'Промводопровод': labeled 8 text objects.
Layer 'Водосток': labeled 51 text objects.
Layer 'Газопровод': labeled 111 text objects.
Layer 'Горизонтали': labeled 472 text objects.
Layer 'Кабель Мосэнерго': labeled 112 text objects.
Layer 'Кабель Мосэнерго проектный': labeled 1 text objects.
Layer 'Кабель электрический': labeled 201 text objects.
Layer 'Кабель электрический проектный': labeled 3 text objects.
Layer 'Кабель связи': labeled 81 text objects.
Layer 'Канализация напорная': labeled 34 text objects.
Layer 'Канализация самотёчная': labeled 89 text objects.
Layer 'Колодцы': labeled 1356 text objects.
Layer 'Общий коллектор': labeled 13 text objects.
Layer 'Теплосеть': labeled 120 text objects.
Layer 'Трубопроводы': labeled 28 text objects.
------------------------------
Total text objects labeled in this batch: 2815


In [13]:
msk_df.to_csv('MSK_in_progress_1.14.csv')

In [14]:
# 1. Filter for unlabeled text objects
# We need rows where l1 is NaN and Text is not empty
mask_unlabeled = (
    msk_df['l1'].isna() & 
    msk_df['Text'].notna() & 
    (msk_df['Text'].str.strip() != "")
)

# 2. Select only relevant columns for the analysis
# We keep 'Layer' and 'Text' to understand the context of each unique string
unlabeled_data = msk_df.loc[mask_unlabeled, ['Layer', 'Text']]

# 3. Get unique combinations of Layer and Text
# This drastically reduces the number of rows for review
unique_unlabeled = unlabeled_data.drop_duplicates(subset=['Layer', 'Text'])

# 4. Sort results for better readability in Excel
unique_unlabeled = unique_unlabeled.sort_values(by=['Layer', 'Text'])

# 5. Export to Excel
output_file = 'unique_unlabeled_texts.xlsx'
try:
    unique_unlabeled.to_excel(output_file, index=False)
    print(f"Extraction complete!")
    print(f"Total objects found: {len(unlabeled_data)}")
    print(f"Unique text/layer combinations: {len(unique_unlabeled)}")
    print(f"Saved to: {output_file}")
except Exception as e:
    print(f"Error while saving Excel: {e}")

Extraction complete!
Total objects found: 20223
Unique text/layer combinations: 923
Saved to: unique_unlabeled_texts.xlsx


In [23]:
# Use a single list of columns: ['l1', 'l2'] instead of [['l1', 'l2']]
msk_df[(msk_df['Layer'] == 'Платформы ЖД')]

,Source_File,Handle,Layer,Type,BlockName,Text,l1,l2
101623,output(1-3)_3_3458ЖДС-21tp.dxf,845A,Платформы ЖД,LWPOLYLINE,NaN,NaN,NaN,NaN
104357,output(1-3)_3_3458ЖДС-21tp.dxf,A8D6,Платформы ЖД,TEXT,NaN,ПЛАТФ.,NaN,NaN


## 1.15 Инженерные сети

In [24]:
# 1. Define the mapping dictionary for whole layers
layer_mapping = {
    '0': ('Служебные элементы', 'Прочее'),
    'Болота': ('Водные объекты', 'Неизвестно'),
    'Вентиляторы': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Геодезические пункты': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Кабели': ('Инженерные сети', 'Прочие кабели'),
    'Кабели проектные': ('Инженерные сети', 'Прочие кабели'),
    'Кабели защиты': ('Инженерные сети', 'Прочие кабели'),
    'Подземные коммуникации': ('Инженерные сети', 'Прочие кабели'),
    'ЛЭП': ('Инженерные сети', 'ЛЭП'),
    'Люки': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Платформы ЖД': ('Здания и сооружения', 'Неизвестно'),
    'Трубы': ('Инженерные сети', 'Прочие трубопроводы'),
    'Трубы под дорогами': ('Инженерные сети', 'Прочие трубопроводы')
}

# 2. Apply the dictionary mapping to all unlabeled text in these layers
labeled_total = 0
for layer, (l1, l2) in layer_mapping.items():
    # Target only unlabeled text objects
    mask = (
        (msk_df['Layer'] == layer) & 
        (msk_df['Text'].notna()) & 
        (msk_df['Text'].str.strip() != "") & 
        (msk_df['l1'].isna())
    )
    
    # Assign labels
    msk_df.loc[mask, ['l1', 'l2']] = [l1, l2]
    
    # Track progress
    count = mask.sum()
    labeled_total += count
    if count > 0:
        print(f"Layer '{layer}': labeled {count} text objects.")

# 3. Specific mapping for elevation patterns in 'Железные дороги'
# The regex ^[+-]?\d+\.\d+$ matches standard float formats like 123.45, +123.45, or -1.5
elevation_pattern = r'^\d{3}\.\d{2}$'

mask_railway_elevations = (
    (msk_df['Layer'] == 'Железные дороги') & 
    (msk_df['Text'].str.match(elevation_pattern, na=False)) & 
    (msk_df['l1'].isna())
)

msk_df.loc[mask_railway_elevations, ['l1', 'l2']] = ['Рельеф', 'Отметки высот']

print(f"\nLayer 'Железные дороги': labeled {mask_railway_elevations.sum()} elevations.")
print("-" * 30)
print(f"Total objects labeled in this batch: {labeled_total + mask_railway_elevations.sum()}")

Layer '0': labeled 12088 text objects.
Layer 'Болота': labeled 39 text objects.
Layer 'Вентиляторы': labeled 429 text objects.
Layer 'Геодезические пункты': labeled 168 text objects.
Layer 'Кабели': labeled 233 text objects.
Layer 'Кабели проектные': labeled 17 text objects.
Layer 'Подземные коммуникации': labeled 60 text objects.
Layer 'ЛЭП': labeled 143 text objects.
Layer 'Люки': labeled 276 text objects.
Layer 'Платформы ЖД': labeled 1 text objects.
Layer 'Трубы': labeled 24 text objects.
Layer 'Трубы под дорогами': labeled 96 text objects.

Layer 'Железные дороги': labeled 179 elevations.
------------------------------
Total objects labeled in this batch: 13753


In [25]:
msk_df.to_csv('MSK_in_progress_1.15.csv')

In [26]:
# 1. Filter for unlabeled text objects
# We need rows where l1 is NaN and Text is not empty
mask_unlabeled = (
    msk_df['l1'].isna() & 
    msk_df['Text'].notna() & 
    (msk_df['Text'].str.strip() != "")
)

# 2. Select only relevant columns for the analysis
# We keep 'Layer' and 'Text' to understand the context of each unique string
unlabeled_data = msk_df.loc[mask_unlabeled, ['Layer', 'Text']]

# 3. Get unique combinations of Layer and Text
# This drastically reduces the number of rows for review
unique_unlabeled = unlabeled_data.drop_duplicates(subset=['Layer', 'Text'])

# 4. Sort results for better readability in Excel
unique_unlabeled = unique_unlabeled.sort_values(by=['Layer', 'Text'])

# 5. Export to Excel
output_file = 'unique_unlabeled_texts.xlsx'
try:
    unique_unlabeled.to_excel(output_file, index=False)
    print(f"Extraction complete!")
    print(f"Total objects found: {len(unlabeled_data)}")
    print(f"Unique text/layer combinations: {len(unique_unlabeled)}")
    print(f"Saved to: {output_file}")
except Exception as e:
    print(f"Error while saving Excel: {e}")

Extraction complete!
Total objects found: 6467
Unique text/layer combinations: 401
Saved to: unique_unlabeled_texts.xlsx


## 1.16 Text left

In [85]:
msk_df[(msk_df['Text'] == '5.6') & (msk_df['Layer'] == 'Фонтаны')]

,Source_File,Handle,Layer,Type,BlockName,Text,l1,l2
102671,output(1-3)_3_3458ЖДС-21tp.dxf,9FD6,Фонтаны,TEXT,NaN,5.6,NaN,NaN


In [32]:
taxonomy_df = {
    "Водные объекты": ["Неизвестно"],

    "Водопроницаемые покрытия": [
        "Галька",
        "Газон спортивный",
        "Гравий",
        "Гранитный отсев",
        "Деревянные настилы",
        "ПГС",
        "Песок",
        "Щебень",
        "Щепа",
        "Грунт",
    ],

    "Здания и сооружения": [
        "Неизвестно",
        "Мосты",
        "Подпорные стенки",
        "Лестницы и пандусы",
        "Фонтаны"
    ],

    "Инженерные сети": [
        "Водопровод",
        "Газоснабжение",
        "Дренаж",
        "Инженерная инфраструктура",
        "Канализация бытовая",
        "Канализация напорная",
        "Канализация самотечная",
        "Ливневая канализация",
        "Наружное освещение",
        "Прочие кабели",
        "Прочие трубопроводы",
        "Сети связи",
        "Теплосеть",
        "Электроснабжение",
        "Столбы и опоры",
        "ЛЭП"
    ],

    "МАФ": [
        "Велоинфраструктура",
        "Дорожные знаки",
        "Игровое оборудование",
        "Информационные конструкции",
        "Калитка",
        "Ворота",
        "Ограждения",
        "Памятники",
        "Прочее",
        "Спортивное оборудование",
        "Уличная мебель",
        "Шлагбаумы",
        "Павильоны",
        "Флагштоки",
        "Вазоны",
        "ИДН"
    ],

    "Озеленение": [
        "Газон",
        "Деревья",
        "Кустарники",
        "Цветники",
        "Газонная решетка"
    ],

    "Рельеф": [
        "Горизонтали",
        "Отметки высот",
        "Откосы",
    ],

    "Служебные элементы": [
        "Зоны СЗЗ",
        "Зоны ПК",
        "Зоны ООПТ",
        "Зоны ОКН",
        "Зоны Береговая полоса",
        "Зоны Водоохранная зона",
        "Зоны Полоса отвода",
        "Зоны Прочее",
        "Границы",
        "Кадастр",
        "Контуры",
        "Координатные кресты",
        "Координаты",
        "Красные линии",
        "Нумерация"
        "Номера деревьев",
        "Прочее",
        "Размеры",
        "Размеры деревьев"
        "Условные обозначения",
        "Штамп",
        "Динамические блоки"
    ],

    "Твердые покрытия": [
        "Асфальт",
        "Бетон",
        "Брусчатка",
        "Гранит",
        "Камень",
        "Плитка",
        "Резиновая крошка",
    ],

    "Элементы благоустройства": [
        "Бортовые камни",
        "Дорожная разметка",
        "ЖД пути",
        "Парковка",
        "Трамвайные пути",
        "Металлические настилы",
    ],

    "Неизвестно": ["Неизвестно"],
}

In [35]:
# 1. Filter for unlabeled text objects
mask_unlabeled = (
    msk_df['l1'].isna() & 
    msk_df['Text'].notna() & 
    (msk_df['Text'].str.strip() != "")
)

# 2. Select relevant columns and get unique combinations
unlabeled_data = msk_df.loc[mask_unlabeled, ['Layer', 'Text']]
unique_unlabeled = unlabeled_data.drop_duplicates(subset=['Layer', 'Text']).copy()

# 3. Sort results for better readability
unique_unlabeled = unique_unlabeled.sort_values(by=['Layer', 'Text'])

# 4. Add empty columns for your manual labeling
unique_unlabeled['l1'] = ""
unique_unlabeled['l2'] = ""

# 5. Taxonomy
flattened_data = [(l1, l2) for l1, l2_list in taxonomy_df.items() for l2 in l2_list]
taxonomy_df = pd.DataFrame(flattened_data, columns=['l1', 'l2'])

# 6. Export to Excel with multiple sheets using ExcelWriter
output_file = 'unique_unlabeled_texts_with_dict.xlsx'
try:
    # Use 'xlsxwriter' or 'openpyxl' engine to write multiple sheets
    with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
        # Sheet 1: The data you need to label
        unique_unlabeled.to_excel(writer, sheet_name='Data_to_Label', index=False)
        
        # Sheet 2: The taxonomy dictionary
        taxonomy_df.to_excel(writer, sheet_name='Taxonomy_Dict', index=False)
        
    print("Extraction complete!")
    print(f"Total objects found: {len(unlabeled_data)}")
    print(f"Unique text/layer combinations: {len(unique_unlabeled)}")
    print(f"Taxonomy categories extracted: {len(taxonomy_df)}")
    print(f"Saved to: {output_file} (Check both sheets!)")
except Exception as e:
    print(f"Error while saving Excel: {e}")

Extraction complete!
Total objects found: 6467
Unique text/layer combinations: 401
Taxonomy categories extracted: 90
Saved to: unique_unlabeled_texts_with_dict.xlsx (Check both sheets!)


In [86]:
excel_file = 'unique_unlabeled_texts_with_dict.xlsx'
sheet_name = 'Data_to_Label'

labeled_df = pd.read_excel(excel_file, sheet_name=sheet_name)
labeled_df = labeled_df.dropna(subset=['l1']).copy()
print(f"Loaded labeled unique pairs from Excel: {len(labeled_df)}")

msk_df = msk_df.merge(labeled_df[['Layer', 'Text', 'l1', 'l2']], 
                      on=['Layer', 'Text'], 
                      how='left', 
                      suffixes=('', '_excel'))

msk_df['l1'] = msk_df['l1'].fillna(msk_df['l1_excel'])
msk_df['l2'] = msk_df['l2'].fillna(msk_df['l2_excel'])
msk_df.drop(columns=['l1_excel', 'l2_excel'], inplace=True)


unlabeled_count = msk_df['l1'].isna().sum()
total_count = len(msk_df)
labeled_percent = ((total_count - unlabeled_count) / total_count) * 100

print("-" * 30)
print("Done! Data updated.")
print(f"Unlabeled rows remaining: {unlabeled_count} out of {total_count}")
print(f"Labeling progress: {labeled_percent:.2f}%")

c:\ProgramData\anaconda3\lib\site-packages\openpyxl\worksheet\_read_only.py:79: UserWarning: Data Validation extension is not supported and will be removed
  for idx, row in parser.parse():


Loaded labeled unique pairs from Excel: 399
------------------------------
Done! Data updated.
Unlabeled rows remaining: 664957 out of 3903465
Labeling progress: 82.96%


In [87]:
msk_df.to_csv('MSK_in_progress_1.16.csv')

## 1.17 Objects left

In [ ]:
mask_unlabeled = msk_df['l1'].isna() 
unlabeled_df = msk_df[mask_unlabeled].copy()

cols_to_ignore = ['Handle', 'Source_File']
subset_cols = [col for col in unlabeled_df.columns if col not in cols_to_ignore]
unlabeled_unique = unlabeled_df.drop_duplicates(subset=subset_cols)

if 'Layer' in unlabeled_unique.columns and 'Text' in unlabeled_unique.columns:
    unlabeled_unique = unlabeled_unique.sort_values(by=['Layer', 'Text'])

output_file = 'objects_remaining.xlsx'
unlabeled_unique.to_excel(output_file, index=False)

In [90]:
mask_unlabeled = msk_df['l1'].isna()
unlabeled_df = msk_df[mask_unlabeled].copy()

cols_to_ignore = ['Handle', 'Source_File']
subset_cols = [col for col in unlabeled_df.columns if col not in cols_to_ignore]
unlabeled_unique = unlabeled_df.drop_duplicates(subset=subset_cols)

if 'BlockName' in unlabeled_unique.columns:
    mask_exclude_blocks = ~unlabeled_unique['BlockName'].str.contains('msdElement|TXT', case=False, na=False)
    unlabeled_unique = unlabeled_unique[mask_exclude_blocks]
    print("Block filtering applied successfully.")

if 'Layer' in unlabeled_unique.columns and 'Text' in unlabeled_unique.columns:
    unlabeled_unique = unlabeled_unique.sort_values(by=['Layer', 'Text'])

output_file = 'objects_remaining_2.xlsx'
unlabeled_unique.to_excel(output_file, index=False)

Block filtering applied successfully.


In [94]:
mask_unlabeled = msk_df['l1'].isna()
unlabeled_df = msk_df[mask_unlabeled].copy()

if 'BlockName' in unlabeled_df.columns:
    unlabeled_df['BlockName'] = unlabeled_df['BlockName'].str.replace(r'[_\d]+', '', regex=True)
    print("Block names cleaned from digits and underscores.")

cols_to_drop = ['Handle', 'Source_File']
unlabeled_df = unlabeled_df.drop(columns=cols_to_drop, errors='ignore')

unlabeled_unique = unlabeled_df.drop_duplicates()

if 'Layer' in unlabeled_unique.columns and 'BlockName' in unlabeled_unique.columns:
    unlabeled_unique = unlabeled_unique.sort_values(by=['Layer', 'BlockName'])

output_file = 'unlabeled_cleaned_blocks.xlsx'
unlabeled_unique.to_excel(output_file, index=False)

Block names cleaned from digits and underscores.


In [105]:
msk_df[msk_df['Layer'] == 'Территории']

,Source_File,Handle,Layer,Type,BlockName,Text,l1,l2
416547,x_los_part_1_tp.dxf,349921,Территории,CIRCLE,NaN,NaN,NaN,NaN
416548,x_los_part_1_tp.dxf,349922,Территории,LINE,NaN,NaN,NaN,NaN
416549,x_los_part_1_tp.dxf,349923,Территории,LINE,NaN,NaN,NaN,NaN
416550,x_los_part_1_tp.dxf,349924,Территории,LINE,NaN,NaN,NaN,NaN
416551,x_los_part_1_tp.dxf,349925,Территории,LINE,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
3884063,x_los_part_4_tp.dxf,77986,Территории,LWPOLYLINE,NaN,NaN,NaN,NaN
3884064,x_los_part_4_tp.dxf,77987,Территории,ELLIPSE,NaN,NaN,NaN,NaN
3884083,x_los_part_4_tp.dxf,7799A,Территории,ARC,NaN,NaN,NaN,NaN
3884084,x_los_part_4_tp.dxf,7799B,Территории,LWPOLYLINE,NaN,NaN,NaN,NaN


In [106]:
import pandas as pd

# ==========================================
# STAGE 1: Labeling by Block Names (BlockName)
# ==========================================
# Using regular expressions: the '|' symbol means "OR". 
# The pattern will find any block name containing these root words.
block_mapping = {
    r'DEREVO|FRUCTO|KIPAR|LISTV|KUST': ('Озеленение', 'Деревья'),
    r'PESOK': ('Водопроницаемые покрытия', 'Песок'),
    r'ЛУГ|GAZON': ('Озеленение', 'Газон'),
    r'DORZN': ('МАФ', 'Дорожные знаки'),
    r'FONAR|FNRM|STOLBG|PROZCT|RADFON|SVETFR': ('Инженерные сети', 'Столбы и опоры'),
    r'SHLAGB': ('МАФ', 'Шлагбаум'),
    r'AFIS': ('МАФ', 'Информационные конструкции'),
    r'VOROTA': ('МАФ', 'Ворота'),
    r'OSTAN': ('МАФ', 'Павильоны')
}

labeled_by_block = 0

# Check if the dataframe has a column for block names ('BlockName' or 'Name')
block_col = 'BlockName' if 'BlockName' in msk_df.columns else 'Name' if 'Name' in msk_df.columns else None

if block_col:
    for pattern, (l1, l2) in block_mapping.items():
        # Find unlabeled rows where the block name contains the required root (case-insensitive)
        mask = (
            msk_df['l1'].isna() & 
            msk_df[block_col].str.contains(pattern, case=False, na=False, regex=True)
        )
        count = mask.sum()
        if count > 0:
            msk_df.loc[mask, ['l1', 'l2']] = [l1, l2]
            labeled_by_block += count

    print(f"Labeled by block names: {labeled_by_block} objects.")
else:
    print("Block name column not found, skipping Stage 1.")


# ==========================================
# STAGE 2: Labeling by Layers (Layer)
# ==========================================
# The grouped layer names (separated by slashes in the prompt) are expanded 
# into a flat dictionary so the script runs quickly and without errors.
layer_mapping = {
    'Береговая линия': ('Водные объекты', 'Неизвестно'),
    'Болота': ('Водные объекты', 'Неизвестно'),
    'Бортовой камень': ('Элементы благоустройства', 'Бортовые камни'),
    'Будка телефонная': ('МАФ', 'Павильоны'),
    'Вентиляторы': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Водопровод': ('Инженерные сети', 'Водопровод'),
    'Водопровод проектный': ('Инженерные сети', 'Водопровод'),
    'Промводопровод': ('Инженерные сети', 'Водопровод'),
    'Водосток': ('Инженерные сети', 'Ливневая канализация'),
    'Водосток проектный': ('Инженерные сети', 'Ливневая канализация'),
    'Выгребные ямы': ('Водопроницаемые покрытия', 'Грунт'),
    'Вышки': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Газопровод': ('Инженерные сети', 'Газоснабжение'),
    'Газопровод проектный': ('Инженерные сети', 'Газоснабжение'),
    'Геодезические пункты': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Горизонтали': ('Рельеф', 'Отметки высот'),
    'Граница заказа': ('Служебные элементы', 'Границы'),
    'Граница площадки': ('Служебные элементы', 'Контуры'),
    'Граница улицы': ('Служебные элементы', 'Контуры'),
    'Громоотвод': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Грунты': ('Водопроницаемые покрытия', 'Грунт'),
    'Дренаж': ('Инженерные сети', 'Дренаж'),
    'Дренаж проектный': ('Инженерные сети', 'Дренаж'),
    'Железные дороги': ('Элементы благоустройства', 'ЖД пути'),
    'Кабели': ('Инженерные сети', 'Прочие кабели'),
    'Кабели проектные': ('Инженерные сети', 'Прочие кабели'),
    'Кабель защиты': ('Инженерные сети', 'Прочие кабели'),
    'Кабель защиты проектный': ('Инженерные сети', 'Прочие кабели'),
    'Кабель Мосэнерго': ('Инженерные сети', 'Электроснабжение'),
    'Кабель Мосэнерго проектный': ('Инженерные сети', 'Электроснабжение'),
    'Кабель электрический': ('Инженерные сети', 'Электроснабжение'),
    'Кабель электрический проектный': ('Инженерные сети', 'Электроснабжение'),
    'Кабель связи': ('Инженерные сети', 'Сети связи'),
    'Кабель связи проектный': ('Инженерные сети', 'Сети связи'),
    'Канализация напорная': ('Инженерные сети', 'Канализация напорная'),
    'Канализация напорная проектная': ('Инженерные сети', 'Канализация напорная'),
    'Канализация самотёчная': ('Инженерные сети', 'Канализация самотечная'),
    'Канализация самотёчная проектная': ('Инженерные сети', 'Канализация самотечная'),
    'Колодцы': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Крышки колодцев': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Люки': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Координатные кресты': ('Служебные элементы', 'Координатные кресты'),
    'ЛЭП': ('Инженерные сети', 'ЛЭП'),
    'Лестницы набережных': ('Здания и сооружения', 'Лестницы и пандусы'),
    'Мосты': ('Здания и сооружения', 'Мосты'),
    'Общий коллектор': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Ограды': ('МАФ', 'Ограждения'),
    'Опоры контакной сети': ('Инженерные сети', 'Столбы и опоры'),
    'Светофоры': ('Инженерные сети', 'Столбы и опоры'),
    'Столбы': ('Инженерные сети', 'Столбы и опоры'),
    'Фонари': ('Инженерные сети', 'Столбы и опоры'),
    'Откосы': ('Рельеф', 'Откосы'),
    'Павильоны': ('МАФ', 'Павильоны'),
    'Памятники': ('МАФ', 'Памятники'),
    'Парапеты': ('Здания и сооружения', 'Подпорные стенки'),
    'Платформы ЖД': ('Здания и сооружения', 'Неизвестно'),
    'Спец сооружения': ('Здания и сооружения', 'Неизвестно'),
    'Подземные коммуникации': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Подземные коммуникации проектные': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Подземный переход': ('Здания и сооружения', 'Подземные'),
    'Подъемные краны': ('МАФ', 'Ограждения'), 
    'Пояснительные подписи': ('Служебные элементы', 'Прочее'),
    'Просеки в лесу': ('Водопроницаемые покрытия', 'Грунт'),
    'Рамки': ('Служебные элементы', 'Штамп'),
    'Теплосеть': ('Инженерные сети', 'Теплосеть'),
    'Теплосеть проектная': ('Инженерные сети', 'Теплосеть'),
    'Трубопроводы': ('Инженерные сети', 'Прочие трубопроводы'),
    'Трубопроводы проектные': ('Инженерные сети', 'Прочие трубопроводы'),
    'Трубы': ('Инженерные сети', 'Прочие трубопроводы'),
    'Трубы под дорогами': ('Инженерные сети', 'Прочие трубопроводы'),
    'Указатель подз коммуникаций': ('Инженерные сети', 'Инженерная инфраструктура'),
    'Фонтаны': ('Инженерные сети', 'Ливневая канализация'),
    'дорисовки': ('Служебные элементы', 'Прочее')
}

labeled_by_layer = 0
for layer, (l1, l2) in layer_mapping.items():
    # Strictly look for unlabeled rows within a specific layer
    mask = (msk_df['l1'].isna()) & (msk_df['Layer'] == layer)
    count = mask.sum()
    if count > 0:
        msk_df.loc[mask, ['l1', 'l2']] = [l1, l2]
        labeled_by_layer += count

# ==========================================
# SUMMARY OF THIS RUN
# ==========================================
print(f"Labeled by layers: {labeled_by_layer} objects.")
print("-" * 30)
print(f"Total labeled in this run: {labeled_by_block + labeled_by_layer} objects.")

# Calculate the remaining unlabeled data
unlabeled_count = msk_df['l1'].isna().sum()
total_count = len(msk_df)
labeled_percent = ((total_count - unlabeled_count) / total_count) * 100

print(f"Remaining unlabeled rows: {unlabeled_count} out of {total_count}")
print(f"Overall labeling progress: {labeled_percent:.2f}%")

Labeled by block names: 6436 objects.
Labeled by layers: 619086 objects.
------------------------------
Total labeled in this run: 625522 objects.
Remaining unlabeled rows: 39435 out of 3903465
Overall labeling progress: 98.99%


In [107]:
msk_df.to_csv('MSK_in_progress_1.17.csv')

In [3]:
msk_df = pd.read_csv('MSK_in_progress_1.17.csv')

C:\Users\artem\AppData\Local\Temp\ipykernel_33024\3535499176.py:1: DtypeWarning: Columns (5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  msk_df = pd.read_csv('MSK_in_progress_1.17.csv')


## 1.18 Objects left

In [4]:
msk_df[msk_df['l1'].isna()]['BlockName'].nunique()

1129

In [5]:
# ==========================================
# EXPORTING REMAINING UNLABELED OBJECTS
# ==========================================

# 1. Filter the dataframe for rows where 'l1' is still NaN
mask_unlabeled = msk_df['l1'].isna()
unlabeled_df = msk_df[mask_unlabeled].copy()

# 2. Drop technical columns that prevent finding true duplicates
cols_to_drop = ['Handle', 'Source_File']
# errors='ignore' ensures the code won't crash if these columns were already dropped
unlabeled_df = unlabeled_df.drop(columns=cols_to_drop, errors='ignore')

# 3. Drop duplicates across all remaining columns to get unique objects
unlabeled_unique = unlabeled_df.drop_duplicates()

# 4. Sort the results for better readability in Excel
# Dynamically check which columns exist to avoid KeyError
sort_cols = []
if 'Layer' in unlabeled_unique.columns:
    sort_cols.append('Layer')
if 'BlockName' in unlabeled_unique.columns:
    sort_cols.append('BlockName')
elif 'Name' in unlabeled_unique.columns:
    sort_cols.append('Name')
if 'Text' in unlabeled_unique.columns:
    sort_cols.append('Text')

if sort_cols:
    unlabeled_unique = unlabeled_unique.sort_values(by=sort_cols)

# 5. Export the clean, unique unlabeled data to Excel
output_file = 'final_unlabeled_objects.xlsx'

try:
    unlabeled_unique.to_excel(output_file, index=False)
    print("-" * 30)
    print("Extraction of remaining unlabeled objects complete!")
    print(f"Total unlabeled objects (raw count): {mask_unlabeled.sum()}")
    print(f"Unique unlabeled objects for manual review: {len(unlabeled_unique)}")
    print(f"Saved to: {output_file}")
except Exception as e:
    print(f"Error while saving Excel: {e}")

------------------------------
Extraction of remaining unlabeled objects complete!
Total unlabeled objects (raw count): 39435
Unique unlabeled objects for manual review: 39435
Saved to: final_unlabeled_objects.xlsx


In [6]:
# Define the block name column (handling different possible names)
block_col = 'BlockName' if 'BlockName' in msk_df.columns else 'Name' if 'Name' in msk_df.columns else None

# ==========================================
# STAGE 3: Labeling by specific patterns and layers
# ==========================================

# 1. Blocks starting with asterisk '*' -> (Служебные элементы, Динамические блоки)
if block_col:
    # ^\* means starts with literal asterisk in regex
    mask_dynamic = (msk_df['l1'].isna()) & (msk_df[block_col].str.contains(r'^\*', na=False, regex=True))
    msk_df.loc[mask_dynamic, ['l1', 'l2']] = ['Служебные элементы', 'Динамические блоки']
    print(f"Labeled dynamic blocks: {mask_dynamic.sum()}")

# 2. Blocks starting with LEG or RL -> (Служебные элементы, Условные обозначения)
if block_col:
    mask_legend = (msk_df['l1'].isna()) & (msk_df[block_col].str.contains(r'^(LEG|RL)', case=False, na=False, regex=True))
    msk_df.loc[mask_legend, ['l1', 'l2']] = ['Служебные элементы', 'Условные обозначения']
    print(f"Labeled legend blocks: {mask_legend.sum()}")

# 3. Specific Text "КАНАВА" in layer "Здания" -> (Водные объекты, Неизвестно)
if 'Text' in msk_df.columns:
    mask_kanava = (msk_df['l1'].isna()) & (msk_df['Layer'] == 'Здания') & (msk_df['Text'] == 'КАНАВА')
    msk_df.loc[mask_kanava, ['l1', 'l2']] = ['Водные объекты', 'Неизвестно']
    print(f"Labeled 'КАНАВА' text: {mask_kanava.sum()}")

# 4. Layer based mapping
layer_mapping_v2 = {
    'Граница площадки (Construction)': ('Служебные элементы', 'Контуры'),
    'граница площадки': ('Служебные элементы', 'Контуры'),
    'Топографические объекты': ('Служебные элементы', 'Контуры'),
    'Трамвайные линии': ('Элементы благоустройства', 'Трамвайные пути')
}

labeled_layers_v2 = 0
for layer, (l1, l2) in layer_mapping_v2.items():
    mask = (msk_df['l1'].isna()) & (msk_df['Layer'] == layer)
    count = mask.sum()
    if count > 0:
        msk_df.loc[mask, ['l1', 'l2']] = [l1, l2]
        labeled_layers_v2 += count
print(f"Labeled by specific layers (v2): {labeled_layers_v2}")

# 5. Catch-all: Everything else in Layer '0' -> (Служебные элементы, Прочее)
# This should be executed AFTER specific block patterns
mask_layer_zero = (msk_df['l1'].isna()) & (msk_df['Layer'] == '0')
msk_df.loc[mask_layer_zero, ['l1', 'l2']] = ['Служебные элементы', 'Прочее']
print(f"Labeled remaining objects in layer '0': {mask_layer_zero.sum()}")

# ==========================================
# FINAL PROGRESS CHECK
# ==========================================
unlabeled_count = msk_df['l1'].isna().sum()
total_count = len(msk_df)
progress = ((total_count - unlabeled_count) / total_count) * 100

print("-" * 30)
print(f"Remaining unlabeled rows: {unlabeled_count} out of {total_count}")
print(f"Current labeling progress: {progress:.2f}%")

Labeled dynamic blocks: 130


C:\Users\artem\AppData\Local\Temp\ipykernel_33024\3029707434.py:17: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_legend = (msk_df['l1'].isna()) & (msk_df[block_col].str.contains(r'^(LEG|RL)', case=False, na=False, regex=True))


Labeled legend blocks: 9502
Labeled 'КАНАВА' text: 1
Labeled by specific layers (v2): 2068
Labeled remaining objects in layer '0': 25666
------------------------------
Remaining unlabeled rows: 2068 out of 3903465
Current labeling progress: 99.95%


In [7]:
# ==========================================
# STAGE 4: Labeling "Территории" Layer
# ==========================================

# 1. Define the target layer and the labels
target_layer = 'Территории'
target_l1 = 'Служебные элементы'
target_l2 = 'Контуры'

# 2. Create a mask for unlabeled rows in the specific layer
# We use .isna() to ensure we don't overwrite any previous manual or specific labeling
mask_territories = (msk_df['l1'].isna()) & (msk_df['Layer'] == target_layer)

count_affected = mask_territories.sum()

if count_affected > 0:
    # 3. Apply the labels
    msk_df.loc[mask_territories, ['l1', 'l2']] = [target_l1, target_l2]
    print(f"Successfully labeled layer '{target_layer}': {count_affected} objects.")
else:
    print(f"No unlabeled objects found in layer '{target_layer}'.")

# ==========================================
# FINAL DATASET SUMMARY
# ==========================================
unlabeled_count = msk_df['l1'].isna().sum()
total_count = len(msk_df)
progress = ((total_count - unlabeled_count) / total_count) * 100

print("-" * 30)
print(f"Final cleanup results:")
print(f"Remaining unlabeled rows: {unlabeled_count} out of {total_count}")
print(f"Total labeling progress: {progress:.2f}%")

# Optional: Save the almost-complete dataset
# msk_df.to_csv('MSK_Dataset_Final_Draft.csv', index=False, encoding='utf-8-sig')

Successfully labeled layer 'Территории': 2068 objects.
------------------------------
Final cleanup results:
Remaining unlabeled rows: 0 out of 3903465
Total labeling progress: 100.00%


In [8]:
msk_df.to_csv('MSK_FINAL.csv')

## Some fixes

In [9]:
df = pd.read_csv('REGIONS_FINAL_v2.csv')

C:\Users\artem\AppData\Local\Temp\ipykernel_33024\3975677991.py:1: DtypeWarning: Columns (4,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('REGIONS_FINAL_v2.csv')


In [11]:
# ==========================================
# EXPORTING 'ЛЭП' RELATED OBJECTS TO EXCEL
# ==========================================

# 1. Define the search pattern (case-insensitive)
search_pattern = 'лэп'

# 2. Identify the block name column if it exists
block_col = 'BlockName' if 'BlockName' in df.columns else 'Name' if 'Name' in df.columns else None

# 3. Create a combined mask for Layer, Text, and BlockName
mask_lep = (df['Layer'].str.contains(search_pattern, case=False, na=False)) | \
           (df['Text'].str.contains(search_pattern, case=False, na=False))

if block_col:
    mask_lep = mask_lep | (df[block_col].str.contains(search_pattern, case=False, na=False))

# 4. Filter the dataframe
lep_df = df[mask_lep].copy()

# 5. Drop duplicates to keep only unique combinations of interest
# We exclude 'Handle' and 'Source_File' to see the unique "types" of LEP objects
cols_to_ignore = ['Handle', 'Source_File']
subset_cols = [col for col in lep_df.columns if col not in cols_to_ignore]
lep_unique = lep_df.drop_duplicates(subset=subset_cols)

# 6. Export to Excel
output_file = 'lep_objects_search.xlsx'

if not lep_unique.empty:
    try:
        lep_unique.to_excel(output_file, index=False)
        print("-" * 30)
        print(f"Success! Found {len(lep_unique)} unique 'ЛЭП' related objects.")
        print(f"Data saved to: {output_file}")
    except Exception as e:
        print(f"Error saving Excel file: {e}")
else:
    print(f"No objects matching '{search_pattern}' were found to export.")

------------------------------
Success! Found 8770 unique 'ЛЭП' related objects.
Data saved to: lep_objects_search.xlsx


In [12]:
# ==========================================
# EXPORTING LINEAR 'ЛЭП' INFRASTRUCTURE
# ==========================================

# 1. Define the geometric types we consider "linear/primitive"
linear_types = ['LINE', 'LWPOLYLINE', 'POLYLINE', 'CIRCLE', 'ARC', 'ELLIPSE']

# 2. Define our search criteria
search_pattern = 'лэп'
target_l1 = 'Инженерные сети'

# 3. Apply the filters
# - Layer contains 'лэп'
# - Type is in our linear_types list
# - l1 matches 'Инженерные сети'
mask_lep_linear = (
    (df['Layer'].str.contains(search_pattern, case=False, na=False)) &
    (df['Type'].isin(linear_types)) &
    (df['l1'] == target_l1)
)

lep_linear_df = df[mask_lep_linear].copy()

# 4. Drop technical duplicates to see unique Layer/Type/Geometric property combinations
# We keep 'Color', 'Linetype', or 'Thickness' if they exist in your df to see styling
cols_to_keep = ['Layer', 'Type', 'l1', 'l2', 'Color', 'Linetype']
available_cols = [col for col in cols_to_keep if col in lep_linear_df.columns]

lep_linear_unique = lep_linear_df.drop_duplicates(subset=available_cols)

# 5. Export to Excel
output_file = 'lep_linear_infrastructure.xlsx'

if not lep_linear_unique.empty:
    try:
        lep_linear_unique.sort_values(by=['Layer', 'Type']).to_excel(output_file, index=False)
        print("-" * 30)
        print("Filter applied successfully!")
        print(f"Total linear LEP objects found: {len(lep_linear_df)}")
        print(f"Unique combinations for review: {len(lep_linear_unique)}")
        print(f"File saved as: {output_file}")
    except Exception as e:
        print(f"Error saving file: {e}")
else:
    print("No objects found matching these specific criteria.")

------------------------------
Filter applied successfully!
Total linear LEP objects found: 2078
Unique combinations for review: 11
File saved as: lep_linear_infrastructure.xlsx


In [ ]:
msk_df.head()

,Source_File,Handle,Layer,Type,BlockName,Text,l1,l2
0,output()_3_ЗАО-20_00052kl.dxf,7B,Красные линии,INSERT,msdElementTypeLineString_614,NaN,Служебные элементы,Красные линии
1,output()_3_ЗАО-20_00052kl.dxf,88,Красные линии,INSERT,msdElementTypeLineString_615,NaN,Служебные элементы,Красные линии
2,output()_3_ЗАО-20_00052kl.dxf,E2,Красные линии,TEXT,NaN,ЛИНИИ ГРАДОСТРОИТЕЛЬНОГО РЕГУЛИРОВАНИЯ НАНЕСЕН...,Служебные элементы,Прочее
3,output()_3_ЗАО-20_00052kl.dxf,E8,Красные линии,TEXT,NaN,ТЕХНИЧЕСКАЯ ЗОНА,Служебные элементы,Зоны Прочее
4,output()_3_ЗАО-20_00052tp.dxf,7B,Горизонтали,INSERT,PIKET,NaN,Рельеф,Отметки высот


In [16]:
# ==========================================
# RELABELING L2: 'ЛЭП' -> 'Электроснабжение'
# ==========================================

# 1. Define the change
old_label = 'ЛЭП'
new_label = 'Электроснабжение'

# 2. Identify rows where l2 matches the old label
mask_lep_update = (msk_df['l2'] == old_label)

# 3. Apply the change and count updates
count_before = mask_lep_update.sum()
if count_before > 0:
    msk_df.loc[mask_lep_update, 'l2'] = new_label
    print(f"Successfully updated {count_before} objects.")
    print(f"Label '{old_label}' has been replaced with '{new_label}' in l2.")
else:
    print(f"No objects with l2 = '{old_label}' were found.")

# 4. Quick verification
print("-" * 30)
print("Unique values in l2 for 'Инженерные сети':")
print(msk_df[msk_df['l1'] == 'Инженерные сети']['l2'].unique())

Successfully updated 4010 objects.
Label 'ЛЭП' has been replaced with 'Электроснабжение' in l2.
------------------------------
Unique values in l2 for 'Инженерные сети':
['Инженерная инфраструктура' 'Столбы и опоры' 'Ливневая канализация'
 'Электроснабжение' 'Сети связи' 'Канализация самотечная' 'Водопровод'
 'Прочие кабели' 'Дренаж' 'Прочие трубопроводы' 'Теплосеть'
 'Канализация напорная' 'Газоснабжение']


In [17]:
msk_df.to_csv('MSK_FINAL_v1.csv', index=False)

In [8]:
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
df.head()

,Source_File,Handle,Layer,Type,BlockName,Text,l1,l2
0,(3) старое Симферопольское шоссе.dxf,A7,013,INSERT,BL_333,NaN,Неизвестно,Неизвестно
1,(3) старое Симферопольское шоссе.dxf,A8,013,INSERT,BL_333,NaN,Неизвестно,Неизвестно
2,(3) старое Симферопольское шоссе.dxf,A9,013,INSERT,BL_333,NaN,Неизвестно,Неизвестно
3,(3) старое Симферопольское шоссе.dxf,AA,013,INSERT,BL_333,NaN,Неизвестно,Неизвестно
4,(3) старое Симферопольское шоссе.dxf,AB,013,INSERT,BL_333,NaN,Неизвестно,Неизвестно


In [9]:
df.to_csv('REGIONS_FINAL_v3.csv', index=False)

In [ ]:
# msk_df = pd.read_csv('MSK_FINAL_v1.csv', low_memory=False)
# df = pd.read_csv('REGIONS_FINAL_v3.csv', low_memory=False)
# codes_df = pd.read_excel('topo_codes_classification.xlsx')

In [18]:
code_mask = codes_df['code'].unique()
len(df[df['Layer'].isin(code_mask)])

156175

In [19]:
# 1. Prepare the reference data from codes_df
# We only need the code and the correct l1/l2 labels
reference_df = codes_df[['code', 'l1', 'l2']].copy()
reference_df.columns = ['Layer', 'l1_ref', 'l2_ref'] # Rename for easy merging

# 2. Merge your dataset with the reference data
# We use 'inner' because you specifically want to compare the 156,175 matching rows
comparison_df = df.merge(reference_df, on='Layer', how='inner')

# 3. Create a mask to find discrepancies
# A row is a mismatch if either l1 or l2 doesn't match the reference
mismatch_mask = (comparison_df['l1'] != comparison_df['l1_ref']) | \
                (comparison_df['l2'] != comparison_df['l2_ref'])

# 4. Filter for mismatches
mismatches = comparison_df[mismatch_mask].copy()

# 5. Export to Excel
output_file = 'labeling_discrepancies.xlsx'

if not mismatches.empty:
    # Select and reorder columns for better visibility in Excel
    # This helps you see the "Current" vs "Reference" labels side-by-side
    result_cols = ['Layer', 'Type', 'Text', 'l1', 'l1_ref', 'l2', 'l2_ref', 'Source_File']
    # Use only columns that actually exist in the dataframe
    final_cols = [col for col in result_cols if col in mismatches.columns]
    
    try:
        mismatches[final_cols].to_excel(output_file, index=False)
        print("-" * 30)
        print(f"Comparison complete!")
        print(f"Total matching layers checked: {len(comparison_df)}")
        print(f"Found {len(mismatches)} rows with differing labels.")
        print(f"Discrepancies saved to: {output_file}")
    except Exception as e:
        print(f"Error saving Excel: {e}")
else:
    print("-" * 30)
    print("Perfect match! All 156,175 rows are labeled exactly as in the reference file.")

------------------------------
Comparison complete!
Total matching layers checked: 156175
Found 64048 rows with differing labels.
Discrepancies saved to: labeling_discrepancies.xlsx


In [22]:
# 1. Оставляем ТОЛЬКО те строки, где текущая разметка не совпадает с эталоном
mask_diff = (comparison_df['l1'] != comparison_df['l1_ref']) | \
            (comparison_df['l2'] != comparison_df['l2_ref'])

real_mismatches = comparison_df[mask_diff].copy()

# 2. Берем нужные колонки
compare_cols = ['Layer', 'l1', 'l2', 'l1_ref', 'l2_ref']
real_mismatches = real_mismatches[compare_cols]

# 3. Удаляем дубликаты, оставляя только уникальные примеры расхождений
unique_diffs = real_mismatches.drop_duplicates().sort_values(by=['Layer'])

# 4. Сохраняем в Excel
unique_diffs.to_excel('real_differences_only.xlsx', index=False)
print(f"Осталось уникальных конфликтов: {len(unique_diffs)}")

Осталось уникальных конфликтов: 162


In [23]:
df[(df['Layer'] == '12492') & (df['l1'] == 'МАФ')]

,Source_File,Handle,Layer,Type,BlockName,Text,l1,l2
362,(3) старое Симферопольское шоссе.dxf,13A9,12492,TEXT,NaN,дор. указ.,МАФ,Дорожные знаки
10681,026-19-Серпухов.dxf,1A4CD,12492,TEXT,NaN,инф.,МАФ,Информационные конструкции
10693,026-19-Серпухов.dxf,1A849,12492,TEXT,NaN,инф.,МАФ,Информационные конструкции
10695,026-19-Серпухов.dxf,1A8FB,12492,TEXT,NaN,инф.,МАФ,Информационные конструкции
10707,026-19-Серпухов.dxf,1AE88,12492,TEXT,NaN,тир,МАФ,Игровое оборудование
...,...,...,...,...,...,...,...,...
2624589,Пушкино 500.dxf,4DB21,12492,TEXT,NaN,дор. ук.,МАФ,Дорожные знаки
2625115,Пушкино 500.dxf,4E542,12492,TEXT,NaN,ков.,МАФ,Ограждения
2626184,Пушкино 500.dxf,50C09,12492,TEXT,NaN,рекл.,МАФ,Информационные конструкции
2626196,Пушкино 500.dxf,50C67,12492,TEXT,NaN,рекл.,МАФ,Информационные конструкции


In [25]:
# ==========================================
# CALCULATING 'TYPE' COLUMN DISTRIBUTION
# ==========================================

# 1. Absolute counts (how many of each type exist)
type_counts = msk_df['Type'].value_counts()

print("--- Distribution by Type (Absolute Counts) ---")
print(type_counts)
print("-" * 45)

# 2. Percentage distribution (normalize=True gives fractions 0.0 to 1.0)
# We multiply by 100 to get readable percentages
type_percentages = msk_df['Type'].value_counts(normalize=True) * 100

print("--- Distribution by Type (Percentages) ---")
# .round(2) keeps 2 decimal places for neatness
print(type_percentages.round(2).astype(str) + '%')
print("-" * 45)

# 3. Optional: Combine both into a nice DataFrame and save to Excel
distribution_df = pd.DataFrame({
    'Count': type_counts,
    'Percentage (%)': type_percentages.round(2)
})

--- Distribution by Type (Absolute Counts) ---
LINE          1166119
LWPOLYLINE     931998
ARC            745312
ELLIPSE        602270
INSERT         198322
TEXT           181546
CIRCLE          55951
HATCH           11266
REGION          10293
MTEXT             261
POINT             127
Name: Type, dtype: int64
---------------------------------------------
--- Distribution by Type (Percentages) ---
LINE          29.87%
LWPOLYLINE    23.88%
ARC           19.09%
ELLIPSE       15.43%
INSERT         5.08%
TEXT           4.65%
CIRCLE         1.43%
HATCH          0.29%
REGION         0.26%
MTEXT          0.01%
POINT           0.0%
Name: Type, dtype: object
---------------------------------------------
